# **B1 experiments - Step 0 - 7 (common setups) + E - Regions-Points Relationships + Semantic Integration


**Common B1 infrastructure: Steps 0–7 are intentionally shared across the B1 experimental notebooks to maintain consistent datasets, models, frozen dev100 subset, and SAN cache definitions.**

# Step 0 - Mount Google Drive and select compute device

In [1]:
# ============================================================
# Step 0 — Mount Google Drive and select compute device
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Mounted at /content/drive
Device: cuda


# Step 1 - Project paths + dataset loader


In [2]:
# ============================================================
# Step 1A — Define project and dataset paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

DATA_ROOT = PROJECT_ROOT / "datasets"

# ------------------------------------------------------------
# FSC-147
# ------------------------------------------------------------

FSC147_ROOT = DATA_ROOT / "FSC147"
FSC147_RAW = FSC147_ROOT / "raw"

FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
FSC147_DENSITY_DIR = (
    FSC147_RAW / "gt_density_map_adaptive_384_VarV2"
)

FSC147_ANNOTATION_FILE = (
    FSC147_RAW / "annotation_FSC147_384.json"
)

FSC147_SPLIT_FILE = (
    FSC147_RAW / "Train_Test_Val_FSC_147.json"
)

FSC147_CLASSES_FILE = (
    FSC147_RAW / "ImageClasses_FSC147.txt"
)

# ------------------------------------------------------------
# OmniCount-191
# ------------------------------------------------------------

OMNICOUNT_ROOT = DATA_ROOT / "OmniCount-191"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)

print("\nFSC-147 root :", FSC147_ROOT)
print("OmniCount root:", OMNICOUNT_ROOT)

PROJECT_ROOT : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
DATA_ROOT    : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets

FSC-147 root : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147
OmniCount root: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/OmniCount-191


In [3]:
# ============================================================
# Step 1B — Clone project repository and import shared modules
# ============================================================

%cd /content

!rm -rf Surrey-MScProject-Zero-Shot-Object-Counting

!git clone \
    https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git

/content
Cloning into 'Surrey-MScProject-Zero-Shot-Object-Counting'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 47 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 65.52 KiB | 771.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.


In [4]:
# ============================================================
# Step 1C — Import project dataset / evaluation utilities
# ============================================================

import sys

REPO_ROOT = (
    "/content/Surrey-MScProject-Zero-Shot-Object-Counting"
)

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.datasets import (
    FSC147Dataset,
    OmniCount191Dataset,
    collate_keep_dicts,
)

from src.evaluation import (
    evaluate_single_label_counts,
)

print("Project modules imported successfully.")

Project modules imported successfully.


# Step 2 - Install / Load SAN

In [5]:
# ============================================================
# Step 2A — Clone official SAN repository
# ============================================================

%cd /content

!rm -rf SAN
!git clone https://github.com/MendelXu/SAN.git

%cd /content/SAN

/content
Cloning into 'SAN'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 183 (delta 36), reused 26 (delta 26), pack-reused 135 (from 1)
Receiving objects: 100% (183/183), 3.17 MiB | 6.90 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/SAN


In [6]:
# ============================================================
# Step 2B — Install SAN dependencies
# ============================================================

!pip install -q \
    cython \
    scipy \
    shapely \
    timm \
    h5py \
    submitit \
    scikit-image \
    wandb \
    setuptools \
    fvcore \
    tabulate \
    regex \
    opencv-python \
    "open_clip_torch==2.16.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cur

In [7]:
# ============================================================
# Step 2C — Install Detectron2
# ============================================================

%cd /content

!pip uninstall -y detectron2
!rm -rf /content/detectron2

!python -m pip install \
    'git+https://github.com/facebookresearch/detectron2.git'

/content
  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-4uw22eky
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-4uw22eky
  Resolved https://github.com/facebookresearch/detectron2.git to commit a2f4a8771ab77e8411c26b27f24f9489a28a2453
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 17.0 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp313-cp313-linux_x86_64.whl size=7049834 sha256=bf1af93cfb94a92ff8a342147318e3546864d44b5e3b9dc376a791594c8715dc
  Stored in directory: /tmp/pip-ephem-wheel-

In [8]:
# ============================================================
# Step 2D — Configure and load SAN-ViT-B/16
# ============================================================

import sys
from pathlib import Path

SAN_ROOT = Path("/content/SAN")

if str(SAN_ROOT) not in sys.path:
    sys.path.insert(0, str(SAN_ROOT))

CHECKPOINT_DIR = SAN_ROOT / "output"
CHECKPOINT_DIR.mkdir(exist_ok=True)

SAN_CHECKPOINT = (
    CHECKPOINT_DIR / "san_vit_b_16.pth"
)

if not SAN_CHECKPOINT.exists():
    !wget -O {SAN_CHECKPOINT} \
        https://huggingface.co/Mendel192/san/resolve/main/san_vit_b_16.pth

print("SAN checkpoint:", SAN_CHECKPOINT)
print("Exists:", SAN_CHECKPOINT.exists())

--2026-09-26 09:11:58--  https://huggingface.co/Mendel192/san/resolve/main/san_vit_b_16.pth
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.118, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/63f599a9cc1dd316869b503b/039ab682651d8629737a75626ef620be728538b7b3b97afcf02d80602183f000?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27san_vit_b_16.pth%3B+filename%3D%22san_vit_b_16.pth%22%3B&user_id=public&X-Xet-Cas-Uid=public&xip=PLrcf3EK65o&Expires=1790417519&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjNmNTk5YTljYzFkZDMxNjg2OWI1MDNiLzAzOWFiNjgyNjUxZDg2Mjk3MzdhNzU2MjZlZjYyMGJlNzI4NTM4YjdiM2I5N2FmY2YwMmQ4MDYwMjE4M2YwMDBcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsaWMmeGlwPVBMcmNmM0VLNjVvIiwiQ29uZGl0aW9uIjp7IkRhdGVM

In [9]:
# ============================================================
# Step 2E — Build SAN model
# ============================================================

%cd /content/SAN

from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config

from san.config import add_san_config
from train_net import Trainer

SAN_CONFIG = (
    "/content/SAN/configs/"
    "san_clip_vit_res4_coco.yaml"
)

cfg = get_cfg()

add_deeplab_config(cfg)
add_san_config(cfg)

cfg.merge_from_file(SAN_CONFIG)

cfg.MODEL.DEVICE = device
cfg.MODEL.WEIGHTS = str(SAN_CHECKPOINT)

cfg.freeze()

san_model = Trainer.build_model(cfg)

print("SAN model constructed.")
print("Model class:", san_model.__class__.__name__)

/content/SAN


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|████████████████████████████████████████| 351M/351M [00:03<00:00, 101MiB/s]
/content/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


SAN model constructed.
Model class: SAN


In [10]:
# ============================================================
# Step 2F — Load SAN pretrained weights
# ============================================================

from detectron2.checkpoint import (
    DetectionCheckpointer,
)

checkpointer = DetectionCheckpointer(
    san_model
)

checkpoint_info = checkpointer.resume_or_load(
    cfg.MODEL.WEIGHTS,
    resume=False,
)

san_model.eval()

print("SAN weights loaded successfully.")
print("Training mode:", san_model.training)
print("Device:", cfg.MODEL.DEVICE)

SAN weights loaded successfully.
Training mode: False
Device: cuda


# Step 3 — Install and load SAM2

In [11]:
# ============================================================
# Step 3A — Install official SAM2 repository
# ============================================================

%cd /content

if not Path("/content/sam2").exists():
    !git clone \
        https://github.com/facebookresearch/sam2.git

%cd /content/sam2

!pip install -q -e .

print("SAM2 repository ready.")

/content
Cloning into 'sam2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 1107 (delta 13), reused 7 (delta 7), pack-reused 1089 (from 2)
Receiving objects: 100% (1107/1107), 134.85 MiB | 26.85 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
detectron2 0.6 requires iopath<0.1.10,>=0.1.7, but you have iopath 0.1.10 which is incompatible.
SAM2 repository ready.


In [12]:
# ============================================================
# Step 3B — Prepare SAM2.1 Hiera-small checkpoint
# ============================================================

SAM2_ROOT = Path("/content/sam2")

SAM2_CHECKPOINT_DIR = (
    SAM2_ROOT / "checkpoints"
)

SAM2_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SAM2_CHECKPOINT = (
    SAM2_CHECKPOINT_DIR /
    "sam2.1_hiera_small.pt"
)

if not SAM2_CHECKPOINT.exists():

    !wget -q \
        -O {SAM2_CHECKPOINT} \
        https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt

print("SAM2 checkpoint:", SAM2_CHECKPOINT)
print("Exists:", SAM2_CHECKPOINT.exists())

SAM2 checkpoint: /content/sam2/checkpoints/sam2.1_hiera_small.pt
Exists: True


In [13]:
# ============================================================
# Step 3C — Build SAM2.1 Hiera-small predictor
# ============================================================

import sys

if str(SAM2_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(SAM2_ROOT),
    )

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import (
    SAM2ImagePredictor,
)

SAM2_MODEL_CFG = (
    "configs/sam2.1/"
    "sam2.1_hiera_s.yaml"
)

sam2_model = build_sam2(
    SAM2_MODEL_CFG,
    str(SAM2_CHECKPOINT),
    device=device,
)

sam2_model.eval()

sam2_predictor = (
    SAM2ImagePredictor(
        sam2_model
    )
)

print("SAM2 loaded successfully.")
print(
    "Predictor:",
    sam2_predictor.__class__.__name__,
)

SAM2 loaded successfully.
Predictor: SAM2ImagePredictor


# Step 4 — Define experiment paths and fixed B1 assumptions

In [14]:
# ============================================================
# Step 4A — Define B1 experiment paths and fixed assumptions
# ============================================================

B1_RESULTS_ROOT = (
    PROJECT_ROOT /
    "results" /
    "B1_experiments"
)

B1_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

B1_CACHE_ROOT = (
    PROJECT_ROOT /
    "cache" /
    "B1_SAN"
)

B1_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "B1 results:",
    B1_RESULTS_ROOT
)

print(
    "SAN cache:",
    B1_CACHE_ROOT
)

B1 results: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments
SAN cache: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/B1_SAN


In [15]:
# ============================================================
# Step 4B — Fixed B1 experimental assumptions
# ============================================================

B1_PRIMARY_DATASET = "fsc147"
B1_SPLIT = "val"

SAN_SHORT_SIDE = 640
SAN_VOCABULARY_MODE = "target+background"

SAM2_PROMPT_MODE = "single_positive"
SAM2_MULTIMASK_OUTPUT = False
SAM2_INSTANCE_SELECTION = "retain_all"

print("B1 fixed assumptions:")
print("Primary dataset      :", B1_PRIMARY_DATASET)
print("Split                :", B1_SPLIT)
print("SAN short side       :", SAN_SHORT_SIDE)
print("SAN vocabulary       :", SAN_VOCABULARY_MODE)
print("SAM2 prompt mode     :", SAM2_PROMPT_MODE)
print("SAM2 multimask       :", SAM2_MULTIMASK_OUTPUT)
print("Instance selection   :", SAM2_INSTANCE_SELECTION)

B1 fixed assumptions:
Primary dataset      : fsc147
Split                : val
SAN short side       : 640
SAN vocabulary       : target+background
SAM2 prompt mode     : single_positive
SAM2 multimask       : False
Instance selection   : retain_all


# Step 5 — Build FSC-147 and OmniCount-191 validation datasets

In [16]:
# ============================================================
# Step 5A — Build validation datasets
# ============================================================

fsc147_val_dataset = FSC147Dataset(
    root=FSC147_RAW,
    split=B1_SPLIT,
    load_images=True,
)

omnicount_val_dataset = OmniCount191Dataset(
    root=OMNICOUNT_ROOT,
    split=B1_SPLIT,
    load_images=True,
)

print(
    "FSC-147 validation images:",
    len(fsc147_val_dataset),
)

print(
    "OmniCount-191 validation images:",
    len(omnicount_val_dataset),
)

FSC-147 validation images: 1286
OmniCount-191 validation images: 3578


In [17]:
# ============================================================
# Step 5B — FSC-147 sanity check
# ============================================================

sample = fsc147_val_dataset[0]

print("Image ID   :", sample["image_id"])
print("Image size :", sample["image"].size)
print(
    "Categories :",
    list(sample["categories"].keys()),
)

for category_name, info in sample["categories"].items():
    print(
        category_name,
        "GT count =",
        info["count"],
    )

Image ID   : 190.jpg
Image size : (576, 384)
Categories : ['seagulls']
seagulls GT count = 13


In [18]:
# ============================================================
# Step 5C — OmniCount-191 sanity check
# ============================================================

sample = omnicount_val_dataset[0]

print("Image ID   :", sample["image_id"])
print("Image size :", sample["image"].size)
print(
    "Categories :",
    list(sample["categories"].keys()),
)

for category_name, info in sample["categories"].items():
    print(
        category_name,
        "GT count =",
        info["count"],
    )

Image ID   : Birds/val/0
Image size : (1920, 1080)
Categories : ['pigeon', 'raven', 'crow']
pigeon GT count = 10
raven GT count = 4
crow GT count = 2


# Step 6 — Define and freeze the FSC development subset

In [19]:
# ============================================================
# Step 6A — Build metadata-only FSC-147 validation dataset
# ============================================================

import pandas as pd
import numpy as np

fsc147_val_metadata = FSC147Dataset(
    root=FSC147_RAW,
    split=B1_SPLIT,
    load_images=False,
)

print(
    "Metadata-only FSC-147 validation images:",
    len(fsc147_val_metadata),
)

Metadata-only FSC-147 validation images: 1286


In [20]:
# ============================================================
# Step 6B — Build FSC-147 validation metadata table
# ============================================================

metadata_rows = []

for idx in range(len(fsc147_val_metadata)):

    sample_meta = fsc147_val_metadata[idx]

    # FSC-147 contains one target category per image
    category_name = next(
        iter(sample_meta["categories"])
    )

    gt_count = sample_meta["categories"][
        category_name
    ]["count"]

    metadata_rows.append(
        {
            "dataset_index": idx,
            "image_id": sample_meta["image_id"],
            "category": category_name,
            "gt_count": int(gt_count),
        }
    )

fsc147_val_metadata_df = pd.DataFrame(
    metadata_rows
)

print(
    "Number of metadata rows:",
    len(fsc147_val_metadata_df),
)

display(
    fsc147_val_metadata_df.head(10)
)

Number of metadata rows: 1286


,dataset_index,image_id,category,gt_count
0,0,190.jpg,seagulls,13
1,1,191.jpg,seagulls,15
2,2,192.jpg,seagulls,19
3,3,194.jpg,peaches,82
4,4,195.jpg,peaches,10
5,5,196.jpg,peaches,85
6,6,197.jpg,peaches,77
7,7,198.jpg,peaches,69
8,8,214.jpg,grapes,64
9,9,215.jpg,grapes,259


In [21]:
# ============================================================
# Step 6C — Inspect FSC-147 GT-count distribution
# ============================================================

print(
    fsc147_val_metadata_df["gt_count"].describe(
        percentiles=[
            0.10,
            0.20,
            0.25,
            0.40,
            0.50,
            0.60,
            0.75,
            0.80,
            0.90,
            0.95,
        ]
    )
)

print(
    "\nNumber of unique categories:",
    fsc147_val_metadata_df["category"].nunique(),
)

count    1286.000000
mean       63.796267
std       123.812072
min         7.000000
10%        10.000000
20%        12.000000
25%        14.000000
40%        20.000000
50%        27.000000
60%        37.000000
75%        63.000000
80%        76.000000
90%       139.500000
95%       215.750000
max      2092.000000
Name: gt_count, dtype: float64

Number of unique categories: 29


In [22]:
# ============================================================
# Step 6D — Inspect FSC-147 validation category distribution
# ============================================================

category_counts = (
    fsc147_val_metadata_df["category"]
    .value_counts()
    .sort_values(ascending=False)
)

print(
    "Number of unique categories:",
    len(category_counts),
)

print("\nImages per category:")
print(category_counts)

print("\nCategory frequency summary:")
print(category_counts.describe())

Number of unique categories: 29

Images per category:
category
chicken wings         157
grapes                155
bottle caps           122
flamingos             102
birds                  87
seagulls               78
books                  76
donuts tray            73
chairs                 72
pills                  55
polka dots             53
peaches                36
kiwis                  25
horses                 23
toilet paper rolls     23
ants                   22
shirts                 20
skateboard             19
camels                 14
milk cartons           14
flower pots            12
fresh cut              11
bullets                11
oyster shells           9
sausages                5
oysters                 4
shallots                4
prawn crackers          2
flowers                 2
Name: count, dtype: int64

Category frequency summary:
count     29.000000
mean      44.344828
std       45.457418
min        2.000000
25%       11.000000
50%       23.000000
75%     

In [23]:
# ============================================================
# Step 6E — Inspect GT-count range by category
# ============================================================

category_gt_summary = (
    fsc147_val_metadata_df
    .groupby("category")["gt_count"]
    .agg(
        num_images="count",
        min_count="min",
        median_count="median",
        mean_count="mean",
        max_count="max",
    )
    .sort_values(
        "num_images",
        ascending=False,
    )
)

display(category_gt_summary)

,num_images,min_count,median_count,mean_count,max_count
category,,,,,
chicken wings,157,8,17.0,18.649682,52
grapes,155,11,53.0,73.851613,757
bottle caps,122,8,50.0,109.991803,1229
flamingos,102,8,17.0,32.480392,402
birds,87,7,47.0,121.781609,2092
seagulls,78,7,14.0,22.038462,119
books,76,8,42.5,92.802632,1022
donuts tray,73,8,12.0,13.698630,54
chairs,72,8,57.5,82.263889,301


In [24]:
# ============================================================
# Step 6F — Construct density-enriched FSC-147 development subset
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Fixed sampling configuration
# ------------------------------------------------------------

RANDOM_SEED = 42

STRATUM_CONFIG = [
    {
        "stratum": "S1_0_20",
        "lower_q": 0.00,
        "upper_q": 0.20,
        "n_samples": 10,
    },
    {
        "stratum": "S2_20_40",
        "lower_q": 0.20,
        "upper_q": 0.40,
        "n_samples": 10,
    },
    {
        "stratum": "S3_40_60",
        "lower_q": 0.40,
        "upper_q": 0.60,
        "n_samples": 20,
    },
    {
        "stratum": "S4_60_80",
        "lower_q": 0.60,
        "upper_q": 0.80,
        "n_samples": 20,
    },
    {
        "stratum": "S5_80_95",
        "lower_q": 0.80,
        "upper_q": 0.95,
        "n_samples": 20,
    },
    {
        "stratum": "S6_95_100",
        "lower_q": 0.95,
        "upper_q": 1.00,
        "n_samples": 20,
    },
]

# ------------------------------------------------------------
# Compute quantile boundaries from the full FSC-147 val set
# ------------------------------------------------------------

gt_counts = fsc147_val_metadata_df["gt_count"]

quantile_boundaries = {
    q: float(gt_counts.quantile(q))
    for q in [
        0.00,
        0.20,
        0.40,
        0.60,
        0.80,
        0.95,
        1.00,
    ]
}

print("Quantile boundaries:")
for q, value in quantile_boundaries.items():
    print(f"{q:>4.2f} : {value:.2f}")

# ------------------------------------------------------------
# Assign every validation image to one density stratum
#
# Important:
# We use rank-based qcut rather than raw threshold intervals.
# This avoids ambiguity when many images share the same GT count
# at a quantile boundary.
# ------------------------------------------------------------

ranked_counts = gt_counts.rank(
    method="first"
)

fsc147_val_metadata_df = (
    fsc147_val_metadata_df.copy()
)

fsc147_val_metadata_df["density_stratum"] = pd.qcut(
    ranked_counts,
    q=[
        0.00,
        0.20,
        0.40,
        0.60,
        0.80,
        0.95,
        1.00,
    ],
    labels=[
        "S1_0_20",
        "S2_20_40",
        "S3_40_60",
        "S4_60_80",
        "S5_80_95",
        "S6_95_100",
    ],
)

# ------------------------------------------------------------
# Sample fixed number of images from each stratum
# ------------------------------------------------------------

selected_parts = []

for config in STRATUM_CONFIG:

    stratum_name = config["stratum"]
    n_samples = config["n_samples"]

    stratum_df = (
        fsc147_val_metadata_df[
            fsc147_val_metadata_df["density_stratum"]
            == stratum_name
        ]
    )

    if len(stratum_df) < n_samples:
        raise ValueError(
            f"{stratum_name} has only "
            f"{len(stratum_df)} images, "
            f"but {n_samples} were requested."
        )

    sampled_df = stratum_df.sample(
        n=n_samples,
        random_state=RANDOM_SEED,
        replace=False,
    )

    selected_parts.append(
        sampled_df
    )

fsc147_dev_100_df = pd.concat(
    selected_parts,
    ignore_index=True,
)

# ------------------------------------------------------------
# Sort for stable / readable output
# ------------------------------------------------------------

stratum_order = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]

fsc147_dev_100_df["density_stratum"] = pd.Categorical(
    fsc147_dev_100_df["density_stratum"],
    categories=stratum_order,
    ordered=True,
)

fsc147_dev_100_df = (
    fsc147_dev_100_df
    .sort_values(
        ["density_stratum", "gt_count", "image_id"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

assert len(fsc147_dev_100_df) == 100

assert (
    fsc147_dev_100_df["image_id"].nunique()
    == 100
)

print(
    "\nDevelopment subset size:",
    len(fsc147_dev_100_df),
)

print(
    "Unique images:",
    fsc147_dev_100_df["image_id"].nunique(),
)

display(
    fsc147_dev_100_df.head(20)
)

Quantile boundaries:
0.00 : 7.00
0.20 : 12.00
0.40 : 20.00
0.60 : 37.00
0.80 : 76.00
0.95 : 215.75
1.00 : 2092.00

Development subset size: 100
Unique images: 100


,dataset_index,image_id,category,gt_count,density_stratum
0,576,3873.jpg,shallots,7,S1_0_20
1,322,1964.jpg,bottle caps,8,S1_0_20
2,153,774.jpg,books,8,S1_0_20
3,360,2304.jpg,camels,9,S1_0_20
4,880,5211.jpg,donuts tray,9,S1_0_20
5,888,5220.jpg,donuts tray,9,S1_0_20
6,559,3771.jpg,donuts tray,10,S1_0_20
7,1081,6306.jpg,pills,11,S1_0_20
8,524,3585.jpg,chicken wings,12,S1_0_20
9,67,575.jpg,flamingos,12,S1_0_20


In [25]:
# ============================================================
# Step 6F-2 — Validate development subset composition
# ============================================================

print("Images per density stratum:")
print(
    fsc147_dev_100_df[
        "density_stratum"
    ].value_counts().sort_index()
)

print(
    "\nGT-count summary by stratum:"
)

stratum_summary = (
    fsc147_dev_100_df
    .groupby(
        "density_stratum",
        observed=True,
    )["gt_count"]
    .agg(
        num_images="count",
        min_count="min",
        median_count="median",
        mean_count="mean",
        max_count="max",
    )
)

display(stratum_summary)

print(
    "\nUnique categories represented:",
    fsc147_dev_100_df[
        "category"
    ].nunique(),
    "/",
    fsc147_val_metadata_df[
        "category"
    ].nunique(),
)

print(
    "\nImages per category in development subset:"
)

dev_category_counts = (
    fsc147_dev_100_df[
        "category"
    ]
    .value_counts()
    .sort_values(
        ascending=False
    )
)

print(dev_category_counts)

Images per density stratum:
density_stratum
S1_0_20      10
S2_20_40     10
S3_40_60     20
S4_60_80     20
S5_80_95     20
S6_95_100    20
Name: count, dtype: int64

GT-count summary by stratum:


,num_images,min_count,median_count,mean_count,max_count
density_stratum,,,,,
S1_0_20,10,7,9.0,9.50,12
S2_20_40,10,14,16.5,16.90,20
S3_40_60,20,21,25.5,26.85,35
S4_60_80,20,38,56.0,54.80,76
S5_80_95,20,85,111.5,118.05,175
S6_95_100,20,219,399.5,595.95,2092



Unique categories represented: 21 / 29

Images per category in development subset:
category
polka dots            11
birds                 11
chairs                10
bottle caps           10
chicken wings          9
grapes                 7
books                  7
flamingos              6
seagulls               5
pills                  5
donuts tray            4
toilet paper rolls     3
camels                 2
kiwis                  2
ants                   2
shallots               1
bullets                1
horses                 1
skateboard             1
shirts                 1
milk cartons           1
Name: count, dtype: int64


In [26]:
# ============================================================
# Step 6F-3 — Density × category cross-check
# ============================================================

density_category_table = pd.crosstab(
    fsc147_dev_100_df["category"],
    fsc147_dev_100_df["density_stratum"],
)

display(
    density_category_table
)

density_stratum,S1_0_20,S2_20_40,S3_40_60,S4_60_80,S5_80_95,S6_95_100
category,,,,,,
ants,0,0,0,2,0,0
birds,0,0,2,2,2,5
books,1,0,1,0,2,3
bottle caps,1,0,1,3,3,2
bullets,0,1,0,0,0,0
camels,1,0,0,1,0,0
chairs,0,0,1,1,7,1
chicken wings,1,3,5,0,0,0
donuts tray,3,1,0,0,0,0


In [27]:
# ============================================================
# Step 6G — Build final FSC-147 development subset
# ============================================================

FINAL_STRATUM_CONFIG = [
    {
        "stratum": "S1_0_20",
        "n_samples": 20,
    },
    {
        "stratum": "S2_20_40",
        "n_samples": 20,
    },
    {
        "stratum": "S3_40_60",
        "n_samples": 20,
    },
    {
        "stratum": "S4_60_80",
        "n_samples": 20,
    },
    {
        "stratum": "S5_80_95",
        "n_samples": 10,
    },
    {
        "stratum": "S6_95_100",
        "n_samples": 10,
    },
]

RANDOM_SEED = 42

selected_parts = []

for config in FINAL_STRATUM_CONFIG:

    stratum_name = config["stratum"]
    n_samples = config["n_samples"]

    stratum_df = (
        fsc147_val_metadata_df[
            fsc147_val_metadata_df["density_stratum"]
            == stratum_name
        ]
    )

    sampled_df = stratum_df.sample(
        n=n_samples,
        random_state=RANDOM_SEED,
        replace=False,
    )

    selected_parts.append(sampled_df)

fsc147_dev_100_df = pd.concat(
    selected_parts,
    ignore_index=True,
)

fsc147_dev_100_df["density_stratum"] = pd.Categorical(
    fsc147_dev_100_df["density_stratum"],
    categories=[
        "S1_0_20",
        "S2_20_40",
        "S3_40_60",
        "S4_60_80",
        "S5_80_95",
        "S6_95_100",
    ],
    ordered=True,
)

fsc147_dev_100_df = (
    fsc147_dev_100_df
    .sort_values(
        ["density_stratum", "gt_count", "image_id"]
    )
    .reset_index(drop=True)
)

assert len(fsc147_dev_100_df) == 100
assert fsc147_dev_100_df["image_id"].nunique() == 100

print("Final development subset size:", len(fsc147_dev_100_df))

print("\nImages per density stratum:")
print(
    fsc147_dev_100_df[
        "density_stratum"
    ].value_counts().sort_index()
)

Final development subset size: 100

Images per density stratum:
density_stratum
S1_0_20      20
S2_20_40     20
S3_40_60     20
S4_60_80     20
S5_80_95     10
S6_95_100    10
Name: count, dtype: int64


In [28]:
# ============================================================
# Step 6H — Validate final development subset
# ============================================================

print("GT-count summary by stratum:")

final_stratum_summary = (
    fsc147_dev_100_df
    .groupby(
        "density_stratum",
        observed=True,
    )["gt_count"]
    .agg(
        num_images="count",
        min_count="min",
        median_count="median",
        mean_count="mean",
        max_count="max",
    )
)

display(final_stratum_summary)

print(
    "\nUnique categories represented:",
    fsc147_dev_100_df["category"].nunique(),
    "/",
    fsc147_val_metadata_df["category"].nunique(),
)

print("\nImages per category:")

print(
    fsc147_dev_100_df[
        "category"
    ]
    .value_counts()
    .sort_values(ascending=False)
)

print("\nDensity × category table:")

display(
    pd.crosstab(
        fsc147_dev_100_df["category"],
        fsc147_dev_100_df["density_stratum"],
    )
)

GT-count summary by stratum:


,num_images,min_count,median_count,mean_count,max_count
density_stratum,,,,,
S1_0_20,20,7,9.0,9.75,12
S2_20_40,20,13,15.0,16.00,20
S3_40_60,20,21,25.5,26.85,35
S4_60_80,20,38,56.0,54.80,76
S5_80_95,10,85,114.5,121.10,175
S6_95_100,10,219,496.0,668.60,2092



Unique categories represented: 21 / 29

Images per category:
category
chicken wings         12
bottle caps           11
birds                 10
seagulls               7
grapes                 7
flamingos              7
pills                  6
books                  6
donuts tray            5
chairs                 5
ants                   4
polka dots             4
toilet paper rolls     4
kiwis                  2
camels                 2
horses                 2
skateboard             2
shallots               1
bullets                1
shirts                 1
milk cartons           1
Name: count, dtype: int64

Density × category table:


density_stratum,S1_0_20,S2_20_40,S3_40_60,S4_60_80,S5_80_95,S6_95_100
category,,,,,,
ants,1,1,0,2,0,0
birds,0,1,2,2,1,4
books,1,2,1,0,1,1
bottle caps,3,1,1,3,3,0
bullets,0,1,0,0,0,0
camels,1,0,0,1,0,0
chairs,1,0,1,1,2,0
chicken wings,2,5,5,0,0,0
donuts tray,3,2,0,0,0,0


In [29]:
# ============================================================
# Step 6H.1 — FSC-147 GT point helper
# ============================================================

def get_gt_points(sample, category):
    """
    Return FSC-147 GT points as Nx2 array [x, y].
    """
    points = sample[
        "categories"
    ][category]["points"]

    return np.asarray(
        points,
        dtype=float,
    ).reshape(-1, 2)

In [30]:
# ============================================================
# Step 6I — Save/freeze FSC-147 development subset
# ============================================================

SUBSET_DIR = (
    B1_RESULTS_ROOT /
    "subsets"
)

SUBSET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FSC147_DEV_100_FILE = (
    SUBSET_DIR /
    "FSC147_val_dev_100.csv"
)

fsc147_dev_100_df.to_csv(
    FSC147_DEV_100_FILE,
    index=False,
)

print(
    "Saved final development subset to:",
    FSC147_DEV_100_FILE
)

Saved final development subset to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/subsets/FSC147_val_dev_100.csv


# Step 7 - SAN output caching

In [31]:
# ============================================================
# Step 7A — Load frozen FSC-147 development subset
# ============================================================

import pandas as pd

FSC147_DEV_100_FILE = (
    B1_RESULTS_ROOT /
    "subsets" /
    "FSC147_val_dev_100.csv"
)

fsc147_dev_100_df = pd.read_csv(
    FSC147_DEV_100_FILE
)

print(
    "Loaded development subset:",
    len(fsc147_dev_100_df),
    "images"
)

print(
    "Unique images:",
    fsc147_dev_100_df["image_id"].nunique()
)

print("\nDensity strata:")
print(
    fsc147_dev_100_df[
        "density_stratum"
    ].value_counts().sort_index()
)

display(
    fsc147_dev_100_df.head()
)

Loaded development subset: 100 images
Unique images: 100

Density strata:
density_stratum
S1_0_20      20
S2_20_40     20
S3_40_60     20
S4_60_80     20
S5_80_95     10
S6_95_100    10
Name: count, dtype: int64


,dataset_index,image_id,category,gt_count,density_stratum
0,576,3873.jpg,shallots,7,S1_0_20
1,322,1964.jpg,bottle caps,8,S1_0_20
2,153,774.jpg,books,8,S1_0_20
3,360,2304.jpg,camels,9,S1_0_20
4,514,3573.jpg,chicken wings,9,S1_0_20


In [32]:
# ============================================================
# Step 7B — SAN preprocessing helper
# ============================================================

import numpy as np
import torch
from PIL import Image

def prepare_san_input(
    image,
    short_side=SAN_SHORT_SIDE,
):
    """
    Prepare one PIL RGB image for SAN inference.

    Returns
    -------
    image_tensor : torch.Tensor
        CHW float tensor after shortest-side resize.
    original_height : int
    original_width : int
    """

    image = image.convert("RGB")

    original_width, original_height = image.size

    scale = (
        short_side /
        min(original_height, original_width)
    )

    resized_width = int(
        round(original_width * scale)
    )

    resized_height = int(
        round(original_height * scale)
    )

    resized_image = image.resize(
        (resized_width, resized_height),
        Image.BILINEAR,
    )

    image_np = np.asarray(
        resized_image
    ).copy()

    image_tensor = (
        torch.from_numpy(image_np)
        .permute(2, 0, 1)
        .float()
    )

    return (
        image_tensor,
        original_height,
        original_width,
    )

In [33]:
# ============================================================
# Step 7C — Single-image SAN inference helper
# ============================================================

def run_san_for_sample(
    sample,
    san_model,
    device,
):
    """
    Run SAN once for one FSC-147 sample.

    Uses vocabulary:
        [target_category, "background"]

    Returns a compact dictionary containing:
        target score map
        background score map
        metadata
    """

    target_categories = list(
        sample["categories"].keys()
    )

    if len(target_categories) != 1:
        raise ValueError(
            "Current FSC development cache expects "
            "exactly one target category per image."
        )

    target_category = target_categories[0]

    san_vocabulary = [
        target_category,
        "background",
    ]

    (
        image_tensor,
        original_height,
        original_width,
    ) = prepare_san_input(
        sample["image"],
        short_side=SAN_SHORT_SIDE,
    )

    inputs = [
        {
            "image": image_tensor.to(device),
            "height": original_height,
            "width": original_width,
            "vocabulary": san_vocabulary,
        }
    ]

    with torch.no_grad():

        outputs = san_model(inputs)

    sem_seg = (
        outputs[0]["sem_seg"]
        .detach()
        .cpu()
        .float()
    )

    if sem_seg.shape[0] != 2:
        raise ValueError(
            f"Expected 2 SAN channels, "
            f"got shape {tuple(sem_seg.shape)}"
        )

    return {
        "image_id": sample["image_id"],
        "target_category": target_category,
        "vocabulary": san_vocabulary,
        "height": original_height,
        "width": original_width,

        "target_score": sem_seg[0],
        "background_score": sem_seg[1],
    }

In [34]:
# ============================================================
# Step 7D — Single-image SAN cache sanity check
# ============================================================

test_row = fsc147_dev_100_df.iloc[0]

test_index = int(
    test_row["dataset_index"]
)

test_sample = fsc147_val_dataset[
    test_index
]

test_san_cache = run_san_for_sample(
    test_sample,
    san_model,
    device,
)

print(
    "Image ID:",
    test_san_cache["image_id"]
)

print(
    "Target category:",
    test_san_cache["target_category"]
)

print(
    "Vocabulary:",
    test_san_cache["vocabulary"]
)

print(
    "Target score shape:",
    tuple(
        test_san_cache[
            "target_score"
        ].shape
    )
)

print(
    "Background score shape:",
    tuple(
        test_san_cache[
            "background_score"
        ].shape
    )
)

print(
    "Target score range:",
    float(
        test_san_cache[
            "target_score"
        ].min()
    ),
    "to",
    float(
        test_san_cache[
            "target_score"
        ].max()
    ),
)

print(
    "Background score range:",
    float(
        test_san_cache[
            "background_score"
        ].min()
    ),
    "to",
    float(
        test_san_cache[
            "background_score"
        ].max()
    ),
)

Image ID: 3873.jpg
Target category: shallots
Vocabulary: ['shallots', 'background']
Target score shape: (384, 384)
Background score shape: (384, 384)
Target score range: 0.010385769419372082 to 6.709410667419434
Background score range: 0.0021597566083073616 to 0.570577085018158


In [35]:
# ============================================================
# Step 7E — Define SAN cache directory
# ============================================================

FSC147_DEV_SAN_CACHE_DIR = (
    B1_CACHE_ROOT /
    "FSC147_val_dev_100"
)

FSC147_DEV_SAN_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "SAN cache directory:",
    FSC147_DEV_SAN_CACHE_DIR
)

SAN cache directory: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/B1_SAN/FSC147_val_dev_100


In [36]:
# ============================================================
# Step 7E.1 — SAN cache loader helper
# ============================================================

from pathlib import Path
import torch


def load_san_cache(image_id):
    """
    Load cached SAN output for one FSC-147 development image.
    """

    cache_file = (
        FSC147_DEV_SAN_CACHE_DIR /
        f"{Path(image_id).stem}.pt"
    )

    if not cache_file.exists():
        raise FileNotFoundError(
            f"SAN cache not found: {cache_file}"
        )

    cached = torch.load(
        cache_file,
        map_location="cpu",
        weights_only=False,
    )

    return cached

In [37]:
# ============================================================
# Step 7F — Cache SAN outputs for frozen development subset
# ============================================================

from tqdm.auto import tqdm
import time

cache_records = []

start_time = time.perf_counter()

for _, row in tqdm(
    fsc147_dev_100_df.iterrows(),
    total=len(fsc147_dev_100_df),
    desc="Caching SAN outputs",
):

    dataset_index = int(
        row["dataset_index"]
    )

    image_id = row["image_id"]

    cache_file = (
        FSC147_DEV_SAN_CACHE_DIR /
        f"{Path(image_id).stem}.pt"
    )

    # --------------------------------------------------------
    # Skip if already cached
    # --------------------------------------------------------

    if cache_file.exists():

        cache_records.append(
            {
                "image_id": image_id,
                "dataset_index": dataset_index,
                "cache_file": str(cache_file),
                "status": "existing",
            }
        )

        continue

    # --------------------------------------------------------
    # Load dataset sample
    # --------------------------------------------------------

    sample = fsc147_val_dataset[
        dataset_index
    ]

    # --------------------------------------------------------
    # SAN inference
    # --------------------------------------------------------

    san_cache = run_san_for_sample(
        sample,
        san_model,
        device,
    )

    # --------------------------------------------------------
    # Save compact cache
    # --------------------------------------------------------

    torch.save(
        san_cache,
        cache_file,
    )

    cache_records.append(
        {
            "image_id": image_id,
            "dataset_index": dataset_index,
            "cache_file": str(cache_file),
            "status": "created",
        }
    )

elapsed_time = (
    time.perf_counter()
    - start_time
)

san_cache_index_df = pd.DataFrame(
    cache_records
)

print(
    "\nCache completed."
)

print(
    "Total images:",
    len(san_cache_index_df)
)

print(
    "Created:",
    (
        san_cache_index_df["status"]
        == "created"
    ).sum()
)

print(
    "Existing:",
    (
        san_cache_index_df["status"]
        == "existing"
    ).sum()
)

print(
    f"Elapsed time: "
    f"{elapsed_time:.1f} seconds"
)

Caching SAN outputs:   0%|          | 0/100 [00:00<?, ?it/s]


Cache completed.
Total images: 100
Created: 0
Existing: 100
Elapsed time: 0.6 seconds


In [38]:
# ============================================================
# Step 7G — Save SAN cache index
# ============================================================

SAN_CACHE_INDEX_FILE = (
    FSC147_DEV_SAN_CACHE_DIR /
    "cache_index.csv"
)

san_cache_index_df.to_csv(
    SAN_CACHE_INDEX_FILE,
    index=False,
)

print(
    "Saved cache index:",
    SAN_CACHE_INDEX_FILE
)

Saved cache index: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/B1_SAN/FSC147_val_dev_100/cache_index.csv


In [39]:
# ============================================================
# Step 7H — Validate SAN development cache
# ============================================================

validation_rows = []

for _, row in tqdm(
    fsc147_dev_100_df.iterrows(),
    total=len(fsc147_dev_100_df),
    desc="Validating SAN cache",
):

    image_id = row["image_id"]

    cache_file = (
        FSC147_DEV_SAN_CACHE_DIR /
        f"{Path(image_id).stem}.pt"
    )

    if not cache_file.exists():

        validation_rows.append(
            {
                "image_id": image_id,
                "valid": False,
                "reason": "missing_file",
            }
        )

        continue

    cached = torch.load(
        cache_file,
        map_location="cpu",
        weights_only=False,
    )

    target_score = cached[
        "target_score"
    ]

    background_score = cached[
        "background_score"
    ]

    expected_shape = (
        int(cached["height"]),
        int(cached["width"]),
    )

    valid = (
        tuple(target_score.shape)
        == expected_shape
        and
        tuple(background_score.shape)
        == expected_shape
        and
        torch.isfinite(
            target_score
        ).all().item()
        and
        torch.isfinite(
            background_score
        ).all().item()
    )

    validation_rows.append(
        {
            "image_id": image_id,
            "valid": valid,
            "reason": (
                "ok"
                if valid
                else "shape_or_finite_error"
            ),
        }
    )

san_cache_validation_df = (
    pd.DataFrame(
        validation_rows
    )
)

print(
    "Valid caches:",
    san_cache_validation_df[
        "valid"
    ].sum(),
    "/",
    len(
        san_cache_validation_df
    ),
)

print(
    "\nValidation status:"
)

print(
    san_cache_validation_df[
        "reason"
    ].value_counts()
)

Validating SAN cache:   0%|          | 0/100 [00:00<?, ?it/s]

Valid caches: 100 / 100

Validation status:
reason
ok    100
Name: count, dtype: int64


# Step 13 — Group E: Region–Point Relationship, Complementarity and Semantic Integration

Group E investigates how the retained **region-based semantic evidence (C1, C2b, C2c)** and **activation-based semantic evidence (D1, D4, D5-768, D6B-T-native, D6B9-strict)** relate to and complement each other.

The experiments proceed in four stages:

- **E0 — Pre-Integration Performance Summary:** establish a harmonised baseline for all retained C and D representations.
- **E1 — Region–Point Relationship Analysis:** examine the spatial relationship between region-based and activation-based evidence.
- **E2 — Complementarity Analysis:** determine whether the representations recover shared or unique ground-truth objects.
- **E3 — Semantic Integration:** design and evaluate integration strategies based on the evidence from E1 and E2.

All experiments use the frozen **FSC-147 dev100 subset** and the common **3%-of-short-side one-to-one localisation evaluation protocol** established in Groups C and D.

## Step 13A — E0: Pre-Integration Performance Summary

Before analysing relationships or designing semantic integration strategies, E0 establishes a common baseline for the **eight retained Group C and Group D representations**.

The final saved C and D handoff results are loaded and compared under the same FSC-147 dev100 subset and canonical localisation protocol. E0 will:

1. verify the retained candidate inventory and handoff data;
2. produce an **overall performance summary**;
3. compare performance across the **S1–S6 density strata**.

No representations are combined or modified in E0. Its purpose is to establish the **pre-integration baseline** against which the subsequent relationship, complementarity and integration experiments can be interpreted.

In [40]:
# ============================================================
# Step 13A.1 — Define Group-E paths and retained candidate inventory
# ============================================================
#
# Purpose
# -------
# Define the final Group-C and Group-D handoff files used by
# Group E, and explicitly record the eight retained
# pre-integration representations.
#
# No experimental results are loaded or analysed in this step.
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Group-C and Group-D result directories
#
# B1_RESULTS_ROOT was defined in Step 4A as:
#
#   PROJECT_ROOT / "results" / "B1_experiments"
# ------------------------------------------------------------

GROUP_C_RESULTS_DIR = (
    B1_RESULTS_ROOT /
    "Group_C"
)

GROUP_D_RESULTS_DIR = (
    B1_RESULTS_ROOT /
    "Group_D"
)


# ------------------------------------------------------------
# Final Group-C and Group-D handoff files
# ------------------------------------------------------------

GROUP_C_E_PER_IMAGE_PATH = (
    GROUP_C_RESULTS_DIR /
    "B1_Group_C_E_candidates_per_image.csv"
)

GROUP_D_E_PER_IMAGE_PATH = (
    GROUP_D_RESULTS_DIR /
    "B1_Group_D_E_candidates_per_image.csv"
)


# ------------------------------------------------------------
# Retained candidate inventory
# ------------------------------------------------------------

GROUP_E_CANDIDATES = pd.DataFrame(
    [
        # ----------------------------------------------------
        # Region-based semantic evidence
        # ----------------------------------------------------
        {
            "candidate_id": "C1",
            "source_group": "C",
            "evidence_type": "region",
            "description":
                "A3c mask + 4-connected components",
            "role_in_E":
                "Conservative region baseline",
        },

        {
            "candidate_id": "C2b",
            "source_group": "C",
            "evidence_type": "region",
            "description":
                "A3c mask + spatial watershed (alpha=0.010)",
            "role_in_E":
                "Higher-recall split-region alternative",
        },

        {
            "candidate_id": "C2c",
            "source_group": "C",
            "evidence_type": "region",
            "description":
                "A3c mask + spatial watershed (alpha=0.020)",
            "role_in_E":
                "Intermediate region alternative",
        },

        # ----------------------------------------------------
        # Activation / point-based semantic evidence
        # ----------------------------------------------------
        {
            "candidate_id": "D1",
            "source_group": "D",
            "evidence_type": "activation_point",
            "description":
                "SAN target-score local maxima",
            "role_in_E":
                "Primary general activation baseline",
        },

        {
            "candidate_id": "D4",
            "source_group": "D",
            "evidence_type": "activation_point",
            "description":
                "D1 activation points gated by A3c semantic mask",
            "role_in_E":
                "Hard region-gating baseline",
        },

        {
            "candidate_id": "D5-768",
            "source_group": "D",
            "evidence_type": "activation_point",
            "description":
                "CLIP-Surgery activation points at 768 resolution",
            "role_in_E":
                "Dense-scene complementary evidence",
        },

        {
            "candidate_id": "D6B-T-native",
            "source_group": "D",
            "evidence_type": "activation_point",
            "description":
                "Candidate-B native query activation + Taylor refinement",
            "role_in_E":
                "Permissive high-recall activation evidence",
        },

        {
            "candidate_id": "D6B9-strict",
            "source_group": "D",
            "evidence_type": "activation_point",
            "description":
                "Candidate-B strict filtered query activation + Taylor refinement",
            "role_in_E":
                "Selective Candidate-B evidence",
        },
    ]
)


# ------------------------------------------------------------
# Expected retained candidates
# ------------------------------------------------------------

EXPECTED_GROUP_E_CANDIDATES = [
    "C1",
    "C2b",
    "C2c",
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]


# ------------------------------------------------------------
# Inventory sanity checks
# ------------------------------------------------------------

assert len(GROUP_E_CANDIDATES) == 8

assert (
    GROUP_E_CANDIDATES["candidate_id"].tolist()
    ==
    EXPECTED_GROUP_E_CANDIDATES
)

assert (
    GROUP_E_CANDIDATES["candidate_id"].is_unique
)

assert (
    (GROUP_E_CANDIDATES["source_group"] == "C").sum()
    == 3
)

assert (
    (GROUP_E_CANDIDATES["source_group"] == "D").sum()
    == 5
)


# ------------------------------------------------------------
# Check final handoff files exist
# ------------------------------------------------------------

assert GROUP_C_E_PER_IMAGE_PATH.exists(), (
    "Group-C handoff file not found:\n"
    f"{GROUP_C_E_PER_IMAGE_PATH}"
)

assert GROUP_D_E_PER_IMAGE_PATH.exists(), (
    "Group-D handoff file not found:\n"
    f"{GROUP_D_E_PER_IMAGE_PATH}"
)


# ------------------------------------------------------------
# Display inventory
# ------------------------------------------------------------

print("=" * 78)
print("Group E — retained pre-integration candidate inventory")
print("=" * 78)

print("\nB1 results root:")
print(B1_RESULTS_ROOT)

print("\nGroup-C handoff:")
print(GROUP_C_E_PER_IMAGE_PATH)

print("\nGroup-D handoff:")
print(GROUP_D_E_PER_IMAGE_PATH)

print("\nRetained candidates:")
display(GROUP_E_CANDIDATES)

print("\nCandidate summary:")
print(
    GROUP_E_CANDIDATES[
        ["source_group", "evidence_type"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nTotal retained candidates:",
    len(GROUP_E_CANDIDATES),
)

print("\nStep 13A.1 complete.")

Group E — retained pre-integration candidate inventory

B1 results root:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments

Group-C handoff:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_C/B1_Group_C_E_candidates_per_image.csv

Group-D handoff:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_D/B1_Group_D_E_candidates_per_image.csv

Retained candidates:


,candidate_id,source_group,evidence_type,description,role_in_E
0,C1,C,region,A3c mask + 4-connected components,Conservative region baseline
1,C2b,C,region,A3c mask + spatial watershed (alpha=0.010),Higher-recall split-region alternative
2,C2c,C,region,A3c mask + spatial watershed (alpha=0.020),Intermediate region alternative
3,D1,D,activation_point,SAN target-score local maxima,Primary general activation baseline
4,D4,D,activation_point,D1 activation points gated by A3c semantic mask,Hard region-gating baseline
5,D5-768,D,activation_point,CLIP-Surgery activation points at 768 resolution,Dense-scene complementary evidence
6,D6B-T-native,D,activation_point,Candidate-B native query activation + Taylor r...,Permissive high-recall activation evidence
7,D6B9-strict,D,activation_point,Candidate-B strict filtered query activation +...,Selective Candidate-B evidence



Candidate summary:
source_group  evidence_type   
C             region              3
D             activation_point    5
Name: count, dtype: int64

Total retained candidates: 8

Step 13A.1 complete.


In [41]:
# ============================================================
# Step 13A.2 — Load and inspect final C/D handoff data
# ============================================================
#
# Purpose
# -------
# Load the final Group-C and Group-D per-image handoff files
# and inspect their actual schemas before harmonisation.
#
# Expected:
#   Group C: 3 candidates x 100 images = 300 rows
#   Group D: 5 candidates x 100 images = 500 rows
#
# IMPORTANT
# ---------
# Group C and Group D were produced by separate notebooks and
# therefore contain some equivalent fields under different
# names. This step only validates the source files.
#
# No columns are renamed, removed or transformed here.
# Harmonisation is performed in Step 13A.3.
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Load final handoff files
# ------------------------------------------------------------

group_c_handoff_df = pd.read_csv(
    GROUP_C_E_PER_IMAGE_PATH
)

group_d_handoff_df = pd.read_csv(
    GROUP_D_E_PER_IMAGE_PATH
)


# ------------------------------------------------------------
# Basic dimensions
# ------------------------------------------------------------

print("=" * 78)
print("Group E — final C/D handoff inspection")
print("=" * 78)

print(
    "\nGroup C shape:",
    group_c_handoff_df.shape,
)

print(
    "Group D shape:",
    group_d_handoff_df.shape,
)


# ------------------------------------------------------------
# Candidate row counts
# ------------------------------------------------------------

print("\nGroup C candidate counts:")

display(
    group_c_handoff_df[
        "candidate_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_rows")
    .to_frame()
)


print("\nGroup D candidate counts:")

display(
    group_d_handoff_df[
        "candidate_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_rows")
    .to_frame()
)


# ------------------------------------------------------------
# Inspect actual schemas
# ------------------------------------------------------------

print("\nGroup C columns:")
print(
    group_c_handoff_df.columns.tolist()
)

print("\nGroup D columns:")
print(
    group_d_handoff_df.columns.tolist()
)


# ------------------------------------------------------------
# Compare common / source-specific columns
# ------------------------------------------------------------

c_columns = set(
    group_c_handoff_df.columns
)

d_columns = set(
    group_d_handoff_df.columns
)

common_columns = sorted(
    c_columns & d_columns
)

c_only_columns = sorted(
    c_columns - d_columns
)

d_only_columns = sorted(
    d_columns - c_columns
)


print("\nCommon columns:")
print(common_columns)

print("\nGroup-C-only columns:")
print(c_only_columns)

print("\nGroup-D-only columns:")
print(d_only_columns)


# ------------------------------------------------------------
# Common required fields
#
# These fields must exist under exactly the same names in
# both handoff files.
# ------------------------------------------------------------

common_required_columns = [
    "candidate_id",
    "experiment_id",
    "candidate_family",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "radius_px",
    "gt_count",
    "pred_count",
    "pred_gt_ratio",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "gt_points_xy",
    "pred_points_xy",
]


missing_common_c = [
    col
    for col in common_required_columns
    if col not in group_c_handoff_df.columns
]

missing_common_d = [
    col
    for col in common_required_columns
    if col not in group_d_handoff_df.columns
]


# ------------------------------------------------------------
# Source-specific required fields
#
# Equivalent information currently uses different names:
#
#   Group C                 Group D
#   -------                 -------
#   width                   image_width
#   height                  image_height
#   point_tolerance_frac    tolerance_frac
#   signed_error            count_error
#   abs_error               abs_count_error
#
# squared_error exists only in C and can be reconstructed
# from count_error after harmonisation if needed.
# ------------------------------------------------------------

c_specific_required = [
    "width",
    "height",
    "point_tolerance_frac",
    "signed_error",
    "abs_error",
]

d_specific_required = [
    "image_width",
    "image_height",
    "tolerance_frac",
    "count_error",
    "abs_count_error",
]


missing_specific_c = [
    col
    for col in c_specific_required
    if col not in group_c_handoff_df.columns
]

missing_specific_d = [
    col
    for col in d_specific_required
    if col not in group_d_handoff_df.columns
]


print("\nRequired-column check:")

print(
    "  Group C common missing:",
    missing_common_c,
)

print(
    "  Group D common missing:",
    missing_common_d,
)

print(
    "  Group C source-specific missing:",
    missing_specific_c,
)

print(
    "  Group D source-specific missing:",
    missing_specific_d,
)


# ------------------------------------------------------------
# Structural sanity checks
# ------------------------------------------------------------

assert len(group_c_handoff_df) == 300, (
    f"Expected 300 Group-C rows, "
    f"found {len(group_c_handoff_df)}."
)

assert len(group_d_handoff_df) == 500, (
    f"Expected 500 Group-D rows, "
    f"found {len(group_d_handoff_df)}."
)


assert (
    group_c_handoff_df["candidate_id"]
    .nunique()
    == 3
)

assert (
    group_d_handoff_df["candidate_id"]
    .nunique()
    == 5
)


# Exactly 100 rows per candidate
assert (
    group_c_handoff_df[
        "candidate_id"
    ]
    .value_counts()
    .eq(100)
    .all()
)

assert (
    group_d_handoff_df[
        "candidate_id"
    ]
    .value_counts()
    .eq(100)
    .all()
)


# One row per candidate-image pair
assert not (
    group_c_handoff_df[
        ["candidate_id", "dataset_index"]
    ]
    .duplicated()
    .any()
)

assert not (
    group_d_handoff_df[
        ["candidate_id", "dataset_index"]
    ]
    .duplicated()
    .any()
)


# Required schemas
assert len(missing_common_c) == 0, (
    "Required common Group-C columns missing: "
    f"{missing_common_c}"
)

assert len(missing_common_d) == 0, (
    "Required common Group-D columns missing: "
    f"{missing_common_d}"
)

assert len(missing_specific_c) == 0, (
    "Required source-specific Group-C columns missing: "
    f"{missing_specific_c}"
)

assert len(missing_specific_d) == 0, (
    "Required source-specific Group-D columns missing: "
    f"{missing_specific_d}"
)


# ------------------------------------------------------------
# Check candidate identities against frozen E inventory
# ------------------------------------------------------------

loaded_candidates = (
    set(
        group_c_handoff_df[
            "candidate_id"
        ].unique()
    )
    |
    set(
        group_d_handoff_df[
            "candidate_id"
        ].unique()
    )
)

expected_candidates = set(
    GROUP_E_CANDIDATES[
        "candidate_id"
    ]
)


print("\nLoaded candidates:")
print(sorted(loaded_candidates))

print("\nExpected candidates:")
print(sorted(expected_candidates))


assert (
    loaded_candidates
    ==
    expected_candidates
), (
    "Loaded candidate identities do not match "
    "the frozen Group-E inventory."
)


# ------------------------------------------------------------
# Verify both handoffs refer to the SAME frozen dev100 images
# ------------------------------------------------------------

c_image_keys = (
    group_c_handoff_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
        ]
    ]
    .drop_duplicates()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)

d_image_keys = (
    group_d_handoff_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
        ]
    ]
    .drop_duplicates()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


print(
    "\nUnique Group-C images:",
    len(c_image_keys),
)

print(
    "Unique Group-D images:",
    len(d_image_keys),
)


assert len(c_image_keys) == 100

assert len(d_image_keys) == 100

assert c_image_keys.equals(d_image_keys), (
    "Group C and Group D do not contain "
    "the same frozen dev100 image set."
)


# ------------------------------------------------------------
# Inspect first rows without modifying data
# ------------------------------------------------------------

print("\nGroup C sample rows:")

display(
    group_c_handoff_df.head(3)
)


print("\nGroup D sample rows:")

display(
    group_d_handoff_df.head(3)
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 78)

print("Step 13A.2 complete.")

print(
    "Group C: 300 rows / 3 candidates / "
    "100 frozen dev images"
)

print(
    "Group D: 500 rows / 5 candidates / "
    "100 frozen dev images"
)

print(
    "C and D use the same frozen dev100 image set."
)

print(
    "Source-specific naming differences identified; "
    "ready for harmonisation in Step 13A.3."
)

print("=" * 78)

Group E — final C/D handoff inspection

Group C shape: (300, 27)
Group D shape: (500, 48)

Group C candidate counts:


,n_rows
candidate_id,
C1,100
C2b,100
C2c,100



Group D candidate counts:


,n_rows
candidate_id,
D1,100
D4,100
D5-768,100
D6B-T-native,100
D6B9-strict,100



Group C columns:
['candidate_id', 'experiment_id', 'candidate_family', 'description', 'dataset_index', 'image_id', 'category', 'density_stratum', 'width', 'height', 'a3c_percentile', 'point_tolerance_frac', 'radius_px', 'gt_count', 'pred_count', 'signed_error', 'abs_error', 'squared_error', 'pred_gt_ratio', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1', 'gt_points_xy', 'pred_points_xy']

Group D columns:
['candidate_id', 'candidate_family', 'experiment_id', 'image_id', 'dataset_index', 'category', 'density_stratum', 'image_width', 'image_height', 'short_side', 'tolerance_frac', 'radius_px', 'gt_count', 'pred_count', 'count_error', 'abs_count_error', 'pred_gt_ratio', 'gt_points_xy', 'pred_points_xy', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1', 'mean_match_dist_px', 'median_match_dist_px', 'min_distance_frac', 'score_threshold', 'a3c_percentile', 'd1_points_before_gate', 'points_rejected_by_gate', 'clip_resolution', 'query_prob_threshold', 'similarity_threshold', 'gaussian_sigma', 'r

,candidate_id,experiment_id,candidate_family,description,dataset_index,image_id,category,density_stratum,width,height,...,squared_error,pred_gt_ratio,tp,fp,fn,precision,recall,f1,gt_points_xy,pred_points_xy
0,C1,B1.C.1,region_based_semantic,4-connected components,576,3873.jpg,shallots,S1_0_20,384,384,...,0,1.000000,2,5,5,0.285714,0.285714,0.285714,"[[207.1295928955078, 104.14720153808594], [87....","[[205.41571044921875, 107.74606323242188], [31..."
1,C2b,B1.C.2b,region_based_semantic,spatial watershed alpha=0.010,576,3873.jpg,shallots,S1_0_20,384,384,...,196,3.000000,2,19,5,0.095238,0.285714,0.142857,"[[207.1295928955078, 104.14720153808594], [87....","[[214.55380249023438, 105.64675903320312], [17..."
2,C2c,B1.C.2c,region_based_semantic,spatial watershed alpha=0.020,576,3873.jpg,shallots,S1_0_20,384,384,...,64,2.142857,2,13,5,0.133333,0.285714,0.181818,"[[207.1295928955078, 104.14720153808594], [87....","[[214.55380249023438, 105.64675903320312], [17..."



Group D sample rows:


,candidate_id,candidate_family,experiment_id,image_id,dataset_index,category,density_stratum,image_width,image_height,short_side,...,use_taylor,selected_queries,rejected_redundant_queries,raw_peaks,admitted_peaks,merge_radius_px,feature_h,feature_w,taylor_valid,taylor_invalid
0,D1,activation_san_final,D1,3873.jpg,576,shallots,S1_0_20,384,384,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D1,activation_san_final,D1,1964.jpg,322,bottle caps,S1_0_20,540,384,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D1,activation_san_final,D1,774.jpg,153,books,S1_0_20,575,384,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Step 13A.2 complete.
Group C: 300 rows / 3 candidates / 100 frozen dev images
Group D: 500 rows / 5 candidates / 100 frozen dev images
C and D use the same frozen dev100 image set.
Source-specific naming differences identified; ready for harmonisation in Step 13A.3.


In [42]:
# ============================================================
# Step 13A.3 — Harmonise C/D schemas and create canonical
#               Group-E dataframe
# ============================================================
#
# Purpose
# -------
# Harmonise equivalent Group-C and Group-D fields, concatenate
# the 300 C rows and 500 D rows, and create one canonical
# 800-row dataframe for subsequent Group-E analysis.
#
# Source-specific diagnostic columns are preserved.
#
# No candidate predictions, points or localisation metrics
# are recomputed in this step.
# ============================================================

import ast
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Work on copies — preserve original handoffs
# ------------------------------------------------------------

c_e_df = group_c_handoff_df.copy()
d_e_df = group_d_handoff_df.copy()


# ------------------------------------------------------------
# Harmonise equivalent source-specific column names
# ------------------------------------------------------------

c_e_df = c_e_df.rename(
    columns={
        "point_tolerance_frac": "tolerance_frac",
        "signed_error": "count_error",
        "abs_error": "abs_count_error",
    }
)

d_e_df = d_e_df.rename(
    columns={
        "image_width": "width",
        "image_height": "height",
    }
)


# ------------------------------------------------------------
# Add squared_error to Group D
# ------------------------------------------------------------

d_e_df["squared_error"] = (
    d_e_df["count_error"].astype(float) ** 2
)


# ------------------------------------------------------------
# Add source-group label
# ------------------------------------------------------------

c_e_df["source_group"] = "C"
d_e_df["source_group"] = "D"


# ------------------------------------------------------------
# Canonical Group-E core columns
# ------------------------------------------------------------

GROUP_E_CORE_COLUMNS = [
    "candidate_id",
    "source_group",
    "candidate_family",
    "experiment_id",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "width",
    "height",
    "tolerance_frac",
    "radius_px",
    "gt_count",
    "pred_count",
    "count_error",
    "abs_count_error",
    "squared_error",
    "pred_gt_ratio",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "gt_points_xy",
    "pred_points_xy",
]


# ------------------------------------------------------------
# Check canonical fields exist
# ------------------------------------------------------------

missing_c_core = [
    col
    for col in GROUP_E_CORE_COLUMNS
    if col not in c_e_df.columns
]

missing_d_core = [
    col
    for col in GROUP_E_CORE_COLUMNS
    if col not in d_e_df.columns
]


assert not missing_c_core, (
    f"Canonical fields missing from Group C: "
    f"{missing_c_core}"
)

assert not missing_d_core, (
    f"Canonical fields missing from Group D: "
    f"{missing_d_core}"
)


# ------------------------------------------------------------
# Concatenate C + D
# ------------------------------------------------------------

group_e_handoff_df = pd.concat(
    [
        c_e_df,
        d_e_df,
    ],
    ignore_index=True,
    sort=False,
)


# ------------------------------------------------------------
# Canonical candidate order
# ------------------------------------------------------------

GROUP_E_CANDIDATE_ORDER = (
    GROUP_E_CANDIDATES[
        "candidate_id"
    ]
    .tolist()
)

group_e_handoff_df["candidate_id"] = pd.Categorical(
    group_e_handoff_df["candidate_id"],
    categories=GROUP_E_CANDIDATE_ORDER,
    ordered=True,
)


# ------------------------------------------------------------
# Basic structural checks
# ------------------------------------------------------------

assert len(group_e_handoff_df) == 800

assert (
    group_e_handoff_df["candidate_id"]
    .nunique()
    == 8
)

assert not (
    group_e_handoff_df[
        ["candidate_id", "dataset_index"]
    ]
    .duplicated()
    .any()
)


candidate_counts = (
    group_e_handoff_df[
        "candidate_id"
    ]
    .value_counts(sort=False)
)

assert candidate_counts.eq(100).all()


# ------------------------------------------------------------
# Count-metric consistency
# ------------------------------------------------------------

assert np.allclose(
    group_e_handoff_df["count_error"].astype(float),
    (
        group_e_handoff_df["pred_count"].astype(float)
        -
        group_e_handoff_df["gt_count"].astype(float)
    ),
)

assert np.allclose(
    group_e_handoff_df["abs_count_error"].astype(float),
    np.abs(
        group_e_handoff_df["count_error"].astype(float)
    ),
)

assert np.allclose(
    group_e_handoff_df["squared_error"].astype(float),
    (
        group_e_handoff_df["count_error"].astype(float)
        ** 2
    ),
)


# ------------------------------------------------------------
# Parse point lists
# ------------------------------------------------------------

def parse_point_list(value):

    if isinstance(value, str):
        return ast.literal_eval(value)

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    raise TypeError(
        f"Unexpected point-list type: {type(value)}"
    )


_gt_points_parsed = (
    group_e_handoff_df["gt_points_xy"]
    .apply(parse_point_list)
)

_pred_points_parsed = (
    group_e_handoff_df["pred_points_xy"]
    .apply(parse_point_list)
)


# ------------------------------------------------------------
# Point-count consistency
# ------------------------------------------------------------

gt_point_counts = (
    _gt_points_parsed
    .apply(len)
    .to_numpy()
)

pred_point_counts = (
    _pred_points_parsed
    .apply(len)
    .to_numpy()
)


assert np.array_equal(
    gt_point_counts,
    group_e_handoff_df["gt_count"]
    .astype(int)
    .to_numpy(),
)

assert np.array_equal(
    pred_point_counts,
    group_e_handoff_df["pred_count"]
    .astype(int)
    .to_numpy(),
)


# ------------------------------------------------------------
# Cross-candidate invariant metadata
# ------------------------------------------------------------

image_invariant_columns = [
    "image_id",
    "category",
    "density_stratum",
    "width",
    "height",
    "gt_count",
    "tolerance_frac",
    "radius_px",
]


invariant_nunique = (
    group_e_handoff_df
    .groupby(
        "dataset_index",
        observed=True,
    )[image_invariant_columns]
    .nunique(dropna=False)
)


bad_invariant_rows = (
    invariant_nunique > 1
).any(axis=1)


assert not bad_invariant_rows.any(), (
    "Cross-candidate invariant metadata mismatch "
    "detected."
)


# ------------------------------------------------------------
# Exactly 8 candidate rows per image
# ------------------------------------------------------------

rows_per_image = (
    group_e_handoff_df
    .groupby(
        "dataset_index",
        observed=True,
    )
    .size()
)

assert len(rows_per_image) == 100

assert rows_per_image.eq(8).all()


# ------------------------------------------------------------
# GT-coordinate consistency
#
# Diagnostic analysis showed that C and D contain the same
# GT points but with tiny floating-point serialization
# differences (maximum observed difference ~3.1e-05 px).
#
# Therefore use an order-invariant numerical comparison with
# a strict absolute tolerance of 1e-4 pixels.
#
# Stored GT coordinates are NOT modified.
# ------------------------------------------------------------

GT_COORD_ATOL = 1e-4


def sorted_point_array(points):

    pts = sorted(
        (float(x), float(y))
        for x, y in points
    )

    return np.asarray(
        pts,
        dtype=float,
    ).reshape(-1, 2)


gt_consistency_failures = []


for dataset_index, image_rows in (
    group_e_handoff_df
    .groupby(
        "dataset_index",
        observed=True,
    )
):

    row_indices = (
        image_rows.index.tolist()
    )

    reference_points = sorted_point_array(
        _gt_points_parsed.loc[
            row_indices[0]
        ]
    )

    for row_idx in row_indices[1:]:

        candidate_points = sorted_point_array(
            _gt_points_parsed.loc[
                row_idx
            ]
        )

        if (
            reference_points.shape
            !=
            candidate_points.shape
        ):

            gt_consistency_failures.append(
                (
                    dataset_index,
                    row_idx,
                    "shape_mismatch",
                )
            )

            continue


        if not np.allclose(
            reference_points,
            candidate_points,
            rtol=0.0,
            atol=GT_COORD_ATOL,
        ):

            max_diff = float(
                np.max(
                    np.abs(
                        reference_points
                        -
                        candidate_points
                    )
                )
            )

            gt_consistency_failures.append(
                (
                    dataset_index,
                    row_idx,
                    max_diff,
                )
            )


assert not gt_consistency_failures, (
    "GT-coordinate mismatch remains after applying "
    f"atol={GT_COORD_ATOL}: "
    f"{gt_consistency_failures[:10]}"
)


# ------------------------------------------------------------
# Display canonical structure
# ------------------------------------------------------------

print("=" * 78)
print("Group E — canonical harmonised handoff")
print("=" * 78)

print(
    "\nCanonical Group-E shape:",
    group_e_handoff_df.shape,
)

print(
    "\nUnique images:",
    group_e_handoff_df[
        "dataset_index"
    ].nunique(),
)


print("\nRows per candidate:")

display(
    candidate_counts
    .rename("n_rows")
    .to_frame()
)


print("\nSource-group counts:")

display(
    group_e_handoff_df[
        "source_group"
    ]
    .value_counts()
    .sort_index()
    .rename("n_rows")
    .to_frame()
)


print(
    "\nGT-coordinate comparison tolerance:",
    GT_COORD_ATOL,
    "pixels",
)


print("\nCanonical core columns:")
print(GROUP_E_CORE_COLUMNS)


print("\nCanonical sample:")

display(
    group_e_handoff_df[
        GROUP_E_CORE_COLUMNS
    ]
    .head(8)
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 78)

print("Step 13A.3 complete.")

print(
    "Canonical Group-E dataframe: "
    "800 rows = 8 candidates x 100 images."
)

print(
    "C/D naming differences harmonised."
)

print(
    "Count metrics, point counts, image metadata and "
    "localisation radius are consistent."
)

print(
    "GT coordinates are numerically consistent across "
    f"candidates within atol={GT_COORD_ATOL} px."
)

print(
    "Original source-specific diagnostic columns preserved."
)

print("=" * 78)

Group E — canonical harmonised handoff

Canonical Group-E shape: (800, 51)

Unique images: 100

Rows per candidate:


,n_rows
candidate_id,
C1,100
C2b,100
C2c,100
D1,100
D4,100
D5-768,100
D6B-T-native,100
D6B9-strict,100



Source-group counts:


,n_rows
source_group,
C,300
D,500



GT-coordinate comparison tolerance: 0.0001 pixels

Canonical core columns:
['candidate_id', 'source_group', 'candidate_family', 'experiment_id', 'dataset_index', 'image_id', 'category', 'density_stratum', 'width', 'height', 'tolerance_frac', 'radius_px', 'gt_count', 'pred_count', 'count_error', 'abs_count_error', 'squared_error', 'pred_gt_ratio', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1', 'gt_points_xy', 'pred_points_xy']

Canonical sample:


,candidate_id,source_group,candidate_family,experiment_id,dataset_index,image_id,category,density_stratum,width,height,...,squared_error,pred_gt_ratio,tp,fp,fn,precision,recall,f1,gt_points_xy,pred_points_xy
0,C1,C,region_based_semantic,B1.C.1,576,3873.jpg,shallots,S1_0_20,384,384,...,0.0,1.000000,2,5,5,0.285714,0.285714,0.285714,"[[207.1295928955078, 104.14720153808594], [87....","[[205.41571044921875, 107.74606323242188], [31..."
1,C2b,C,region_based_semantic,B1.C.2b,576,3873.jpg,shallots,S1_0_20,384,384,...,196.0,3.000000,2,19,5,0.095238,0.285714,0.142857,"[[207.1295928955078, 104.14720153808594], [87....","[[214.55380249023438, 105.64675903320312], [17..."
2,C2c,C,region_based_semantic,B1.C.2c,576,3873.jpg,shallots,S1_0_20,384,384,...,64.0,2.142857,2,13,5,0.133333,0.285714,0.181818,"[[207.1295928955078, 104.14720153808594], [87....","[[214.55380249023438, 105.64675903320312], [17..."
3,C1,C,region_based_semantic,B1.C.1,322,1964.jpg,bottle caps,S1_0_20,540,384,...,9.0,1.375000,5,6,3,0.454545,0.625000,0.526316,"[[287.3999938964844, 73.09200286865234], [47.2...","[[290.562744140625, 67.60237121582031], [245.2..."
4,C2b,C,region_based_semantic,B1.C.2b,322,1964.jpg,bottle caps,S1_0_20,540,384,...,144.0,2.500000,5,15,3,0.250000,0.625000,0.357143,"[[287.3999938964844, 73.09200286865234], [47.2...","[[290.6514587402344, 67.6655502319336], [256.4..."
5,C2c,C,region_based_semantic,B1.C.2c,322,1964.jpg,bottle caps,S1_0_20,540,384,...,64.0,2.000000,5,11,3,0.312500,0.625000,0.416667,"[[287.3999938964844, 73.09200286865234], [47.2...","[[290.562744140625, 67.60237121582031], [245.2..."
6,C1,C,region_based_semantic,B1.C.1,153,774.jpg,books,S1_0_20,575,384,...,16.0,0.500000,0,4,8,0.000000,0.000000,0.000000,"[[496.26092529296875, 193.14337158203125], [41...","[[130.41702270507812, 110.52379608154297], [36..."
7,C2b,C,region_based_semantic,B1.C.2b,153,774.jpg,books,S1_0_20,575,384,...,81.0,2.125000,0,17,8,0.000000,0.000000,0.000000,"[[496.26092529296875, 193.14337158203125], [41...","[[125.53883361816406, 114.87833404541016], [16..."



Step 13A.3 complete.
Canonical Group-E dataframe: 800 rows = 8 candidates x 100 images.
C/D naming differences harmonised.
Count metrics, point counts, image metadata and localisation radius are consistent.
GT coordinates are numerically consistent across candidates within atol=0.0001 px.
Original source-specific diagnostic columns preserved.


In [43]:
# ============================================================
# TEMP DIAGNOSTIC — Investigate C/D GT-coordinate mismatch
# ============================================================
#
# Purpose
# -------
# Identify which dev100 images have different GT-coordinate
# signatures across the eight retained candidates, and
# determine whether the differences are:
#
#   1. point ordering only,
#   2. small numerical / rounding differences, or
#   3. genuinely different GT coordinates.
#
# This cell does NOT modify any data.
# ============================================================

import ast
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Parse GT point lists
# ------------------------------------------------------------

def parse_gt_points(value):
    if isinstance(value, str):
        return ast.literal_eval(value)

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    raise TypeError(
        f"Unexpected GT point-list type: {type(value)}"
    )


diag_df = group_e_handoff_df[
    [
        "dataset_index",
        "image_id",
        "candidate_id",
        "source_group",
        "gt_count",
        "gt_points_xy",
    ]
].copy()


diag_df["_gt_points"] = (
    diag_df["gt_points_xy"]
    .apply(parse_gt_points)
)


# ------------------------------------------------------------
# Exact signature — preserves original point order
# ------------------------------------------------------------

diag_df["_signature_exact"] = (
    diag_df["_gt_points"]
    .apply(
        lambda pts: tuple(
            (float(x), float(y))
            for x, y in pts
        )
    )
)


# ------------------------------------------------------------
# Order-invariant signature
#
# Sort coordinates so identical point sets with different
# ordering compare equal.
# ------------------------------------------------------------

diag_df["_signature_sorted"] = (
    diag_df["_gt_points"]
    .apply(
        lambda pts: tuple(
            sorted(
                (float(x), float(y))
                for x, y in pts
            )
        )
    )
)


# ------------------------------------------------------------
# Find exact-order mismatches
# ------------------------------------------------------------

exact_nunique = (
    diag_df
    .groupby("dataset_index")["_signature_exact"]
    .nunique()
)

exact_mismatch_ids = (
    exact_nunique[
        exact_nunique > 1
    ]
    .index
    .tolist()
)


# ------------------------------------------------------------
# Find mismatches that remain after ignoring point order
# ------------------------------------------------------------

sorted_nunique = (
    diag_df
    .groupby("dataset_index")["_signature_sorted"]
    .nunique()
)

set_mismatch_ids = (
    sorted_nunique[
        sorted_nunique > 1
    ]
    .index
    .tolist()
)


print("=" * 78)
print("GT-coordinate mismatch diagnostic")
print("=" * 78)

print(
    "\nImages failing exact ordered comparison:",
    len(exact_mismatch_ids),
)

print(
    "Images still failing after ignoring point order:",
    len(set_mismatch_ids),
)


# ------------------------------------------------------------
# Check whether mismatch is ONLY ordering
# ------------------------------------------------------------

ordering_only_ids = sorted(
    set(exact_mismatch_ids)
    -
    set(set_mismatch_ids)
)

print(
    "Images differing only in point ordering:",
    len(ordering_only_ids),
)


# ------------------------------------------------------------
# Display first mismatching image(s)
# ------------------------------------------------------------

if exact_mismatch_ids:

    first_ids = exact_mismatch_ids[:5]

    print(
        "\nFirst mismatching dataset_index values:",
        first_ids,
    )

    display(
        diag_df[
            diag_df["dataset_index"].isin(first_ids)
        ][
            [
                "dataset_index",
                "image_id",
                "candidate_id",
                "source_group",
                "gt_count",
                "gt_points_xy",
            ]
        ]
        .sort_values(
            ["dataset_index", "candidate_id"]
        )
    )


# ------------------------------------------------------------
# For genuine set mismatches, compare C1 against D1
# numerically after sorting.
# ------------------------------------------------------------

def sorted_point_array(points):
    pts = sorted(
        (float(x), float(y))
        for x, y in points
    )

    return np.asarray(
        pts,
        dtype=float,
    ).reshape(-1, 2)


comparison_rows = []


for dataset_index in set_mismatch_ids:

    image_rows = diag_df[
        diag_df["dataset_index"] == dataset_index
    ]

    c1_row = image_rows[
        image_rows["candidate_id"] == "C1"
    ].iloc[0]

    d1_row = image_rows[
        image_rows["candidate_id"] == "D1"
    ].iloc[0]

    c_pts = sorted_point_array(
        c1_row["_gt_points"]
    )

    d_pts = sorted_point_array(
        d1_row["_gt_points"]
    )

    same_count = (
        len(c_pts) == len(d_pts)
    )

    if same_count and len(c_pts) > 0:

        abs_diff = np.abs(
            c_pts - d_pts
        )

        max_abs_diff = float(
            abs_diff.max()
        )

        mean_abs_diff = float(
            abs_diff.mean()
        )

    elif same_count:

        max_abs_diff = 0.0
        mean_abs_diff = 0.0

    else:

        max_abs_diff = np.nan
        mean_abs_diff = np.nan


    comparison_rows.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                c1_row["image_id"],

            "gt_count_C1":
                len(c_pts),

            "gt_count_D1":
                len(d_pts),

            "same_point_count":
                same_count,

            "max_abs_coordinate_diff":
                max_abs_diff,

            "mean_abs_coordinate_diff":
                mean_abs_diff,
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


if len(comparison_df) > 0:

    print(
        "\nC1 vs D1 numerical comparison "
        "for genuine set mismatches:"
    )

    display(
        comparison_df
        .sort_values(
            "max_abs_coordinate_diff",
            ascending=False,
        )
        .head(20)
    )

else:

    print(
        "\nNo genuine coordinate-set mismatch remains "
        "after ignoring point order."
    )


print("\nDiagnostic complete.")

GT-coordinate mismatch diagnostic

Images failing exact ordered comparison: 100
Images still failing after ignoring point order: 100
Images differing only in point ordering: 0

First mismatching dataset_index values: [9, 14, 20, 31, 40]


,dataset_index,image_id,candidate_id,source_group,gt_count,gt_points_xy
273,9,215.jpg,C1,C,259,"[[23.545284271240234, 4.3632001876831055], [30..."
274,9,215.jpg,C2b,C,259,"[[23.545284271240234, 4.3632001876831055], [30..."
275,9,215.jpg,C2c,C,259,"[[23.545284271240234, 4.3632001876831055], [30..."
391,9,215.jpg,D1,D,259,"[[23.545284280936453, 4.3632], [30.43625418060..."
491,9,215.jpg,D4,D,259,"[[23.545284280936453, 4.3632], [30.43625418060..."
591,9,215.jpg,D5-768,D,259,"[[23.545284271240234, 4.3632001876831055], [30..."
782,9,215.jpg,D6B-T-native,D,259,"[[23.545284280936453, 4.3632], [30.43625418060..."
783,9,215.jpg,D6B9-strict,D,259,"[[23.545284280936453, 4.3632], [30.43625418060..."
216,14,220.jpg,C1,C,59,"[[295.3531799316406, 234.7010955810547], [234...."
217,14,220.jpg,C2b,C,59,"[[295.3531799316406, 234.7010955810547], [234...."



C1 vs D1 numerical comparison for genuine set mismatches:


,dataset_index,image_id,gt_count_C1,gt_count_D1,same_point_count,max_abs_coordinate_diff,mean_abs_coordinate_diff
96,1256,7013.jpg,139,139,True,0.000031,0.000006
97,1257,7436.jpg,283,283,True,0.000030,0.000006
24,183,838.jpg,113,113,True,0.000030,0.000006
77,938,5866.jpg,267,267,True,0.000030,0.000005
89,1157,7544.jpg,67,67,True,0.000030,0.000004
25,184,840.jpg,637,637,True,0.000030,0.000005
7,59,567.jpg,10,10,True,0.000030,0.000007
85,1102,7088.jpg,110,110,True,0.000030,0.000004
32,253,975.jpg,718,718,True,0.000030,0.000005
31,236,948.jpg,24,24,True,0.000030,0.000005



Diagnostic complete.


In [44]:
# ============================================================
# Step 13A.4 — E0 Overall Pre-Integration Performance Summary
# ============================================================
#
# Purpose
# -------
# Produce a harmonised overall performance summary for the
# eight retained Group-C and Group-D representations.
#
# Metrics:
#   - MAE
#   - RMSE
#   - Mean signed error
#   - Mean Pred/GT ratio
#   - Aggregate Pred/GT ratio
#   - Macro Precision @ 3%
#   - Macro Recall @ 3%
#   - Macro F1 @ 3%
#
# IMPORTANT
# ---------
# All metrics are computed directly from the canonical
# group_e_handoff_df created in Step 13A.3.
#
# No candidate predictions or localisation matches are
# recomputed here.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Compute one overall summary row per retained candidate
# ------------------------------------------------------------

overall_rows = []


for candidate_id in GROUP_E_CANDIDATE_ORDER:

    candidate_df = (
        group_e_handoff_df[
            group_e_handoff_df["candidate_id"]
            == candidate_id
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Structural check
    # --------------------------------------------------------

    assert len(candidate_df) == 100, (
        f"{candidate_id}: expected 100 rows, "
        f"found {len(candidate_df)}."
    )


    # --------------------------------------------------------
    # Count metrics
    # --------------------------------------------------------

    mae = float(
        candidate_df[
            "abs_count_error"
        ].mean()
    )


    rmse = float(
        np.sqrt(
            candidate_df[
                "squared_error"
            ].mean()
        )
    )


    mean_signed_error = float(
        candidate_df[
            "count_error"
        ].mean()
    )


    # --------------------------------------------------------
    # Candidate-burden / count-ratio metrics
    #
    # Mean Pred/GT:
    #   mean_i (Pred_i / GT_i)
    #
    # Aggregate Pred/GT:
    #   sum_i Pred_i / sum_i GT_i
    #
    # These are intentionally reported separately because
    # they answer different questions.
    # --------------------------------------------------------

    mean_pred_gt_ratio = float(
        candidate_df[
            "pred_gt_ratio"
        ].mean()
    )


    total_gt = int(
        candidate_df[
            "gt_count"
        ].sum()
    )


    total_pred = int(
        candidate_df[
            "pred_count"
        ].sum()
    )


    aggregate_pred_gt_ratio = float(
        total_pred / total_gt
    )


    # --------------------------------------------------------
    # Macro localisation metrics
    #
    # The per-image precision / recall / F1 values were
    # produced previously using the canonical 3%-of-short-side
    # one-to-one matching protocol.
    #
    # E0 reports their macro mean over the 100 images.
    # --------------------------------------------------------

    macro_precision = float(
        candidate_df[
            "precision"
        ].mean()
    )


    macro_recall = float(
        candidate_df[
            "recall"
        ].mean()
    )


    macro_f1 = float(
        candidate_df[
            "f1"
        ].mean()
    )


    # --------------------------------------------------------
    # Candidate metadata
    # --------------------------------------------------------

    source_group = (
        candidate_df[
            "source_group"
        ]
        .iloc[0]
    )


    candidate_family = (
        candidate_df[
            "candidate_family"
        ]
        .iloc[0]
    )


    # --------------------------------------------------------
    # Store summary row
    # --------------------------------------------------------

    overall_rows.append(
        {
            "candidate_id":
                candidate_id,

            "source_group":
                source_group,

            "candidate_family":
                candidate_family,

            "n_images":
                len(candidate_df),

            "mae":
                mae,

            "rmse":
                rmse,

            "mean_signed_error":
                mean_signed_error,

            "mean_pred_gt_ratio":
                mean_pred_gt_ratio,

            "aggregate_pred_gt_ratio":
                aggregate_pred_gt_ratio,

            "macro_precision_at_3pct":
                macro_precision,

            "macro_recall_at_3pct":
                macro_recall,

            "macro_f1_at_3pct":
                macro_f1,

            "sum_gt":
                total_gt,

            "sum_pred":
                total_pred,
        }
    )


# ------------------------------------------------------------
# Create summary dataframe
# ------------------------------------------------------------

e0_overall_summary_df = pd.DataFrame(
    overall_rows
)


# Preserve frozen candidate order
e0_overall_summary_df["candidate_id"] = pd.Categorical(
    e0_overall_summary_df["candidate_id"],
    categories=GROUP_E_CANDIDATE_ORDER,
    ordered=True,
)

e0_overall_summary_df = (
    e0_overall_summary_df
    .sort_values("candidate_id")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(e0_overall_summary_df) == 8

assert (
    e0_overall_summary_df[
        "n_images"
    ]
    .eq(100)
    .all()
)


# All candidates must share the same total GT count
assert (
    e0_overall_summary_df[
        "sum_gt"
    ]
    .nunique()
    == 1
), (
    "Candidates do not share the same total GT count."
)


# Verify aggregate Pred/GT calculation
expected_aggregate_ratio = (
    e0_overall_summary_df["sum_pred"]
    /
    e0_overall_summary_df["sum_gt"]
)

assert np.allclose(
    e0_overall_summary_df[
        "aggregate_pred_gt_ratio"
    ],
    expected_aggregate_ratio,
)


# ------------------------------------------------------------
# Display concise E0 overall table
# ------------------------------------------------------------

E0_OVERALL_DISPLAY_COLUMNS = [
    "candidate_id",
    "source_group",
    "mae",
    "rmse",
    "mean_signed_error",
    "mean_pred_gt_ratio",
    "aggregate_pred_gt_ratio",
    "macro_precision_at_3pct",
    "macro_recall_at_3pct",
    "macro_f1_at_3pct",
]


e0_overall_display_df = (
    e0_overall_summary_df[
        E0_OVERALL_DISPLAY_COLUMNS
    ]
    .copy()
)


# Round DISPLAY ONLY.
# Full-precision values remain in e0_overall_summary_df.
numeric_display_columns = [
    "mae",
    "rmse",
    "mean_signed_error",
    "mean_pred_gt_ratio",
    "aggregate_pred_gt_ratio",
    "macro_precision_at_3pct",
    "macro_recall_at_3pct",
    "macro_f1_at_3pct",
]


e0_overall_display_df[
    numeric_display_columns
] = (
    e0_overall_display_df[
        numeric_display_columns
    ]
    .round(4)
)


print("=" * 100)
print("E0 — Overall Pre-Integration Performance Summary")
print("=" * 100)

display(
    e0_overall_display_df
)


# ------------------------------------------------------------
# Supporting totals
# ------------------------------------------------------------

print("\nSupporting count totals:")

display(
    e0_overall_summary_df[
        [
            "candidate_id",
            "sum_gt",
            "sum_pred",
            "aggregate_pred_gt_ratio",
        ]
    ]
    .assign(
        aggregate_pred_gt_ratio=lambda x:
            x["aggregate_pred_gt_ratio"].round(4)
    )
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 100)

print("Step 13A.4 complete.")

print(
    "Overall pre-integration performance computed for "
    "all 8 retained candidates."
)

print(
    "Count metrics and macro localisation metrics were "
    "derived from the canonical 800-row Group-E dataframe."
)

print(
    "No predictions or localisation matches were recomputed."
)

print("=" * 100)

E0 — Overall Pre-Integration Performance Summary


,candidate_id,source_group,mae,rmse,mean_signed_error,mean_pred_gt_ratio,aggregate_pred_gt_ratio,macro_precision_at_3pct,macro_recall_at_3pct,macro_f1_at_3pct
0,C1,C,79.67,263.5414,-74.91,0.7986,0.2543,0.4031,0.2573,0.2687
1,C2b,C,88.79,257.5129,-39.71,2.2940,0.6047,0.3053,0.4250,0.2861
2,C2c,C,80.08,260.1300,-58.74,1.4963,0.4152,0.3725,0.3860,0.3178
3,D1,D,94.25,254.9215,-23.93,2.8301,0.7618,0.3770,0.4603,0.3452
4,D4,D,80.77,260.6916,-58.51,1.5923,0.4175,0.3880,0.3628,0.3247
5,D5-768,D,138.20,249.4203,40.80,5.3620,1.4062,0.2774,0.5484,0.2852
6,D6B-T-native,D,166.72,288.7728,58.68,5.8786,1.5842,0.3038,0.6475,0.3136
7,D6B9-strict,D,90.02,265.9052,-54.94,1.6777,0.4531,0.4430,0.3958,0.3277



Supporting count totals:


,candidate_id,sum_gt,sum_pred,aggregate_pred_gt_ratio
0,C1,10045,2554,0.2543
1,C2b,10045,6074,0.6047
2,C2c,10045,4171,0.4152
3,D1,10045,7652,0.7618
4,D4,10045,4194,0.4175
5,D5-768,10045,14125,1.4062
6,D6B-T-native,10045,15913,1.5842
7,D6B9-strict,10045,4551,0.4531



Step 13A.4 complete.
Overall pre-integration performance computed for all 8 retained candidates.
Count metrics and macro localisation metrics were derived from the canonical 800-row Group-E dataframe.
No predictions or localisation matches were recomputed.


In [45]:
# ============================================================
# Step 13A.5 — E0 Density-Stratified Pre-Integration Summary
# ============================================================
#
# Purpose
# -------
# Compare the eight retained pre-integration representations
# across the frozen FSC-147 dev100 density strata.
#
# Main tables:
#   1. Macro F1 @ 3% by S1-S6
#   2. Macro Recall @ 3% by S1-S6
#   3. Aggregate Pred/GT by S1-S6
#
# These tables complement the overall summary from Step 13A.4
# and expose density-dependent behaviour before E1/E2.
#
# No predictions or localisation matches are recomputed.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Frozen density-stratum order
# ------------------------------------------------------------

DENSITY_ORDER = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]


# ------------------------------------------------------------
# Validate density strata
# ------------------------------------------------------------

observed_strata = set(
    group_e_handoff_df[
        "density_stratum"
    ].dropna().unique()
)

expected_strata = set(
    DENSITY_ORDER
)

assert observed_strata == expected_strata, (
    "Observed density strata do not match the frozen "
    f"S1-S6 definition.\n"
    f"Observed: {sorted(observed_strata)}\n"
    f"Expected: {DENSITY_ORDER}"
)


# ------------------------------------------------------------
# Build density-stratified summary
# ------------------------------------------------------------

density_rows = []


for candidate_id in GROUP_E_CANDIDATE_ORDER:

    candidate_df = (
        group_e_handoff_df[
            group_e_handoff_df["candidate_id"]
            == candidate_id
        ]
        .copy()
    )


    for density_stratum in DENSITY_ORDER:

        stratum_df = (
            candidate_df[
                candidate_df["density_stratum"]
                == density_stratum
            ]
            .copy()
        )


        assert len(stratum_df) > 0, (
            f"No rows found for "
            f"{candidate_id} / {density_stratum}."
        )


        n_images = len(
            stratum_df
        )


        sum_gt = int(
            stratum_df[
                "gt_count"
            ].sum()
        )


        sum_pred = int(
            stratum_df[
                "pred_count"
            ].sum()
        )


        aggregate_pred_gt_ratio = float(
            sum_pred / sum_gt
        )


        density_rows.append(
            {
                "candidate_id":
                    candidate_id,

                "density_stratum":
                    density_stratum,

                "n_images":
                    n_images,

                "sum_gt":
                    sum_gt,

                "sum_pred":
                    sum_pred,

                "macro_precision_at_3pct":
                    float(
                        stratum_df[
                            "precision"
                        ].mean()
                    ),

                "macro_recall_at_3pct":
                    float(
                        stratum_df[
                            "recall"
                        ].mean()
                    ),

                "macro_f1_at_3pct":
                    float(
                        stratum_df[
                            "f1"
                        ].mean()
                    ),

                "aggregate_pred_gt_ratio":
                    aggregate_pred_gt_ratio,

                # Retained for later analysis if needed
                "mae":
                    float(
                        stratum_df[
                            "abs_count_error"
                        ].mean()
                    ),

                "mean_signed_error":
                    float(
                        stratum_df[
                            "count_error"
                        ].mean()
                    ),
            }
        )


e0_density_summary_df = pd.DataFrame(
    density_rows
)


# ------------------------------------------------------------
# Apply frozen ordering
# ------------------------------------------------------------

e0_density_summary_df["candidate_id"] = pd.Categorical(
    e0_density_summary_df["candidate_id"],
    categories=GROUP_E_CANDIDATE_ORDER,
    ordered=True,
)

e0_density_summary_df["density_stratum"] = pd.Categorical(
    e0_density_summary_df["density_stratum"],
    categories=DENSITY_ORDER,
    ordered=True,
)


e0_density_summary_df = (
    e0_density_summary_df
    .sort_values(
        [
            "candidate_id",
            "density_stratum",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Structural sanity checks
#
# 8 candidates x 6 strata = 48 rows
# ------------------------------------------------------------

assert len(e0_density_summary_df) == 48

assert not (
    e0_density_summary_df[
        [
            "candidate_id",
            "density_stratum",
        ]
    ]
    .duplicated()
    .any()
)


# ------------------------------------------------------------
# Verify every candidate has the frozen
# 20 / 20 / 20 / 20 / 10 / 10 allocation
# ------------------------------------------------------------

EXPECTED_STRATUM_N = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}


for candidate_id in GROUP_E_CANDIDATE_ORDER:

    candidate_density = (
        e0_density_summary_df[
            e0_density_summary_df[
                "candidate_id"
            ]
            == candidate_id
        ]
    )

    for density_stratum in DENSITY_ORDER:

        actual_n = int(
            candidate_density.loc[
                candidate_density[
                    "density_stratum"
                ]
                == density_stratum,
                "n_images",
            ].iloc[0]
        )

        expected_n = (
            EXPECTED_STRATUM_N[
                density_stratum
            ]
        )

        assert actual_n == expected_n, (
            f"{candidate_id} / {density_stratum}: "
            f"expected {expected_n} images, "
            f"found {actual_n}."
        )


# ------------------------------------------------------------
# Verify GT totals are identical across candidates
# within each stratum
# ------------------------------------------------------------

gt_totals_check = (
    e0_density_summary_df
    .groupby(
        "density_stratum",
        observed=True,
    )["sum_gt"]
    .nunique()
)

assert gt_totals_check.eq(1).all(), (
    "GT totals differ between candidates "
    "within at least one density stratum."
)


# ------------------------------------------------------------
# Helper — create candidate x stratum table
# ------------------------------------------------------------

def make_density_table(
    value_column,
    decimals=4,
):

    table = (
        e0_density_summary_df
        .pivot(
            index="candidate_id",
            columns="density_stratum",
            values=value_column,
        )
        .reindex(
            index=GROUP_E_CANDIDATE_ORDER,
            columns=DENSITY_ORDER,
        )
    )

    return table.round(decimals)


# ------------------------------------------------------------
# Table 1 — Macro F1 @ 3%
# ------------------------------------------------------------

e0_f1_by_density_df = make_density_table(
    "macro_f1_at_3pct"
)


print("=" * 100)
print("E0 — Macro F1 @ 3% by Density Stratum")
print("=" * 100)

display(
    e0_f1_by_density_df
)


# ------------------------------------------------------------
# Table 2 — Macro Recall @ 3%
# ------------------------------------------------------------

e0_recall_by_density_df = make_density_table(
    "macro_recall_at_3pct"
)


print("\n" + "=" * 100)
print("E0 — Macro Recall @ 3% by Density Stratum")
print("=" * 100)

display(
    e0_recall_by_density_df
)


# ------------------------------------------------------------
# Table 3 — Aggregate Pred/GT
# ------------------------------------------------------------

e0_burden_by_density_df = make_density_table(
    "aggregate_pred_gt_ratio"
)


print("\n" + "=" * 100)
print("E0 — Aggregate Pred/GT by Density Stratum")
print("=" * 100)

display(
    e0_burden_by_density_df
)


# ------------------------------------------------------------
# Supporting stratum composition
#
# Take one candidate only because GT composition is identical
# across all eight candidates.
# ------------------------------------------------------------

reference_candidate = (
    GROUP_E_CANDIDATE_ORDER[0]
)

e0_stratum_composition_df = (
    e0_density_summary_df[
        e0_density_summary_df[
            "candidate_id"
        ]
        == reference_candidate
    ][
        [
            "density_stratum",
            "n_images",
            "sum_gt",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("Frozen dev100 Density-Stratum Composition")
print("=" * 100)

display(
    e0_stratum_composition_df
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 100)

print("Step 13A.5 complete.")

print(
    "Density-stratified F1, recall and candidate burden "
    "computed for all 8 retained candidates."
)

print(
    "All candidates use the same frozen S1-S6 image "
    "allocation and GT population."
)

print(
    "No density-dependent routing or candidate selection "
    "has been introduced."
)

print("=" * 100)

E0 — Macro F1 @ 3% by Density Stratum


density_stratum,S1_0_20,S2_20_40,S3_40_60,S4_60_80,S5_80_95,S6_95_100
candidate_id,,,,,,
C1,0.2740,0.2652,0.2209,0.3444,0.3363,0.1418
C2b,0.2208,0.2500,0.2741,0.4073,0.3718,0.1846
C2c,0.2546,0.3053,0.3259,0.4232,0.3960,0.1638
D1,0.2043,0.3647,0.3592,0.4241,0.4976,0.2497
D4,0.2347,0.3623,0.3549,0.4117,0.3819,0.1377
D5-768,0.1279,0.2140,0.2726,0.4355,0.3759,0.3761
D6B-T-native,0.2163,0.2997,0.3220,0.3775,0.4268,0.2782
D6B9-strict,0.3242,0.4296,0.3092,0.3095,0.4142,0.1183



E0 — Macro Recall @ 3% by Density Stratum


density_stratum,S1_0_20,S2_20_40,S3_40_60,S4_60_80,S5_80_95,S6_95_100
candidate_id,,,,,,
C1,0.3214,0.2989,0.2097,0.2827,0.2605,0.0874
C2b,0.4787,0.5112,0.4530,0.4403,0.3641,0.1199
C2c,0.4261,0.4766,0.4074,0.3969,0.3429,0.1032
D1,0.4509,0.5425,0.4821,0.4909,0.4820,0.1884
D4,0.3832,0.4577,0.3970,0.3809,0.3076,0.0832
D5-768,0.5292,0.6523,0.5704,0.6061,0.4778,0.2904
D6B-T-native,0.6802,0.7785,0.6701,0.6687,0.5951,0.2847
D6B9-strict,0.4860,0.5604,0.3949,0.3067,0.3804,0.0814



E0 — Aggregate Pred/GT by Density Stratum


density_stratum,S1_0_20,S2_20_40,S3_40_60,S4_60_80,S5_80_95,S6_95_100
candidate_id,,,,,,
C1,1.2308,1.2250,0.7467,0.5164,0.3840,0.0733
C2b,3.8923,3.6562,2.2793,1.1223,0.7077,0.1247
C2c,2.5333,2.3375,1.4358,0.8020,0.5285,0.0956
D1,5.5487,3.8531,2.5885,1.5173,0.9438,0.1707
D4,3.2000,2.1812,1.4060,0.8659,0.5054,0.0832
D5-768,10.2564,7.9531,4.5456,2.7026,1.7440,0.3089
D6B-T-native,10.3795,8.9062,5.5345,3.2901,1.9794,0.3087
D6B9-strict,2.7128,2.6625,1.8939,0.8294,0.6515,0.0681



Frozen dev100 Density-Stratum Composition


,density_stratum,n_images,sum_gt
0,S1_0_20,20,195
1,S2_20_40,20,320
2,S3_40_60,20,537
3,S4_60_80,20,1096
4,S5_80_95,10,1211
5,S6_95_100,10,6686



Step 13A.5 complete.
Density-stratified F1, recall and candidate burden computed for all 8 retained candidates.
All candidates use the same frozen S1-S6 image allocation and GT population.
No density-dependent routing or candidate selection has been introduced.


In [46]:
# ============================================================
# Step 13A.6 — E0 Sanity Checks and Freeze Baseline
# ============================================================
#
# Purpose
# -------
# Perform the final consolidated validation of the E0
# pre-integration baseline before moving to E1.
#
# This step verifies:
#   1. retained candidate set;
#   2. canonical 800-row structure;
#   3. common dev100 image population;
#   4. GT and image-metadata consistency;
#   5. canonical 3%-of-short-side localisation protocol;
#   6. frozen S1-S6 allocation;
#   7. essential metric completeness;
#   8. E0 overall and density-summary completeness.
#
# IMPORTANT
# ---------
# This cell does NOT modify predictions, GT points,
# localisation matches, or E0 metrics.
#
# Passing this cell formally freezes the E0 baseline.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Frozen E0 specification
# ------------------------------------------------------------

EXPECTED_CANDIDATES = [
    "C1",
    "C2b",
    "C2c",
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]

EXPECTED_SOURCE_COUNTS = {
    "C": 300,
    "D": 500,
}

EXPECTED_STRATUM_N = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}

EXPECTED_TOLERANCE_FRAC = 0.03


# ------------------------------------------------------------
# 1. Candidate-set validation
# ------------------------------------------------------------

actual_candidates = (
    group_e_handoff_df[
        "candidate_id"
    ]
    .astype(str)
    .unique()
    .tolist()
)

assert set(actual_candidates) == set(EXPECTED_CANDIDATES), (
    "Unexpected retained candidate set.\n"
    f"Expected: {EXPECTED_CANDIDATES}\n"
    f"Observed: {actual_candidates}"
)

assert GROUP_E_CANDIDATE_ORDER == EXPECTED_CANDIDATES, (
    "GROUP_E_CANDIDATE_ORDER differs from the frozen "
    "E0 candidate order."
)


# ------------------------------------------------------------
# 2. Canonical dataframe structure
#
# 8 candidates x 100 images = 800 rows
# ------------------------------------------------------------

assert len(group_e_handoff_df) == 800, (
    f"Expected 800 Group-E rows, "
    f"found {len(group_e_handoff_df)}."
)

assert (
    group_e_handoff_df[
        "dataset_index"
    ]
    .nunique()
    == 100
), (
    "Expected exactly 100 unique dev100 images."
)


candidate_counts = (
    group_e_handoff_df[
        "candidate_id"
    ]
    .value_counts()
)

for candidate_id in EXPECTED_CANDIDATES:

    assert (
        candidate_counts.get(
            candidate_id,
            0,
        )
        == 100
    ), (
        f"{candidate_id}: expected 100 rows."
    )


assert not (
    group_e_handoff_df[
        [
            "candidate_id",
            "dataset_index",
        ]
    ]
    .duplicated()
    .any()
), (
    "Duplicate candidate/image rows detected."
)


# ------------------------------------------------------------
# 3. Source-group validation
# ------------------------------------------------------------

source_counts = (
    group_e_handoff_df[
        "source_group"
    ]
    .value_counts()
    .to_dict()
)

assert source_counts == EXPECTED_SOURCE_COUNTS, (
    "Unexpected C/D source-group row counts.\n"
    f"Expected: {EXPECTED_SOURCE_COUNTS}\n"
    f"Observed: {source_counts}"
)


# ------------------------------------------------------------
# 4. Verify identical dev100 image population
#    across all eight candidates
# ------------------------------------------------------------

reference_candidate = EXPECTED_CANDIDATES[0]

reference_image_ids = set(
    group_e_handoff_df.loc[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str)
        == reference_candidate,
        "dataset_index",
    ]
)


for candidate_id in EXPECTED_CANDIDATES:

    candidate_image_ids = set(
        group_e_handoff_df.loc[
            group_e_handoff_df[
                "candidate_id"
            ].astype(str)
            == candidate_id,
            "dataset_index",
        ]
    )

    assert (
        candidate_image_ids
        == reference_image_ids
    ), (
        f"{candidate_id}: dev100 image population "
        "does not match the reference candidate."
    )


# ------------------------------------------------------------
# 5. Cross-candidate invariant metadata
# ------------------------------------------------------------

INVARIANT_COLUMNS = [
    "image_id",
    "category",
    "density_stratum",
    "width",
    "height",
    "gt_count",
    "tolerance_frac",
    "radius_px",
]


invariant_check = (
    group_e_handoff_df
    .groupby(
        "dataset_index",
        observed=True,
    )[INVARIANT_COLUMNS]
    .nunique(dropna=False)
)


assert (
    invariant_check <= 1
).all().all(), (
    "At least one image has inconsistent GT or "
    "metadata across retained candidates."
)


# ------------------------------------------------------------
# 6. Canonical localisation protocol
#
# tolerance_frac must be 0.03
#
# radius_px must satisfy:
#
#   max(1, round(0.03 * min(width, height)))
# ------------------------------------------------------------

assert np.allclose(
    group_e_handoff_df[
        "tolerance_frac"
    ].astype(float),
    EXPECTED_TOLERANCE_FRAC,
    rtol=0.0,
    atol=1e-12,
), (
    "Non-canonical localisation tolerance detected."
)


expected_radius = np.maximum(
    1,
    np.rint(
        EXPECTED_TOLERANCE_FRAC
        *
        np.minimum(
            group_e_handoff_df[
                "width"
            ].astype(float),
            group_e_handoff_df[
                "height"
            ].astype(float),
        )
    ),
).astype(int)


actual_radius = (
    group_e_handoff_df[
        "radius_px"
    ]
    .astype(int)
    .to_numpy()
)


assert np.array_equal(
    actual_radius,
    expected_radius,
), (
    "radius_px is inconsistent with the canonical "
    "3%-of-short-side rule."
)


# ------------------------------------------------------------
# 7. Count / localisation identity checks
# ------------------------------------------------------------

assert np.array_equal(
    (
        group_e_handoff_df["tp"]
        +
        group_e_handoff_df["fn"]
    )
    .astype(int)
    .to_numpy(),
    group_e_handoff_df[
        "gt_count"
    ]
    .astype(int)
    .to_numpy(),
), (
    "GT localisation identity failed: TP + FN != GT."
)


assert np.array_equal(
    (
        group_e_handoff_df["tp"]
        +
        group_e_handoff_df["fp"]
    )
    .astype(int)
    .to_numpy(),
    group_e_handoff_df[
        "pred_count"
    ]
    .astype(int)
    .to_numpy(),
), (
    "Prediction localisation identity failed: "
    "TP + FP != predicted count."
)


assert np.allclose(
    group_e_handoff_df[
        "count_error"
    ].astype(float),
    (
        group_e_handoff_df[
            "pred_count"
        ].astype(float)
        -
        group_e_handoff_df[
            "gt_count"
        ].astype(float)
    ),
), (
    "Count-error identity failed."
)


# ------------------------------------------------------------
# 8. Essential metric completeness
# ------------------------------------------------------------

ESSENTIAL_COLUMNS = [
    "gt_count",
    "pred_count",
    "count_error",
    "abs_count_error",
    "squared_error",
    "pred_gt_ratio",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
]


missing_values = (
    group_e_handoff_df[
        ESSENTIAL_COLUMNS
    ]
    .isna()
    .sum()
)


assert (
    missing_values == 0
).all(), (
    "Missing values detected in essential E0 metrics:\n"
    f"{missing_values[missing_values > 0]}"
)


# Metric bounds
for metric in [
    "precision",
    "recall",
    "f1",
]:

    values = (
        group_e_handoff_df[
            metric
        ]
        .astype(float)
    )

    assert (
        (values >= 0.0)
        &
        (values <= 1.0)
    ).all(), (
        f"{metric} contains values outside [0, 1]."
    )


# ------------------------------------------------------------
# 9. Frozen density-stratum allocation
#
# Check one row per physical image, rather than all
# eight candidate copies.
# ------------------------------------------------------------

dev100_reference_df = (
    group_e_handoff_df[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str)
        == reference_candidate
    ]
    .copy()
)


observed_stratum_counts = (
    dev100_reference_df[
        "density_stratum"
    ]
    .value_counts()
    .to_dict()
)


assert observed_stratum_counts == EXPECTED_STRATUM_N, (
    "Frozen S1-S6 allocation mismatch.\n"
    f"Expected: {EXPECTED_STRATUM_N}\n"
    f"Observed: {observed_stratum_counts}"
)


# ------------------------------------------------------------
# 10. E0 overall-summary validation
# ------------------------------------------------------------

assert len(
    e0_overall_summary_df
) == 8, (
    "E0 overall summary should contain 8 rows."
)


overall_candidates = set(
    e0_overall_summary_df[
        "candidate_id"
    ]
    .astype(str)
)

assert overall_candidates == set(
    EXPECTED_CANDIDATES
), (
    "E0 overall summary candidate set mismatch."
)


assert (
    e0_overall_summary_df[
        "n_images"
    ]
    .astype(int)
    .eq(100)
    .all()
), (
    "E0 overall summary does not contain "
    "100 images per candidate."
)


assert (
    e0_overall_summary_df[
        "sum_gt"
    ]
    .nunique()
    == 1
), (
    "E0 overall summary GT total differs "
    "between candidates."
)


# ------------------------------------------------------------
# 11. E0 density-summary validation
#
# 8 candidates x 6 strata = 48 rows
# ------------------------------------------------------------

assert len(
    e0_density_summary_df
) == 48, (
    "E0 density summary should contain 48 rows."
)


assert not (
    e0_density_summary_df[
        [
            "candidate_id",
            "density_stratum",
        ]
    ]
    .duplicated()
    .any()
), (
    "Duplicate candidate/stratum rows found "
    "in E0 density summary."
)


density_candidates = set(
    e0_density_summary_df[
        "candidate_id"
    ]
    .astype(str)
)

assert density_candidates == set(
    EXPECTED_CANDIDATES
), (
    "E0 density-summary candidate set mismatch."
)


density_strata = set(
    e0_density_summary_df[
        "density_stratum"
    ]
    .astype(str)
)

assert density_strata == set(
    EXPECTED_STRATUM_N.keys()
), (
    "E0 density-summary stratum set mismatch."
)


# ------------------------------------------------------------
# 12. Cross-check 13A.4 summary against canonical dataframe
#
# This is deliberately independent of the 13A.4 calculation
# for several key metrics.
# ------------------------------------------------------------

for candidate_id in EXPECTED_CANDIDATES:

    raw_df = (
        group_e_handoff_df[
            group_e_handoff_df[
                "candidate_id"
            ].astype(str)
            == candidate_id
        ]
    )

    summary_row = (
        e0_overall_summary_df[
            e0_overall_summary_df[
                "candidate_id"
            ].astype(str)
            == candidate_id
        ]
        .iloc[0]
    )


    expected_mae = float(
        np.mean(
            np.abs(
                raw_df[
                    "pred_count"
                ].to_numpy(dtype=float)
                -
                raw_df[
                    "gt_count"
                ].to_numpy(dtype=float)
            )
        )
    )


    expected_rmse = float(
        np.sqrt(
            np.mean(
                (
                    raw_df[
                        "pred_count"
                    ].to_numpy(dtype=float)
                    -
                    raw_df[
                        "gt_count"
                    ].to_numpy(dtype=float)
                )
                ** 2
            )
        )
    )


    expected_precision = float(
        raw_df[
            "precision"
        ].mean()
    )

    expected_recall = float(
        raw_df[
            "recall"
        ].mean()
    )

    expected_f1 = float(
        raw_df[
            "f1"
        ].mean()
    )


    assert np.isclose(
        summary_row["mae"],
        expected_mae,
    )

    assert np.isclose(
        summary_row["rmse"],
        expected_rmse,
    )

    assert np.isclose(
        summary_row[
            "macro_precision_at_3pct"
        ],
        expected_precision,
    )

    assert np.isclose(
        summary_row[
            "macro_recall_at_3pct"
        ],
        expected_recall,
    )

    assert np.isclose(
        summary_row[
            "macro_f1_at_3pct"
        ],
        expected_f1,
    )


# ------------------------------------------------------------
# 13. Freeze E0 baseline objects
#
# Deep copies prevent later E1/E2/E3 dataframe operations
# from accidentally modifying the validated E0 baseline.
# ------------------------------------------------------------

E0_FROZEN_HANDOFF_DF = (
    group_e_handoff_df
    .copy(deep=True)
)

E0_FROZEN_OVERALL_SUMMARY_DF = (
    e0_overall_summary_df
    .copy(deep=True)
)

E0_FROZEN_DENSITY_SUMMARY_DF = (
    e0_density_summary_df
    .copy(deep=True)
)


# ------------------------------------------------------------
# Compact freeze summary
# ------------------------------------------------------------

freeze_summary_df = pd.DataFrame(
    {
        "check": [
            "Retained candidates",
            "Canonical rows",
            "Unique dev100 images",
            "Group-C rows",
            "Group-D rows",
            "Localisation tolerance",
            "Density strata",
            "Overall-summary rows",
            "Density-summary rows",
        ],

        "validated_value": [
            len(EXPECTED_CANDIDATES),
            len(E0_FROZEN_HANDOFF_DF),
            E0_FROZEN_HANDOFF_DF[
                "dataset_index"
            ].nunique(),
            EXPECTED_SOURCE_COUNTS["C"],
            EXPECTED_SOURCE_COUNTS["D"],
            "3% of short side",
            "20/20/20/20/10/10",
            len(
                E0_FROZEN_OVERALL_SUMMARY_DF
            ),
            len(
                E0_FROZEN_DENSITY_SUMMARY_DF
            ),
        ],
    }
)


print("=" * 100)
print("E0 — Final Sanity Check and Baseline Freeze")
print("=" * 100)

display(
    freeze_summary_df
)


print("\nFrozen candidates:")
print(
    "  "
    + ", ".join(
        EXPECTED_CANDIDATES
    )
)


print("\n" + "=" * 100)

print("Step 13A.6 complete.")

print(
    "E0 baseline validated and frozen."
)

print(
    "Canonical baseline: "
    "8 candidates x 100 images = 800 rows."
)

print(
    "All retained candidates use the same dev100 "
    "population, GT metadata, density strata and "
    "canonical 3%-of-short-side localisation protocol."
)

print(
    "E0 overall and density-stratified summaries "
    "are internally consistent with the canonical handoff."
)

print(
    "Subsequent E1/E2/E3 analyses must not modify "
    "the frozen candidate predictions or E0 "
    "evaluation protocol."
)

print("=" * 100)

E0 — Final Sanity Check and Baseline Freeze


,check,validated_value
0,Retained candidates,8
1,Canonical rows,800
2,Unique dev100 images,100
3,Group-C rows,300
4,Group-D rows,500
5,Localisation tolerance,3% of short side
6,Density strata,20/20/20/20/10/10
7,Overall-summary rows,8
8,Density-summary rows,48



Frozen candidates:
  C1, C2b, C2c, D1, D4, D5-768, D6B-T-native, D6B9-strict

Step 13A.6 complete.
E0 baseline validated and frozen.
Canonical baseline: 8 candidates x 100 images = 800 rows.
All retained candidates use the same dev100 population, GT metadata, density strata and canonical 3%-of-short-side localisation protocol.
E0 overall and density-stratified summaries are internally consistent with the canonical handoff.
Subsequent E1/E2/E3 analyses must not modify the frozen candidate predictions or E0 evaluation protocol.


## Step 13B — E1: Region–Point Relationship Analysis

E1 investigates the spatial relationship between the retained **region-based semantic evidence** (C1, C2b, C2c) and **activation-based semantic points** (D1, D4, D5-768, D6B-T-native, D6B9-strict).

The analysis begins with **C1 ↔ D1** as the primary baseline pair before extending the same framework to the other retained representations.

For each image, E1 examines:

1. how many activation points fall **inside versus outside semantic regions**;
2. how many activation points are associated with each region (**zero, one, or multiple**);
3. whether multiple activation points within a region indicate potentially unresolved objects in merged regions;
4. how these relationships change across the frozen **S1–S6 density strata**.

E1 is a **relationship analysis**, not an integration experiment. Candidate predictions are not modified, and no new fusion rule is introduced at this stage.

The results will provide evidence for:

- **E2 — Complementarity Analysis**, which determines whether the representations recover shared or unique GT objects; and
- **E3 — Semantic Integration Strategy**, which designs and evaluates integration rules based on the evidence from E1 and E2.

All analysis uses the frozen E0 baseline and the same FSC-147 dev100 evaluation population.

In [47]:
# ============================================================
# Step 13B.1 — Reconstruct and validate frozen C1/C2b/C2c
#               region maps
# ============================================================
#
# Purpose
# -------
# Reconstruct the actual labelled region maps underlying the
# three frozen Group-C representations:
#
#   C1  = A3c mask + 4-connected components
#   C2b = A3c mask + spatial watershed, alpha = 0.010
#   C2c = A3c mask + spatial watershed, alpha = 0.020
#
# The reconstruction uses the existing frozen SAN cache.
#
# Validation
# ----------
# For every candidate/image:
#
#   1. reconstructed number of regions must equal the frozen
#      E0 predicted count;
#
#   2. reconstructed region centroids must reproduce the
#      frozen E0 C reference points;
#
#   3. C1/C2b/C2c must have identical foreground support,
#      because all three originate from the same A3c mask.
#
# No SAN inference is performed.
# No frozen E0 prediction is modified.
# ============================================================

import json
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy import ndimage as ndi
from skimage.measure import label, regionprops
from skimage.feature import peak_local_max
from skimage.segmentation import watershed


# ------------------------------------------------------------
# Frozen Group-C configuration
# ------------------------------------------------------------

E1_A3C_PERCENTILE = 90.0

E1_REGION_CONFIGS = {
    "C1": {
        "strategy": "connected_components",
        "alpha": None,
    },
    "C2b": {
        "strategy": "watershed_spatial",
        "alpha": 0.010,
    },
    "C2c": {
        "strategy": "watershed_spatial",
        "alpha": 0.020,
    },
}


# ------------------------------------------------------------
# A3c semantic-mask reconstruction
# ------------------------------------------------------------

def e1_make_a3c_mask(
    target_score,
    percentile=90.0,
):

    if hasattr(target_score, "detach"):
        target_score = (
            target_score
            .detach()
            .cpu()
            .numpy()
        )

    target_score = np.asarray(
        target_score,
        dtype=np.float32,
    )

    threshold = float(
        np.percentile(
            target_score,
            float(percentile),
        )
    )

    mask = (
        target_score >= threshold
    )

    return (
        mask.astype(bool),
        threshold,
    )


# ------------------------------------------------------------
# Labelled regions -> geometric centroid reference points
# ------------------------------------------------------------

def e1_labelled_regions_to_points(
    labelled_regions,
):

    labelled_regions = np.asarray(
        labelled_regions,
        dtype=np.int32,
    )

    props = regionprops(
        labelled_regions
    )

    points = []
    region_areas = []

    for region in props:

        # skimage centroid is (row, col) = (y, x)
        y, x = region.centroid

        points.append(
            [x, y]
        )

        region_areas.append(
            int(region.area)
        )

    points = np.asarray(
        points,
        dtype=float,
    ).reshape(-1, 2)

    return {
        "labelled_regions":
            labelled_regions,

        "points":
            points,

        "num_regions":
            len(props),

        "num_points":
            len(points),

        "region_areas":
            region_areas,
    }


# ------------------------------------------------------------
# C1 — ordinary 4-connected components
# ------------------------------------------------------------

def e1_reconstruct_c1(
    mask,
):

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    labelled_regions = label(
        mask,
        connectivity=1,
    )

    result = (
        e1_labelled_regions_to_points(
            labelled_regions
        )
    )

    result["strategy"] = (
        "connected_components"
    )

    result["parameter"] = None

    return result


# ------------------------------------------------------------
# C2b / C2c — spatial watershed
# ------------------------------------------------------------

def e1_reconstruct_watershed(
    mask,
    alpha,
):

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    height, width = mask.shape

    min_distance = max(
        1,
        int(
            round(
                float(alpha)
                * min(
                    height,
                    width,
                )
            )
        ),
    )

    # Original 4-connected components
    cc_labels = label(
        mask,
        connectivity=1,
    )

    output_labels = np.zeros(
        mask.shape,
        dtype=np.int32,
    )

    next_label = 1
    total_markers = 0

    original_components = int(
        cc_labels.max()
    )

    # Process each connected component independently
    for component_id in range(
        1,
        original_components + 1,
    ):

        component_mask = (
            cc_labels == component_id
        )

        if not component_mask.any():
            continue

        distance = (
            ndi.distance_transform_edt(
                component_mask
            )
        )

        peak_coords = peak_local_max(
            distance,
            labels=component_mask,
            min_distance=min_distance,
            exclude_border=False,
        )

        # Safety fallback:
        # every non-empty component must have >= 1 marker
        if len(peak_coords) == 0:

            max_pos = np.unravel_index(
                np.argmax(distance),
                distance.shape,
            )

            peak_coords = np.asarray(
                [max_pos]
            )

        markers = np.zeros(
            mask.shape,
            dtype=np.int32,
        )

        for marker_id, (
            row,
            col,
        ) in enumerate(
            peak_coords,
            start=1,
        ):

            markers[
                row,
                col,
            ] = marker_id

        total_markers += len(
            peak_coords
        )

        ws_labels = watershed(
            -distance,
            markers=markers,
            mask=component_mask,
        )

        local_region_ids = np.unique(
            ws_labels
        )

        local_region_ids = (
            local_region_ids[
                local_region_ids > 0
            ]
        )

        for local_id in local_region_ids:

            output_labels[
                ws_labels == local_id
            ] = next_label

            next_label += 1

    result = (
        e1_labelled_regions_to_points(
            output_labels
        )
    )

    result["strategy"] = (
        "watershed_spatial"
    )

    result["parameter"] = float(
        alpha
    )

    result["min_distance_px"] = int(
        min_distance
    )

    result["original_components"] = int(
        original_components
    )

    result["num_markers"] = int(
        total_markers
    )

    return result


# ------------------------------------------------------------
# Helper — parse frozen E0 point coordinates
# ------------------------------------------------------------

def e1_parse_points(
    value,
):

    if isinstance(value, str):
        points = json.loads(value)
    else:
        points = value

    return np.asarray(
        points,
        dtype=float,
    ).reshape(-1, 2)


# ------------------------------------------------------------
# Select the frozen Group-C E0 rows
# ------------------------------------------------------------

e1_frozen_c_df = (
    E0_FROZEN_HANDOFF_DF[
        E0_FROZEN_HANDOFF_DF[
            "candidate_id"
        ].isin(
            [
                "C1",
                "C2b",
                "C2c",
            ]
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# Validate frozen Group-C handoff
# ------------------------------------------------------------

assert len(e1_frozen_c_df) == 300, (
    f"Expected 300 Group-C rows, "
    f"got {len(e1_frozen_c_df)}."
)

e1_c_counts = (
    e1_frozen_c_df[
        "candidate_id"
    ]
    .value_counts()
)

print("=" * 100)
print("Frozen Group-C handoff")
print("=" * 100)

print(
    e1_c_counts
    .reindex(
        ["C1", "C2b", "C2c"]
    )
)

for candidate_id in [
    "C1",
    "C2b",
    "C2c",
]:

    n_rows = int(
        e1_c_counts.get(
            candidate_id,
            0,
        )
    )

    assert n_rows == 100, (
        f"{candidate_id}: "
        f"expected 100 rows, "
        f"got {n_rows}."
    )

print(
    "\nFrozen Group-C handoff check: PASS"
)


# ------------------------------------------------------------
# Storage
#
# E1_REGION_MAPS[dataset_index]["C1"]
# E1_REGION_MAPS[dataset_index]["C2b"]
# E1_REGION_MAPS[dataset_index]["C2c"]
#
# Each contains the actual integer-labelled region map,
# centroid points and reconstruction metadata.
# ------------------------------------------------------------

E1_REGION_MAPS = {}

validation_rows = []


# ------------------------------------------------------------
# Use one C1 row per dev100 image as image metadata reference
# ------------------------------------------------------------

reference_images_df = (
    e1_frozen_c_df[
        e1_frozen_c_df[
            "candidate_id"
        ]
        ==
        "C1"
    ]
    [
        [
            "dataset_index",
            "image_id",
            "density_stratum",
            "width",
            "height",
        ]
    ]
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    reference_images_df
) == 100


# ------------------------------------------------------------
# Reconstruct all three region representations
# ------------------------------------------------------------

for _, image_row in tqdm(
    reference_images_df.iterrows(),
    total=len(reference_images_df),
    desc="Reconstructing C1/C2b/C2c regions",
):

    dataset_index = int(
        image_row[
            "dataset_index"
        ]
    )

    image_id = str(
        image_row[
            "image_id"
        ]
    )

    width = int(
        image_row[
            "width"
        ]
    )

    height = int(
        image_row[
            "height"
        ]
    )

    density_stratum = str(
        image_row[
            "density_stratum"
        ]
    )

    # --------------------------------------------------------
    # Load existing frozen SAN cache
    # --------------------------------------------------------

    san_cache = load_san_cache(
        image_id
    )

    # --------------------------------------------------------
    # Reconstruct frozen A3c mask
    # --------------------------------------------------------

    semantic_mask, a3c_threshold = (
        e1_make_a3c_mask(
            san_cache[
                "target_score"
            ],
            percentile=
                E1_A3C_PERCENTILE,
        )
    )


    assert semantic_mask.shape == (
        height,
        width,
    ), (
        f"{image_id}: "
        f"semantic-mask shape "
        f"{semantic_mask.shape} != "
        f"expected {(height, width)}"
    )


    # --------------------------------------------------------
    # Reconstruct C1 / C2b / C2c
    # --------------------------------------------------------

    reconstructed = {}

    reconstructed["C1"] = (
        e1_reconstruct_c1(
            semantic_mask
        )
    )

    reconstructed["C2b"] = (
        e1_reconstruct_watershed(
            semantic_mask,
            alpha=0.010,
        )
    )

    reconstructed["C2c"] = (
        e1_reconstruct_watershed(
            semantic_mask,
            alpha=0.020,
        )
    )


    # --------------------------------------------------------
    # Foreground-support sanity
    #
    # C1/C2b/C2c all originate from the same A3c mask.
    # Their partitions differ, but their foreground support
    # must therefore be identical.
    # --------------------------------------------------------

    foreground_c1 = (
        reconstructed[
            "C1"
        ][
            "labelled_regions"
        ]
        > 0
    )

    foreground_c2b = (
        reconstructed[
            "C2b"
        ][
            "labelled_regions"
        ]
        > 0
    )

    foreground_c2c = (
        reconstructed[
            "C2c"
        ][
            "labelled_regions"
        ]
        > 0
    )


    assert np.array_equal(
        foreground_c1,
        semantic_mask,
    ), (
        f"{image_id}: "
        "C1 foreground does not "
        "reproduce A3c mask."
    )


    assert np.array_equal(
        foreground_c2b,
        semantic_mask,
    ), (
        f"{image_id}: "
        "C2b foreground does not "
        "reproduce A3c mask."
    )


    assert np.array_equal(
        foreground_c2c,
        semantic_mask,
    ), (
        f"{image_id}: "
        "C2c foreground does not "
        "reproduce A3c mask."
    )


    # --------------------------------------------------------
    # Validate against frozen E0 Group-C evidence
    # --------------------------------------------------------

    for candidate_id in [
        "C1",
        "C2b",
        "C2c",
    ]:

        frozen_rows = (
            e1_frozen_c_df[
                (
                    e1_frozen_c_df[
                        "candidate_id"
                    ]
                    ==
                    candidate_id
                )
                &
                (
                    e1_frozen_c_df[
                        "dataset_index"
                    ]
                    ==
                    dataset_index
                )
            ]
        )

        assert len(
            frozen_rows
        ) == 1, (
            f"{image_id} / {candidate_id}: "
            f"expected exactly one frozen row, "
            f"got {len(frozen_rows)}."
        )

        frozen_row = (
            frozen_rows.iloc[0]
        )


        frozen_points = (
            e1_parse_points(
                frozen_row[
                    "pred_points_xy"
                ]
            )
        )


        reconstructed_points = (
            np.asarray(
                reconstructed[
                    candidate_id
                ][
                    "points"
                ],
                dtype=float,
            )
            .reshape(-1, 2)
        )


        frozen_count = int(
            frozen_row[
                "pred_count"
            ]
        )

        reconstructed_count = int(
            reconstructed[
                candidate_id
            ][
                "num_regions"
            ]
        )


        # ----------------------------------------------------
        # Count validation
        # ----------------------------------------------------

        count_match = (
            reconstructed_count
            ==
            frozen_count
        )


        # ----------------------------------------------------
        # Centroid-coordinate validation
        #
        # Use the same small numerical tolerance adopted in
        # E0 for serialised floating-point coordinates.
        # ----------------------------------------------------

        if (
            len(frozen_points)
            ==
            len(reconstructed_points)
        ):

            point_match = np.allclose(
                reconstructed_points,
                frozen_points,
                rtol=0.0,
                atol=1e-4,
            )

            if len(
                frozen_points
            ) > 0:

                max_point_diff = float(
                    np.max(
                        np.abs(
                            reconstructed_points
                            -
                            frozen_points
                        )
                    )
                )

            else:

                max_point_diff = 0.0

        else:

            point_match = False
            max_point_diff = np.nan


        validation_rows.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "density_stratum":
                    density_stratum,

                "candidate_id":
                    candidate_id,

                "frozen_count":
                    frozen_count,

                "reconstructed_count":
                    reconstructed_count,

                "count_match":
                    bool(
                        count_match
                    ),

                "point_match_atol_1e4":
                    bool(
                        point_match
                    ),

                "max_point_abs_diff_px":
                    max_point_diff,

                "foreground_pixels":
                    int(
                        semantic_mask.sum()
                    ),

                "a3c_threshold":
                    float(
                        a3c_threshold
                    ),
            }
        )


        assert count_match, (
            f"{image_id} / {candidate_id}: "
            f"reconstructed count "
            f"{reconstructed_count} != "
            f"frozen count {frozen_count}."
        )


        assert point_match, (
            f"{image_id} / {candidate_id}: "
            "reconstructed centroid points "
            "do not reproduce frozen E0 points."
        )


    # --------------------------------------------------------
    # Store validated maps for E1
    # --------------------------------------------------------

    E1_REGION_MAPS[
        dataset_index
    ] = {
        "image_id":
            image_id,

        "density_stratum":
            density_stratum,

        "width":
            width,

        "height":
            height,

        "a3c_percentile":
            E1_A3C_PERCENTILE,

        "a3c_threshold":
            float(
                a3c_threshold
            ),

        "semantic_mask":
            semantic_mask,

        "C1":
            reconstructed[
                "C1"
            ],

        "C2b":
            reconstructed[
                "C2b"
            ],

        "C2c":
            reconstructed[
                "C2c"
            ],
    }


# ------------------------------------------------------------
# Validation dataframe
# ------------------------------------------------------------

e1_region_reconstruction_validation_df = (
    pd.DataFrame(
        validation_rows
    )
)


# ------------------------------------------------------------
# Final consolidated sanity checks
# ------------------------------------------------------------

assert len(
    E1_REGION_MAPS
) == 100, (
    f"Expected 100 reconstructed images, "
    f"got {len(E1_REGION_MAPS)}."
)


assert len(
    e1_region_reconstruction_validation_df
) == 300, (
    "Expected 300 validation rows "
    "(3 candidates × 100 images)."
)


assert (
    e1_region_reconstruction_validation_df[
        "count_match"
    ]
    .all()
), (
    "At least one reconstructed region count "
    "does not match frozen E0."
)


assert (
    e1_region_reconstruction_validation_df[
        "point_match_atol_1e4"
    ]
    .all()
), (
    "At least one reconstructed centroid set "
    "does not match frozen E0."
)


# ------------------------------------------------------------
# Compact validation summary
# ------------------------------------------------------------

reconstruction_summary_df = (
    e1_region_reconstruction_validation_df
    .groupby(
        "candidate_id",
        observed=True,
    )
    .agg(
        n_images=(
            "dataset_index",
            "size",
        ),

        count_matches=(
            "count_match",
            "sum",
        ),

        point_matches=(
            "point_match_atol_1e4",
            "sum",
        ),

        max_centroid_diff_px=(
            "max_point_abs_diff_px",
            "max",
        ),

        total_regions=(
            "reconstructed_count",
            "sum",
        ),
    )
    .reindex(
        [
            "C1",
            "C2b",
            "C2c",
        ]
    )
    .reset_index()
)


print("\n" + "=" * 100)

print(
    "E1 — Frozen Group-C Region Reconstruction Validation"
)

print("=" * 100)

display(
    reconstruction_summary_df
)


print(
    "\nStored region-map images:",
    len(E1_REGION_MAPS),
)

print(
    "Validation rows:",
    len(
        e1_region_reconstruction_validation_df
    ),
)


print(
    "\nShared foreground support:"
)

print(
    "  C1  == A3c : PASS"
)

print(
    "  C2b == A3c : PASS"
)

print(
    "  C2c == A3c : PASS"
)


print("\n" + "=" * 100)

print("Step 13B.1 complete.")

print(
    "Frozen C1/C2b/C2c region maps "
    "successfully reconstructed."
)

print(
    "Region counts and centroid reference points "
    "reproduce the frozen E0 Group-C evidence."
)

print(
    "No SAN inference or E0 prediction was modified."
)

print("=" * 100)

Frozen Group-C handoff
candidate_id
C1     100
C2b    100
C2c    100
Name: count, dtype: int64

Frozen Group-C handoff check: PASS


Reconstructing C1/C2b/C2c regions:   0%|          | 0/100 [00:00<?, ?it/s]


E1 — Frozen Group-C Region Reconstruction Validation


,candidate_id,n_images,count_matches,point_matches,max_centroid_diff_px,total_regions
0,C1,100,100,100,0.00003,2554
1,C2b,100,100,100,0.00003,6074
2,C2c,100,100,100,0.00003,4171



Stored region-map images: 100
Validation rows: 300

Shared foreground support:
  C1  == A3c : PASS
  C2b == A3c : PASS
  C2c == A3c : PASS

Step 13B.1 complete.
Frozen C1/C2b/C2c region maps successfully reconstructed.
Region counts and centroid reference points reproduce the frozen E0 Group-C evidence.
No SAN inference or E0 prediction was modified.


In [48]:
# ============================================================
# Step 13B.1a — Save validated Group-C region-map cache
# ============================================================
#
# Purpose
# -------
# Persist the validated region maps reconstructed in Step 13B.1
# so that later E1/E2/E3 analyses do not need to reconstruct
# them after a Colab runtime restart.
#
# Saved per image:
#
#   semantic_mask  : frozen A3c foreground mask
#   C1_labels      : C1 4-connected-component labels
#   C2b_labels     : C2b watershed labels (alpha = 0.010)
#   C2c_labels     : C2c watershed labels (alpha = 0.020)
#
# A cache_index.csv is also saved with reconstruction metadata.
#
# After saving, every file is loaded again and compared exactly
# against the validated in-memory Step 13B.1 representation.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# E1 derived-region cache
#
# Keep this separate from B1_CACHE_ROOT, which is specifically
# the SAN cache:
#
#   PROJECT_ROOT / cache / B1_SAN
#
# E1 region maps are derived Group-E artifacts and therefore
# receive their own cache directory.
# ------------------------------------------------------------

E1_REGION_CACHE_ROOT = (
    PROJECT_ROOT
    / "cache"
    / "B1_E1_region_maps"
)

E1_REGION_CACHE_DIR = (
    E1_REGION_CACHE_ROOT
    / "FSC147_val_dev_100"
)

E1_REGION_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

E1_REGION_CACHE_INDEX_PATH = (
    E1_REGION_CACHE_DIR
    / "cache_index.csv"
)

print(
    "E1 region-map cache:",
    E1_REGION_CACHE_DIR
)


print("=" * 100)
print("E1 — Saving validated Group-C region-map cache")
print("=" * 100)

print(
    "\nCache directory:"
)

print(
    E1_REGION_CACHE_DIR
)


# ------------------------------------------------------------
# Basic pre-save validation
# ------------------------------------------------------------

assert "E1_REGION_MAPS" in globals(), (
    "E1_REGION_MAPS is not available. "
    "Run Step 13B.1 first."
)

assert len(
    E1_REGION_MAPS
) == 100, (
    f"Expected 100 validated region-map images, "
    f"got {len(E1_REGION_MAPS)}."
)


# ------------------------------------------------------------
# Save one compressed NPZ per image
# ------------------------------------------------------------

cache_index_rows = []


for dataset_index in sorted(
    E1_REGION_MAPS.keys()
):

    item = E1_REGION_MAPS[
        dataset_index
    ]

    image_id = str(
        item["image_id"]
    )

    semantic_mask = np.asarray(
        item["semantic_mask"],
        dtype=bool,
    )

    c1_labels = np.asarray(
        item["C1"]["labelled_regions"],
        dtype=np.int32,
    )

    c2b_labels = np.asarray(
        item["C2b"]["labelled_regions"],
        dtype=np.int32,
    )

    c2c_labels = np.asarray(
        item["C2c"]["labelled_regions"],
        dtype=np.int32,
    )


    # --------------------------------------------------------
    # Pre-save structural checks
    # --------------------------------------------------------

    expected_shape = (
        int(item["height"]),
        int(item["width"]),
    )

    assert (
        semantic_mask.shape
        ==
        expected_shape
    )

    assert (
        c1_labels.shape
        ==
        expected_shape
    )

    assert (
        c2b_labels.shape
        ==
        expected_shape
    )

    assert (
        c2c_labels.shape
        ==
        expected_shape
    )


    # All three region representations must preserve
    # exactly the same A3c foreground support.
    assert np.array_equal(
        c1_labels > 0,
        semantic_mask,
    )

    assert np.array_equal(
        c2b_labels > 0,
        semantic_mask,
    )

    assert np.array_equal(
        c2c_labels > 0,
        semantic_mask,
    )


    # --------------------------------------------------------
    # Use dataset index in filename to guarantee uniqueness.
    # image_id is retained inside the NPZ and cache index.
    # --------------------------------------------------------

    cache_filename = (
        f"{dataset_index:04d}_{image_id}.npz"
    )

    cache_path = (
        E1_REGION_CACHE_DIR
        / cache_filename
    )


    # --------------------------------------------------------
    # Save compressed arrays + essential metadata
    # --------------------------------------------------------

    np.savez_compressed(
        cache_path,

        dataset_index=np.int32(
            dataset_index
        ),

        image_id=np.asarray(
            image_id
        ),

        density_stratum=np.asarray(
            str(
                item["density_stratum"]
            )
        ),

        width=np.int32(
            item["width"]
        ),

        height=np.int32(
            item["height"]
        ),

        a3c_percentile=np.float32(
            item["a3c_percentile"]
        ),

        a3c_threshold=np.float32(
            item["a3c_threshold"]
        ),

        semantic_mask=semantic_mask,

        C1_labels=c1_labels,

        C2b_labels=c2b_labels,

        C2c_labels=c2c_labels,
    )


    # --------------------------------------------------------
    # Index metadata
    # --------------------------------------------------------

    cache_index_rows.append(
        {
            "dataset_index":
                int(dataset_index),

            "image_id":
                image_id,

            "density_stratum":
                str(
                    item[
                        "density_stratum"
                    ]
                ),

            "width":
                int(
                    item["width"]
                ),

            "height":
                int(
                    item["height"]
                ),

            "a3c_percentile":
                float(
                    item[
                        "a3c_percentile"
                    ]
                ),

            "a3c_threshold":
                float(
                    item[
                        "a3c_threshold"
                    ]
                ),

            "foreground_pixels":
                int(
                    semantic_mask.sum()
                ),

            "C1_num_regions":
                int(
                    item[
                        "C1"
                    ][
                        "num_regions"
                    ]
                ),

            "C2b_num_regions":
                int(
                    item[
                        "C2b"
                    ][
                        "num_regions"
                    ]
                ),

            "C2c_num_regions":
                int(
                    item[
                        "C2c"
                    ][
                        "num_regions"
                    ]
                ),

            "cache_filename":
                cache_filename,
        }
    )


# ------------------------------------------------------------
# Save cache index
# ------------------------------------------------------------

e1_region_cache_index_df = (
    pd.DataFrame(
        cache_index_rows
    )
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    e1_region_cache_index_df
) == 100


e1_region_cache_index_df.to_csv(
    E1_REGION_CACHE_INDEX_PATH,
    index=False,
)


# ------------------------------------------------------------
# Round-trip validation
#
# Load every saved file and compare it EXACTLY with the
# validated in-memory Step 13B.1 representation.
# ------------------------------------------------------------

roundtrip_rows = []


for _, row in e1_region_cache_index_df.iterrows():

    dataset_index = int(
        row["dataset_index"]
    )

    cache_path = (
        E1_REGION_CACHE_DIR
        / row["cache_filename"]
    )

    assert cache_path.exists(), (
        f"Missing cache file: {cache_path}"
    )


    original = E1_REGION_MAPS[
        dataset_index
    ]


    with np.load(
        cache_path,
        allow_pickle=False,
    ) as cached:

        saved_dataset_index = int(
            cached[
                "dataset_index"
            ]
        )

        saved_image_id = str(
            cached[
                "image_id"
            ].item()
        )

        saved_semantic_mask = (
            cached[
                "semantic_mask"
            ]
        )

        saved_c1 = (
            cached[
                "C1_labels"
            ]
        )

        saved_c2b = (
            cached[
                "C2b_labels"
            ]
        )

        saved_c2c = (
            cached[
                "C2c_labels"
            ]
        )


        metadata_match = (
            saved_dataset_index
            ==
            dataset_index
            and
            saved_image_id
            ==
            str(
                original[
                    "image_id"
                ]
            )
        )


        semantic_match = (
            np.array_equal(
                saved_semantic_mask,
                original[
                    "semantic_mask"
                ],
            )
        )


        c1_match = (
            np.array_equal(
                saved_c1,
                original[
                    "C1"
                ][
                    "labelled_regions"
                ],
            )
        )


        c2b_match = (
            np.array_equal(
                saved_c2b,
                original[
                    "C2b"
                ][
                    "labelled_regions"
                ],
            )
        )


        c2c_match = (
            np.array_equal(
                saved_c2c,
                original[
                    "C2c"
                ][
                    "labelled_regions"
                ],
            )
        )


        roundtrip_rows.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    saved_image_id,

                "metadata_match":
                    metadata_match,

                "semantic_mask_match":
                    semantic_match,

                "C1_match":
                    c1_match,

                "C2b_match":
                    c2b_match,

                "C2c_match":
                    c2c_match,
            }
        )


# ------------------------------------------------------------
# Round-trip summary
# ------------------------------------------------------------

e1_region_cache_validation_df = (
    pd.DataFrame(
        roundtrip_rows
    )
)


validation_columns = [
    "metadata_match",
    "semantic_mask_match",
    "C1_match",
    "C2b_match",
    "C2c_match",
]


assert len(
    e1_region_cache_validation_df
) == 100


for column in validation_columns:

    assert (
        e1_region_cache_validation_df[
            column
        ]
        .all()
    ), (
        f"Cache round-trip validation failed: "
        f"{column}"
    )


# ------------------------------------------------------------
# Cache-level region-count validation
# ------------------------------------------------------------

assert (
    e1_region_cache_index_df[
        "C1_num_regions"
    ]
    .sum()
    ==
    2554
)

assert (
    e1_region_cache_index_df[
        "C2b_num_regions"
    ]
    .sum()
    ==
    6074
)

assert (
    e1_region_cache_index_df[
        "C2c_num_regions"
    ]
    .sum()
    ==
    4171
)


# ------------------------------------------------------------
# Display compact cache summary
# ------------------------------------------------------------

cache_summary_df = pd.DataFrame(
    {
        "representation": [
            "A3c semantic mask",
            "C1 labelled regions",
            "C2b labelled regions",
            "C2c labelled regions",
        ],

        "n_images": [
            len(
                e1_region_cache_index_df
            ),
            len(
                e1_region_cache_index_df
            ),
            len(
                e1_region_cache_index_df
            ),
            len(
                e1_region_cache_index_df
            ),
        ],

        "total_regions": [
            np.nan,
            int(
                e1_region_cache_index_df[
                    "C1_num_regions"
                ]
                .sum()
            ),
            int(
                e1_region_cache_index_df[
                    "C2b_num_regions"
                ]
                .sum()
            ),
            int(
                e1_region_cache_index_df[
                    "C2c_num_regions"
                ]
                .sum()
            ),
        ],

        "roundtrip_match": [
            int(
                e1_region_cache_validation_df[
                    "semantic_mask_match"
                ]
                .sum()
            ),
            int(
                e1_region_cache_validation_df[
                    "C1_match"
                ]
                .sum()
            ),
            int(
                e1_region_cache_validation_df[
                    "C2b_match"
                ]
                .sum()
            ),
            int(
                e1_region_cache_validation_df[
                    "C2c_match"
                ]
                .sum()
            ),
        ],
    }
)


print("\nSaved files:", len(e1_region_cache_index_df))

print(
    "Cache index:",
    E1_REGION_CACHE_INDEX_PATH,
)

print(
    "\nRound-trip validation:"
)

display(
    cache_summary_df
)


print("\n" + "=" * 100)

print("Step 13B.1a complete.")

print(
    "Validated A3c/C1/C2b/C2c region maps "
    "saved successfully."
)

print(
    "All 100 cache files passed exact "
    "round-trip array validation."
)

print(
    "Frozen region totals preserved: "
    "C1=2554, C2b=6074, C2c=4171."
)

print(
    "The saved cache can now be reused "
    "after a Colab runtime restart."
)

print("=" * 100)

E1 region-map cache: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/B1_E1_region_maps/FSC147_val_dev_100
E1 — Saving validated Group-C region-map cache

Cache directory:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/B1_E1_region_maps/FSC147_val_dev_100

Saved files: 100
Cache index: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/B1_E1_region_maps/FSC147_val_dev_100/cache_index.csv

Round-trip validation:


,representation,n_images,total_regions,roundtrip_match
0,A3c semantic mask,100,NaN,100
1,C1 labelled regions,100,2554.0,100
2,C2b labelled regions,100,6074.0,100
3,C2c labelled regions,100,4171.0,100



Step 13B.1a complete.
Validated A3c/C1/C2b/C2c region maps saved successfully.
All 100 cache files passed exact round-trip array validation.
Frozen region totals preserved: C1=2554, C2b=6074, C2c=4171.
The saved cache can now be reused after a Colab runtime restart.


In [49]:
# ============================================================
# Step 13B.2 — C1 ↔ D1 point–region relationship
# ============================================================
#
# Purpose
# -------
# Analyse the structural relationship between:
#
#   C1 = A3c foreground partitioned by 4-connected components
#   D1 = SAN target-score local-max activation points
#
# Questions:
#
#   1. How many D1 points lie inside / outside C1 foreground?
#   2. How many C1 regions contain 0, 1, or >=2 D1 points?
#   3. How many D1 points occur inside occupied C1 regions?
#   4. How does this relationship vary across S1-S6?
#
# IMPORTANT:
# This is a representation-relationship analysis only.
# GT correctness is NOT analysed here.
# ============================================================

import ast
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "group_e_handoff_df" in globals(), (
    "group_e_handoff_df not found. "
    "Run E0 / Step 13A first."
)

assert "E1_REGION_MAPS" in globals(), (
    "E1_REGION_MAPS not found. "
    "Run Step 13B.1 first."
)

assert len(E1_REGION_MAPS) == 100


# ------------------------------------------------------------
# Helper — robustly parse stored XY point lists
# ------------------------------------------------------------

def e1_parse_points_xy(value):

    if value is None:
        return np.empty((0, 2), dtype=np.float32)

    if isinstance(value, str):

        value = value.strip()

        if (
            value == ""
            or value.lower() == "nan"
        ):
            return np.empty(
                (0, 2),
                dtype=np.float32,
            )

        value = ast.literal_eval(value)

    arr = np.asarray(
        value,
        dtype=np.float32,
    )

    if arr.size == 0:
        return np.empty(
            (0, 2),
            dtype=np.float32,
        )

    arr = arr.reshape(-1, 2)

    return arr


# ------------------------------------------------------------
# Extract frozen D1 rows
# ------------------------------------------------------------

d1_e1_df = (
    group_e_handoff_df[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str) == "D1"
    ]
    .copy()
    .sort_values(
        "dataset_index"
    )
    .reset_index(drop=True)
)


assert len(d1_e1_df) == 100, (
    f"Expected 100 D1 rows, got {len(d1_e1_df)}."
)

assert (
    d1_e1_df["dataset_index"].nunique()
    == 100
)


# ------------------------------------------------------------
# Per-image relationship analysis
# ------------------------------------------------------------

image_rows = []

region_rows = []


for _, d1_row in d1_e1_df.iterrows():

    dataset_index = int(
        d1_row["dataset_index"]
    )

    image_id = str(
        d1_row["image_id"]
    )

    density_stratum = str(
        d1_row["density_stratum"]
    )


    # --------------------------------------------------------
    # Load validated C1 labelled region map
    # --------------------------------------------------------

    region_item = E1_REGION_MAPS[
        dataset_index
    ]

    assert (
        str(region_item["image_id"])
        == image_id
    )

    c1_labels = np.asarray(
        region_item[
            "C1"
        ][
            "labelled_regions"
        ]
    )

    height, width = c1_labels.shape

    n_c1_regions = int(
        region_item[
            "C1"
        ][
            "num_regions"
        ]
    )


    # --------------------------------------------------------
    # Load frozen D1 points
    #
    # Coordinates are stored as (x, y).
    # --------------------------------------------------------

    d1_points = e1_parse_points_xy(
        d1_row["pred_points_xy"]
    )

    n_d1_points = len(
        d1_points
    )


    # --------------------------------------------------------
    # Assign each D1 point to a C1 region.
    #
    # label = 0  -> outside C1/A3c foreground
    # label > 0  -> inside a C1 region
    # --------------------------------------------------------

    point_region_labels = []

    n_out_of_bounds = 0


    for x, y in d1_points:

        # Frozen coordinates should already be valid image
        # coordinates. Use nearest integer pixel only for
        # indexing the discrete C1 label map.

        xi = int(
            np.rint(x)
        )

        yi = int(
            np.rint(y)
        )


        if not (
            0 <= xi < width
            and
            0 <= yi < height
        ):

            n_out_of_bounds += 1

            point_region_labels.append(
                -1
            )

            continue


        point_region_labels.append(
            int(
                c1_labels[
                    yi,
                    xi,
                ]
            )
        )


    point_region_labels = np.asarray(
        point_region_labels,
        dtype=np.int32,
    )


    # --------------------------------------------------------
    # D1 point occupancy
    # --------------------------------------------------------

    n_d1_inside = int(
        np.sum(
            point_region_labels > 0
        )
    )

    n_d1_outside = int(
        np.sum(
            point_region_labels == 0
        )
    )


    assert (
        n_d1_inside
        +
        n_d1_outside
        +
        n_out_of_bounds
        ==
        n_d1_points
    )


    # --------------------------------------------------------
    # Count D1 points per C1 region
    # --------------------------------------------------------

    points_per_region = np.zeros(
        n_c1_regions + 1,
        dtype=np.int32,
    )


    valid_region_labels = (
        point_region_labels[
            point_region_labels > 0
        ]
    )


    if len(valid_region_labels) > 0:

        unique_labels, counts = np.unique(
            valid_region_labels,
            return_counts=True,
        )

        points_per_region[
            unique_labels
        ] = counts


    region_counts = (
        points_per_region[
            1:
        ]
    )


    n_regions_zero = int(
        np.sum(
            region_counts == 0
        )
    )

    n_regions_one = int(
        np.sum(
            region_counts == 1
        )
    )

    n_regions_multi = int(
        np.sum(
            region_counts >= 2
        )
    )

    n_regions_occupied = (
        n_regions_one
        +
        n_regions_multi
    )


    assert (
        n_regions_zero
        +
        n_regions_one
        +
        n_regions_multi
        ==
        n_c1_regions
    )


    # --------------------------------------------------------
    # Per-region rows
    # --------------------------------------------------------

    for region_label in range(
        1,
        n_c1_regions + 1,
    ):

        n_points = int(
            points_per_region[
                region_label
            ]
        )


        if n_points == 0:
            occupancy_class = "0"
        elif n_points == 1:
            occupancy_class = "1"
        else:
            occupancy_class = "2+"


        region_rows.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "density_stratum":
                    density_stratum,

                "c1_region_label":
                    region_label,

                "n_d1_points":
                    n_points,

                "occupancy_class":
                    occupancy_class,
            }
        )


    # --------------------------------------------------------
    # Per-image row
    # --------------------------------------------------------

    image_rows.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "density_stratum":
                density_stratum,

            "n_c1_regions":
                n_c1_regions,

            "n_d1_points":
                n_d1_points,

            "n_d1_inside_c1":
                n_d1_inside,

            "n_d1_outside_c1":
                n_d1_outside,

            "n_d1_out_of_bounds":
                n_out_of_bounds,

            "d1_inside_fraction":
                (
                    n_d1_inside
                    / n_d1_points
                    if n_d1_points > 0
                    else np.nan
                ),

            "n_c1_regions_zero_d1":
                n_regions_zero,

            "n_c1_regions_one_d1":
                n_regions_one,

            "n_c1_regions_multi_d1":
                n_regions_multi,

            "n_c1_regions_occupied":
                n_regions_occupied,

            "c1_region_occupied_fraction":
                (
                    n_regions_occupied
                    / n_c1_regions
                    if n_c1_regions > 0
                    else np.nan
                ),

            "c1_region_multi_fraction":
                (
                    n_regions_multi
                    / n_c1_regions
                    if n_c1_regions > 0
                    else np.nan
                ),
        }
    )


# ------------------------------------------------------------
# Dataframes
# ------------------------------------------------------------

e1_c1_d1_image_df = pd.DataFrame(
    image_rows
)

e1_c1_d1_region_df = pd.DataFrame(
    region_rows
)


# ------------------------------------------------------------
# Structural sanity checks
# ------------------------------------------------------------

assert len(
    e1_c1_d1_image_df
) == 100


assert (
    e1_c1_d1_image_df[
        "n_c1_regions"
    ].sum()
    ==
    2554
)


assert (
    e1_c1_d1_image_df[
        "n_d1_points"
    ].sum()
    ==
    7652
)


assert len(
    e1_c1_d1_region_df
) == 2554


assert (
    e1_c1_d1_image_df[
        "n_d1_out_of_bounds"
    ].sum()
    ==
    0
), (
    "At least one frozen D1 point lies outside "
    "the image boundary."
)


# ------------------------------------------------------------
# Overall relationship summary
# ------------------------------------------------------------

total_c1_regions = int(
    e1_c1_d1_image_df[
        "n_c1_regions"
    ].sum()
)

total_d1_points = int(
    e1_c1_d1_image_df[
        "n_d1_points"
    ].sum()
)

total_inside = int(
    e1_c1_d1_image_df[
        "n_d1_inside_c1"
    ].sum()
)

total_outside = int(
    e1_c1_d1_image_df[
        "n_d1_outside_c1"
    ].sum()
)

total_zero = int(
    e1_c1_d1_image_df[
        "n_c1_regions_zero_d1"
    ].sum()
)

total_one = int(
    e1_c1_d1_image_df[
        "n_c1_regions_one_d1"
    ].sum()
)

total_multi = int(
    e1_c1_d1_image_df[
        "n_c1_regions_multi_d1"
    ].sum()
)

total_occupied = (
    total_one
    +
    total_multi
)


e1_c1_d1_overall_df = pd.DataFrame(
    [
        {
            "n_images":
                100,

            "total_c1_regions":
                total_c1_regions,

            "total_d1_points":
                total_d1_points,

            "d1_inside_c1":
                total_inside,

            "d1_outside_c1":
                total_outside,

            "d1_inside_fraction":
                (
                    total_inside
                    / total_d1_points
                ),

            "c1_regions_zero_d1":
                total_zero,

            "c1_regions_one_d1":
                total_one,

            "c1_regions_multi_d1":
                total_multi,

            "c1_regions_occupied":
                total_occupied,

            "c1_occupied_fraction":
                (
                    total_occupied
                    / total_c1_regions
                ),

            "c1_multi_fraction":
                (
                    total_multi
                    / total_c1_regions
                ),
        }
    ]
)


# ------------------------------------------------------------
# Density-stratified relationship summary
# ------------------------------------------------------------

density_rows = []


for stratum in DENSITY_ORDER:

    sdf = e1_c1_d1_image_df[
        e1_c1_d1_image_df[
            "density_stratum"
        ]
        ==
        stratum
    ]


    n_images = len(sdf)

    c1_regions = int(
        sdf[
            "n_c1_regions"
        ].sum()
    )

    d1_points = int(
        sdf[
            "n_d1_points"
        ].sum()
    )

    inside = int(
        sdf[
            "n_d1_inside_c1"
        ].sum()
    )

    outside = int(
        sdf[
            "n_d1_outside_c1"
        ].sum()
    )

    zero = int(
        sdf[
            "n_c1_regions_zero_d1"
        ].sum()
    )

    one = int(
        sdf[
            "n_c1_regions_one_d1"
        ].sum()
    )

    multi = int(
        sdf[
            "n_c1_regions_multi_d1"
        ].sum()
    )

    occupied = (
        one
        +
        multi
    )


    density_rows.append(
        {
            "density_stratum":
                stratum,

            "n_images":
                n_images,

            "c1_regions":
                c1_regions,

            "d1_points":
                d1_points,

            "d1_inside_fraction":
                (
                    inside / d1_points
                    if d1_points > 0
                    else np.nan
                ),

            "c1_occupied_fraction":
                (
                    occupied / c1_regions
                    if c1_regions > 0
                    else np.nan
                ),

            "c1_zero_fraction":
                (
                    zero / c1_regions
                    if c1_regions > 0
                    else np.nan
                ),

            "c1_one_fraction":
                (
                    one / c1_regions
                    if c1_regions > 0
                    else np.nan
                ),

            "c1_multi_fraction":
                (
                    multi / c1_regions
                    if c1_regions > 0
                    else np.nan
                ),
        }
    )


e1_c1_d1_density_df = pd.DataFrame(
    density_rows
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 100)
print(
    "E1 — C1 ↔ D1 Point–Region Relationship"
)
print("=" * 100)


print(
    "\nOverall relationship:"
)

display(
    e1_c1_d1_overall_df.round(4)
)


print(
    "\nRelationship by density stratum:"
)

display(
    e1_c1_d1_density_df.round(4)
)


print(
    "\nC1 region occupancy distribution:"
)

occupancy_summary_df = (
    e1_c1_d1_region_df[
        "occupancy_class"
    ]
    .value_counts()
    .reindex(
        ["0", "1", "2+"],
        fill_value=0,
    )
    .rename_axis(
        "D1_points_in_C1_region"
    )
    .reset_index(
        name="n_regions"
    )
)

occupancy_summary_df[
    "fraction_of_C1_regions"
] = (
    occupancy_summary_df[
        "n_regions"
    ]
    /
    len(
        e1_c1_d1_region_df
    )
)

display(
    occupancy_summary_df.round(4)
)


print("\n" + "=" * 100)

print("Step 13B.2 complete.")

print(
    "C1 ↔ D1 structural relationship measured "
    "for all 100 dev100 images."
)

print(
    "No GT correctness or semantic fusion was "
    "introduced in this step."
)

print("=" * 100)

E1 — C1 ↔ D1 Point–Region Relationship

Overall relationship:


,n_images,total_c1_regions,total_d1_points,d1_inside_c1,d1_outside_c1,d1_inside_fraction,c1_regions_zero_d1,c1_regions_one_d1,c1_regions_multi_d1,c1_regions_occupied,c1_occupied_fraction,c1_multi_fraction
0,100,2554,7652,4194,3458,0.5481,399,1528,627,2155,0.8438,0.2455



Relationship by density stratum:


,density_stratum,n_images,c1_regions,d1_points,d1_inside_fraction,c1_occupied_fraction,c1_zero_fraction,c1_one_fraction,c1_multi_fraction
0,S1_0_20,20,240,1082,0.5767,0.8708,0.1292,0.4375,0.4333
1,S2_20_40,20,392,1233,0.5661,0.8520,0.1480,0.5485,0.3036
2,S3_40_60,20,401,1390,0.5432,0.7930,0.2070,0.4988,0.2943
3,S4_60_80,20,566,1663,0.5707,0.9346,0.0654,0.7032,0.2314
4,S5_80_95,10,465,1143,0.5354,0.8624,0.1376,0.6559,0.2065
5,S6_95_100,10,490,1141,0.4873,0.7429,0.2571,0.6224,0.1204



C1 region occupancy distribution:


,D1_points_in_C1_region,n_regions,fraction_of_C1_regions
0,0,399,0.1562
1,1,1528,0.5983
2,2+,627,0.2455



Step 13B.2 complete.
C1 ↔ D1 structural relationship measured for all 100 dev100 images.
No GT correctness or semantic fusion was introduced in this step.


In [50]:
# ============================================================
# Step 13B.3 — C1 ↔ D1 region multiplicity analysis
# ============================================================
#
# Purpose
# -------
# Extend the structural C1 ↔ D1 analysis from Step 13B.2 by
# examining how D1 activation points are distributed across
# C1 semantic regions.
#
# Questions:
#
#   1. What is the full distribution of D1 points per C1 region?
#   2. Among multi-point regions, how many contain 2, 3, 4,
#      5, or larger numbers of D1 points?
#   3. How much of the inside-C1 D1 evidence is concentrated
#      in single-point versus multi-point regions?
#   4. How extreme are the highest-multiplicity C1 regions?
#
# IMPORTANT:
# This remains a representation-relationship analysis.
# No GT correctness and no fusion rule are introduced.
#
# Requires:
#   e1_c1_d1_image_df
#   e1_c1_d1_region_df
# from Step 13B.2.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "e1_c1_d1_image_df" in globals(), (
    "e1_c1_d1_image_df not found. "
    "Run Step 13B.2 first."
)

assert "e1_c1_d1_region_df" in globals(), (
    "e1_c1_d1_region_df not found. "
    "Run Step 13B.2 first."
)

assert len(e1_c1_d1_image_df) == 100

assert len(e1_c1_d1_region_df) == 2554


# ------------------------------------------------------------
# 1. Exact D1-points-per-C1-region distribution
# ------------------------------------------------------------

exact_multiplicity_df = (
    e1_c1_d1_region_df[
        "n_d1_points"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "n_d1_points"
    )
    .reset_index(
        name="n_c1_regions"
    )
)

exact_multiplicity_df[
    "fraction_of_c1_regions"
] = (
    exact_multiplicity_df[
        "n_c1_regions"
    ]
    /
    len(e1_c1_d1_region_df)
)

exact_multiplicity_df[
    "d1_points_contributed"
] = (
    exact_multiplicity_df[
        "n_d1_points"
    ]
    *
    exact_multiplicity_df[
        "n_c1_regions"
    ]
)


# ------------------------------------------------------------
# 2. More interpretable multiplicity bins
#
# Keep 0/1/2/3/4 exact, then group the long tail.
# ------------------------------------------------------------

def e1_multiplicity_bin(n):

    n = int(n)

    if n == 0:
        return "0"

    if n == 1:
        return "1"

    if n == 2:
        return "2"

    if n == 3:
        return "3"

    if n == 4:
        return "4"

    if 5 <= n <= 9:
        return "5-9"

    return "10+"


MULTIPLICITY_BIN_ORDER = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5-9",
    "10+",
]


e1_c1_d1_region_df[
    "multiplicity_bin"
] = (
    e1_c1_d1_region_df[
        "n_d1_points"
    ]
    .apply(
        e1_multiplicity_bin
    )
)


multiplicity_bin_df = (
    e1_c1_d1_region_df
    .groupby(
        "multiplicity_bin",
        observed=True,
    )
    .agg(
        n_c1_regions=(
            "c1_region_label",
            "size",
        ),

        total_d1_points=(
            "n_d1_points",
            "sum",
        ),

        mean_d1_points=(
            "n_d1_points",
            "mean",
        ),

        max_d1_points=(
            "n_d1_points",
            "max",
        ),
    )
    .reindex(
        MULTIPLICITY_BIN_ORDER,
        fill_value=0,
    )
    .reset_index()
)


multiplicity_bin_df[
    "fraction_of_c1_regions"
] = (
    multiplicity_bin_df[
        "n_c1_regions"
    ]
    /
    len(e1_c1_d1_region_df)
)


# ------------------------------------------------------------
# 3. D1 evidence concentration
#
# Important:
# D1 points outside C1 foreground are NOT represented in
# e1_c1_d1_region_df. Therefore this analysis concerns only
# the D1 points that fall inside C1 regions.
# ------------------------------------------------------------

total_inside_d1 = int(
    e1_c1_d1_image_df[
        "n_d1_inside_c1"
    ].sum()
)

total_outside_d1 = int(
    e1_c1_d1_image_df[
        "n_d1_outside_c1"
    ].sum()
)

total_d1 = int(
    e1_c1_d1_image_df[
        "n_d1_points"
    ].sum()
)


inside_from_single_regions = int(
    e1_c1_d1_region_df.loc[
        e1_c1_d1_region_df[
            "n_d1_points"
        ] == 1,
        "n_d1_points",
    ].sum()
)


inside_from_multi_regions = int(
    e1_c1_d1_region_df.loc[
        e1_c1_d1_region_df[
            "n_d1_points"
        ] >= 2,
        "n_d1_points",
    ].sum()
)


assert (
    inside_from_single_regions
    +
    inside_from_multi_regions
    ==
    total_inside_d1
)


assert (
    total_inside_d1
    +
    total_outside_d1
    ==
    total_d1
)


e1_c1_d1_evidence_concentration_df = pd.DataFrame(
    [
        {
            "relationship":
                "Outside C1 foreground",

            "n_d1_points":
                total_outside_d1,

            "fraction_of_all_d1":
                (
                    total_outside_d1
                    / total_d1
                ),

            "fraction_of_inside_c1_d1":
                np.nan,
        },

        {
            "relationship":
                "Inside C1: 1 D1 point in region",

            "n_d1_points":
                inside_from_single_regions,

            "fraction_of_all_d1":
                (
                    inside_from_single_regions
                    / total_d1
                ),

            "fraction_of_inside_c1_d1":
                (
                    inside_from_single_regions
                    / total_inside_d1
                ),
        },

        {
            "relationship":
                "Inside C1: >=2 D1 points in region",

            "n_d1_points":
                inside_from_multi_regions,

            "fraction_of_all_d1":
                (
                    inside_from_multi_regions
                    / total_d1
                ),

            "fraction_of_inside_c1_d1":
                (
                    inside_from_multi_regions
                    / total_inside_d1
                ),
        },
    ]
)


# ------------------------------------------------------------
# 4. Multi-point-region-specific summary
# ------------------------------------------------------------

multi_region_df = (
    e1_c1_d1_region_df[
        e1_c1_d1_region_df[
            "n_d1_points"
        ] >= 2
    ]
    .copy()
)


assert len(
    multi_region_df
) == 627


e1_c1_d1_multi_summary_df = pd.DataFrame(
    [
        {
            "n_multi_regions":
                len(
                    multi_region_df
                ),

            "fraction_of_all_c1_regions":
                (
                    len(multi_region_df)
                    /
                    len(e1_c1_d1_region_df)
                ),

            "total_d1_points_in_multi_regions":
                int(
                    multi_region_df[
                        "n_d1_points"
                    ].sum()
                ),

            "mean_d1_points_per_multi_region":
                float(
                    multi_region_df[
                        "n_d1_points"
                    ].mean()
                ),

            "median_d1_points_per_multi_region":
                float(
                    multi_region_df[
                        "n_d1_points"
                    ].median()
                ),

            "max_d1_points_in_one_c1_region":
                int(
                    multi_region_df[
                        "n_d1_points"
                    ].max()
                ),
        }
    ]
)


# ------------------------------------------------------------
# 5. Highest-multiplicity C1 regions
#
# Useful later for qualitative inspection / failure analysis.
# ------------------------------------------------------------

e1_c1_d1_top_multi_regions_df = (
    multi_region_df[
        [
            "dataset_index",
            "image_id",
            "density_stratum",
            "c1_region_label",
            "n_d1_points",
        ]
    ]
    .sort_values(
        [
            "n_d1_points",
            "dataset_index",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .head(20)
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 6. Sanity checks
# ------------------------------------------------------------

assert (
    exact_multiplicity_df[
        "n_c1_regions"
    ].sum()
    ==
    2554
)


assert (
    exact_multiplicity_df[
        "d1_points_contributed"
    ].sum()
    ==
    total_inside_d1
)


assert (
    multiplicity_bin_df[
        "n_c1_regions"
    ].sum()
    ==
    2554
)


assert (
    multiplicity_bin_df[
        "total_d1_points"
    ].sum()
    ==
    total_inside_d1
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 100)

print(
    "E1 — C1 ↔ D1 Region Multiplicity Analysis"
)

print("=" * 100)


print(
    "\nExact D1-points-per-C1-region distribution:"
)

display(
    exact_multiplicity_df.round(4)
)


print(
    "\nGrouped multiplicity distribution:"
)

display(
    multiplicity_bin_df.round(4)
)


print(
    "\nD1 evidence concentration:"
)

display(
    e1_c1_d1_evidence_concentration_df.round(4)
)


print(
    "\nMulti-point C1 region summary:"
)

display(
    e1_c1_d1_multi_summary_df.round(4)
)


print(
    "\nTop 20 highest-multiplicity C1 regions:"
)

display(
    e1_c1_d1_top_multi_regions_df
)


print("\n" + "=" * 100)

print("Step 13B.3 complete.")

print(
    "C1 ↔ D1 multiplicity structure characterised "
    "without using GT correctness."
)

print(
    "The results quantify how much activation evidence "
    "is concentrated within multi-point C1 regions."
)

print("=" * 100)

E1 — C1 ↔ D1 Region Multiplicity Analysis

Exact D1-points-per-C1-region distribution:


,n_d1_points,n_c1_regions,fraction_of_c1_regions,d1_points_contributed
0,0,399,0.1562,0
1,1,1528,0.5983,1528
2,2,294,0.1151,588
3,3,128,0.0501,384
4,4,47,0.0184,188
5,5,39,0.0153,195
6,6,29,0.0114,174
7,7,15,0.0059,105
8,8,13,0.0051,104
9,9,11,0.0043,99



Grouped multiplicity distribution:


,multiplicity_bin,n_c1_regions,total_d1_points,mean_d1_points,max_d1_points,fraction_of_c1_regions
0,0,399,0,0.0000,0,0.1562
1,1,1528,1528,1.0000,1,0.5983
2,2,294,588,2.0000,2,0.1151
3,3,128,384,3.0000,3,0.0501
4,4,47,188,4.0000,4,0.0184
5,5-9,107,677,6.3271,9,0.0419
6,10+,51,829,16.2549,31,0.0200



D1 evidence concentration:


...


Multi-point C1 region summary:


,n_multi_regions,fraction_of_all_c1_regions,total_d1_points_in_multi_regions,mean_d1_points_per_multi_region,median_d1_points_per_multi_region,max_d1_points_in_one_c1_region
0,627,0.2455,2666,4.252,3.0,31



Top 20 highest-multiplicity C1 regions:


,dataset_index,image_id,density_stratum,c1_region_label,n_d1_points
0,1061,6286.jpg,S4_60_80,1,31
1,31,241.jpg,S3_40_60,1,30
2,609,4535.jpg,S3_40_60,2,30
3,372,2412.jpg,S2_20_40,2,28
4,1260,7100.jpg,S3_40_60,2,27
5,919,5739.jpg,S4_60_80,4,26
6,180,835.jpg,S3_40_60,1,24
7,1269,7060.jpg,S4_60_80,4,24
8,1192,6969.jpg,S6_95_100,13,23
9,40,252.jpg,S3_40_60,1,22



Step 13B.3 complete.
C1 ↔ D1 multiplicity structure characterised without using GT correctness.
The results quantify how much activation evidence is concentrated within multi-point C1 regions.


In [51]:
# ============================================================
# Step 13B.4 — All retained C × D region–point relationships
# ============================================================
#
# Purpose
# -------
# Generalise the validated C1 ↔ D1 relationship analysis to:
#
#   C = {C1, C2b, C2c}
#   D = {D1, D4, D5-768, D6B-T-native, D6B9-strict}
#
# giving 3 x 5 = 15 C×D relationships.
#
# For each pair measure:
#
#   Point-level:
#     - D points inside / outside common semantic foreground
#
#   Region-level:
#     - C regions containing 0 D points
#     - C regions containing exactly 1 D point
#     - C regions containing >=2 D points
#     - occupied-region fraction
#     - multi-point-region fraction
#
#   Multiplicity:
#     - total D points inside multi-point regions
#     - mean / median / maximum D points per multi-point region
#
# Also produce density-stratified S1-S6 summaries.
#
# IMPORTANT:
#   - No GT correctness is used.
#   - No fusion rule is introduced.
#   - Frozen C/D predictions are not modified.
# ============================================================

import ast
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "group_e_handoff_df" in globals(), (
    "group_e_handoff_df not found. Run Step 13A first."
)

assert "E1_REGION_MAPS" in globals(), (
    "E1_REGION_MAPS not found. Run Step 13B.1 first."
)

assert len(E1_REGION_MAPS) == 100


# ------------------------------------------------------------
# Frozen retained representations
# ------------------------------------------------------------

E1_C_VARIANTS = [
    "C1",
    "C2b",
    "C2c",
]

E1_D_VARIANTS = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]

assert len(E1_C_VARIANTS) * len(E1_D_VARIANTS) == 15


# ------------------------------------------------------------
# Point parser
#
# Reuse Step 13B.2 helper if available; otherwise define it.
# ------------------------------------------------------------

if "e1_parse_points_xy" not in globals():

    def e1_parse_points_xy(value):

        if value is None:
            return np.empty(
                (0, 2),
                dtype=np.float32,
            )

        if isinstance(value, str):

            value = value.strip()

            if (
                value == ""
                or value.lower() == "nan"
            ):
                return np.empty(
                    (0, 2),
                    dtype=np.float32,
                )

            value = ast.literal_eval(value)

        arr = np.asarray(
            value,
            dtype=np.float32,
        )

        if arr.size == 0:
            return np.empty(
                (0, 2),
                dtype=np.float32,
            )

        return arr.reshape(-1, 2)


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

pair_image_rows = []
pair_region_rows = []


# ------------------------------------------------------------
# Analyse all 15 C×D pairs
# ------------------------------------------------------------

for c_variant in E1_C_VARIANTS:

    for d_variant in E1_D_VARIANTS:

        print(
            f"Analysing {c_variant} ↔ {d_variant} ..."
        )

        d_df = (
            group_e_handoff_df[
                group_e_handoff_df[
                    "candidate_id"
                ].astype(str)
                ==
                d_variant
            ]
            .copy()
            .sort_values(
                "dataset_index"
            )
            .reset_index(drop=True)
        )

        assert len(d_df) == 100, (
            f"{d_variant}: expected 100 rows, "
            f"got {len(d_df)}."
        )

        assert (
            d_df["dataset_index"].nunique()
            == 100
        )


        # ----------------------------------------------------
        # Image loop
        # ----------------------------------------------------

        for _, d_row in d_df.iterrows():

            dataset_index = int(
                d_row["dataset_index"]
            )

            image_id = str(
                d_row["image_id"]
            )

            density_stratum = str(
                d_row["density_stratum"]
            )

            region_item = E1_REGION_MAPS[
                dataset_index
            ]

            assert (
                str(region_item["image_id"])
                == image_id
            )


            # ------------------------------------------------
            # Load labelled C region map
            # ------------------------------------------------

            c_labels = np.asarray(
                region_item[
                    c_variant
                ][
                    "labelled_regions"
                ]
            )

            height, width = (
                c_labels.shape
            )

            n_c_regions = int(
                region_item[
                    c_variant
                ][
                    "num_regions"
                ]
            )


            # ------------------------------------------------
            # Frozen D points
            # ------------------------------------------------

            d_points = e1_parse_points_xy(
                d_row["pred_points_xy"]
            )

            n_d_points = len(
                d_points
            )


            # ------------------------------------------------
            # Assign D points to C regions
            #
            # label 0 = outside common A3c foreground
            # label >0 = inside a C region
            # ------------------------------------------------

            point_region_labels = []

            n_out_of_bounds = 0


            for x, y in d_points:

                xi = int(
                    np.rint(x)
                )

                yi = int(
                    np.rint(y)
                )

                if not (
                    0 <= xi < width
                    and
                    0 <= yi < height
                ):

                    n_out_of_bounds += 1

                    point_region_labels.append(
                        -1
                    )

                    continue


                point_region_labels.append(
                    int(
                        c_labels[
                            yi,
                            xi,
                        ]
                    )
                )


            point_region_labels = np.asarray(
                point_region_labels,
                dtype=np.int32,
            )


            # ------------------------------------------------
            # Point-level relationship
            # ------------------------------------------------

            n_inside = int(
                np.sum(
                    point_region_labels > 0
                )
            )

            n_outside = int(
                np.sum(
                    point_region_labels == 0
                )
            )

            assert (
                n_inside
                +
                n_outside
                +
                n_out_of_bounds
                ==
                n_d_points
            )


            # ------------------------------------------------
            # D points per C region
            # ------------------------------------------------

            points_per_region = np.zeros(
                n_c_regions + 1,
                dtype=np.int32,
            )

            valid_labels = (
                point_region_labels[
                    point_region_labels > 0
                ]
            )


            if len(valid_labels) > 0:

                unique_labels, counts = np.unique(
                    valid_labels,
                    return_counts=True,
                )

                assert (
                    unique_labels.max()
                    <= n_c_regions
                )

                points_per_region[
                    unique_labels
                ] = counts


            region_counts = (
                points_per_region[
                    1:
                ]
            )


            # ------------------------------------------------
            # Region occupancy
            # ------------------------------------------------

            n_zero = int(
                np.sum(
                    region_counts == 0
                )
            )

            n_one = int(
                np.sum(
                    region_counts == 1
                )
            )

            n_multi = int(
                np.sum(
                    region_counts >= 2
                )
            )

            n_occupied = (
                n_one
                +
                n_multi
            )


            assert (
                n_zero
                +
                n_one
                +
                n_multi
                ==
                n_c_regions
            )


            # ------------------------------------------------
            # Multi-point-region statistics
            # ------------------------------------------------

            multi_counts = (
                region_counts[
                    region_counts >= 2
                ]
            )


            if len(multi_counts) > 0:

                multi_total_points = int(
                    multi_counts.sum()
                )

                multi_mean = float(
                    multi_counts.mean()
                )

                multi_median = float(
                    np.median(
                        multi_counts
                    )
                )

                multi_max = int(
                    multi_counts.max()
                )

            else:

                multi_total_points = 0
                multi_mean = np.nan
                multi_median = np.nan
                multi_max = 0


            # ------------------------------------------------
            # Per-region rows
            # ------------------------------------------------

            for region_label in range(
                1,
                n_c_regions + 1,
            ):

                n_points = int(
                    points_per_region[
                        region_label
                    ]
                )

                if n_points == 0:
                    occupancy_class = "0"

                elif n_points == 1:
                    occupancy_class = "1"

                else:
                    occupancy_class = "2+"


                pair_region_rows.append(
                    {
                        "c_variant":
                            c_variant,

                        "d_variant":
                            d_variant,

                        "dataset_index":
                            dataset_index,

                        "image_id":
                            image_id,

                        "density_stratum":
                            density_stratum,

                        "c_region_label":
                            region_label,

                        "n_d_points":
                            n_points,

                        "occupancy_class":
                            occupancy_class,
                    }
                )


            # ------------------------------------------------
            # Per-image row
            # ------------------------------------------------

            pair_image_rows.append(
                {
                    "c_variant":
                        c_variant,

                    "d_variant":
                        d_variant,

                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_id,

                    "density_stratum":
                        density_stratum,

                    "n_c_regions":
                        n_c_regions,

                    "n_d_points":
                        n_d_points,

                    "n_d_inside":
                        n_inside,

                    "n_d_outside":
                        n_outside,

                    "n_d_out_of_bounds":
                        n_out_of_bounds,

                    "n_c_regions_zero":
                        n_zero,

                    "n_c_regions_one":
                        n_one,

                    "n_c_regions_multi":
                        n_multi,

                    "n_c_regions_occupied":
                        n_occupied,

                    "multi_total_d_points":
                        multi_total_points,

                    "multi_mean_d_points":
                        multi_mean,

                    "multi_median_d_points":
                        multi_median,

                    "multi_max_d_points":
                        multi_max,
                }
            )


# ------------------------------------------------------------
# Build canonical E1 all-pair dataframes
# ------------------------------------------------------------

e1_all_pairs_image_df = pd.DataFrame(
    pair_image_rows
)

e1_all_pairs_region_df = pd.DataFrame(
    pair_region_rows
)


# ------------------------------------------------------------
# Structural sanity checks
# ------------------------------------------------------------

assert len(
    e1_all_pairs_image_df
) == 1500, (
    "Expected 15 pairs × 100 images = 1500 rows."
)


pair_counts = (
    e1_all_pairs_image_df
    .groupby(
        [
            "c_variant",
            "d_variant",
        ]
    )
    .size()
)

assert len(pair_counts) == 15

assert (
    pair_counts == 100
).all()


assert (
    e1_all_pairs_image_df[
        "n_d_out_of_bounds"
    ].sum()
    ==
    0
), (
    "At least one frozen D point lies outside "
    "an image boundary."
)


# ------------------------------------------------------------
# Expected total region rows
#
# Each D representation is mapped onto every C partition.
#
# C1  = 2554 regions × 5 D variants
# C2b = 6074 regions × 5 D variants
# C2c = 4171 regions × 5 D variants
# ------------------------------------------------------------

expected_region_rows = (
    (
        2554
        +
        6074
        +
        4171
    )
    *
    5
)

assert len(
    e1_all_pairs_region_df
) == expected_region_rows, (
    f"Expected {expected_region_rows} region rows, "
    f"got {len(e1_all_pairs_region_df)}."
)


# ------------------------------------------------------------
# Overall summary for each of the 15 pairs
# ------------------------------------------------------------

overall_rows = []


for c_variant in E1_C_VARIANTS:

    for d_variant in E1_D_VARIANTS:

        sdf = (
            e1_all_pairs_image_df[
                (
                    e1_all_pairs_image_df[
                        "c_variant"
                    ]
                    ==
                    c_variant
                )
                &
                (
                    e1_all_pairs_image_df[
                        "d_variant"
                    ]
                    ==
                    d_variant
                )
            ]
        )


        total_c_regions = int(
            sdf[
                "n_c_regions"
            ].sum()
        )

        total_d_points = int(
            sdf[
                "n_d_points"
            ].sum()
        )

        inside = int(
            sdf[
                "n_d_inside"
            ].sum()
        )

        outside = int(
            sdf[
                "n_d_outside"
            ].sum()
        )

        zero = int(
            sdf[
                "n_c_regions_zero"
            ].sum()
        )

        one = int(
            sdf[
                "n_c_regions_one"
            ].sum()
        )

        multi = int(
            sdf[
                "n_c_regions_multi"
            ].sum()
        )

        occupied = (
            one
            +
            multi
        )


        # --------------------------------------------
        # Region-level multiplicity data for this pair
        # --------------------------------------------

        rdf = (
            e1_all_pairs_region_df[
                (
                    e1_all_pairs_region_df[
                        "c_variant"
                    ]
                    ==
                    c_variant
                )
                &
                (
                    e1_all_pairs_region_df[
                        "d_variant"
                    ]
                    ==
                    d_variant
                )
            ]
        )


        multi_values = (
            rdf.loc[
                rdf[
                    "n_d_points"
                ] >= 2,
                "n_d_points",
            ]
            .to_numpy()
        )


        if len(multi_values) > 0:

            multi_d_points = int(
                multi_values.sum()
            )

            mean_multi = float(
                multi_values.mean()
            )

            median_multi = float(
                np.median(
                    multi_values
                )
            )

            max_multi = int(
                multi_values.max()
            )

        else:

            multi_d_points = 0
            mean_multi = np.nan
            median_multi = np.nan
            max_multi = 0


        overall_rows.append(
            {
                "c_variant":
                    c_variant,

                "d_variant":
                    d_variant,

                "n_c_regions":
                    total_c_regions,

                "n_d_points":
                    total_d_points,

                "d_inside_fraction":
                    (
                        inside
                        / total_d_points
                        if total_d_points > 0
                        else np.nan
                    ),

                "d_outside_fraction":
                    (
                        outside
                        / total_d_points
                        if total_d_points > 0
                        else np.nan
                    ),

                "c_zero_fraction":
                    (
                        zero
                        / total_c_regions
                    ),

                "c_one_fraction":
                    (
                        one
                        / total_c_regions
                    ),

                "c_multi_fraction":
                    (
                        multi
                        / total_c_regions
                    ),

                "c_occupied_fraction":
                    (
                        occupied
                        / total_c_regions
                    ),

                "n_multi_regions":
                    multi,

                "d_points_in_multi_regions":
                    multi_d_points,

                "mean_d_points_per_multi_region":
                    mean_multi,

                "median_d_points_per_multi_region":
                    median_multi,

                "max_d_points_per_region":
                    max_multi,
            }
        )


e1_all_pairs_overall_df = pd.DataFrame(
    overall_rows
)


# ------------------------------------------------------------
# Density-stratified summary: 15 pairs × 6 strata = 90 rows
# ------------------------------------------------------------

density_rows = []


for c_variant in E1_C_VARIANTS:

    for d_variant in E1_D_VARIANTS:

        pair_df = (
            e1_all_pairs_image_df[
                (
                    e1_all_pairs_image_df[
                        "c_variant"
                    ]
                    ==
                    c_variant
                )
                &
                (
                    e1_all_pairs_image_df[
                        "d_variant"
                    ]
                    ==
                    d_variant
                )
            ]
        )


        for stratum in DENSITY_ORDER:

            sdf = (
                pair_df[
                    pair_df[
                        "density_stratum"
                    ]
                    ==
                    stratum
                ]
            )


            total_c_regions = int(
                sdf[
                    "n_c_regions"
                ].sum()
            )

            total_d_points = int(
                sdf[
                    "n_d_points"
                ].sum()
            )

            inside = int(
                sdf[
                    "n_d_inside"
                ].sum()
            )

            zero = int(
                sdf[
                    "n_c_regions_zero"
                ].sum()
            )

            one = int(
                sdf[
                    "n_c_regions_one"
                ].sum()
            )

            multi = int(
                sdf[
                    "n_c_regions_multi"
                ].sum()
            )

            occupied = (
                one
                +
                multi
            )


            density_rows.append(
                {
                    "c_variant":
                        c_variant,

                    "d_variant":
                        d_variant,

                    "density_stratum":
                        stratum,

                    "n_images":
                        len(sdf),

                    "n_c_regions":
                        total_c_regions,

                    "n_d_points":
                        total_d_points,

                    "d_inside_fraction":
                        (
                            inside
                            / total_d_points
                            if total_d_points > 0
                            else np.nan
                        ),

                    "c_zero_fraction":
                        (
                            zero
                            / total_c_regions
                            if total_c_regions > 0
                            else np.nan
                        ),

                    "c_one_fraction":
                        (
                            one
                            / total_c_regions
                            if total_c_regions > 0
                            else np.nan
                        ),

                    "c_multi_fraction":
                        (
                            multi
                            / total_c_regions
                            if total_c_regions > 0
                            else np.nan
                        ),

                    "c_occupied_fraction":
                        (
                            occupied
                            / total_c_regions
                            if total_c_regions > 0
                            else np.nan
                        ),
                }
            )


e1_all_pairs_density_df = pd.DataFrame(
    density_rows
)


assert len(
    e1_all_pairs_density_df
) == 90


# ------------------------------------------------------------
# Important consistency check:
#
# C1/C2b/C2c have identical binary foreground support.
# Therefore, for a fixed D representation, the inside fraction
# MUST be identical across all three C representations.
# ------------------------------------------------------------

inside_consistency_rows = []


for d_variant in E1_D_VARIANTS:

    vals = (
        e1_all_pairs_overall_df[
            e1_all_pairs_overall_df[
                "d_variant"
            ]
            ==
            d_variant
        ]
        .set_index(
            "c_variant"
        )[
            "d_inside_fraction"
        ]
    )


    spread = float(
        vals.max()
        -
        vals.min()
    )


    inside_consistency_rows.append(
        {
            "d_variant":
                d_variant,

            "C1_inside_fraction":
                vals["C1"],

            "C2b_inside_fraction":
                vals["C2b"],

            "C2c_inside_fraction":
                vals["C2c"],

            "max_minus_min":
                spread,

            "consistent":
                bool(
                    np.isclose(
                        spread,
                        0.0,
                        atol=1e-12,
                    )
                ),
        }
    )


e1_foreground_consistency_df = pd.DataFrame(
    inside_consistency_rows
)


assert (
    e1_foreground_consistency_df[
        "consistent"
    ]
).all(), (
    "C1/C2b/C2c do not appear to share "
    "identical foreground support."
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 110)

print(
    "E1 — All Retained C × D Region–Point Relationships"
)

print("=" * 110)


print(
    "\nOverall 15-pair relationship summary:"
)

display(
    e1_all_pairs_overall_df.round(4)
)


print(
    "\nShared-foreground consistency check:"
)

display(
    e1_foreground_consistency_df.round(6)
)


print(
    "\nDensity-stratified summary:"
)

display(
    e1_all_pairs_density_df.round(4)
)


print("\nStructural checks:")

print(
    f"  Pair-image rows: "
    f"{len(e1_all_pairs_image_df)} "
    f"(expected 1500)"
)

print(
    f"  Pair-region rows: "
    f"{len(e1_all_pairs_region_df)} "
    f"(expected {expected_region_rows})"
)

print(
    f"  Overall pair summaries: "
    f"{len(e1_all_pairs_overall_df)} "
    f"(expected 15)"
)

print(
    f"  Density summaries: "
    f"{len(e1_all_pairs_density_df)} "
    f"(expected 90)"
)


print("\n" + "=" * 110)

print("Step 13B.4 complete.")

print(
    "All 15 retained C×D structural relationships "
    "analysed using frozen predictions."
)

print(
    "No GT correctness or fusion rule was introduced."
)

print("=" * 110)

Analysing C1 ↔ D1 ...
Analysing C1 ↔ D4 ...
Analysing C1 ↔ D5-768 ...
Analysing C1 ↔ D6B-T-native ...
Analysing C1 ↔ D6B9-strict ...
Analysing C2b ↔ D1 ...
Analysing C2b ↔ D4 ...
Analysing C2b ↔ D5-768 ...
Analysing C2b ↔ D6B-T-native ...
Analysing C2b ↔ D6B9-strict ...
Analysing C2c ↔ D1 ...
Analysing C2c ↔ D4 ...
Analysing C2c ↔ D5-768 ...
Analysing C2c ↔ D6B-T-native ...
Analysing C2c ↔ D6B9-strict ...
E1 — All Retained C × D Region–Point Relationships

Overall 15-pair relationship summary:


,c_variant,d_variant,n_c_regions,n_d_points,d_inside_fraction,d_outside_fraction,c_zero_fraction,c_one_fraction,c_multi_fraction,c_occupied_fraction,n_multi_regions,d_points_in_multi_regions,mean_d_points_per_multi_region,median_d_points_per_multi_region,max_d_points_per_region
0,C1,D1,2554,7652,0.5481,0.4519,0.1562,0.5983,0.2455,0.8438,627,2666,4.2520,3.0,31
1,C1,D4,2554,4194,1.0000,0.0000,0.1562,0.5983,0.2455,0.8438,627,2666,4.2520,3.0,31
2,C1,D5-768,2554,14125,0.1935,0.8065,0.6347,0.2001,0.1652,0.3653,422,2222,5.2654,3.0,33
3,C1,D6B-T-native,2554,15913,0.2594,0.7406,0.4714,0.2980,0.2306,0.5286,589,3367,5.7165,3.0,50
4,C1,D6B9-strict,2554,4551,0.3507,0.6493,0.6797,0.2177,0.1026,0.3203,262,1040,3.9695,3.0,31
5,C2b,D1,6074,7652,0.5481,0.4519,0.4409,0.4745,0.0846,0.5591,514,1312,2.5525,2.0,22
6,C2b,D4,6074,4194,1.0000,0.0000,0.4409,0.4745,0.0846,0.5591,514,1312,2.5525,2.0,22
7,C2b,D5-768,6074,14125,0.1935,0.8065,0.7097,0.2097,0.0805,0.2903,489,1459,2.9836,2.0,33
8,C2b,D6B-T-native,6074,15913,0.2594,0.7406,0.5850,0.2921,0.1230,0.4150,747,2354,3.1513,2.0,31
9,C2b,D6B9-strict,6074,4551,0.3507,0.6493,0.7842,0.1846,0.0313,0.2158,190,475,2.5000,2.0,9



Shared-foreground consistency check:


,d_variant,C1_inside_fraction,C2b_inside_fraction,C2c_inside_fraction,max_minus_min,consistent
0,D1,0.548092,0.548092,0.548092,0.0,True
1,D4,1.000000,1.000000,1.000000,0.0,True
2,D5-768,0.193487,0.193487,0.193487,0.0,True
3,D6B-T-native,0.259411,0.259411,0.259411,0.0,True
4,D6B9-strict,0.350692,0.350692,0.350692,0.0,True



Density-stratified summary:


,c_variant,d_variant,density_stratum,n_images,n_c_regions,n_d_points,d_inside_fraction,c_zero_fraction,c_one_fraction,c_multi_fraction,c_occupied_fraction
0,C1,D1,S1_0_20,20,240,1082,0.5767,0.1292,0.4375,0.4333,0.8708
1,C1,D1,S2_20_40,20,392,1233,0.5661,0.1480,0.5485,0.3036,0.8520
2,C1,D1,S3_40_60,20,401,1390,0.5432,0.2070,0.4988,0.2943,0.7930
3,C1,D1,S4_60_80,20,566,1663,0.5707,0.0654,0.7032,0.2314,0.9346
4,C1,D1,S5_80_95,10,465,1143,0.5354,0.1376,0.6559,0.2065,0.8624
...,...,...,...,...,...,...,...,...,...,...,...
85,C2c,D6B9-strict,S2_20_40,20,748,852,0.3592,0.6684,0.2727,0.0588,0.3316
86,C2c,D6B9-strict,S3_40_60,20,771,1017,0.3235,0.6978,0.2179,0.0843,0.3022
87,C2c,D6B9-strict,S4_60_80,20,879,909,0.3421,0.7156,0.2491,0.0353,0.2844
88,C2c,D6B9-strict,S5_80_95,10,640,789,0.3118,0.6516,0.3219,0.0266,0.3484



Structural checks:
  Pair-image rows: 1500 (expected 1500)
  Pair-region rows: 63995 (expected 63995)
  Overall pair summaries: 15 (expected 15)
  Density summaries: 90 (expected 90)

Step 13B.4 complete.
All 15 retained C×D structural relationships analysed using frozen predictions.
No GT correctness or fusion rule was introduced.


In [52]:
# ============================================================
# Step 13B.5 — Cross-Representation Comparison and Diagnostics
# ============================================================
#
# Purpose
# -------
# Summarise the structural trade-offs identified in Step 13B.4.
#
# Part A — Hold D fixed:
#          Compare C1 → C2c → C2b partitioning.
#
# Part B — Hold C fixed:
#          Compare D1 / D4 / D5 / D6 variants.
#
# Part C — Quantify changes relative to interpretable baselines:
#          C-side baseline = C1
#          D-side baseline = D1
#
# IMPORTANT:
#   - No GT correctness.
#   - No ranking / winner selection.
#   - No fusion rule.
#   - Frozen predictions are not modified.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "e1_all_pairs_overall_df" in globals(), (
    "e1_all_pairs_overall_df not found. Run Step 13B.4 first."
)

assert "e1_all_pairs_density_df" in globals(), (
    "e1_all_pairs_density_df not found. Run Step 13B.4 first."
)

assert len(e1_all_pairs_overall_df) == 15
assert len(e1_all_pairs_density_df) == 90


# ------------------------------------------------------------
# Display order
#
# C order is deliberately:
#   C1 → C2c → C2b
#
# because this represents increasingly fine partitioning
# of the SAME A3c foreground support.
# ------------------------------------------------------------

C_COMPARE_ORDER = [
    "C1",
    "C2c",
    "C2b",
]

D_COMPARE_ORDER = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]


# ============================================================
# Part A
# Hold D fixed — effect of C partitioning
# ============================================================

c_partition_rows = []


for d_variant in D_COMPARE_ORDER:

    sdf = (
        e1_all_pairs_overall_df[
            e1_all_pairs_overall_df[
                "d_variant"
            ]
            ==
            d_variant
        ]
        .set_index("c_variant")
    )


    for c_variant in C_COMPARE_ORDER:

        row = sdf.loc[c_variant]

        c_partition_rows.append(
            {
                "d_variant":
                    d_variant,

                "c_variant":
                    c_variant,

                "n_c_regions":
                    int(
                        row["n_c_regions"]
                    ),

                "d_inside_fraction":
                    float(
                        row["d_inside_fraction"]
                    ),

                "c_zero_fraction":
                    float(
                        row["c_zero_fraction"]
                    ),

                "c_one_fraction":
                    float(
                        row["c_one_fraction"]
                    ),

                "c_multi_fraction":
                    float(
                        row["c_multi_fraction"]
                    ),

                "c_occupied_fraction":
                    float(
                        row["c_occupied_fraction"]
                    ),

                "mean_d_points_per_multi_region":
                    float(
                        row[
                            "mean_d_points_per_multi_region"
                        ]
                    ),

                "median_d_points_per_multi_region":
                    float(
                        row[
                            "median_d_points_per_multi_region"
                        ]
                    ),

                "max_d_points_per_region":
                    int(
                        row["max_d_points_per_region"]
                    ),
            }
        )


e1_c_partition_comparison_df = pd.DataFrame(
    c_partition_rows
)


# ------------------------------------------------------------
# Changes relative to C1, for each fixed D
# ------------------------------------------------------------

c_delta_rows = []


for d_variant in D_COMPARE_ORDER:

    sdf = (
        e1_all_pairs_overall_df[
            e1_all_pairs_overall_df[
                "d_variant"
            ]
            ==
            d_variant
        ]
        .set_index("c_variant")
    )

    baseline = sdf.loc["C1"]


    for c_variant in [
        "C2c",
        "C2b",
    ]:

        row = sdf.loc[c_variant]

        c_delta_rows.append(
            {
                "d_variant":
                    d_variant,

                "comparison":
                    f"{c_variant} - C1",

                "delta_n_regions":
                    int(
                        row["n_c_regions"]
                        -
                        baseline["n_c_regions"]
                    ),

                "delta_zero_fraction":
                    float(
                        row["c_zero_fraction"]
                        -
                        baseline["c_zero_fraction"]
                    ),

                "delta_one_fraction":
                    float(
                        row["c_one_fraction"]
                        -
                        baseline["c_one_fraction"]
                    ),

                "delta_multi_fraction":
                    float(
                        row["c_multi_fraction"]
                        -
                        baseline["c_multi_fraction"]
                    ),

                "delta_occupied_fraction":
                    float(
                        row["c_occupied_fraction"]
                        -
                        baseline["c_occupied_fraction"]
                    ),

                "delta_mean_multi":
                    float(
                        row[
                            "mean_d_points_per_multi_region"
                        ]
                        -
                        baseline[
                            "mean_d_points_per_multi_region"
                        ]
                    ),

                "delta_max_multi":
                    int(
                        row["max_d_points_per_region"]
                        -
                        baseline["max_d_points_per_region"]
                    ),
            }
        )


e1_c_partition_delta_df = pd.DataFrame(
    c_delta_rows
)


# ============================================================
# Part B
# Hold C fixed — effect of D representation
# ============================================================

d_representation_rows = []


for c_variant in C_COMPARE_ORDER:

    sdf = (
        e1_all_pairs_overall_df[
            e1_all_pairs_overall_df[
                "c_variant"
            ]
            ==
            c_variant
        ]
        .set_index("d_variant")
    )


    for d_variant in D_COMPARE_ORDER:

        row = sdf.loc[d_variant]

        d_representation_rows.append(
            {
                "c_variant":
                    c_variant,

                "d_variant":
                    d_variant,

                "n_d_points":
                    int(
                        row["n_d_points"]
                    ),

                "d_inside_fraction":
                    float(
                        row["d_inside_fraction"]
                    ),

                "d_outside_fraction":
                    float(
                        row["d_outside_fraction"]
                    ),

                "c_zero_fraction":
                    float(
                        row["c_zero_fraction"]
                    ),

                "c_one_fraction":
                    float(
                        row["c_one_fraction"]
                    ),

                "c_multi_fraction":
                    float(
                        row["c_multi_fraction"]
                    ),

                "c_occupied_fraction":
                    float(
                        row["c_occupied_fraction"]
                    ),

                "d_points_in_multi_regions":
                    int(
                        row[
                            "d_points_in_multi_regions"
                        ]
                    ),
            }
        )


e1_d_representation_comparison_df = pd.DataFrame(
    d_representation_rows
)


# ------------------------------------------------------------
# Changes relative to D1, for each fixed C
#
# NOTE:
# D4 is derived from D1 by hard A3c gating, so this comparison
# is descriptive rather than treating D4 as independent evidence.
# ------------------------------------------------------------

d_delta_rows = []


for c_variant in C_COMPARE_ORDER:

    sdf = (
        e1_all_pairs_overall_df[
            e1_all_pairs_overall_df[
                "c_variant"
            ]
            ==
            c_variant
        ]
        .set_index("d_variant")
    )

    baseline = sdf.loc["D1"]


    for d_variant in [
        "D4",
        "D5-768",
        "D6B-T-native",
        "D6B9-strict",
    ]:

        row = sdf.loc[d_variant]

        d_delta_rows.append(
            {
                "c_variant":
                    c_variant,

                "comparison":
                    f"{d_variant} - D1",

                "delta_n_d_points":
                    int(
                        row["n_d_points"]
                        -
                        baseline["n_d_points"]
                    ),

                "delta_inside_fraction":
                    float(
                        row["d_inside_fraction"]
                        -
                        baseline["d_inside_fraction"]
                    ),

                "delta_outside_fraction":
                    float(
                        row["d_outside_fraction"]
                        -
                        baseline["d_outside_fraction"]
                    ),

                "delta_zero_fraction":
                    float(
                        row["c_zero_fraction"]
                        -
                        baseline["c_zero_fraction"]
                    ),

                "delta_one_fraction":
                    float(
                        row["c_one_fraction"]
                        -
                        baseline["c_one_fraction"]
                    ),

                "delta_multi_fraction":
                    float(
                        row["c_multi_fraction"]
                        -
                        baseline["c_multi_fraction"]
                    ),

                "delta_occupied_fraction":
                    float(
                        row["c_occupied_fraction"]
                        -
                        baseline["c_occupied_fraction"]
                    ),
            }
        )


e1_d_representation_delta_df = pd.DataFrame(
    d_delta_rows
)


# ============================================================
# Part C
# Compact pivot tables for interpretation
# ============================================================

# ------------------------------------------------------------
# C partition trade-off:
# zero / one / multi fractions
# ------------------------------------------------------------

e1_zero_pivot_df = (
    e1_all_pairs_overall_df
    .pivot(
        index="d_variant",
        columns="c_variant",
        values="c_zero_fraction",
    )
    .reindex(
        index=D_COMPARE_ORDER,
        columns=C_COMPARE_ORDER,
    )
)


e1_one_pivot_df = (
    e1_all_pairs_overall_df
    .pivot(
        index="d_variant",
        columns="c_variant",
        values="c_one_fraction",
    )
    .reindex(
        index=D_COMPARE_ORDER,
        columns=C_COMPARE_ORDER,
    )
)


e1_multi_pivot_df = (
    e1_all_pairs_overall_df
    .pivot(
        index="d_variant",
        columns="c_variant",
        values="c_multi_fraction",
    )
    .reindex(
        index=D_COMPARE_ORDER,
        columns=C_COMPARE_ORDER,
    )
)


# ------------------------------------------------------------
# D spatial relationship to common A3c foreground
#
# Since foreground support is identical across C1/C2b/C2c,
# use C1 only to avoid redundant reporting.
# ------------------------------------------------------------

e1_d_foreground_summary_df = (
    e1_all_pairs_overall_df[
        e1_all_pairs_overall_df[
            "c_variant"
        ]
        ==
        "C1"
    ][
        [
            "d_variant",
            "n_d_points",
            "d_inside_fraction",
            "d_outside_fraction",
        ]
    ]
    .set_index("d_variant")
    .reindex(D_COMPARE_ORDER)
    .reset_index()
)


# ============================================================
# Sanity checks
# ============================================================

assert len(
    e1_c_partition_comparison_df
) == 15

assert len(
    e1_c_partition_delta_df
) == 10

assert len(
    e1_d_representation_comparison_df
) == 15

assert len(
    e1_d_representation_delta_df
) == 12

assert e1_zero_pivot_df.shape == (5, 3)
assert e1_one_pivot_df.shape == (5, 3)
assert e1_multi_pivot_df.shape == (5, 3)

assert len(
    e1_d_foreground_summary_df
) == 5


# ============================================================
# Display
# ============================================================

print("=" * 110)
print(
    "E1 — Cross-Representation Comparison and Diagnostics"
)
print("=" * 110)


print(
    "\nA. Hold D fixed — effect of C partitioning:"
)

display(
    e1_c_partition_comparison_df.round(4)
)


print(
    "\nA2. Change from C1 when using C2c / C2b:"
)

display(
    e1_c_partition_delta_df.round(4)
)


print(
    "\nB. Hold C fixed — effect of D representation:"
)

display(
    e1_d_representation_comparison_df.round(4)
)


print(
    "\nB2. Change from D1 for each fixed C:"
)

display(
    e1_d_representation_delta_df.round(4)
)


print(
    "\nC1. Fraction of C regions with ZERO D points:"
)

display(
    e1_zero_pivot_df.round(4)
)


print(
    "\nC2. Fraction of C regions with EXACTLY ONE D point:"
)

display(
    e1_one_pivot_df.round(4)
)


print(
    "\nC3. Fraction of C regions with MULTIPLE D points:"
)

display(
    e1_multi_pivot_df.round(4)
)


print(
    "\nC4. D relationship to common A3c foreground:"
)

display(
    e1_d_foreground_summary_df.round(4)
)


print("\nStructural checks:")
print(
    f"  C-partition comparison rows: "
    f"{len(e1_c_partition_comparison_df)}"
)
print(
    f"  C-partition delta rows: "
    f"{len(e1_c_partition_delta_df)}"
)
print(
    f"  D-representation comparison rows: "
    f"{len(e1_d_representation_comparison_df)}"
)
print(
    f"  D-representation delta rows: "
    f"{len(e1_d_representation_delta_df)}"
)


print("\n" + "=" * 110)
print("Step 13B.5 complete.")
print(
    "Cross-representation structural trade-offs "
    "summarised without GT correctness or fusion."
)
print("=" * 110)

E1 — Cross-Representation Comparison and Diagnostics

A. Hold D fixed — effect of C partitioning:


,d_variant,c_variant,n_c_regions,d_inside_fraction,c_zero_fraction,c_one_fraction,c_multi_fraction,c_occupied_fraction,mean_d_points_per_multi_region,median_d_points_per_multi_region,max_d_points_per_region
0,D1,C1,2554,0.5481,0.1562,0.5983,0.2455,0.8438,4.2520,3.0,31
1,D1,C2c,4171,0.5481,0.2467,0.6023,0.1510,0.7533,2.6698,2.0,22
2,D1,C2b,6074,0.5481,0.4409,0.4745,0.0846,0.5591,2.5525,2.0,22
3,D4,C1,2554,1.0000,0.1562,0.5983,0.2455,0.8438,4.2520,3.0,31
4,D4,C2c,4171,1.0000,0.2467,0.6023,0.1510,0.7533,2.6698,2.0,22
5,D4,C2b,6074,1.0000,0.4409,0.4745,0.0846,0.5591,2.5525,2.0,22
6,D5-768,C1,2554,0.1935,0.6347,0.2001,0.1652,0.3653,5.2654,3.0,33
7,D5-768,C2c,4171,0.1935,0.6207,0.2510,0.1283,0.3793,3.1514,2.0,33
8,D5-768,C2b,6074,0.1935,0.7097,0.2097,0.0805,0.2903,2.9836,2.0,33
9,D6B-T-native,C1,2554,0.2594,0.4714,0.2980,0.2306,0.5286,5.7165,3.0,50



A2. Change from C1 when using C2c / C2b:


,d_variant,comparison,delta_n_regions,delta_zero_fraction,delta_one_fraction,delta_multi_fraction,delta_occupied_fraction,delta_mean_multi,delta_max_multi
0,D1,C2c - C1,1617,0.0905,0.0040,-0.0945,-0.0905,-1.5822,-9
1,D1,C2b - C1,3520,0.2847,-0.1238,-0.1609,-0.2847,-1.6995,-9
2,D4,C2c - C1,1617,0.0905,0.0040,-0.0945,-0.0905,-1.5822,-9
3,D4,C2b - C1,3520,0.2847,-0.1238,-0.1609,-0.2847,-1.6995,-9
4,D5-768,C2c - C1,1617,-0.0140,0.0509,-0.0370,0.0140,-2.1140,0
5,D5-768,C2b - C1,3520,0.0751,0.0097,-0.0847,-0.0751,-2.2818,0
6,D6B-T-native,C2c - C1,1617,-0.0104,0.0458,-0.0355,0.0104,-2.4069,-19
7,D6B-T-native,C2b - C1,3520,0.1135,-0.0059,-0.1076,-0.1135,-2.5652,-19
8,D6B9-strict,C2c - C1,1617,0.0184,0.0333,-0.0518,-0.0184,-1.3798,-22
9,D6B9-strict,C2b - C1,3520,0.1044,-0.0331,-0.0713,-0.1044,-1.4695,-22



B. Hold C fixed — effect of D representation:


...


C3. Fraction of C regions with MULTIPLE D points:


c_variant,C1,C2c,C2b
d_variant,,,
D1,0.2455,0.1510,0.0846
D4,0.2455,0.1510,0.0846
D5-768,0.1652,0.1283,0.0805
D6B-T-native,0.2306,0.1952,0.1230
D6B9-strict,0.1026,0.0508,0.0313



C4. D relationship to common A3c foreground:


,d_variant,n_d_points,d_inside_fraction,d_outside_fraction
0,D1,7652,0.5481,0.4519
1,D4,4194,1.0000,0.0000
2,D5-768,14125,0.1935,0.8065
3,D6B-T-native,15913,0.2594,0.7406
4,D6B9-strict,4551,0.3507,0.6493



Structural checks:
  C-partition comparison rows: 15
  C-partition delta rows: 10
  D-representation comparison rows: 15
  D-representation delta rows: 12

Step 13B.5 complete.
Cross-representation structural trade-offs summarised without GT correctness or fusion.


In [53]:
# ============================================================
# Step 13B.6 — E1 Summary and Freeze Relationship Evidence
# ============================================================
#
# Purpose
# -------
# Finalise and freeze Group E1.
#
# E1 has characterised the structural relationship between the
# frozen Group-C region evidence and Group-D activation evidence.
#
# This step:
#   1. validates the final E1 artefacts,
#   2. saves the important E1 dataframes,
#   3. records the frozen E1 protocol / candidate inventory,
#   4. prints a compact reproducibility summary.
#
# IMPORTANT:
#   - No GT correctness is introduced.
#   - No fusion strategy is introduced.
#   - No C/D prediction is modified.
#   - E1 is frozen after this step.
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

required_e1_objects = [
    "e1_all_pairs_image_df",
    "e1_all_pairs_region_df",
    "e1_all_pairs_overall_df",
    "e1_all_pairs_density_df",
    "e1_foreground_consistency_df",
    "e1_c_partition_comparison_df",
    "e1_c_partition_delta_df",
    "e1_d_representation_comparison_df",
    "e1_d_representation_delta_df",
]

for name in required_e1_objects:
    assert name in globals(), (
        f"{name} not found. "
        "Run Steps 13B.4–13B.5 first."
    )


# ------------------------------------------------------------
# Frozen candidate inventory
# ------------------------------------------------------------

E1_FROZEN_C_VARIANTS = [
    "C1",
    "C2b",
    "C2c",
]

E1_FROZEN_D_VARIANTS = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]

E1_EXPECTED_PAIRS = (
    len(E1_FROZEN_C_VARIANTS)
    *
    len(E1_FROZEN_D_VARIANTS)
)

assert E1_EXPECTED_PAIRS == 15


# ------------------------------------------------------------
# Final structural validation
# ------------------------------------------------------------

# 15 C×D pairs × 100 images
assert len(e1_all_pairs_image_df) == 1500

assert (
    e1_all_pairs_image_df[
        ["c_variant", "d_variant"]
    ]
    .drop_duplicates()
    .shape[0]
    == 15
)

pair_sizes = (
    e1_all_pairs_image_df
    .groupby(
        ["c_variant", "d_variant"]
    )
    .size()
)

assert (pair_sizes == 100).all()


# Expected region rows:
# (2554 + 6074 + 4171) × 5 D variants
expected_region_rows = (
    2554 + 6074 + 4171
) * 5

assert (
    len(e1_all_pairs_region_df)
    ==
    expected_region_rows
)


# Overall and density summaries
assert len(e1_all_pairs_overall_df) == 15
assert len(e1_all_pairs_density_df) == 90


# No D points outside image boundaries
assert (
    e1_all_pairs_image_df[
        "n_d_out_of_bounds"
    ].sum()
    == 0
)


# C1 / C2b / C2c must share identical foreground support.
assert (
    e1_foreground_consistency_df[
        "consistent"
    ]
).all()


# ------------------------------------------------------------
# E1 output directory
# ------------------------------------------------------------

GROUP_E_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "Group_E"
)

E1_RESULTS_DIR = (
    GROUP_E_RESULTS_DIR
    / "E1_region_point_relationship"
)

E1_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Save canonical E1 artefacts
# ------------------------------------------------------------

E1_SAVE_TABLES = {
    "E1_all_pairs_per_image.csv":
        e1_all_pairs_image_df,

    "E1_all_pairs_per_region.csv":
        e1_all_pairs_region_df,

    "E1_all_pairs_overall_summary.csv":
        e1_all_pairs_overall_df,

    "E1_all_pairs_density_summary.csv":
        e1_all_pairs_density_df,

    "E1_foreground_consistency.csv":
        e1_foreground_consistency_df,

    "E1_C_partition_comparison.csv":
        e1_c_partition_comparison_df,

    "E1_C_partition_delta_from_C1.csv":
        e1_c_partition_delta_df,

    "E1_D_representation_comparison.csv":
        e1_d_representation_comparison_df,

    "E1_D_representation_delta_from_D1.csv":
        e1_d_representation_delta_df,
}


for filename, df in E1_SAVE_TABLES.items():

    path = (
        E1_RESULTS_DIR
        / filename
    )

    df.to_csv(
        path,
        index=False,
    )


# ------------------------------------------------------------
# Frozen E1 protocol metadata
# ------------------------------------------------------------

e1_freeze_metadata = {

    "stage":
        "Group E1",

    "stage_name":
        "Region–Point Relationship Analysis",

    "dataset":
        "FSC147",

    "evaluation_subset":
        "frozen FSC147 validation dev100",

    "n_images":
        100,

    "c_variants":
        E1_FROZEN_C_VARIANTS,

    "d_variants":
        E1_FROZEN_D_VARIANTS,

    "n_c_d_pairs":
        E1_EXPECTED_PAIRS,

    "c_foreground_support":
        "shared frozen A3c foreground",

    "relationship_definition": {
        "outside_region_label":
            0,

        "inside_region_label":
            ">0",

        "region_occupancy_classes":
            [
                "0 D points",
                "1 D point",
                "2+ D points",
            ],
    },

    "gt_used_for_correctness":
        False,

    "fusion_rule_introduced":
        False,

    "predictions_modified":
        False,

    "interpretation_boundary":
        (
            "E1 characterises structural agreement, "
            "disagreement, occupancy and multiplicity only. "
            "It does not determine whether C-only, D-only "
            "or shared evidence corresponds to true objects."
        ),

    "next_stage":
        (
            "E2 — GT-based complementarity analysis"
        ),
}


E1_METADATA_PATH = (
    E1_RESULTS_DIR
    / "E1_freeze_metadata.json"
)

with open(
    E1_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        e1_freeze_metadata,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Verify saved files
# ------------------------------------------------------------

saved_files = sorted(
    E1_RESULTS_DIR.glob("*")
)

expected_saved_count = (
    len(E1_SAVE_TABLES)
    + 1
)

assert (
    len(saved_files)
    >= expected_saved_count
)


# ------------------------------------------------------------
# Compact frozen E1 summary
# ------------------------------------------------------------

print("=" * 110)
print(
    "GROUP E1 — FINAL SUMMARY AND FREEZE"
)
print("=" * 110)

print(
    f"\nDataset / subset:"
    f"\n  FSC147 frozen validation dev100"
)

print(
    f"\nFrozen C representations:"
    f"\n  {E1_FROZEN_C_VARIANTS}"
)

print(
    f"\nFrozen D representations:"
    f"\n  {E1_FROZEN_D_VARIANTS}"
)

print(
    f"\nC×D relationships analysed:"
    f"\n  {E1_EXPECTED_PAIRS}"
)

print(
    f"\nPair-image rows:"
    f"\n  {len(e1_all_pairs_image_df)}"
)

print(
    f"\nPair-region rows:"
    f"\n  {len(e1_all_pairs_region_df)}"
)

print(
    "\nShared C foreground support:"
    "\n  PASS — C1 / C2b / C2c use identical "
    "frozen A3c foreground support."
)

print(
    "\nE1 analysis covered:"
    "\n  - D evidence inside vs outside C foreground"
    "\n  - C regions with 0 / 1 / 2+ D points"
    "\n  - region occupancy"
    "\n  - activation multiplicity"
    "\n  - C-partition comparison"
    "\n  - D-representation comparison"
    "\n  - density-stratified structural relationships"
)

print(
    "\nE1 interpretation boundary:"
    "\n  Structural relationship only."
    "\n  No GT correctness."
    "\n  No semantic fusion decision."
    "\n  No candidate removed or added."
)

print(
    f"\nSaved E1 artefacts:"
    f"\n  {E1_RESULTS_DIR}"
)

for path in saved_files:
    print(
        f"    - {path.name}"
    )

print("\n" + "=" * 110)

print(
    "Step 13B.6 complete — Group E1 frozen."
)

print(
    "Next: E2 — GT-based complementarity analysis."
)

print("=" * 110)

GROUP E1 — FINAL SUMMARY AND FREEZE

Dataset / subset:
  FSC147 frozen validation dev100

Frozen C representations:
  ['C1', 'C2b', 'C2c']

Frozen D representations:
  ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']

C×D relationships analysed:
  15

Pair-image rows:
  1500

Pair-region rows:
  63995

Shared C foreground support:
  PASS — C1 / C2b / C2c use identical frozen A3c foreground support.

E1 analysis covered:
  - D evidence inside vs outside C foreground
  - C regions with 0 / 1 / 2+ D points
  - region occupancy
  - activation multiplicity
  - C-partition comparison
  - D-representation comparison
  - density-stratified structural relationships

E1 interpretation boundary:
  Structural relationship only.
  No GT correctness.
  No semantic fusion decision.
  No candidate removed or added.

Saved E1 artefacts:
  /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_E/E1_region_point_relationship
    - E1_C_partition_comparison.csv
    - E1_C_p

In [54]:
# ============================================================
# Step 13C.1 — Define and Validate GT-Centric Complementarity
#              Protocol
# ============================================================
#
# Purpose
# -------
# Define and freeze the E2 protocol for measuring
# complementarity between Group-C reference points and
# Group-D activation points relative to FSC-147 GT objects.
#
# For every image and C×D pair:
#
#   1. Match frozen C points independently to GT
#   2. Match frozen D points independently to GT
#
# using the SAME canonical one-to-one matcher already frozen
# in Group D (Step 11D.6A) and used for the corrected Group-C
# evaluation.
#
# Canonical tolerance:
#
#   radius_px = max(1, round(0.03 * min(H, W)))
#
# GT classes:
#
#   BOTH
#   C_ONLY
#   D_ONLY
#   NEITHER
#
# IMPORTANT
# ---------
# - No C↔D matching is required for GT complementarity.
# - No fusion prediction is created.
# - No candidate is removed or added.
# - This cell defines/validates protocol only.
# ============================================================


import ast
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


# ============================================================
# 13C.1.1 — Preconditions
# ============================================================

assert "group_e_handoff_df" in globals(), (
    "group_e_handoff_df not found. "
    "Run Step 13A first."
)

assert len(group_e_handoff_df) == 800, (
    "Expected frozen Group-E handoff to contain "
    "8 candidates × 100 images = 800 rows."
)


# ============================================================
# 13C.1.2 — Frozen E2 candidate inventory
# ============================================================

E2_C_VARIANTS = [
    "C1",
    "C2b",
    "C2c",
]

E2_D_VARIANTS = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]

assert len(E2_C_VARIANTS) == 3
assert len(E2_D_VARIANTS) == 5
assert (
    len(E2_C_VARIANTS)
    * len(E2_D_VARIANTS)
    == 15
)


# ============================================================
# 13C.1.3 — Frozen canonical localisation setting
# ============================================================

E2_POINT_TOLERANCE_FRAC = 0.03


def e2_match_radius_px(
    image_width,
    image_height,
):
    """
    Frozen canonical B1 localisation tolerance.
    """

    return max(
        1,
        int(
            round(
                E2_POINT_TOLERANCE_FRAC
                *
                min(
                    int(image_width),
                    int(image_height),
                )
            )
        ),
    )


# ============================================================
# 13C.1.4 — Canonical one-to-one matcher
#
# EXACT algorithm frozen in:
#
#   Group D — Step 11D.6A
#
# and reused in:
#
#   Group ABC — corrected Step 10F
#
# Output:
#   matches[:, 0] = pred index
#   matches[:, 1] = GT index
# ============================================================

def e2_match_points_one_to_one(
    pred_xy,
    gt_xy,
    radius_px,
):
    """
    Canonical memory-efficient one-to-one matching.

    Objective:
        1. maximise number of matches within radius_px
        2. among maximum-cardinality matchings,
           minimise total Euclidean distance

    Each predicted point and GT point can be matched
    at most once.

    Returns
    -------
    dict:
        matches:
            shape (TP, 2)
            columns = [pred_index, gt_index]

        distances:
            Euclidean distance of accepted matches

        tp, fp, fn
    """

    pred_xy = np.asarray(
        pred_xy,
        dtype=np.float32,
    ).reshape(-1, 2)

    gt_xy = np.asarray(
        gt_xy,
        dtype=np.float32,
    ).reshape(-1, 2)

    n_pred = len(pred_xy)
    n_gt = len(gt_xy)


    # --------------------------------------------------------
    # Empty cases
    # --------------------------------------------------------

    if n_pred == 0 or n_gt == 0:

        return {
            "matches": np.empty(
                (0, 2),
                dtype=np.int64,
            ),

            "distances": np.empty(
                (0,),
                dtype=np.float32,
            ),

            "tp": 0,

            "fp": n_pred,

            "fn": n_gt,
        }


    # --------------------------------------------------------
    # Pairwise Euclidean distances
    # --------------------------------------------------------

    dist_matrix = cdist(
        pred_xy,
        gt_xy,
        metric="euclidean",
    )

    valid = (
        dist_matrix
        <= radius_px
    )


    # --------------------------------------------------------
    # Lexicographic matching objective
    #
    # Maximum-cardinality first,
    # then minimum total distance.
    # --------------------------------------------------------

    max_possible_matches = min(
        n_pred,
        n_gt,
    )

    unmatched_cost = (
        (max_possible_matches + 1)
        *
        (float(radius_px) + 1.0)
    )

    forbidden_cost = (
        unmatched_cost
        *
        (max_possible_matches + 2)
    )


    # --------------------------------------------------------
    # Rectangular assignment matrix
    #
    # Columns:
    #   0 ... n_gt-1 = real GT
    #   n_gt ...      = dummy unmatched columns
    # --------------------------------------------------------

    cost_matrix = np.full(
        (
            n_pred,
            n_gt + n_pred,
        ),
        forbidden_cost,
        dtype=np.float64,
    )


    real_costs = cost_matrix[
        :,
        :n_gt,
    ]

    real_costs[valid] = (
        dist_matrix[valid]
    )


    # Any prediction may remain unmatched
    cost_matrix[
        :,
        n_gt:
    ] = unmatched_cost


    # --------------------------------------------------------
    # Hungarian assignment
    # --------------------------------------------------------

    row_ind, col_ind = (
        linear_sum_assignment(
            cost_matrix
        )
    )


    # Real prediction → real GT assignments
    real_match_mask = (
        col_ind < n_gt
    )

    pred_idx = row_ind[
        real_match_mask
    ]

    gt_idx = col_ind[
        real_match_mask
    ]


    matched_distances = (
        dist_matrix[
            pred_idx,
            gt_idx,
        ]
    )


    # Safety check
    accepted = (
        matched_distances
        <= radius_px
    )

    pred_idx = pred_idx[
        accepted
    ]

    gt_idx = gt_idx[
        accepted
    ]

    matched_distances = (
        matched_distances[
            accepted
        ]
    )


    tp = len(pred_idx)
    fp = n_pred - tp
    fn = n_gt - tp


    if tp > 0:

        matches = np.column_stack(
            [
                pred_idx,
                gt_idx,
            ]
        ).astype(
            np.int64
        )

    else:

        matches = np.empty(
            (0, 2),
            dtype=np.int64,
        )


    return {
        "matches":
            matches,

        "distances":
            matched_distances.astype(
                np.float32
            ),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "fn":
            int(fn),
    }


# ============================================================
# 13C.1.5 — Robust point parser
# ============================================================

def e2_parse_points_xy(value):

    if value is None:

        return np.empty(
            (0, 2),
            dtype=np.float32,
        )


    if isinstance(value, str):

        value = value.strip()

        if (
            value == ""
            or value.lower() == "nan"
        ):

            return np.empty(
                (0, 2),
                dtype=np.float32,
            )

        value = ast.literal_eval(
            value
        )


    arr = np.asarray(
        value,
        dtype=np.float32,
    )


    if arr.size == 0:

        return np.empty(
            (0, 2),
            dtype=np.float32,
        )


    arr = arr.reshape(
        -1,
        2,
    )

    assert np.isfinite(
        arr
    ).all()

    return arr


# ============================================================
# 13C.1.6 — Required handoff schema
# ============================================================

required_columns = {
    "candidate_id",
    "dataset_index",
    "image_id",
    "density_stratum",
    "pred_points_xy",
    "gt_points_xy",
    "gt_count",
}

missing_columns = (
    required_columns
    -
    set(
        group_e_handoff_df.columns
    )
)

assert not missing_columns, (
    "Missing required Group-E columns: "
    f"{sorted(missing_columns)}"
)


# ============================================================
# 13C.1.7 — Validate GT consistency across all 8 candidates
# ============================================================

def e2_canonical_point_tuple(value):

    arr = e2_parse_points_xy(
        value
    )

    return tuple(
        tuple(
            float(v)
            for v in point
        )
        for point in arr
    )


gt_consistency_rows = []


for dataset_index, sdf in (
    group_e_handoff_df
    .groupby(
        "dataset_index"
    )
):

    assert len(sdf) == 8, (
        f"dataset_index={dataset_index}: "
        f"expected 8 rows, got {len(sdf)}."
    )


    image_ids = (
        sdf["image_id"]
        .astype(str)
        .unique()
    )

    assert len(image_ids) == 1


    gt_counts = (
        sdf["gt_count"]
        .astype(int)
        .unique()
    )

    assert len(gt_counts) == 1


    gt_versions = [
        e2_canonical_point_tuple(v)
        for v in sdf["gt_points_xy"]
    ]

    first_gt = (
        gt_versions[0]
    )

    all_same_gt = all(
        gt == first_gt
        for gt in gt_versions
    )


    assert all_same_gt, (
        f"GT coordinate mismatch at "
        f"dataset_index={dataset_index}."
    )


    assert (
        len(first_gt)
        ==
        int(gt_counts[0])
    ), (
        f"GT count mismatch at "
        f"dataset_index={dataset_index}."
    )


    gt_consistency_rows.append(
        {
            "dataset_index":
                int(dataset_index),

            "image_id":
                str(image_ids[0]),

            "gt_count":
                int(gt_counts[0]),

            "gt_consistent":
                True,
        }
    )


e2_gt_consistency_df = pd.DataFrame(
    gt_consistency_rows
)


assert len(
    e2_gt_consistency_df
) == 100

assert (
    e2_gt_consistency_df[
        "gt_consistent"
    ]
).all()


# ============================================================
# 13C.1.8 — Image-size helper
# ============================================================

def e2_get_image_size(row):

    if (
        "width" in row.index
        and
        "height" in row.index
        and
        pd.notna(row["width"])
        and
        pd.notna(row["height"])
    ):

        return (
            int(row["width"]),
            int(row["height"]),
        )


    if (
        "image_width" in row.index
        and
        "image_height" in row.index
        and
        pd.notna(row["image_width"])
        and
        pd.notna(row["image_height"])
    ):

        return (
            int(row["image_width"]),
            int(row["image_height"]),
        )


    raise ValueError(
        "Could not recover image size for "
        f"image_id={row.get('image_id', 'UNKNOWN')}."
    )


# ============================================================
# 13C.1.9 — E2 matcher wrapper
#
# Because the frozen matcher explicitly returns:
#
#   matches[:, 0] = pred index
#   matches[:, 1] = GT index
#
# there is NO need to infer orientation.
# ============================================================

def e2_match_pred_to_gt(
    pred_points_xy,
    gt_points_xy,
    radius_px,
):

    result = (
        e2_match_points_one_to_one(
            pred_xy=pred_points_xy,
            gt_xy=gt_points_xy,
            radius_px=radius_px,
        )
    )


    matches = np.asarray(
        result["matches"],
        dtype=np.int64,
    ).reshape(-1, 2)


    matched_pred_indices = set(
        int(i)
        for i in matches[:, 0]
    )

    matched_gt_indices = set(
        int(i)
        for i in matches[:, 1]
    )


    assert (
        len(matched_pred_indices)
        ==
        int(result["tp"])
    )

    assert (
        len(matched_gt_indices)
        ==
        int(result["tp"])
    )


    return {
        "matched_pred_indices":
            matched_pred_indices,

        "matched_gt_indices":
            matched_gt_indices,

        "matches":
            matches,

        "distances":
            result["distances"],

        "tp":
            int(result["tp"]),

        "fp":
            int(result["fp"]),

        "fn":
            int(result["fn"]),
    }


# ============================================================
# 13C.1.10 — GT-centric complementarity classifier
# ============================================================

E2_GT_CLASSES = [
    "BOTH",
    "C_ONLY",
    "D_ONLY",
    "NEITHER",
]


def e2_classify_gt_complementarity(
    n_gt,
    c_matched_gt_indices,
    d_matched_gt_indices,
):

    c_set = set(
        c_matched_gt_indices
    )

    d_set = set(
        d_matched_gt_indices
    )


    valid_gt_indices = set(
        range(
            int(n_gt)
        )
    )


    assert c_set.issubset(
        valid_gt_indices
    )

    assert d_set.issubset(
        valid_gt_indices
    )


    labels = []


    for gt_idx in range(
        int(n_gt)
    ):

        in_c = (
            gt_idx in c_set
        )

        in_d = (
            gt_idx in d_set
        )


        if in_c and in_d:

            label = "BOTH"

        elif in_c:

            label = "C_ONLY"

        elif in_d:

            label = "D_ONLY"

        else:

            label = "NEITHER"


        labels.append(
            label
        )


    return np.asarray(
        labels,
        dtype=object,
    )


# ============================================================
# 13C.1.11 — Synthetic protocol validation
# ============================================================

test_labels = (
    e2_classify_gt_complementarity(
        n_gt=6,

        c_matched_gt_indices={
            0,
            1,
            3,
        },

        d_matched_gt_indices={
            1,
            2,
            3,
            4,
        },
    )
)


expected_labels = np.asarray(
    [
        "C_ONLY",
        "BOTH",
        "D_ONLY",
        "BOTH",
        "D_ONLY",
        "NEITHER",
    ],
    dtype=object,
)


assert np.array_equal(
    test_labels,
    expected_labels,
)


# ============================================================
# 13C.1.12 — Validate matcher itself on a tiny known example
# ============================================================

_test_pred = np.asarray(
    [
        [10.0, 10.0],
        [50.0, 50.0],
        [90.0, 90.0],
    ],
    dtype=np.float32,
)

_test_gt = np.asarray(
    [
        [11.0, 10.0],
        [51.0, 50.0],
        [200.0, 200.0],
    ],
    dtype=np.float32,
)


_test_match = (
    e2_match_pred_to_gt(
        pred_points_xy=
            _test_pred,

        gt_points_xy=
            _test_gt,

        radius_px=3,
    )
)


assert (
    _test_match["tp"] == 2
)

assert (
    _test_match["fp"] == 1
)

assert (
    _test_match["fn"] == 1
)

assert (
    _test_match[
        "matched_pred_indices"
    ]
    ==
    {0, 1}
)

assert (
    _test_match[
        "matched_gt_indices"
    ]
    ==
    {0, 1}
)


# ============================================================
# 13C.1.13 — Required E2 identities
# ============================================================
#
# These will be asserted for every image/pair in E2:
#
# BOTH + C_ONLY + D_ONLY + NEITHER = GT
#
# C recovered =
#     BOTH + C_ONLY
#
# D recovered =
#     BOTH + D_ONLY
#
# Oracle union =
#     BOTH + C_ONLY + D_ONLY
#
# ============================================================


# ============================================================
# 13C.1.14 — Protocol summary
# ============================================================

print("=" * 110)
print(
    "E2 — GT-CENTRIC COMPLEMENTARITY PROTOCOL"
)
print("=" * 110)

print(
    "\nFrozen C representations:"
)
print(
    f"  {E2_C_VARIANTS}"
)

print(
    "\nFrozen D representations:"
)
print(
    f"  {E2_D_VARIANTS}"
)

print(
    "\nC×D pairs:"
)
print(
    "  15"
)

print(
    "\nGT consistency:"
)
print(
    f"  PASS — "
    f"{len(e2_gt_consistency_df)}/100 images"
)

print(
    "\nCanonical matcher:"
)
print(
    "  PASS — exact frozen Group-D "
    "memory-efficient one-to-one algorithm"
)

print(
    "\nMatch orientation:"
)
print(
    "  matches[:, 0] = prediction index"
)
print(
    "  matches[:, 1] = GT index"
)

print(
    "\nCanonical tolerance:"
)
print(
    "  radius = max(1, round(0.03 × min(H, W)))"
)

print(
    "\nMatching rule:"
)
print(
    "  C → GT independently"
)
print(
    "  D → GT independently"
)

print(
    "\nGT complementarity classes:"
)
for label in E2_GT_CLASSES:
    print(
        f"  - {label}"
    )

print(
    "\nRequired identities:"
)
print(
    "  BOTH + C_ONLY + D_ONLY + NEITHER = GT"
)
print(
    "  C recovered = BOTH + C_ONLY"
)
print(
    "  D recovered = BOTH + D_ONLY"
)
print(
    "  Oracle union = BOTH + C_ONLY + D_ONLY"
)

print(
    "\nInterpretation boundary:"
)
print(
    "  GT recovery overlap/complementarity only."
)
print(
    "  No semantic fusion rule introduced."
)

print("\n" + "=" * 110)
print(
    "Step 13C.1 protocol definition complete."
)
print("=" * 110)

E2 — GT-CENTRIC COMPLEMENTARITY PROTOCOL

Frozen C representations:
  ['C1', 'C2b', 'C2c']

Frozen D representations:
  ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']

C×D pairs:
  15

GT consistency:
  PASS — 100/100 images

Canonical matcher:
  PASS — exact frozen Group-D memory-efficient one-to-one algorithm

Match orientation:
  matches[:, 0] = prediction index
  matches[:, 1] = GT index

Canonical tolerance:
  radius = max(1, round(0.03 × min(H, W)))

Matching rule:
  C → GT independently
  D → GT independently

GT complementarity classes:
  - BOTH
  - C_ONLY
  - D_ONLY
  - NEITHER

Required identities:
  BOTH + C_ONLY + D_ONLY + NEITHER = GT
  C recovered = BOTH + C_ONLY
  D recovered = BOTH + D_ONLY
  Oracle union = BOTH + C_ONLY + D_ONLY

Interpretation boundary:
  GT recovery overlap/complementarity only.
  No semantic fusion rule introduced.

Step 13C.1 protocol definition complete.


In [55]:
# ============================================================
# Step 13C.2 — C1 ↔ D1 GT Complementarity Baseline
# ============================================================
#
# Purpose
# -------
# First GT-based complementarity experiment.
#
# For every frozen dev100 image:
#
#   1. Match C1 reference points independently to GT
#   2. Match D1 activation points independently to GT
#   3. Classify every GT object as:
#
#        BOTH
#        C_ONLY
#        D_ONLY
#        NEITHER
#
# This answers:
#
#   - How much GT recovery is shared by C1 and D1?
#   - How much is uniquely recovered by C1?
#   - How much is uniquely recovered by D1?
#   - How much is missed by both?
#
# Also validates that C1/D1 marginal TP counts reproduce the
# frozen localisation results from E0.
#
# IMPORTANT
# ---------
# - C1 and D1 are matched independently to GT.
# - No C↔D matching is used.
# - No fusion prediction is created.
# - "Oracle union" is diagnostic only.
# ============================================================


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_e2_objects = [
    "group_e_handoff_df",
    "e2_parse_points_xy",
    "e2_get_image_size",
    "e2_match_radius_px",
    "e2_match_pred_to_gt",
    "e2_classify_gt_complementarity",
]

for name in required_e2_objects:
    assert name in globals(), (
        f"{name} not found. Run Step 13C.1 first."
    )


# ------------------------------------------------------------
# 2. Select frozen C1 and D1 rows
# ------------------------------------------------------------

c1_df = (
    group_e_handoff_df[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str)
        ==
        "C1"
    ]
    .copy()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


d1_df = (
    group_e_handoff_df[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str)
        ==
        "D1"
    ]
    .copy()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


assert len(c1_df) == 100
assert len(d1_df) == 100

assert (
    c1_df["dataset_index"].to_numpy()
    ==
    d1_df["dataset_index"].to_numpy()
).all()

assert (
    c1_df["image_id"].astype(str).to_numpy()
    ==
    d1_df["image_id"].astype(str).to_numpy()
).all()


# ------------------------------------------------------------
# 3. Storage
# ------------------------------------------------------------

image_rows = []
gt_rows = []


# ------------------------------------------------------------
# 4. Analyse each image
# ------------------------------------------------------------

for i in range(100):

    c_row = c1_df.iloc[i]
    d_row = d1_df.iloc[i]

    dataset_index = int(
        c_row["dataset_index"]
    )

    image_id = str(
        c_row["image_id"]
    )

    density_stratum = str(
        c_row["density_stratum"]
    )


    # --------------------------------------------------------
    # Validate common image metadata
    # --------------------------------------------------------

    assert dataset_index == int(
        d_row["dataset_index"]
    )

    assert image_id == str(
        d_row["image_id"]
    )

    assert density_stratum == str(
        d_row["density_stratum"]
    )


    # --------------------------------------------------------
    # Parse points
    # --------------------------------------------------------

    gt_points = e2_parse_points_xy(
        c_row["gt_points_xy"]
    )

    c_points = e2_parse_points_xy(
        c_row["pred_points_xy"]
    )

    d_points = e2_parse_points_xy(
        d_row["pred_points_xy"]
    )


    n_gt = len(gt_points)
    n_c = len(c_points)
    n_d = len(d_points)


    assert n_gt == int(
        c_row["gt_count"]
    )

    assert n_gt == int(
        d_row["gt_count"]
    )


    # --------------------------------------------------------
    # Image size and canonical tolerance
    # --------------------------------------------------------

    width, height = (
        e2_get_image_size(
            c_row
        )
    )

    radius_px = (
        e2_match_radius_px(
            image_width=width,
            image_height=height,
        )
    )


    # --------------------------------------------------------
    # Independent C1 → GT matching
    # --------------------------------------------------------

    c_match = (
        e2_match_pred_to_gt(
            pred_points_xy=c_points,
            gt_points_xy=gt_points,
            radius_px=radius_px,
        )
    )


    # --------------------------------------------------------
    # Independent D1 → GT matching
    # --------------------------------------------------------

    d_match = (
        e2_match_pred_to_gt(
            pred_points_xy=d_points,
            gt_points_xy=gt_points,
            radius_px=radius_px,
        )
    )


    c_gt_set = (
        c_match[
            "matched_gt_indices"
        ]
    )

    d_gt_set = (
        d_match[
            "matched_gt_indices"
        ]
    )


    # --------------------------------------------------------
    # GT-centric four-way classification
    # --------------------------------------------------------

    gt_classes = (
        e2_classify_gt_complementarity(
            n_gt=n_gt,
            c_matched_gt_indices=c_gt_set,
            d_matched_gt_indices=d_gt_set,
        )
    )


    class_counts = {
        label: int(
            np.sum(
                gt_classes == label
            )
        )
        for label in E2_GT_CLASSES
    }


    n_both = (
        class_counts["BOTH"]
    )

    n_c_only = (
        class_counts["C_ONLY"]
    )

    n_d_only = (
        class_counts["D_ONLY"]
    )

    n_neither = (
        class_counts["NEITHER"]
    )


    # --------------------------------------------------------
    # Required identities
    # --------------------------------------------------------

    assert (
        n_both
        +
        n_c_only
        +
        n_d_only
        +
        n_neither
        ==
        n_gt
    )


    # C recovered GT
    assert (
        n_both
        +
        n_c_only
        ==
        c_match["tp"]
    )


    # D recovered GT
    assert (
        n_both
        +
        n_d_only
        ==
        d_match["tp"]
    )


    oracle_union_tp = (
        n_both
        +
        n_c_only
        +
        n_d_only
    )


    assert (
        oracle_union_tp
        ==
        len(
            c_gt_set
            |
            d_gt_set
        )
    )


    # --------------------------------------------------------
    # Per-GT rows
    # --------------------------------------------------------

    for gt_idx, (
        gt_point,
        gt_class,
    ) in enumerate(
        zip(
            gt_points,
            gt_classes,
        )
    ):

        gt_rows.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "density_stratum":
                    density_stratum,

                "gt_index":
                    int(gt_idx),

                "gt_x":
                    float(
                        gt_point[0]
                    ),

                "gt_y":
                    float(
                        gt_point[1]
                    ),

                "c_variant":
                    "C1",

                "d_variant":
                    "D1",

                "c_recovered":
                    bool(
                        gt_idx
                        in
                        c_gt_set
                    ),

                "d_recovered":
                    bool(
                        gt_idx
                        in
                        d_gt_set
                    ),

                "complementarity_class":
                    str(
                        gt_class
                    ),

                "radius_px":
                    int(
                        radius_px
                    ),
            }
        )


    # --------------------------------------------------------
    # Per-image row
    # --------------------------------------------------------

    image_rows.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "density_stratum":
                density_stratum,

            "gt_count":
                n_gt,

            "radius_px":
                radius_px,

            "c_variant":
                "C1",

            "d_variant":
                "D1",

            "n_c_candidates":
                n_c,

            "n_d_candidates":
                n_d,

            "c_tp":
                int(
                    c_match["tp"]
                ),

            "c_fp":
                int(
                    c_match["fp"]
                ),

            "c_fn":
                int(
                    c_match["fn"]
                ),

            "d_tp":
                int(
                    d_match["tp"]
                ),

            "d_fp":
                int(
                    d_match["fp"]
                ),

            "d_fn":
                int(
                    d_match["fn"]
                ),

            "both":
                n_both,

            "c_only":
                n_c_only,

            "d_only":
                n_d_only,

            "neither":
                n_neither,

            "oracle_union_tp":
                oracle_union_tp,

            "c_recall":
                (
                    c_match["tp"]
                    / n_gt
                    if n_gt > 0
                    else np.nan
                ),

            "d_recall":
                (
                    d_match["tp"]
                    / n_gt
                    if n_gt > 0
                    else np.nan
                ),

            "oracle_union_recall":
                (
                    oracle_union_tp
                    / n_gt
                    if n_gt > 0
                    else np.nan
                ),
        }
    )


# ------------------------------------------------------------
# 5. Canonical C1 ↔ D1 E2 dataframes
# ------------------------------------------------------------

e2_c1_d1_image_df = pd.DataFrame(
    image_rows
)

e2_c1_d1_gt_df = pd.DataFrame(
    gt_rows
)


assert len(
    e2_c1_d1_image_df
) == 100


# Frozen dev100 contains 10,045 GT objects
assert len(
    e2_c1_d1_gt_df
) == 10045


# ------------------------------------------------------------
# 6. Overall GT-centric complementarity summary
# ------------------------------------------------------------

overall_counts = (
    e2_c1_d1_gt_df[
        "complementarity_class"
    ]
    .value_counts()
    .reindex(
        E2_GT_CLASSES,
        fill_value=0,
    )
)


total_gt = len(
    e2_c1_d1_gt_df
)


e2_c1_d1_overall_df = pd.DataFrame(
    [
        {
            "c_variant":
                "C1",

            "d_variant":
                "D1",

            "n_images":
                100,

            "total_gt":
                total_gt,

            "both":
                int(
                    overall_counts["BOTH"]
                ),

            "c_only":
                int(
                    overall_counts["C_ONLY"]
                ),

            "d_only":
                int(
                    overall_counts["D_ONLY"]
                ),

            "neither":
                int(
                    overall_counts["NEITHER"]
                ),

            "both_fraction":
                (
                    overall_counts["BOTH"]
                    / total_gt
                ),

            "c_only_fraction":
                (
                    overall_counts["C_ONLY"]
                    / total_gt
                ),

            "d_only_fraction":
                (
                    overall_counts["D_ONLY"]
                    / total_gt
                ),

            "neither_fraction":
                (
                    overall_counts["NEITHER"]
                    / total_gt
                ),

            "c_recovered":
                int(
                    overall_counts["BOTH"]
                    +
                    overall_counts["C_ONLY"]
                ),

            "d_recovered":
                int(
                    overall_counts["BOTH"]
                    +
                    overall_counts["D_ONLY"]
                ),

            "oracle_union_recovered":
                int(
                    total_gt
                    -
                    overall_counts["NEITHER"]
                ),

            "c_micro_recall":
                (
                    (
                        overall_counts["BOTH"]
                        +
                        overall_counts["C_ONLY"]
                    )
                    / total_gt
                ),

            "d_micro_recall":
                (
                    (
                        overall_counts["BOTH"]
                        +
                        overall_counts["D_ONLY"]
                    )
                    / total_gt
                ),

            "oracle_union_micro_recall":
                (
                    (
                        total_gt
                        -
                        overall_counts["NEITHER"]
                    )
                    / total_gt
                ),
        }
    ]
)


# ------------------------------------------------------------
# 7. Density-stratified complementarity
# ------------------------------------------------------------

density_rows = []


for stratum in DENSITY_ORDER:

    sdf = (
        e2_c1_d1_gt_df[
            e2_c1_d1_gt_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    counts = (
        sdf[
            "complementarity_class"
        ]
        .value_counts()
        .reindex(
            E2_GT_CLASSES,
            fill_value=0,
        )
    )


    n_gt_s = len(sdf)

    n_images_s = (
        e2_c1_d1_image_df[
            e2_c1_d1_image_df[
                "density_stratum"
            ]
            ==
            stratum
        ][
            "dataset_index"
        ]
        .nunique()
    )


    both = int(
        counts["BOTH"]
    )

    c_only = int(
        counts["C_ONLY"]
    )

    d_only = int(
        counts["D_ONLY"]
    )

    neither = int(
        counts["NEITHER"]
    )


    density_rows.append(
        {
            "density_stratum":
                stratum,

            "n_images":
                int(
                    n_images_s
                ),

            "total_gt":
                n_gt_s,

            "both":
                both,

            "c_only":
                c_only,

            "d_only":
                d_only,

            "neither":
                neither,

            "both_fraction":
                (
                    both / n_gt_s
                    if n_gt_s > 0
                    else np.nan
                ),

            "c_only_fraction":
                (
                    c_only / n_gt_s
                    if n_gt_s > 0
                    else np.nan
                ),

            "d_only_fraction":
                (
                    d_only / n_gt_s
                    if n_gt_s > 0
                    else np.nan
                ),

            "neither_fraction":
                (
                    neither / n_gt_s
                    if n_gt_s > 0
                    else np.nan
                ),

            "c_micro_recall":
                (
                    (both + c_only)
                    / n_gt_s
                    if n_gt_s > 0
                    else np.nan
                ),

            "d_micro_recall":
                (
                    (both + d_only)
                    / n_gt_s
                    if n_gt_s > 0
                    else np.nan
                ),

            "oracle_union_micro_recall":
                (
                    (
                        both
                        +
                        c_only
                        +
                        d_only
                    )
                    / n_gt_s
                    if n_gt_s > 0
                    else np.nan
                ),
        }
    )


e2_c1_d1_density_df = pd.DataFrame(
    density_rows
)


assert len(
    e2_c1_d1_density_df
) == 6


# ------------------------------------------------------------
# 8. Reproduce frozen marginal localisation results
#
# Important validation:
#
# The C1 and D1 TP totals obtained here should equal the
# corresponding canonical TP totals already stored in the
# frozen Group-E handoff.
# ------------------------------------------------------------

e2_c1_tp_total = int(
    e2_c1_d1_image_df[
        "c_tp"
    ].sum()
)

e2_d1_tp_total = int(
    e2_c1_d1_image_df[
        "d_tp"
    ].sum()
)


# Extract frozen handoff TP totals if canonical_tp exists.
if "canonical_tp" in group_e_handoff_df.columns:

    frozen_c1_tp_total = int(
        group_e_handoff_df.loc[
            group_e_handoff_df[
                "candidate_id"
            ].astype(str)
            ==
            "C1",
            "canonical_tp",
        ].sum()
    )

    frozen_d1_tp_total = int(
        group_e_handoff_df.loc[
            group_e_handoff_df[
                "candidate_id"
            ].astype(str)
            ==
            "D1",
            "canonical_tp",
        ].sum()
    )

    assert (
        e2_c1_tp_total
        ==
        frozen_c1_tp_total
    ), (
        "C1 TP total does not reproduce "
        "the frozen handoff result."
    )

    assert (
        e2_d1_tp_total
        ==
        frozen_d1_tp_total
    ), (
        "D1 TP total does not reproduce "
        "the frozen handoff result."
    )

    marginal_validation = "PASS"

else:

    frozen_c1_tp_total = np.nan
    frozen_d1_tp_total = np.nan

    marginal_validation = (
        "SKIPPED — canonical_tp column "
        "not present in handoff"
    )


# ------------------------------------------------------------
# 9. Additional identities
# ------------------------------------------------------------

overall = (
    e2_c1_d1_overall_df
    .iloc[0]
)


assert (
    int(overall["both"])
    +
    int(overall["c_only"])
    ==
    e2_c1_tp_total
)


assert (
    int(overall["both"])
    +
    int(overall["d_only"])
    ==
    e2_d1_tp_total
)


assert (
    int(overall["both"])
    +
    int(overall["c_only"])
    +
    int(overall["d_only"])
    +
    int(overall["neither"])
    ==
    10045
)


# ------------------------------------------------------------
# 10. Display
# ------------------------------------------------------------

print("=" * 110)
print(
    "E2 — C1 ↔ D1 GT COMPLEMENTARITY BASELINE"
)
print("=" * 110)


print(
    "\nOverall GT-centric complementarity:"
)

display(
    e2_c1_d1_overall_df.round(4)
)


print(
    "\nDensity-stratified GT complementarity:"
)

display(
    e2_c1_d1_density_df.round(4)
)


print(
    "\nGT complementarity class counts:"
)

display(
    pd.DataFrame(
        {
            "class":
                E2_GT_CLASSES,

            "count":
                [
                    int(
                        overall_counts[
                            label
                        ]
                    )
                    for label
                    in E2_GT_CLASSES
                ],

            "fraction_of_GT":
                [
                    float(
                        overall_counts[
                            label
                        ]
                        / total_gt
                    )
                    for label
                    in E2_GT_CLASSES
                ],
        }
    ).round(4)
)


print(
    "\nMarginal localisation validation:"
)

print(
    f"  E2 C1 TP total: "
    f"{e2_c1_tp_total}"
)

print(
    f"  E2 D1 TP total: "
    f"{e2_d1_tp_total}"
)

if (
    marginal_validation
    ==
    "PASS"
):

    print(
        f"  Frozen C1 TP total: "
        f"{frozen_c1_tp_total}"
    )

    print(
        f"  Frozen D1 TP total: "
        f"{frozen_d1_tp_total}"
    )


print(
    f"  Validation: "
    f"{marginal_validation}"
)


print(
    "\nProtocol identities:"
)

print(
    "  BOTH + C_ONLY = C1 recovered GT: PASS"
)

print(
    "  BOTH + D_ONLY = D1 recovered GT: PASS"
)

print(
    "  BOTH + C_ONLY + D_ONLY + NEITHER = GT: PASS"
)


print("\n" + "=" * 110)

print(
    "Step 13C.2 complete."
)

print(
    "C1 ↔ D1 GT recovery overlap and unique recovery "
    "characterised without introducing a fusion rule."
)

print("=" * 110)

E2 — C1 ↔ D1 GT COMPLEMENTARITY BASELINE

Overall GT-centric complementarity:


,c_variant,d_variant,n_images,total_gt,both,c_only,d_only,neither,both_fraction,c_only_fraction,d_only_fraction,neither_fraction,c_recovered,d_recovered,oracle_union_recovered,c_micro_recall,d_micro_recall,oracle_union_micro_recall
0,C1,D1,100,10045,868,424,1547,7206,0.0864,0.0422,0.154,0.7174,1292,2415,2839,0.1286,0.2404,0.2826



Density-stratified GT complementarity:


,density_stratum,n_images,total_gt,both,c_only,d_only,neither,both_fraction,c_only_fraction,d_only_fraction,neither_fraction,c_micro_recall,d_micro_recall,oracle_union_micro_recall
0,S1_0_20,20,195,33,30,56,76,0.1692,0.1538,0.2872,0.3897,0.3231,0.4564,0.6103
1,S2_20_40,20,320,77,17,94,132,0.2406,0.0531,0.2938,0.4125,0.2938,0.5344,0.5875
2,S3_40_60,20,537,70,40,183,244,0.1304,0.0745,0.3408,0.4544,0.2048,0.4711,0.5456
3,S4_60_80,20,1096,247,67,297,485,0.2254,0.0611,0.2710,0.4425,0.2865,0.4964,0.5575
4,S5_80_95,10,1211,226,76,351,558,0.1866,0.0628,0.2898,0.4608,0.2494,0.4765,0.5392
5,S6_95_100,10,6686,215,194,566,5711,0.0322,0.0290,0.0847,0.8542,0.0612,0.1168,0.1458



GT complementarity class counts:


,class,count,fraction_of_GT
0,BOTH,868,0.0864
1,C_ONLY,424,0.0422
2,D_ONLY,1547,0.1540
3,NEITHER,7206,0.7174



Marginal localisation validation:
  E2 C1 TP total: 1292
  E2 D1 TP total: 2415
  Validation: SKIPPED — canonical_tp column not present in handoff

Protocol identities:
  BOTH + C_ONLY = C1 recovered GT: PASS
  BOTH + D_ONLY = D1 recovered GT: PASS
  BOTH + C_ONLY + D_ONLY + NEITHER = GT: PASS

Step 13C.2 complete.
C1 ↔ D1 GT recovery overlap and unique recovery characterised without introducing a fusion rule.


In [56]:
# ============================================================
# Step 13C.3 — C1 ↔ D1 Candidate-Side Disagreement
#              and Burden Analysis
# ============================================================
#
# Purpose
# -------
# Complement the GT-centric analysis from Step 13C.2 with a
# candidate-centric view.
#
# For C1 and D1 separately, quantify:
#
#   - total generated candidates
#   - candidates matched to GT
#   - unmatched candidates
#   - candidate precision / GT-support rate
#
# For D1 specifically, also use the frozen E1 relationship to
# split candidates into:
#
#   - inside C1 foreground
#   - outside C1 foreground
#
# and ask how much GT-supported evidence exists in each group.
#
# This directly investigates the cost of hard C gating:
#
#   D4 = D1 restricted to A3c/C foreground.
#
# IMPORTANT
# ---------
# - No fusion prediction is created.
# - No candidate is removed.
# - GT support is defined ONLY by the frozen canonical
#   one-to-one matcher from Step 13C.1.
# - E1 spatial relationship and E2 GT correctness remain
#   conceptually distinct.
# ============================================================


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_objects = [
    "group_e_handoff_df",
    "E1_REGION_MAPS",
    "e2_parse_points_xy",
    "e2_get_image_size",
    "e2_match_radius_px",
    "e2_match_pred_to_gt",
    "e2_c1_d1_image_df",
    "e2_c1_d1_gt_df",
]

for name in required_objects:
    assert name in globals(), (
        f"{name} not found. "
        "Run E1 and Steps 13C.1–13C.2 first."
    )


# ------------------------------------------------------------
# 2. Select frozen C1 / D1 rows
# ------------------------------------------------------------

c1_df = (
    group_e_handoff_df[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str)
        ==
        "C1"
    ]
    .copy()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


d1_df = (
    group_e_handoff_df[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str)
        ==
        "D1"
    ]
    .copy()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


assert len(c1_df) == 100
assert len(d1_df) == 100


# ------------------------------------------------------------
# 3. Storage
# ------------------------------------------------------------

candidate_rows = []
image_rows = []


# ------------------------------------------------------------
# 4. Per-image candidate analysis
# ------------------------------------------------------------

for i in range(100):

    c_row = c1_df.iloc[i]
    d_row = d1_df.iloc[i]

    dataset_index = int(
        c_row["dataset_index"]
    )

    image_id = str(
        c_row["image_id"]
    )

    density_stratum = str(
        c_row["density_stratum"]
    )


    assert dataset_index == int(
        d_row["dataset_index"]
    )

    assert image_id == str(
        d_row["image_id"]
    )


    # --------------------------------------------------------
    # Parse frozen points
    # --------------------------------------------------------

    gt_points = e2_parse_points_xy(
        c_row["gt_points_xy"]
    )

    c_points = e2_parse_points_xy(
        c_row["pred_points_xy"]
    )

    d_points = e2_parse_points_xy(
        d_row["pred_points_xy"]
    )


    # --------------------------------------------------------
    # Canonical tolerance
    # --------------------------------------------------------

    width, height = e2_get_image_size(
        c_row
    )

    radius_px = e2_match_radius_px(
        image_width=width,
        image_height=height,
    )


    # --------------------------------------------------------
    # Frozen independent matching
    # --------------------------------------------------------

    c_match = e2_match_pred_to_gt(
        pred_points_xy=c_points,
        gt_points_xy=gt_points,
        radius_px=radius_px,
    )

    d_match = e2_match_pred_to_gt(
        pred_points_xy=d_points,
        gt_points_xy=gt_points,
        radius_px=radius_px,
    )


    c_matched_pred = (
        c_match[
            "matched_pred_indices"
        ]
    )

    d_matched_pred = (
        d_match[
            "matched_pred_indices"
        ]
    )


    # --------------------------------------------------------
    # Recover frozen C1 region label map
    #
    # C1 foreground:
    #     label > 0
    #
    # Background:
    #     label == 0
    # --------------------------------------------------------

    region_item = (
        E1_REGION_MAPS[
            dataset_index
        ]
    )

    c1_labels = np.asarray(
        region_item[
            "C1"
        ][
            "labelled_regions"
        ]
    )


    assert c1_labels.shape == (
        height,
        width,
    ), (
        f"Region-map shape mismatch for "
        f"{image_id}: "
        f"{c1_labels.shape} vs "
        f"{(height, width)}"
    )


    # --------------------------------------------------------
    # C1 candidate rows
    #
    # C1 points are frozen representative points (centroids)
    # derived from C1 connected-component regions.
    #
    # IMPORTANT:
    # A geometric centroid of an irregular / concave region
    # is not guaranteed to lie on a foreground pixel.
    #
    # Therefore we do NOT test C1 reference points by looking
    # up the centroid pixel in the C1 label map.
    #
    # For C1, the candidate-side question here is simply:
    #
    #     GT matched vs unmatched
    #
    # The inside/outside-C1 analysis is meaningful for D1,
    # because D1 is generated independently from the C regions.
    # --------------------------------------------------------

    for pred_idx, point in enumerate(
        c_points
    ):

        x = float(point[0])
        y = float(point[1])

        gt_matched = (
            pred_idx
            in
            c_matched_pred
        )

        candidate_rows.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "density_stratum":
                    density_stratum,

                "branch":
                    "C1",

                "candidate_index":
                    int(pred_idx),

                "x":
                    x,

                "y":
                    y,

                # Not applicable to C1 itself.
                # C1 candidates originate from C1 regions.
                "inside_c1_foreground":
                    np.nan,

                "c1_region_label":
                    np.nan,

                "gt_matched":
                    bool(gt_matched),

                "radius_px":
                    int(radius_px),
            }
        )


    # --------------------------------------------------------
    # D1 candidate rows
    #
    # This is the important E1 ↔ E2 bridge:
    #
    #   spatial relationship:
    #       inside / outside C1 foreground
    #
    #   GT correctness:
    #       matched / unmatched
    # --------------------------------------------------------

    d_inside = 0
    d_outside = 0

    d_inside_matched = 0
    d_inside_unmatched = 0

    d_outside_matched = 0
    d_outside_unmatched = 0


    for pred_idx, point in enumerate(
        d_points
    ):

        x = float(point[0])
        y = float(point[1])

        xi = int(
            np.clip(
                np.rint(x),
                0,
                width - 1,
            )
        )

        yi = int(
            np.clip(
                np.rint(y),
                0,
                height - 1,
            )
        )

        region_label = int(
            c1_labels[
                yi,
                xi,
            ]
        )

        inside_c1 = (
            region_label > 0
        )

        gt_matched = (
            pred_idx
            in
            d_matched_pred
        )


        if inside_c1:

            d_inside += 1

            if gt_matched:
                d_inside_matched += 1
            else:
                d_inside_unmatched += 1

        else:

            d_outside += 1

            if gt_matched:
                d_outside_matched += 1
            else:
                d_outside_unmatched += 1


        candidate_rows.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "density_stratum":
                    density_stratum,

                "branch":
                    "D1",

                "candidate_index":
                    int(pred_idx),

                "x":
                    x,

                "y":
                    y,

                "inside_c1_foreground":
                    bool(inside_c1),

                "c1_region_label":
                    region_label,

                "gt_matched":
                    bool(gt_matched),

                "radius_px":
                    int(radius_px),
            }
        )


    # --------------------------------------------------------
    # Per-image identities
    # --------------------------------------------------------

    assert (
        d_inside
        +
        d_outside
        ==
        len(d_points)
    )

    assert (
        d_inside_matched
        +
        d_inside_unmatched
        ==
        d_inside
    )

    assert (
        d_outside_matched
        +
        d_outside_unmatched
        ==
        d_outside
    )

    assert (
        d_inside_matched
        +
        d_outside_matched
        ==
        d_match["tp"]
    )


    # --------------------------------------------------------
    # Per-image summary
    # --------------------------------------------------------

    image_rows.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "density_stratum":
                density_stratum,

            "gt_count":
                len(gt_points),

            "n_c1_candidates":
                len(c_points),

            "c1_matched":
                int(
                    c_match["tp"]
                ),

            "c1_unmatched":
                int(
                    c_match["fp"]
                ),

            "n_d1_candidates":
                len(d_points),

            "d1_matched":
                int(
                    d_match["tp"]
                ),

            "d1_unmatched":
                int(
                    d_match["fp"]
                ),

            "d1_inside_c1":
                int(d_inside),

            "d1_outside_c1":
                int(d_outside),

            "d1_inside_matched":
                int(
                    d_inside_matched
                ),

            "d1_inside_unmatched":
                int(
                    d_inside_unmatched
                ),

            "d1_outside_matched":
                int(
                    d_outside_matched
                ),

            "d1_outside_unmatched":
                int(
                    d_outside_unmatched
                ),
        }
    )


# ------------------------------------------------------------
# 5. Canonical candidate-side dataframes
# ------------------------------------------------------------

e2_c1_d1_candidate_df = pd.DataFrame(
    candidate_rows
)

e2_c1_d1_candidate_image_df = pd.DataFrame(
    image_rows
)


assert len(
    e2_c1_d1_candidate_image_df
) == 100


# ------------------------------------------------------------
# 6. Global branch-level summary
# ------------------------------------------------------------

branch_rows = []


for branch in [
    "C1",
    "D1",
]:

    sdf = (
        e2_c1_d1_candidate_df[
            e2_c1_d1_candidate_df[
                "branch"
            ]
            ==
            branch
        ]
    )

    total = len(sdf)

    matched = int(
        sdf[
            "gt_matched"
        ].sum()
    )

    unmatched = (
        total
        -
        matched
    )


    branch_rows.append(
        {
            "branch":
                branch,

            "total_candidates":
                total,

            "gt_matched_candidates":
                matched,

            "unmatched_candidates":
                unmatched,

            "candidate_gt_support_rate":
                (
                    matched
                    / total
                    if total > 0
                    else np.nan
                ),

            "unmatched_fraction":
                (
                    unmatched
                    / total
                    if total > 0
                    else np.nan
                ),
        }
    )


e2_c1_d1_branch_summary_df = pd.DataFrame(
    branch_rows
)


# ------------------------------------------------------------
# 7. D1 inside/outside C1 foreground × GT support
# ------------------------------------------------------------

d1_candidate_df = (
    e2_c1_d1_candidate_df[
        e2_c1_d1_candidate_df[
            "branch"
        ]
        ==
        "D1"
    ]
    .copy()
)


d1_relation_rows = []


for relation_name, relation_value in [
    (
        "INSIDE_C1_FOREGROUND",
        True,
    ),
    (
        "OUTSIDE_C1_FOREGROUND",
        False,
    ),
]:

    sdf = (
        d1_candidate_df[
            d1_candidate_df[
                "inside_c1_foreground"
            ]
            ==
            relation_value
        ]
    )

    total = len(sdf)

    matched = int(
        sdf[
            "gt_matched"
        ].sum()
    )

    unmatched = (
        total
        -
        matched
    )


    d1_relation_rows.append(
        {
            "relationship":
                relation_name,

            "total_d1_candidates":
                total,

            "gt_matched":
                matched,

            "unmatched":
                unmatched,

            "gt_support_rate":
                (
                    matched
                    / total
                    if total > 0
                    else np.nan
                ),

            "fraction_of_all_d1":
                (
                    total
                    / len(d1_candidate_df)
                    if len(d1_candidate_df) > 0
                    else np.nan
                ),

            "fraction_of_all_d1_tp":
                (
                    matched
                    /
                    int(
                        d1_candidate_df[
                            "gt_matched"
                        ].sum()
                    )
                    if int(
                        d1_candidate_df[
                            "gt_matched"
                        ].sum()
                    ) > 0
                    else np.nan
                ),
        }
    )


e2_d1_inside_outside_df = pd.DataFrame(
    d1_relation_rows
)


# ------------------------------------------------------------
# 8. 2×2 contingency table for D1
#
# Rows:
#     inside / outside C1 foreground
#
# Columns:
#     GT matched / unmatched
# ------------------------------------------------------------

e2_d1_contingency_df = pd.crosstab(
    d1_candidate_df[
        "inside_c1_foreground"
    ].map(
        {
            True:
                "INSIDE_C1",

            False:
                "OUTSIDE_C1",
        }
    ),

    d1_candidate_df[
        "gt_matched"
    ].map(
        {
            True:
                "GT_MATCHED",

            False:
                "UNMATCHED",
        }
    ),
)


# ------------------------------------------------------------
# 9. Density-stratified D1 inside/outside analysis
# ------------------------------------------------------------

density_rows = []


for stratum in DENSITY_ORDER:

    sdf = (
        d1_candidate_df[
            d1_candidate_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    inside = (
        sdf[
            "inside_c1_foreground"
        ]
    )

    matched = (
        sdf[
            "gt_matched"
        ]
    )


    n_total = len(sdf)

    n_inside = int(
        inside.sum()
    )

    n_outside = (
        n_total
        -
        n_inside
    )

    n_inside_matched = int(
        (
            inside
            &
            matched
        ).sum()
    )

    n_outside_matched = int(
        (
            (~inside)
            &
            matched
        ).sum()
    )


    density_rows.append(
        {
            "density_stratum":
                stratum,

            "n_d1_candidates":
                n_total,

            "inside_c1":
                n_inside,

            "outside_c1":
                n_outside,

            "inside_fraction":
                (
                    n_inside
                    / n_total
                    if n_total > 0
                    else np.nan
                ),

            "inside_gt_matched":
                n_inside_matched,

            "outside_gt_matched":
                n_outside_matched,

            "inside_gt_support_rate":
                (
                    n_inside_matched
                    / n_inside
                    if n_inside > 0
                    else np.nan
                ),

            "outside_gt_support_rate":
                (
                    n_outside_matched
                    / n_outside
                    if n_outside > 0
                    else np.nan
                ),

            "fraction_of_d1_tp_outside":
                (
                    n_outside_matched
                    /
                    (
                        n_inside_matched
                        +
                        n_outside_matched
                    )
                    if (
                        n_inside_matched
                        +
                        n_outside_matched
                    ) > 0
                    else np.nan
                ),
        }
    )


e2_d1_inside_outside_density_df = (
    pd.DataFrame(
        density_rows
    )
)


# ------------------------------------------------------------
# 10. Critical E1 consistency checks
#
# E1 C1↔D1 found:
#
#   total D1 candidates = 7652
#   inside C1           = 4194
#   outside C1          = 3458
#
# These must reproduce exactly.
# ------------------------------------------------------------

assert len(
    d1_candidate_df
) == 7652


total_inside = int(
    d1_candidate_df[
        "inside_c1_foreground"
    ].sum()
)

total_outside = (
    len(d1_candidate_df)
    -
    total_inside
)


assert total_inside == 4194, (
    f"E1 consistency failure: "
    f"expected 4194 inside D1 points, "
    f"got {total_inside}."
)

assert total_outside == 3458, (
    f"E1 consistency failure: "
    f"expected 3458 outside D1 points, "
    f"got {total_outside}."
)


# ------------------------------------------------------------
# 11. Critical E2 consistency checks
#
# From 13C.2:
#
#   C1 recovered GT = 1292
#   D1 recovered GT = 2415
#
# Since matching is one-to-one, the number of matched
# candidates must equal those TP totals.
# ------------------------------------------------------------

c1_candidate_tp = int(
    e2_c1_d1_candidate_df.loc[
        e2_c1_d1_candidate_df[
            "branch"
        ]
        ==
        "C1",
        "gt_matched",
    ].sum()
)

d1_candidate_tp = int(
    e2_c1_d1_candidate_df.loc[
        e2_c1_d1_candidate_df[
            "branch"
        ]
        ==
        "D1",
        "gt_matched",
    ].sum()
)


assert c1_candidate_tp == 1292, (
    f"Expected C1 TP=1292, "
    f"got {c1_candidate_tp}."
)

assert d1_candidate_tp == 2415, (
    f"Expected D1 TP=2415, "
    f"got {d1_candidate_tp}."
)


# ------------------------------------------------------------
# 12. Additional identities
# ------------------------------------------------------------

assert (
    int(
        e2_c1_d1_candidate_image_df[
            "d1_inside_matched"
        ].sum()
    )
    +
    int(
        e2_c1_d1_candidate_image_df[
            "d1_outside_matched"
        ].sum()
    )
    ==
    2415
)


assert (
    int(
        e2_c1_d1_candidate_image_df[
            "d1_inside_unmatched"
        ].sum()
    )
    +
    int(
        e2_c1_d1_candidate_image_df[
            "d1_outside_unmatched"
        ].sum()
    )
    ==
    7652 - 2415
)


# ------------------------------------------------------------
# 13. Display
# ------------------------------------------------------------

print("=" * 110)
print(
    "E2 — C1 ↔ D1 CANDIDATE-SIDE DISAGREEMENT "
    "AND BURDEN ANALYSIS"
)
print("=" * 110)


print(
    "\nA. Branch-level candidate burden:"
)

display(
    e2_c1_d1_branch_summary_df.round(4)
)


print(
    "\nB. D1 relationship to C1 foreground × GT support:"
)

display(
    e2_d1_inside_outside_df.round(4)
)


print(
    "\nC. D1 2×2 contingency table:"
)

display(
    e2_d1_contingency_df
)


print(
    "\nD. Density-stratified D1 inside/outside GT support:"
)

display(
    e2_d1_inside_outside_density_df.round(4)
)


print(
    "\nConsistency checks:"
)

print(
    f"  D1 candidates: "
    f"{len(d1_candidate_df)}"
)

print(
    f"  D1 inside C1 foreground: "
    f"{total_inside}"
)

print(
    f"  D1 outside C1 foreground: "
    f"{total_outside}"
)

print(
    f"  C1 GT-matched candidates: "
    f"{c1_candidate_tp}"
)

print(
    f"  D1 GT-matched candidates: "
    f"{d1_candidate_tp}"
)

print(
    "  E1 inside/outside totals reproduced: PASS"
)

print(
    "  E2 marginal TP totals reproduced: PASS"
)


print("\n" + "=" * 110)

print(
    "Step 13C.3 complete."
)

print(
    "Candidate-side GT support and disagreement burden "
    "characterised without introducing a fusion rule."
)

print("=" * 110)

E2 — C1 ↔ D1 CANDIDATE-SIDE DISAGREEMENT AND BURDEN ANALYSIS

A. Branch-level candidate burden:


,branch,total_candidates,gt_matched_candidates,unmatched_candidates,candidate_gt_support_rate,unmatched_fraction
0,C1,2554,1292,1262,0.5059,0.4941
1,D1,7652,2415,5237,0.3156,0.6844



B. D1 relationship to C1 foreground × GT support:


,relationship,total_d1_candidates,gt_matched,unmatched,gt_support_rate,fraction_of_all_d1,fraction_of_all_d1_tp
0,INSIDE_C1_FOREGROUND,4194,1605,2589,0.3827,0.5481,0.6646
1,OUTSIDE_C1_FOREGROUND,3458,810,2648,0.2342,0.4519,0.3354



C. D1 2×2 contingency table:


gt_matched,GT_MATCHED,UNMATCHED
inside_c1_foreground,,
INSIDE_C1,1605,2589
OUTSIDE_C1,810,2648



D. Density-stratified D1 inside/outside GT support:


,density_stratum,n_d1_candidates,inside_c1,outside_c1,inside_fraction,inside_gt_matched,outside_gt_matched,inside_gt_support_rate,outside_gt_support_rate,fraction_of_d1_tp_outside
0,S1_0_20,1082,624,458,0.5767,77,12,0.1234,0.0262,0.1348
1,S2_20_40,1233,698,535,0.5661,142,29,0.2034,0.0542,0.1696
2,S3_40_60,1390,755,635,0.5432,207,46,0.2742,0.0724,0.1818
3,S4_60_80,1663,949,714,0.5707,425,119,0.4478,0.1667,0.2188
4,S5_80_95,1143,612,531,0.5354,347,230,0.5670,0.4331,0.3986
5,S6_95_100,1141,556,585,0.4873,407,374,0.7320,0.6393,0.4789



Consistency checks:
  D1 candidates: 7652
  D1 inside C1 foreground: 4194
  D1 outside C1 foreground: 3458
  C1 GT-matched candidates: 1292
  D1 GT-matched candidates: 2415
  E1 inside/outside totals reproduced: PASS
  E2 marginal TP totals reproduced: PASS

Step 13C.3 complete.
Candidate-side GT support and disagreement burden characterised without introducing a fusion rule.


In [57]:
# ============================================================
# Step 13C.4 — Extend GT Complementarity Analysis to All
#              Retained C×D Pairs
# ============================================================
#
# Purpose
# -------
# Generalise the validated C1↔D1 GT-centric analysis from
# Step 13C.2 to all frozen:
#
#     3 C variants × 5 D variants = 15 C×D pairs
#
# For every image and every pair:
#
#   1. Match C independently to GT
#   2. Match D independently to GT
#   3. Classify every GT object as:
#
#        BOTH
#        C_ONLY
#        D_ONLY
#        NEITHER
#
# Report:
#
#   - overall GT complementarity
#   - density-stratified complementarity
#   - marginal C/D micro recall
#   - oracle-union micro recall
#   - unique recovery contributed by each branch
#
# IMPORTANT
# ---------
# - Uses frozen candidates only.
# - Uses canonical 3%-short-side one-to-one matching.
# - No fusion prediction is created.
# - Oracle union is diagnostic only.
# - Density strata are analysis strata, NOT inference rules.
# ============================================================


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_objects = [
    "group_e_handoff_df",
    "E2_C_VARIANTS",
    "E2_D_VARIANTS",
    "E2_GT_CLASSES",
    "DENSITY_ORDER",
    "e2_parse_points_xy",
    "e2_get_image_size",
    "e2_match_radius_px",
    "e2_match_pred_to_gt",
    "e2_classify_gt_complementarity",
    "e2_c1_d1_overall_df",
]

for name in required_objects:
    assert name in globals(), (
        f"{name} not found. "
        "Run Steps 13C.1–13C.3 first."
    )


assert E2_C_VARIANTS == [
    "C1",
    "C2b",
    "C2c",
]

assert E2_D_VARIANTS == [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]


# ------------------------------------------------------------
# 2. Build frozen candidate lookup
#
# Key:
#     (candidate_id, dataset_index)
#
# This avoids repeatedly filtering the 800-row handoff.
# ------------------------------------------------------------

e2_candidate_lookup = {}

for _, row in group_e_handoff_df.iterrows():

    key = (
        str(row["candidate_id"]),
        int(row["dataset_index"]),
    )

    assert key not in e2_candidate_lookup, (
        f"Duplicate candidate/image key: {key}"
    )

    e2_candidate_lookup[key] = row


assert len(e2_candidate_lookup) == 800


# ------------------------------------------------------------
# 3. Frozen dev100 image inventory
# ------------------------------------------------------------

e2_dataset_indices = sorted(
    group_e_handoff_df[
        "dataset_index"
    ]
    .astype(int)
    .unique()
)

assert len(e2_dataset_indices) == 100


# ------------------------------------------------------------
# 4. Storage
# ------------------------------------------------------------

pair_image_rows = []
pair_gt_rows = []


# ------------------------------------------------------------
# 5. Analyse all 15 C×D pairs
# ------------------------------------------------------------

for c_variant in E2_C_VARIANTS:

    for d_variant in E2_D_VARIANTS:

        print(
            f"Analysing "
            f"{c_variant} ↔ {d_variant} ..."
        )

        for dataset_index in e2_dataset_indices:

            c_row = e2_candidate_lookup[
                (
                    c_variant,
                    dataset_index,
                )
            ]

            d_row = e2_candidate_lookup[
                (
                    d_variant,
                    dataset_index,
                )
            ]


            # ----------------------------------------------
            # Common metadata
            # ----------------------------------------------

            image_id = str(
                c_row["image_id"]
            )

            density_stratum = str(
                c_row["density_stratum"]
            )


            assert image_id == str(
                d_row["image_id"]
            )

            assert density_stratum == str(
                d_row["density_stratum"]
            )

            assert int(
                c_row["gt_count"]
            ) == int(
                d_row["gt_count"]
            )


            # ----------------------------------------------
            # Parse GT and candidate points
            # ----------------------------------------------

            gt_points = e2_parse_points_xy(
                c_row["gt_points_xy"]
            )

            c_points = e2_parse_points_xy(
                c_row["pred_points_xy"]
            )

            d_points = e2_parse_points_xy(
                d_row["pred_points_xy"]
            )


            n_gt = len(gt_points)

            assert n_gt == int(
                c_row["gt_count"]
            )


            # ----------------------------------------------
            # Canonical radius
            # ----------------------------------------------

            width, height = e2_get_image_size(
                c_row
            )

            radius_px = e2_match_radius_px(
                image_width=width,
                image_height=height,
            )


            # ----------------------------------------------
            # Independent C → GT matching
            # ----------------------------------------------

            c_match = e2_match_pred_to_gt(
                pred_points_xy=c_points,
                gt_points_xy=gt_points,
                radius_px=radius_px,
            )


            # ----------------------------------------------
            # Independent D → GT matching
            # ----------------------------------------------

            d_match = e2_match_pred_to_gt(
                pred_points_xy=d_points,
                gt_points_xy=gt_points,
                radius_px=radius_px,
            )


            c_gt_set = (
                c_match[
                    "matched_gt_indices"
                ]
            )

            d_gt_set = (
                d_match[
                    "matched_gt_indices"
                ]
            )


            # ----------------------------------------------
            # Four-way GT classification
            # ----------------------------------------------

            gt_classes = (
                e2_classify_gt_complementarity(
                    n_gt=n_gt,
                    c_matched_gt_indices=c_gt_set,
                    d_matched_gt_indices=d_gt_set,
                )
            )


            class_counts = {
                label: int(
                    np.sum(
                        gt_classes == label
                    )
                )
                for label in E2_GT_CLASSES
            }


            n_both = (
                class_counts["BOTH"]
            )

            n_c_only = (
                class_counts["C_ONLY"]
            )

            n_d_only = (
                class_counts["D_ONLY"]
            )

            n_neither = (
                class_counts["NEITHER"]
            )


            # ----------------------------------------------
            # Required identities
            # ----------------------------------------------

            assert (
                n_both
                + n_c_only
                + n_d_only
                + n_neither
                ==
                n_gt
            )

            assert (
                n_both
                + n_c_only
                ==
                c_match["tp"]
            )

            assert (
                n_both
                + n_d_only
                ==
                d_match["tp"]
            )


            oracle_union_tp = (
                n_both
                + n_c_only
                + n_d_only
            )


            assert oracle_union_tp == len(
                c_gt_set | d_gt_set
            )


            # ----------------------------------------------
            # Per-image result
            # ----------------------------------------------

            pair_image_rows.append(
                {
                    "c_variant":
                        c_variant,

                    "d_variant":
                        d_variant,

                    "dataset_index":
                        int(dataset_index),

                    "image_id":
                        image_id,

                    "density_stratum":
                        density_stratum,

                    "gt_count":
                        n_gt,

                    "radius_px":
                        int(radius_px),

                    "n_c_candidates":
                        len(c_points),

                    "n_d_candidates":
                        len(d_points),

                    "c_tp":
                        int(c_match["tp"]),

                    "c_fp":
                        int(c_match["fp"]),

                    "c_fn":
                        int(c_match["fn"]),

                    "d_tp":
                        int(d_match["tp"]),

                    "d_fp":
                        int(d_match["fp"]),

                    "d_fn":
                        int(d_match["fn"]),

                    "both":
                        n_both,

                    "c_only":
                        n_c_only,

                    "d_only":
                        n_d_only,

                    "neither":
                        n_neither,

                    "oracle_union_tp":
                        oracle_union_tp,
                }
            )


            # ----------------------------------------------
            # Per-GT result
            # ----------------------------------------------

            for gt_idx, (
                gt_point,
                gt_class,
            ) in enumerate(
                zip(
                    gt_points,
                    gt_classes,
                )
            ):

                pair_gt_rows.append(
                    {
                        "c_variant":
                            c_variant,

                        "d_variant":
                            d_variant,

                        "dataset_index":
                            int(dataset_index),

                        "image_id":
                            image_id,

                        "density_stratum":
                            density_stratum,

                        "gt_index":
                            int(gt_idx),

                        "gt_x":
                            float(
                                gt_point[0]
                            ),

                        "gt_y":
                            float(
                                gt_point[1]
                            ),

                        "c_recovered":
                            bool(
                                gt_idx
                                in
                                c_gt_set
                            ),

                        "d_recovered":
                            bool(
                                gt_idx
                                in
                                d_gt_set
                            ),

                        "complementarity_class":
                            str(gt_class),

                        "radius_px":
                            int(radius_px),
                    }
                )


# ------------------------------------------------------------
# 6. Canonical all-pair dataframes
# ------------------------------------------------------------

e2_all_pairs_image_df = pd.DataFrame(
    pair_image_rows
)

e2_all_pairs_gt_df = pd.DataFrame(
    pair_gt_rows
)


# 15 pairs × 100 images
assert len(
    e2_all_pairs_image_df
) == 1500


# Every pair sees the same 10,045 GT objects
assert len(
    e2_all_pairs_gt_df
) == (
    15 * 10045
)


assert (
    e2_all_pairs_image_df[
        [
            "c_variant",
            "d_variant",
        ]
    ]
    .drop_duplicates()
    .shape[0]
    ==
    15
)


# ------------------------------------------------------------
# 7. Overall summary for all 15 pairs
# ------------------------------------------------------------

overall_rows = []


for c_variant in E2_C_VARIANTS:

    for d_variant in E2_D_VARIANTS:

        sdf = (
            e2_all_pairs_gt_df[
                (
                    e2_all_pairs_gt_df[
                        "c_variant"
                    ]
                    ==
                    c_variant
                )
                &
                (
                    e2_all_pairs_gt_df[
                        "d_variant"
                    ]
                    ==
                    d_variant
                )
            ]
        )


        counts = (
            sdf[
                "complementarity_class"
            ]
            .value_counts()
            .reindex(
                E2_GT_CLASSES,
                fill_value=0,
            )
        )


        total_gt = len(sdf)

        assert total_gt == 10045


        both = int(
            counts["BOTH"]
        )

        c_only = int(
            counts["C_ONLY"]
        )

        d_only = int(
            counts["D_ONLY"]
        )

        neither = int(
            counts["NEITHER"]
        )


        c_recovered = (
            both
            +
            c_only
        )

        d_recovered = (
            both
            +
            d_only
        )

        oracle_union = (
            both
            +
            c_only
            +
            d_only
        )


        overall_rows.append(
            {
                "c_variant":
                    c_variant,

                "d_variant":
                    d_variant,

                "total_gt":
                    total_gt,

                "both":
                    both,

                "c_only":
                    c_only,

                "d_only":
                    d_only,

                "neither":
                    neither,

                "both_fraction":
                    both / total_gt,

                "c_only_fraction":
                    c_only / total_gt,

                "d_only_fraction":
                    d_only / total_gt,

                "neither_fraction":
                    neither / total_gt,

                "c_recovered":
                    c_recovered,

                "d_recovered":
                    d_recovered,

                "oracle_union_recovered":
                    oracle_union,

                "c_micro_recall":
                    c_recovered / total_gt,

                "d_micro_recall":
                    d_recovered / total_gt,

                "oracle_union_micro_recall":
                    oracle_union / total_gt,

                # Additional GT coverage available from D
                # beyond C.
                "d_unique_gain_over_c":
                    d_only / total_gt,

                # Additional GT coverage available from C
                # beyond D.
                "c_unique_gain_over_d":
                    c_only / total_gt,

                # Diagnostic oracle gain over the stronger
                # marginal branch.
                "oracle_gain_over_best_branch":
                    (
                        oracle_union
                        -
                        max(
                            c_recovered,
                            d_recovered,
                        )
                    )
                    / total_gt,
            }
        )


e2_all_pairs_overall_df = pd.DataFrame(
    overall_rows
)


assert len(
    e2_all_pairs_overall_df
) == 15


# ------------------------------------------------------------
# 8. Density-stratified summary
#
# 15 pairs × 6 strata = 90 rows
# ------------------------------------------------------------

density_rows = []


for c_variant in E2_C_VARIANTS:

    for d_variant in E2_D_VARIANTS:

        for stratum in DENSITY_ORDER:

            sdf = (
                e2_all_pairs_gt_df[
                    (
                        e2_all_pairs_gt_df[
                            "c_variant"
                        ]
                        ==
                        c_variant
                    )
                    &
                    (
                        e2_all_pairs_gt_df[
                            "d_variant"
                        ]
                        ==
                        d_variant
                    )
                    &
                    (
                        e2_all_pairs_gt_df[
                            "density_stratum"
                        ]
                        ==
                        stratum
                    )
                ]
            )


            counts = (
                sdf[
                    "complementarity_class"
                ]
                .value_counts()
                .reindex(
                    E2_GT_CLASSES,
                    fill_value=0,
                )
            )


            n_gt = len(sdf)

            both = int(
                counts["BOTH"]
            )

            c_only = int(
                counts["C_ONLY"]
            )

            d_only = int(
                counts["D_ONLY"]
            )

            neither = int(
                counts["NEITHER"]
            )


            c_recovered = (
                both
                +
                c_only
            )

            d_recovered = (
                both
                +
                d_only
            )

            oracle_union = (
                both
                +
                c_only
                +
                d_only
            )


            n_images = (
                e2_all_pairs_image_df[
                    (
                        e2_all_pairs_image_df[
                            "c_variant"
                        ]
                        ==
                        c_variant
                    )
                    &
                    (
                        e2_all_pairs_image_df[
                            "d_variant"
                        ]
                        ==
                        d_variant
                    )
                    &
                    (
                        e2_all_pairs_image_df[
                            "density_stratum"
                        ]
                        ==
                        stratum
                    )
                ][
                    "dataset_index"
                ]
                .nunique()
            )


            density_rows.append(
                {
                    "c_variant":
                        c_variant,

                    "d_variant":
                        d_variant,

                    "density_stratum":
                        stratum,

                    "n_images":
                        int(n_images),

                    "total_gt":
                        n_gt,

                    "both":
                        both,

                    "c_only":
                        c_only,

                    "d_only":
                        d_only,

                    "neither":
                        neither,

                    "both_fraction":
                        (
                            both / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "c_only_fraction":
                        (
                            c_only / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "d_only_fraction":
                        (
                            d_only / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "neither_fraction":
                        (
                            neither / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "c_micro_recall":
                        (
                            c_recovered / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "d_micro_recall":
                        (
                            d_recovered / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "oracle_union_micro_recall":
                        (
                            oracle_union / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "d_unique_gain_over_c":
                        (
                            d_only / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "c_unique_gain_over_d":
                        (
                            c_only / n_gt
                            if n_gt > 0
                            else np.nan
                        ),

                    "oracle_gain_over_best_branch":
                        (
                            (
                                oracle_union
                                -
                                max(
                                    c_recovered,
                                    d_recovered,
                                )
                            )
                            / n_gt
                            if n_gt > 0
                            else np.nan
                        ),
                }
            )


e2_all_pairs_density_df = pd.DataFrame(
    density_rows
)


assert len(
    e2_all_pairs_density_df
) == 90


# ------------------------------------------------------------
# 9. Internal consistency:
#    marginal recovery must not depend on partner branch
#
# Example:
#   C1 TP must be identical whether paired with
#   D1, D4, D5, D6...
#
# Likewise for each D representation across C1/C2b/C2c.
# ------------------------------------------------------------

c_marginal_check = (
    e2_all_pairs_overall_df
    .groupby("c_variant")[
        "c_recovered"
    ]
    .nunique()
)

d_marginal_check = (
    e2_all_pairs_overall_df
    .groupby("d_variant")[
        "d_recovered"
    ]
    .nunique()
)


assert (
    c_marginal_check == 1
).all(), (
    "C marginal recovery changed with D partner."
)

assert (
    d_marginal_check == 1
).all(), (
    "D marginal recovery changed with C partner."
)


# ------------------------------------------------------------
# 10. Reproduce 13C.2 C1↔D1 exactly
# ------------------------------------------------------------

baseline_all = (
    e2_all_pairs_overall_df[
        (
            e2_all_pairs_overall_df[
                "c_variant"
            ]
            ==
            "C1"
        )
        &
        (
            e2_all_pairs_overall_df[
                "d_variant"
            ]
            ==
            "D1"
        )
    ]
    .iloc[0]
)


baseline_13c2 = (
    e2_c1_d1_overall_df
    .iloc[0]
)


for column in [
    "total_gt",
    "both",
    "c_only",
    "d_only",
    "neither",
    "c_recovered",
    "d_recovered",
    "oracle_union_recovered",
]:

    assert int(
        baseline_all[column]
    ) == int(
        baseline_13c2[column]
    ), (
        f"13C.2 reproduction failed for "
        f"{column}: "
        f"{baseline_all[column]} vs "
        f"{baseline_13c2[column]}"
    )


# ------------------------------------------------------------
# 11. Global identity checks for every pair
# ------------------------------------------------------------

for _, row in (
    e2_all_pairs_overall_df
    .iterrows()
):

    assert (
        int(row["both"])
        +
        int(row["c_only"])
        +
        int(row["d_only"])
        +
        int(row["neither"])
        ==
        int(row["total_gt"])
    )

    assert (
        int(row["both"])
        +
        int(row["c_only"])
        ==
        int(row["c_recovered"])
    )

    assert (
        int(row["both"])
        +
        int(row["d_only"])
        ==
        int(row["d_recovered"])
    )

    assert (
        int(row["both"])
        +
        int(row["c_only"])
        +
        int(row["d_only"])
        ==
        int(
            row[
                "oracle_union_recovered"
            ]
        )
    )


# ------------------------------------------------------------
# 12. Compact comparison table
#
# Do NOT rank/select a fusion strategy yet.
# This is descriptive E2 evidence.
# ------------------------------------------------------------

e2_all_pairs_compact_df = (
    e2_all_pairs_overall_df[
        [
            "c_variant",
            "d_variant",
            "both",
            "c_only",
            "d_only",
            "neither",
            "c_micro_recall",
            "d_micro_recall",
            "oracle_union_micro_recall",
            "c_unique_gain_over_d",
            "d_unique_gain_over_c",
            "oracle_gain_over_best_branch",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 13. Display
# ------------------------------------------------------------

print("=" * 120)
print(
    "E2 — GT COMPLEMENTARITY ACROSS ALL "
    "15 C×D PAIRS"
)
print("=" * 120)


print(
    "\nA. Overall GT complementarity:"
)

display(
    e2_all_pairs_compact_df.round(4)
)


print(
    "\nB. Density-stratified GT complementarity "
    "(90 rows = 15 pairs × 6 strata):"
)

display(
    e2_all_pairs_density_df[
        [
            "c_variant",
            "d_variant",
            "density_stratum",
            "n_images",
            "total_gt",
            "both",
            "c_only",
            "d_only",
            "neither",
            "c_micro_recall",
            "d_micro_recall",
            "oracle_union_micro_recall",
            "c_unique_gain_over_d",
            "d_unique_gain_over_c",
            "oracle_gain_over_best_branch",
        ]
    ].round(4)
)


print(
    "\nConsistency checks:"
)

print(
    f"  C×D pairs: "
    f"{len(e2_all_pairs_overall_df)} / 15"
)

print(
    f"  Pair-image rows: "
    f"{len(e2_all_pairs_image_df)} / 1500"
)

print(
    f"  Pair-GT rows: "
    f"{len(e2_all_pairs_gt_df)} / "
    f"{15 * 10045}"
)

print(
    "  C marginal recovery invariant "
    "across D partners: PASS"
)

print(
    "  D marginal recovery invariant "
    "across C partners: PASS"
)

print(
    "  C1↔D1 reproduces Step 13C.2: PASS"
)

print(
    "  All GT complementarity identities: PASS"
)


print("\n" + "=" * 120)

print(
    "Step 13C.4 complete."
)

print(
    "GT recovery overlap and unique recovery characterised "
    "for all 15 frozen C×D pairs without introducing "
    "a fusion rule."
)

print("=" * 120)

Analysing C1 ↔ D1 ...
Analysing C1 ↔ D4 ...
Analysing C1 ↔ D5-768 ...
Analysing C1 ↔ D6B-T-native ...
Analysing C1 ↔ D6B9-strict ...
Analysing C2b ↔ D1 ...
Analysing C2b ↔ D4 ...
Analysing C2b ↔ D5-768 ...
Analysing C2b ↔ D6B-T-native ...
Analysing C2b ↔ D6B9-strict ...
Analysing C2c ↔ D1 ...
Analysing C2c ↔ D4 ...
Analysing C2c ↔ D5-768 ...
Analysing C2c ↔ D6B-T-native ...
Analysing C2c ↔ D6B9-strict ...
E2 — GT COMPLEMENTARITY ACROSS ALL 15 C×D PAIRS

A. Overall GT complementarity:


,c_variant,d_variant,both,c_only,d_only,neither,c_micro_recall,d_micro_recall,oracle_union_micro_recall,c_unique_gain_over_d,d_unique_gain_over_c,oracle_gain_over_best_branch
0,C1,D1,868,424,1547,7206,0.1286,0.2404,0.2826,0.0422,0.1540,0.0422
1,C1,D4,866,426,743,8010,0.1286,0.1602,0.2026,0.0424,0.0740,0.0424
2,C1,D5-768,704,588,2628,6125,0.1286,0.3317,0.3902,0.0585,0.2616,0.0585
3,C1,D6B-T-native,844,448,2429,6324,0.1286,0.3258,0.3704,0.0446,0.2418,0.0446
4,C1,D6B9-strict,539,753,1068,7685,0.1286,0.1600,0.2349,0.0750,0.1063,0.0750
5,C2b,D1,1268,746,1147,6884,0.2005,0.2404,0.3147,0.0743,0.1142,0.0743
6,C2b,D4,1265,749,344,7687,0.2005,0.1602,0.2347,0.0746,0.0342,0.0342
7,C2b,D5-768,1103,911,2229,5802,0.2005,0.3317,0.4224,0.0907,0.2219,0.0907
8,C2b,D6B-T-native,1270,744,2003,6028,0.2005,0.3258,0.3999,0.0741,0.1994,0.0741
9,C2b,D6B9-strict,767,1247,840,7191,0.2005,0.1600,0.2841,0.1241,0.0836,0.0836



B. Density-stratified GT complementarity (90 rows = 15 pairs × 6 strata):


,c_variant,d_variant,density_stratum,n_images,total_gt,both,c_only,d_only,neither,c_micro_recall,d_micro_recall,oracle_union_micro_recall,c_unique_gain_over_d,d_unique_gain_over_c,oracle_gain_over_best_branch
0,C1,D1,S1_0_20,20,195,33,30,56,76,0.3231,0.4564,0.6103,0.1538,0.2872,0.1538
1,C1,D1,S2_20_40,20,320,77,17,94,132,0.2938,0.5344,0.5875,0.0531,0.2938,0.0531
2,C1,D1,S3_40_60,20,537,70,40,183,244,0.2048,0.4711,0.5456,0.0745,0.3408,0.0745
3,C1,D1,S4_60_80,20,1096,247,67,297,485,0.2865,0.4964,0.5575,0.0611,0.2710,0.0611
4,C1,D1,S5_80_95,10,1211,226,76,351,558,0.2494,0.4765,0.5392,0.0628,0.2898,0.0628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,C2c,D6B9-strict,S2_20_40,20,320,100,50,82,88,0.4688,0.5688,0.7250,0.1562,0.2562,0.1562
86,C2c,D6B9-strict,S3_40_60,20,537,108,111,101,217,0.4078,0.3892,0.5959,0.2067,0.1881,0.1881
87,C2c,D6B9-strict,S4_60_80,20,1096,165,273,163,495,0.3996,0.2993,0.5484,0.2491,0.1487,0.1487
88,C2c,D6B9-strict,S5_80_95,10,1211,216,177,274,544,0.3245,0.4046,0.5508,0.1462,0.2263,0.1462



Consistency checks:
  C×D pairs: 15 / 15
  Pair-image rows: 1500 / 1500
  Pair-GT rows: 150675 / 150675
  C marginal recovery invariant across D partners: PASS
  D marginal recovery invariant across C partners: PASS
  C1↔D1 reproduces Step 13C.2: PASS
  All GT complementarity identities: PASS

Step 13C.4 complete.
GT recovery overlap and unique recovery characterised for all 15 frozen C×D pairs without introducing a fusion rule.


In [58]:
# ============================================================
# Step 13C.5 — Cross-Representation GT Complementarity
#              and Candidate-Burden Comparison
# ============================================================
#
# Purpose
# -------
# Bring together the two key E2 dimensions:
#
#   1. GT recovery / complementarity
#   2. Candidate burden / GT support
#
# No new matching is performed.
# All quantities are derived from the frozen results produced
# in Step 13C.4.
#
# IMPORTANT
# ---------
# This is a descriptive comparison.
#
# It does NOT:
#   - create a fusion rule
#   - remove candidates
#   - choose density-dependent routing
#   - declare an E3 winner automatically
#
# The purpose is to identify which C/D representations are
# scientifically justified for E3 integration experiments.
# ============================================================


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_objects = [
    "e2_all_pairs_image_df",
    "e2_all_pairs_gt_df",
    "e2_all_pairs_overall_df",
    "e2_all_pairs_density_df",
    "E2_C_VARIANTS",
    "E2_D_VARIANTS",
    "DENSITY_ORDER",
]

for name in required_objects:
    assert name in globals(), (
        f"{name} not found. "
        "Run Step 13C.4 first."
    )


assert len(
    e2_all_pairs_overall_df
) == 15

assert len(
    e2_all_pairs_image_df
) == 1500


# ------------------------------------------------------------
# 2. Extract marginal C candidate statistics
#
# Candidate counts / TP / FP for a C representation must be
# invariant to which D representation it was paired with.
# ------------------------------------------------------------

c_rows = []

for c_variant in E2_C_VARIANTS:

    pair_stats = []

    for d_variant in E2_D_VARIANTS:

        sdf = (
            e2_all_pairs_image_df[
                (
                    e2_all_pairs_image_df[
                        "c_variant"
                    ]
                    ==
                    c_variant
                )
                &
                (
                    e2_all_pairs_image_df[
                        "d_variant"
                    ]
                    ==
                    d_variant
                )
            ]
        )

        assert len(sdf) == 100

        pair_stats.append(
            {
                "total_candidates":
                    int(
                        sdf[
                            "n_c_candidates"
                        ].sum()
                    ),

                "tp":
                    int(
                        sdf[
                            "c_tp"
                        ].sum()
                    ),

                "fp":
                    int(
                        sdf[
                            "c_fp"
                        ].sum()
                    ),
            }
        )


    # All five D partners must reproduce the same C marginal.
    assert len(
        {
            x["total_candidates"]
            for x in pair_stats
        }
    ) == 1

    assert len(
        {
            x["tp"]
            for x in pair_stats
        }
    ) == 1

    assert len(
        {
            x["fp"]
            for x in pair_stats
        }
    ) == 1


    total_candidates = (
        pair_stats[0][
            "total_candidates"
        ]
    )

    tp = pair_stats[0]["tp"]
    fp = pair_stats[0]["fp"]


    c_rows.append(
        {
            "representation":
                c_variant,

            "branch":
                "C",

            "total_candidates":
                total_candidates,

            "gt_matched":
                tp,

            "unmatched":
                fp,

            "gt_support_rate":
                (
                    tp / total_candidates
                    if total_candidates > 0
                    else np.nan
                ),

            "unmatched_fraction":
                (
                    fp / total_candidates
                    if total_candidates > 0
                    else np.nan
                ),

            "micro_recall":
                tp / 10045,
        }
    )


e2_c_marginal_burden_df = pd.DataFrame(
    c_rows
)


# ------------------------------------------------------------
# 3. Extract marginal D candidate statistics
#
# Must be invariant to C partner.
# ------------------------------------------------------------

d_rows = []

for d_variant in E2_D_VARIANTS:

    pair_stats = []

    for c_variant in E2_C_VARIANTS:

        sdf = (
            e2_all_pairs_image_df[
                (
                    e2_all_pairs_image_df[
                        "c_variant"
                    ]
                    ==
                    c_variant
                )
                &
                (
                    e2_all_pairs_image_df[
                        "d_variant"
                    ]
                    ==
                    d_variant
                )
            ]
        )

        assert len(sdf) == 100

        pair_stats.append(
            {
                "total_candidates":
                    int(
                        sdf[
                            "n_d_candidates"
                        ].sum()
                    ),

                "tp":
                    int(
                        sdf[
                            "d_tp"
                        ].sum()
                    ),

                "fp":
                    int(
                        sdf[
                            "d_fp"
                        ].sum()
                    ),
            }
        )


    assert len(
        {
            x["total_candidates"]
            for x in pair_stats
        }
    ) == 1

    assert len(
        {
            x["tp"]
            for x in pair_stats
        }
    ) == 1

    assert len(
        {
            x["fp"]
            for x in pair_stats
        }
    ) == 1


    total_candidates = (
        pair_stats[0][
            "total_candidates"
        ]
    )

    tp = pair_stats[0]["tp"]
    fp = pair_stats[0]["fp"]


    d_rows.append(
        {
            "representation":
                d_variant,

            "branch":
                "D",

            "total_candidates":
                total_candidates,

            "gt_matched":
                tp,

            "unmatched":
                fp,

            "gt_support_rate":
                (
                    tp / total_candidates
                    if total_candidates > 0
                    else np.nan
                ),

            "unmatched_fraction":
                (
                    fp / total_candidates
                    if total_candidates > 0
                    else np.nan
                ),

            "micro_recall":
                tp / 10045,
        }
    )


e2_d_marginal_burden_df = pd.DataFrame(
    d_rows
)


# ------------------------------------------------------------
# 4. Combined marginal representation table
# ------------------------------------------------------------

e2_marginal_burden_df = pd.concat(
    [
        e2_c_marginal_burden_df,
        e2_d_marginal_burden_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 5. Build lookup dictionaries
# ------------------------------------------------------------

c_burden_lookup = (
    e2_c_marginal_burden_df
    .set_index("representation")
    .to_dict("index")
)

d_burden_lookup = (
    e2_d_marginal_burden_df
    .set_index("representation")
    .to_dict("index")
)


# ------------------------------------------------------------
# 6. Pair-level recovery + burden comparison
#
# NOTE:
#
# naive_union_candidates = C candidates + D candidates
#
# This is NOT an actual fused prediction count because some
# C/D candidates may spatially overlap.
#
# It is deliberately labelled "naive" and used only as a
# simple burden indicator.
# ------------------------------------------------------------

comparison_rows = []


for _, row in (
    e2_all_pairs_overall_df
    .iterrows()
):

    c_variant = str(
        row["c_variant"]
    )

    d_variant = str(
        row["d_variant"]
    )


    c_stats = (
        c_burden_lookup[
            c_variant
        ]
    )

    d_stats = (
        d_burden_lookup[
            d_variant
        ]
    )


    c_candidates = int(
        c_stats[
            "total_candidates"
        ]
    )

    d_candidates = int(
        d_stats[
            "total_candidates"
        ]
    )


    naive_union_candidates = (
        c_candidates
        +
        d_candidates
    )


    oracle_recovered = int(
        row[
            "oracle_union_recovered"
        ]
    )


    # Descriptive only:
    # how many raw branch candidates exist per GT object
    # recovered by the oracle union.
    #
    # This is NOT precision of a fused system.
    raw_candidates_per_oracle_gt = (
        naive_union_candidates
        /
        oracle_recovered
        if oracle_recovered > 0
        else np.nan
    )


    comparison_rows.append(
        {
            "c_variant":
                c_variant,

            "d_variant":
                d_variant,

            # ------------------------------
            # Marginal recovery
            # ------------------------------

            "c_micro_recall":
                float(
                    row[
                        "c_micro_recall"
                    ]
                ),

            "d_micro_recall":
                float(
                    row[
                        "d_micro_recall"
                    ]
                ),

            # ------------------------------
            # Complementarity
            # ------------------------------

            "both":
                int(
                    row["both"]
                ),

            "c_only":
                int(
                    row["c_only"]
                ),

            "d_only":
                int(
                    row["d_only"]
                ),

            "neither":
                int(
                    row["neither"]
                ),

            "oracle_union_recovered":
                oracle_recovered,

            "oracle_union_micro_recall":
                float(
                    row[
                        "oracle_union_micro_recall"
                    ]
                ),

            "oracle_gain_over_best_branch":
                float(
                    row[
                        "oracle_gain_over_best_branch"
                    ]
                ),

            # ------------------------------
            # C burden
            # ------------------------------

            "c_candidates":
                c_candidates,

            "c_gt_matched":
                int(
                    c_stats[
                        "gt_matched"
                    ]
                ),

            "c_unmatched":
                int(
                    c_stats[
                        "unmatched"
                    ]
                ),

            "c_gt_support_rate":
                float(
                    c_stats[
                        "gt_support_rate"
                    ]
                ),

            # ------------------------------
            # D burden
            # ------------------------------

            "d_candidates":
                d_candidates,

            "d_gt_matched":
                int(
                    d_stats[
                        "gt_matched"
                    ]
                ),

            "d_unmatched":
                int(
                    d_stats[
                        "unmatched"
                    ]
                ),

            "d_gt_support_rate":
                float(
                    d_stats[
                        "gt_support_rate"
                    ]
                ),

            # ------------------------------
            # Simple combined burden
            # ------------------------------

            "naive_union_candidates":
                naive_union_candidates,

            "raw_candidates_per_oracle_gt":
                raw_candidates_per_oracle_gt,
        }
    )


e2_recovery_burden_comparison_df = (
    pd.DataFrame(
        comparison_rows
    )
)


assert len(
    e2_recovery_burden_comparison_df
) == 15


# ------------------------------------------------------------
# 7. Add descriptive deltas relative to C1↔D1 baseline
#
# This helps interpretation without creating a ranking score.
# ------------------------------------------------------------

baseline = (
    e2_recovery_burden_comparison_df[
        (
            e2_recovery_burden_comparison_df[
                "c_variant"
            ]
            ==
            "C1"
        )
        &
        (
            e2_recovery_burden_comparison_df[
                "d_variant"
            ]
            ==
            "D1"
        )
    ]
    .iloc[0]
)


e2_recovery_burden_comparison_df[
    "delta_oracle_recall_vs_C1_D1"
] = (
    e2_recovery_burden_comparison_df[
        "oracle_union_micro_recall"
    ]
    -
    float(
        baseline[
            "oracle_union_micro_recall"
        ]
    )
)


e2_recovery_burden_comparison_df[
    "delta_raw_candidates_vs_C1_D1"
] = (
    e2_recovery_burden_comparison_df[
        "naive_union_candidates"
    ]
    -
    int(
        baseline[
            "naive_union_candidates"
        ]
    )
)


# ------------------------------------------------------------
# 8. Density robustness summary
#
# For each pair, summarise oracle recall across S1–S6.
#
# We do NOT use density strata as routing rules.
# This only characterises robustness.
# ------------------------------------------------------------

density_summary_rows = []


for c_variant in E2_C_VARIANTS:

    for d_variant in E2_D_VARIANTS:

        sdf = (
            e2_all_pairs_density_df[
                (
                    e2_all_pairs_density_df[
                        "c_variant"
                    ]
                    ==
                    c_variant
                )
                &
                (
                    e2_all_pairs_density_df[
                        "d_variant"
                    ]
                    ==
                    d_variant
                )
            ]
            .copy()
        )


        assert len(sdf) == 6


        lookup = (
            sdf
            .set_index(
                "density_stratum"
            )[
                "oracle_union_micro_recall"
            ]
            .to_dict()
        )


        density_summary_rows.append(
            {
                "c_variant":
                    c_variant,

                "d_variant":
                    d_variant,

                **{
                    f"{stratum}_oracle_recall":
                        float(
                            lookup[
                                stratum
                            ]
                        )
                    for stratum
                    in DENSITY_ORDER
                },
            }
        )


e2_density_robustness_df = pd.DataFrame(
    density_summary_rows
)


# ------------------------------------------------------------
# 9. Sanity checks against known baseline values
# ------------------------------------------------------------

c1_stats = (
    e2_c_marginal_burden_df[
        e2_c_marginal_burden_df[
            "representation"
        ]
        ==
        "C1"
    ]
    .iloc[0]
)

d1_stats = (
    e2_d_marginal_burden_df[
        e2_d_marginal_burden_df[
            "representation"
        ]
        ==
        "D1"
    ]
    .iloc[0]
)


# From 13C.3
assert int(
    c1_stats["total_candidates"]
) == 2554

assert int(
    c1_stats["gt_matched"]
) == 1292

assert int(
    d1_stats["total_candidates"]
) == 7652

assert int(
    d1_stats["gt_matched"]
) == 2415


# From 13C.4 baseline
assert np.isclose(
    float(
        baseline[
            "oracle_union_micro_recall"
        ]
    ),
    0.2826,
    atol=5e-5,
)


# ------------------------------------------------------------
# 10. Display marginal representation characteristics
# ------------------------------------------------------------

print("=" * 125)
print(
    "E2 — CROSS-REPRESENTATION GT RECOVERY "
    "AND CANDIDATE-BURDEN COMPARISON"
)
print("=" * 125)


print(
    "\nA. Marginal representation characteristics:"
)

display(
    e2_marginal_burden_df.round(4)
)


# ------------------------------------------------------------
# 11. Display pair-level recovery/burden table
# ------------------------------------------------------------

print(
    "\nB. C×D recovery–burden comparison:"
)

display(
    e2_recovery_burden_comparison_df[
        [
            "c_variant",
            "d_variant",

            "c_micro_recall",
            "d_micro_recall",

            "c_only",
            "d_only",

            "oracle_union_micro_recall",
            "oracle_gain_over_best_branch",

            "c_candidates",
            "c_gt_support_rate",

            "d_candidates",
            "d_gt_support_rate",

            "naive_union_candidates",
            "raw_candidates_per_oracle_gt",

            "delta_oracle_recall_vs_C1_D1",
            "delta_raw_candidates_vs_C1_D1",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 12. Display density robustness
# ------------------------------------------------------------

print(
    "\nC. Density-stratified oracle recovery:"
)

display(
    e2_density_robustness_df.round(4)
)


# ------------------------------------------------------------
# 13. Interpretation aids
#
# These are deliberately NOT a composite score or ranking.
# They expose useful trade-offs for the E3 decision.
# ------------------------------------------------------------

print(
    "\nD. Selected descriptive trade-off views:"
)


# Highest oracle recovery:
oracle_view = (
    e2_recovery_burden_comparison_df[
        [
            "c_variant",
            "d_variant",
            "oracle_union_micro_recall",
            "oracle_gain_over_best_branch",
            "naive_union_candidates",
            "raw_candidates_per_oracle_gt",
        ]
    ]
    .sort_values(
        "oracle_union_micro_recall",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nOracle-recovery view "
    "(diagnostic, not a fusion ranking):"
)

display(
    oracle_view.round(4)
)


# Lowest raw candidate burden:
burden_view = (
    e2_recovery_burden_comparison_df[
        [
            "c_variant",
            "d_variant",
            "oracle_union_micro_recall",
            "naive_union_candidates",
            "raw_candidates_per_oracle_gt",
        ]
    ]
    .sort_values(
        "naive_union_candidates",
        ascending=True,
    )
    .reset_index(drop=True)
)


print(
    "\nRaw-candidate-burden view "
    "(diagnostic, not a fusion ranking):"
)

display(
    burden_view.round(4)
)


# ------------------------------------------------------------
# 14. Final checks
# ------------------------------------------------------------

print(
    "\nConsistency checks:"
)

print(
    f"  C representations: "
    f"{len(e2_c_marginal_burden_df)} / 3"
)

print(
    f"  D representations: "
    f"{len(e2_d_marginal_burden_df)} / 5"
)

print(
    f"  C×D comparison rows: "
    f"{len(e2_recovery_burden_comparison_df)} / 15"
)

print(
    f"  Density robustness rows: "
    f"{len(e2_density_robustness_df)} / 15"
)

print(
    "  C marginal statistics invariant "
    "across D partners: PASS"
)

print(
    "  D marginal statistics invariant "
    "across C partners: PASS"
)

print(
    "  C1 / D1 candidate totals reproduce "
    "Step 13C.3: PASS"
)

print(
    "  C1↔D1 oracle recovery reproduces "
    "Step 13C.4: PASS"
)


print("\n" + "=" * 125)

print(
    "Step 13C.5 complete."
)

print(
    "GT complementarity and candidate burden compared "
    "across all frozen C×D representations."
)

print(
    "No semantic fusion rule has been selected."
)

print("=" * 125)

E2 — CROSS-REPRESENTATION GT RECOVERY AND CANDIDATE-BURDEN COMPARISON

A. Marginal representation characteristics:


,representation,branch,total_candidates,gt_matched,unmatched,gt_support_rate,unmatched_fraction,micro_recall
0,C1,C,2554,1292,1262,0.5059,0.4941,0.1286
1,C2b,C,6074,2014,4060,0.3316,0.6684,0.2005
2,C2c,C,4171,1785,2386,0.4280,0.5720,0.1777
3,D1,D,7652,2415,5237,0.3156,0.6844,0.2404
4,D4,D,4194,1609,2585,0.3836,0.6164,0.1602
5,D5-768,D,14125,3332,10793,0.2359,0.7641,0.3317
6,D6B-T-native,D,15913,3273,12640,0.2057,0.7943,0.3258
7,D6B9-strict,D,4551,1607,2944,0.3531,0.6469,0.1600



B. C×D recovery–burden comparison:


,c_variant,d_variant,c_micro_recall,d_micro_recall,c_only,d_only,oracle_union_micro_recall,oracle_gain_over_best_branch,c_candidates,c_gt_support_rate,d_candidates,d_gt_support_rate,naive_union_candidates,raw_candidates_per_oracle_gt,delta_oracle_recall_vs_C1_D1,delta_raw_candidates_vs_C1_D1
0,C1,D1,0.1286,0.2404,424,1547,0.2826,0.0422,2554,0.5059,7652,0.3156,10206,3.5949,0.0000,0
1,C1,D4,0.1286,0.1602,426,743,0.2026,0.0424,2554,0.5059,4194,0.3836,6748,3.3160,-0.0800,-3458
2,C1,D5-768,0.1286,0.3317,588,2628,0.3902,0.0585,2554,0.5059,14125,0.2359,16679,4.2548,0.1076,6473
3,C1,D6B-T-native,0.1286,0.3258,448,2429,0.3704,0.0446,2554,0.5059,15913,0.2057,18467,4.9629,0.0878,8261
4,C1,D6B9-strict,0.1286,0.1600,753,1068,0.2349,0.0750,2554,0.5059,4551,0.3531,7105,3.0106,-0.0477,-3101
5,C2b,D1,0.2005,0.2404,746,1147,0.3147,0.0743,6074,0.3316,7652,0.3156,13726,4.3423,0.0321,3520
6,C2b,D4,0.2005,0.1602,749,344,0.2347,0.0342,6074,0.3316,4194,0.3836,10268,4.3545,-0.0479,62
7,C2b,D5-768,0.2005,0.3317,911,2229,0.4224,0.0907,6074,0.3316,14125,0.2359,20199,4.7605,0.1398,9993
8,C2b,D6B-T-native,0.2005,0.3258,744,2003,0.3999,0.0741,6074,0.3316,15913,0.2057,21987,5.4735,0.1173,11781
9,C2b,D6B9-strict,0.2005,0.1600,1247,840,0.2841,0.0836,6074,0.3316,4551,0.3531,10625,3.7228,0.0015,419



C. Density-stratified oracle recovery:


...


D. Selected descriptive trade-off views:

Oracle-recovery view (diagnostic, not a fusion ranking):


,c_variant,d_variant,oracle_union_micro_recall,oracle_gain_over_best_branch,naive_union_candidates,raw_candidates_per_oracle_gt
0,C2b,D5-768,0.4224,0.0907,20199,4.7605
1,C2c,D5-768,0.4099,0.0781,18296,4.4440
2,C2b,D6B-T-native,0.3999,0.0741,21987,5.4735
3,C1,D5-768,0.3902,0.0585,16679,4.2548
4,C2c,D6B-T-native,0.3864,0.0605,20084,5.1750
5,C1,D6B-T-native,0.3704,0.0446,18467,4.9629
6,C2b,D1,0.3147,0.0743,13726,4.3423
7,C2c,D1,0.3000,0.0595,11823,3.9240
8,C2b,D6B9-strict,0.2841,0.0836,10625,3.7228
9,C1,D1,0.2826,0.0422,10206,3.5949



Raw-candidate-burden view (diagnostic, not a fusion ranking):


,c_variant,d_variant,oracle_union_micro_recall,naive_union_candidates,raw_candidates_per_oracle_gt
0,C1,D4,0.2026,6748,3.3160
1,C1,D6B9-strict,0.2349,7105,3.0106
2,C2c,D4,0.2199,8365,3.7868
3,C2c,D6B9-strict,0.2648,8722,3.2789
4,C1,D1,0.2826,10206,3.5949
5,C2b,D4,0.2347,10268,4.3545
6,C2b,D6B9-strict,0.2841,10625,3.7228
7,C2c,D1,0.3000,11823,3.9240
8,C2b,D1,0.3147,13726,4.3423
9,C1,D5-768,0.3902,16679,4.2548



Consistency checks:
  C representations: 3 / 3
  D representations: 5 / 5
  C×D comparison rows: 15 / 15
  Density robustness rows: 15 / 15
  C marginal statistics invariant across D partners: PASS
  D marginal statistics invariant across C partners: PASS
  C1 / D1 candidate totals reproduce Step 13C.3: PASS
  C1↔D1 oracle recovery reproduces Step 13C.4: PASS

Step 13C.5 complete.
GT complementarity and candidate burden compared across all frozen C×D representations.
No semantic fusion rule has been selected.


In [59]:
# ============================================================
# Step 13C.6 — E2 Summary and Freeze
# ============================================================
#
# Purpose
# -------
# Freeze the completed Group-E2 GT-based complementarity
# analysis.
#
# E2 established:
#
#   1. GT recovery overlap between C and D
#   2. GT uniquely recovered by C
#   3. GT uniquely recovered by D
#   4. GT recovered by neither
#   5. candidate burden / GT-support trade-offs
#   6. density-stratified complementarity
#
# IMPORTANT
# ---------
# E2 does NOT define a semantic fusion rule.
# E2 does NOT select a final C+D representation.
# E2 does NOT use density strata as inference rules.
#
# E3 will use the frozen E1/E2 evidence to design and
# evaluate semantic integration strategies.
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_objects = [
    "e2_c1_d1_gt_df",
    "e2_c1_d1_overall_df",
    "e2_c1_d1_density_df",
    "e2_c1_d1_candidate_df",
    "e2_c1_d1_candidate_image_df",
    "e2_c1_d1_branch_summary_df",
    "e2_d1_inside_outside_df",
    "e2_d1_inside_outside_density_df",
    "e2_all_pairs_image_df",
    "e2_all_pairs_gt_df",
    "e2_all_pairs_overall_df",
    "e2_all_pairs_density_df",
    "e2_marginal_burden_df",
    "e2_recovery_burden_comparison_df",
    "e2_density_robustness_df",
    "E2_C_VARIANTS",
    "E2_D_VARIANTS",
    "DENSITY_ORDER",
]

for name in required_objects:
    assert name in globals(), (
        f"{name} not found. "
        "Run Steps 13C.1–13C.5 first."
    )


# ------------------------------------------------------------
# 2. Final structural checks
# ------------------------------------------------------------

assert E2_C_VARIANTS == [
    "C1",
    "C2b",
    "C2c",
]

assert E2_D_VARIANTS == [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]

assert len(e2_all_pairs_overall_df) == 15
assert len(e2_all_pairs_density_df) == 90
assert len(e2_all_pairs_image_df) == 1500
assert len(e2_all_pairs_gt_df) == 15 * 10045

assert len(e2_marginal_burden_df) == 8
assert len(e2_recovery_burden_comparison_df) == 15
assert len(e2_density_robustness_df) == 15


# ------------------------------------------------------------
# 3. E2 output directory
# ------------------------------------------------------------

GROUP_E_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "Group_E"
)

E2_RESULTS_DIR = (
    GROUP_E_RESULTS_DIR
    / "E2_GT_complementarity"
)

E2_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 4. Save C1↔D1 baseline artefacts
# ------------------------------------------------------------

e2_c1_d1_gt_df.to_csv(
    E2_RESULTS_DIR
    / "E2_C1_D1_per_GT.csv",
    index=False,
)

e2_c1_d1_overall_df.to_csv(
    E2_RESULTS_DIR
    / "E2_C1_D1_overall_summary.csv",
    index=False,
)

e2_c1_d1_density_df.to_csv(
    E2_RESULTS_DIR
    / "E2_C1_D1_density_summary.csv",
    index=False,
)

e2_c1_d1_candidate_df.to_csv(
    E2_RESULTS_DIR
    / "E2_C1_D1_candidate_analysis.csv",
    index=False,
)

e2_c1_d1_candidate_image_df.to_csv(
    E2_RESULTS_DIR
    / "E2_C1_D1_candidate_per_image.csv",
    index=False,
)

e2_c1_d1_branch_summary_df.to_csv(
    E2_RESULTS_DIR
    / "E2_C1_D1_branch_burden_summary.csv",
    index=False,
)

e2_d1_inside_outside_df.to_csv(
    E2_RESULTS_DIR
    / "E2_D1_inside_outside_C1_summary.csv",
    index=False,
)

e2_d1_inside_outside_density_df.to_csv(
    E2_RESULTS_DIR
    / "E2_D1_inside_outside_C1_density.csv",
    index=False,
)


# ------------------------------------------------------------
# 5. Save all-pair GT complementarity artefacts
# ------------------------------------------------------------

e2_all_pairs_image_df.to_csv(
    E2_RESULTS_DIR
    / "E2_all_pairs_per_image.csv",
    index=False,
)

e2_all_pairs_gt_df.to_csv(
    E2_RESULTS_DIR
    / "E2_all_pairs_per_GT.csv",
    index=False,
)

e2_all_pairs_overall_df.to_csv(
    E2_RESULTS_DIR
    / "E2_all_pairs_overall_summary.csv",
    index=False,
)

e2_all_pairs_density_df.to_csv(
    E2_RESULTS_DIR
    / "E2_all_pairs_density_summary.csv",
    index=False,
)


# ------------------------------------------------------------
# 6. Save recovery–burden comparison artefacts
# ------------------------------------------------------------

e2_marginal_burden_df.to_csv(
    E2_RESULTS_DIR
    / "E2_marginal_representation_burden.csv",
    index=False,
)

e2_recovery_burden_comparison_df.to_csv(
    E2_RESULTS_DIR
    / "E2_recovery_burden_comparison.csv",
    index=False,
)

e2_density_robustness_df.to_csv(
    E2_RESULTS_DIR
    / "E2_density_robustness.csv",
    index=False,
)


# ------------------------------------------------------------
# 7. Extract descriptive reference results
#
# These are recorded for traceability only.
# They do NOT constitute automatic E3 selection.
# ------------------------------------------------------------

baseline_row = (
    e2_recovery_burden_comparison_df[
        (
            e2_recovery_burden_comparison_df[
                "c_variant"
            ] == "C1"
        )
        &
        (
            e2_recovery_burden_comparison_df[
                "d_variant"
            ] == "D1"
        )
    ]
    .iloc[0]
)


max_oracle_idx = (
    e2_recovery_burden_comparison_df[
        "oracle_union_micro_recall"
    ]
    .idxmax()
)

max_oracle_row = (
    e2_recovery_burden_comparison_df
    .loc[max_oracle_idx]
)


# ------------------------------------------------------------
# 8. Freeze metadata
# ------------------------------------------------------------

e2_freeze_metadata = {
    "experiment":
        "B1_Group_E2_GT_complementarity",

    "dataset":
        "FSC147",

    "subset":
        "frozen_validation_dev100",

    "n_images":
        100,

    "total_gt":
        10045,

    "c_variants":
        list(E2_C_VARIANTS),

    "d_variants":
        list(E2_D_VARIANTS),

    "n_pairs":
        15,

    "localisation_protocol": {
        "matching":
            "independent one-to-one C->GT and D->GT",

        "radius":
            "max(1, round(0.03 * min(H, W)))",

        "gt_classes": [
            "BOTH",
            "C_ONLY",
            "D_ONLY",
            "NEITHER",
        ],
    },

    "density_strata":
        list(DENSITY_ORDER),

    "baseline_pair": {
        "c_variant":
            "C1",

        "d_variant":
            "D1",

        "oracle_union_micro_recall":
            float(
                baseline_row[
                    "oracle_union_micro_recall"
                ]
            ),

        "naive_union_candidates":
            int(
                baseline_row[
                    "naive_union_candidates"
                ]
            ),
    },

    "max_observed_oracle_recovery_pair": {
        "c_variant":
            str(
                max_oracle_row[
                    "c_variant"
                ]
            ),

        "d_variant":
            str(
                max_oracle_row[
                    "d_variant"
                ]
            ),

        "oracle_union_micro_recall":
            float(
                max_oracle_row[
                    "oracle_union_micro_recall"
                ]
            ),

        "naive_union_candidates":
            int(
                max_oracle_row[
                    "naive_union_candidates"
                ]
            ),

        "interpretation":
            (
                "Descriptive oracle result only; "
                "not a selected fusion configuration."
            ),
    },

    "interpretation_boundary": [
        "E2 measures GT recovery overlap and complementarity.",
        "E2 measures candidate burden and GT support.",
        "Oracle union is diagnostic and is not a realizable fusion result.",
        "No semantic fusion rule has been introduced.",
        "No final C+D representation has been selected.",
        "Density strata are evaluation strata only.",
    ],

    "next_stage":
        "E3_semantic_integration_strategy",
}


E2_FREEZE_METADATA_PATH = (
    E2_RESULTS_DIR
    / "E2_freeze_metadata.json"
)


with open(
    E2_FREEZE_METADATA_PATH,
    "w",
) as f:

    json.dump(
        e2_freeze_metadata,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Confirm files exist
# ------------------------------------------------------------

expected_files = [
    "E2_C1_D1_per_GT.csv",
    "E2_C1_D1_overall_summary.csv",
    "E2_C1_D1_density_summary.csv",
    "E2_C1_D1_candidate_analysis.csv",
    "E2_C1_D1_candidate_per_image.csv",
    "E2_C1_D1_branch_burden_summary.csv",
    "E2_D1_inside_outside_C1_summary.csv",
    "E2_D1_inside_outside_C1_density.csv",
    "E2_all_pairs_per_image.csv",
    "E2_all_pairs_per_GT.csv",
    "E2_all_pairs_overall_summary.csv",
    "E2_all_pairs_density_summary.csv",
    "E2_marginal_representation_burden.csv",
    "E2_recovery_burden_comparison.csv",
    "E2_density_robustness.csv",
    "E2_freeze_metadata.json",
]


for filename in expected_files:

    path = (
        E2_RESULTS_DIR
        / filename
    )

    assert path.exists(), (
        f"Missing E2 artefact: {path}"
    )


# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("=" * 110)
print(
    "GROUP E2 — FINAL SUMMARY AND FREEZE"
)
print("=" * 110)


print(
    "\nDataset / subset:"
)

print(
    "  FSC147 frozen validation dev100"
)


print(
    "\nFrozen C representations:"
)

print(
    f"  {E2_C_VARIANTS}"
)


print(
    "\nFrozen D representations:"
)

print(
    f"  {E2_D_VARIANTS}"
)


print(
    "\nC×D combinations analysed:"
)

print(
    "  15"
)


print(
    "\nGT objects per C×D pair:"
)

print(
    "  10045"
)


print(
    "\nE2 analysis covered:"
)

print(
    "  - GT recovery by both C and D"
)

print(
    "  - GT uniquely recovered by C"
)

print(
    "  - GT uniquely recovered by D"
)

print(
    "  - GT recovered by neither"
)

print(
    "  - marginal branch recall"
)

print(
    "  - oracle-union GT recovery"
)

print(
    "  - candidate GT-support / unmatched burden"
)

print(
    "  - density-stratified complementarity"
)

print(
    "  - cross-representation recovery–burden trade-offs"
)


print(
    "\nReference baseline:"
)

print(
    "  C1 + D1"
)

print(
    f"  Oracle union recall: "
    f"{baseline_row['oracle_union_micro_recall']:.4f}"
)

print(
    f"  Naive raw candidate pool: "
    f"{int(baseline_row['naive_union_candidates'])}"
)


print(
    "\nMaximum observed oracle recovery:"
)

print(
    f"  "
    f"{max_oracle_row['c_variant']} + "
    f"{max_oracle_row['d_variant']}"
)

print(
    f"  Oracle union recall: "
    f"{max_oracle_row['oracle_union_micro_recall']:.4f}"
)

print(
    f"  Naive raw candidate pool: "
    f"{int(max_oracle_row['naive_union_candidates'])}"
)


print(
    "\nInterpretation boundary:"
)

print(
    "  E2 measures GT recovery complementarity and burden."
)

print(
    "  Oracle union is diagnostic, not a fusion result."
)

print(
    "  No semantic fusion rule has been introduced."
)

print(
    "  No final C+D representation has been selected."
)

print(
    "  Density strata remain evaluation strata only."
)


print(
    "\nSaved E2 artefacts:"
)

print(
    f"  {E2_RESULTS_DIR}"
)

for filename in expected_files:
    print(
        f"    - {filename}"
    )


print("\n" + "=" * 110)

print(
    "Step 13C.6 complete — Group E2 frozen."
)

print(
    "Next: E3 — Semantic Integration Strategy."
)

print("=" * 110)

GROUP E2 — FINAL SUMMARY AND FREEZE

Dataset / subset:
  FSC147 frozen validation dev100

Frozen C representations:
  ['C1', 'C2b', 'C2c']

Frozen D representations:
  ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']

C×D combinations analysed:
  15

GT objects per C×D pair:
  10045

E2 analysis covered:
  - GT recovery by both C and D
  - GT uniquely recovered by C
  - GT uniquely recovered by D
  - GT recovered by neither
  - marginal branch recall
  - oracle-union GT recovery
  - candidate GT-support / unmatched burden
  - density-stratified complementarity
  - cross-representation recovery–burden trade-offs

Reference baseline:
  C1 + D1
  Oracle union recall: 0.2826
  Naive raw candidate pool: 10206

Maximum observed oracle recovery:
  C2b + D5-768
  Oracle union recall: 0.4224
  Naive raw candidate pool: 20199

Interpretation boundary:
  E2 measures GT recovery complementarity and burden.
  Oracle union is diagnostic, not a fusion result.
  No semantic fusion rule has been i

### **E2 — Summary and Interpretation**

E2 analysed the ground-truth complementarity of the retained region-based (C) and activation-based (D) semantic representations using the frozen FSC-147 dev100 subset and the canonical 3%-of-short-side one-to-one localisation protocol.

The results confirm that the **C and D branches** provide genuinely **complementary semantic evidence**. Each branch recovers GT objects missed by the other, although this comes with different candidate-burden trade-offs.

Among the **C representations**, C2b provides the highest GT recovery, while C1 is substantially more selective and C2c provides an intermediate operating point.

Among the **D representations**, D5-768 and D6B-T-native provide substantially greater GT coverage, but at the cost of large numbers of unmatched candidates. D1 provides a more selective alternative.

Across the **15 retained C×D combinations**, **C2b + D5-768** provides the maximum observed oracle-union GT recall (0.4224), compared with 0.2826 for the C1 + D1 reference configuration. However, this higher recoverable coverage is accompanied by a substantially larger raw candidate pool (20,199 versus 10,206). The oracle result therefore demonstrates available complementary information rather than the performance of a realizable fused predictor.

The **density-stratified analysis** further shows that **semantic evidence** is considerably **stronger in sparse-to-moderate** scenes than the overall micro recall suggests. Some high-recovery combinations provide substantial oracle GT coverage through S1–S4, whereas performance deteriorates markedly in S5 and especially S6. This indicates that extreme-density scenes remain a major unresolved challenge. Conversely, high-recall activation representations can substantially over-generate candidates, particularly outside the extreme-density regime.

The results also provide evidence against simple hard intersection of the C and D branches. D candidates inside the C foreground are generally better supported, but valid D-only evidence also exists outside the C foreground. Hard semantic gating therefore improves selectivity at the cost of discarding useful complementary evidence.

Overall, E2 establishes a **clear recovery–burden trade-off** rather than identifying a single optimal C+D configuration. The evidence motivates E3 to investigate semantic integration strategies that preserve complementary C/D recovery while suppressing redundant and unsupported candidates. Evaluation will continue to report both overall and S1–S6 performance, with particular attention to strong sparse-to-moderate counting performance and graceful degradation as scene density increases. Density strata remain evaluation categories only and are not used as inference-time routing information.

**No final C+D representation or fusion rule is selected** at the end of E2.

## **Step 13D — E3: Semantic Integration Strategy**

### Step 13D.1 — E3 Objectives, Design Space and Evaluation Protocol

#### Objective

E3 investigates how the frozen region-based semantic evidence (C) and activation-based semantic evidence (D) should be integrated to produce a more useful set of semantic reference-point candidates.

The objective is not simply to maximise the number of GT objects recoverable by the union of C and D. E2 showed that increased recovery can be accompanied by substantial candidate over-generation. E3 therefore aims to preserve useful complementary evidence while suppressing redundant and unsupported candidates.

No geometric/depth information is introduced in E3. Marigold remains excluded so that the contribution of semantic integration can be evaluated independently before geometric refinement is added.

---

### Evidence motivating E3

E1 established that C and D provide structurally related but non-equivalent semantic evidence. Activation points occur both inside and outside the region-based semantic foreground, while individual C regions may contain zero, one, or multiple D points.

E2 established that these structural differences correspond to genuine GT complementarity. C recovers some GT objects missed by D, while D recovers additional GT objects missed by C. However, the higher-recall representations also introduce substantial unmatched-candidate burden.

The E2 results additionally showed that hard C-region gating of D evidence is potentially too restrictive: D candidates inside the C foreground are generally better supported, but valid GT-matched D candidates also occur outside it.

Therefore, neither simple hard intersection nor unrestricted union is assumed to be an adequate final integration strategy.

---

### E3 design principle

E3 will investigate semantic integration as a recovery–selectivity problem:

- preserve C–D agreement where it provides strong semantic evidence;
- retain useful complementary evidence from either branch;
- reduce redundant or weakly supported candidates;
- avoid automatically discarding D-only evidence outside C regions;
- avoid excessive candidate generation from permissive activation representations.

The integration mechanism must operate without access to GT annotations, GT counts, or GT-derived density strata.

---

### Candidate representations

The frozen E2 representation set remains:

**Region branch (C):**
- C1
- C2b
- C2c

**Activation branch (D):**
- D1
- D4
- D5-768
- D6B-T-native
- D6B9-strict

E2 does not establish a single optimal C+D pair.

C2b + D5-768 provides the maximum observed oracle-union GT recovery, but also carries a high raw candidate burden. C1 + D1 remains the simple reference configuration, while other retained representations provide different recovery–selectivity operating points.

E3 will therefore avoid selecting a final representation solely from oracle recovery.

---

### Fusion design space

E3 will consider progressively more structured integration strategies.

**1. Simple point-set baselines**

These establish lower-complexity references such as:
- C candidates alone;
- D candidates alone;
- C ∪ D with spatial duplicate consolidation;
- agreement-focused / intersection-like alternatives.

These baselines determine whether more structured integration is actually necessary.

**2. Agreement-aware integration**

C–D spatial agreement can be treated as stronger evidence without requiring that every retained candidate be supported by both branches.

This allows agreement to increase confidence while preserving potentially useful C-only and D-only evidence.

**3. Region-conditioned activation integration**

The C representation can provide spatial/region context for D evidence rather than acting as a hard binary gate.

Examples include distinguishing:
- D evidence inside a C region;
- multiple D candidates within one C region;
- D evidence near but outside C foreground;
- isolated D evidence with weak region support.

**4. Region-guided reference-point generation**

Instead of only combining two pre-existing point sets, C regions may constrain where semantic reference points should be generated. Continuous semantic activation within a region may be used to locate a more representative point than the geometric region centroid.

This represents a stronger form of semantic integration and will only be introduced after simpler fusion baselines are established.

---

### Evaluation protocol

All E3 strategies will use the same frozen FSC-147 dev100 subset and the same canonical one-to-one localisation protocol used in E2:

\[
r = \max(1,\operatorname{round}(0.03\min(H,W))).
\]

Each strategy will be evaluated Overall and separately for S1–S6.

The primary evaluation dimensions are:

1. **Counting behaviour**
   - MAE
   - RMSE
   - mean signed error
   - Pred/GT behaviour

2. **Localisation quality**
   - precision
   - recall
   - F1 under the canonical matching protocol

3. **Candidate burden**
   - number of generated reference points
   - unmatched candidate burden
   - redundancy where relevant

4. **Density-stratified behaviour**
   - S1–S6 count and localisation performance.

Particular attention will be paid to strong performance in sparse-to-moderate scenes (especially S1–S3), while seeking graceful degradation through S4–S5 and the challenging S6 regime.

S1–S6 remain evaluation strata only. They are not supplied to the integration method and are not used for inference-time routing.

---

### Experimental controls

To preserve interpretability:

- C and D representations remain frozen during E3.
- The FSC-147 dev100 subset remains unchanged.
- The localisation tolerance remains unchanged.
- No Marigold/geometric prior is introduced.
- No SAM2 mask generation is required during semantic-fusion screening because, under the frozen one-reference-point → one-mask protocol, candidate/reference-point count remains the appropriate screening proxy.
- Major integration components will be introduced incrementally rather than simultaneously.
- Parameter sweeps, where required, will be small and explicitly bounded.
- GT information will be used only for evaluation and parameter comparison, never as an inference-time input.

---

### E3 decision criterion

No single scalar metric will automatically determine the selected integration strategy.

Selection will consider the joint evidence from:

- counting error;
- localisation precision/recall/F1;
- candidate burden;
- performance stability across S1–S6;
- implementation complexity and interpretability.

A strategy that achieves higher overall recovery by substantially degrading sparse-scene counting or generating excessive unsupported candidates will not automatically be preferred.

The selected semantic integration strategy will subsequently be frozen before introducing the Marigold geometric prior.

---

### Deferred adaptive extension

E3 initially evaluates fixed, density-agnostic semantic integration.

If the experiments reveal a systematic crossover in which different integration behaviour is consistently preferable for different scene conditions, a later extension may investigate class-specific adaptive fusion using observable, GT-free estimates of scene difficulty or complexity.

Such an adaptive mechanism is not part of the initial E3 experiments and will only be introduced if justified by the fixed-pipeline results.

### Step 13D.2 — Simple Semantic-Fusion Baselines

Before introducing structured region-conditioned integration, E3 first evaluates simple point-level fusion baselines using the frozen C and D representations.

The purpose is to establish whether complementary C/D evidence can already improve semantic reference-point generation using low-complexity integration rules.

Three fusion families are considered:

#### F1 — Union with spatial consolidation

Candidates from C and D are combined:

\[
P_{F1} = P_C \cup P_D
\]

Spatially redundant C/D candidates representing approximately the same location are consolidated so that simple branch overlap does not automatically produce two reference points.

F1 is a permissive baseline. It aims to preserve complementary evidence from both branches but is expected to retain substantial unmatched-candidate burden when D is over-generative.

#### F2 — C–D agreement

Only spatially associated C/D evidence is retained:

\[
P_{F2} = P_C \cap_{\delta} P_D
\]

where \(\delta\) denotes an inference-time spatial association criterion.

F2 is a conservative baseline intended to test the precision benefit and recall cost of requiring cross-representation agreement.

#### F3 — C-anchored agreement

All C reference points are retained, while D contributes supporting information only through spatial agreement with C.

Conceptually:

\[
P_{F3} = P_C + \text{D-supported refinement of }P_C
\]

At this simple-baseline stage, F3 does not introduce unsupported D-only candidates. It tests whether D evidence can improve the location of C-derived reference points without increasing the number of C hypotheses.

---

The original frozen C-only and D-only representations remain marginal reference baselines and are not recomputed.

No GT information is used to determine C–D correspondence. The canonical 3%-of-short-side GT matching radius remains an evaluation criterion only and is not automatically reused as the fusion radius.


Before evaluating F1–F3, E3 first constructs and validates a GT-free set of C–D association primitives. These describe complementary topological, multiplicity, spatial, region-context and semantic-strength relationships without yet imposing a fusion rule. The subsequent baseline experiments determine how subsets of these primitives can be used for integration.

#### Step 13D.2a.1 — Inventory Available C/D Evidence and Define the Association Schema

Before constructing C–D associations, this step verifies the inference-time information available from the frozen C and D representations.

The association infrastructure is GT-free. No GT points, GT counts, density strata, localisation matches, or evaluation-derived information are used to construct the associations.

For each C region, the intended core information is:

- region identity;
- region mask / labelled support;
- centroid/reference point;
- region area in pixels;
- region area normalised by image area;
- D-point multiplicity within the region.

For each D point, the intended core information is:

- point coordinates;
- inside/outside C foreground;
- containing C-region identity, if inside;
- nearest C-region identity;
- distance to the nearest C-region boundary/support;
- distance to the relevant/nearest C centroid;
- corresponding image-normalised distances.

Where available, representation-specific D activation/peak scores will be preserved as optional contextual evidence. These scores are not assumed to be directly comparable across different D representations.

C-region semantic strength may also be preserved later as optional contextual evidence, but is not required for the initial simple-fusion baselines.

No distance threshold or fusion decision is introduced at this stage.

In [60]:
# ============================================================
# Step 13D.2a.1 — Inventory available C/D evidence
# ============================================================

print("=" * 100)
print("STEP 13D.2a.1 — AVAILABLE C/D EVIDENCE INVENTORY")
print("=" * 100)

# ------------------------------------------------------------
# 1. Relevant objects currently available in notebook memory
# ------------------------------------------------------------
keywords = (
    "group_c", "group_d", "canonical", "e1_region",
    "e2_", "san_cache"
)

available_objects = sorted(
    name for name in globals()
    if any(k in name.lower() for k in keywords)
)

print("\nRelevant notebook objects:")
for name in available_objects:
    obj = globals()[name]
    shape = getattr(obj, "shape", None)
    print(f"  {name:<45} type={type(obj).__name__:<15} shape={shape}")


# ------------------------------------------------------------
# 2. Inspect frozen canonical Group-E dataframe if available
# ------------------------------------------------------------
if "e_group_df" in globals():
    print("\n" + "-" * 100)
    print("e_group_df columns:")
    print("-" * 100)

    for col in e_group_df.columns:
        print(f"  {col}")

    print("\nCandidate IDs:")
    print(sorted(e_group_df["candidate_id"].unique()))

else:
    print("\n'e_group_df' not found — use the object inventory above "
          "to identify the canonical Group-E dataframe.")


# ------------------------------------------------------------
# 3. Inspect E1 reconstructed region-map structure
# ------------------------------------------------------------
if "E1_REGION_MAPS" in globals():
    print("\n" + "-" * 100)
    print("E1_REGION_MAPS structure:")
    print("-" * 100)

    first_key = next(iter(E1_REGION_MAPS))
    first_item = E1_REGION_MAPS[first_key]

    print("First dataset_index:", first_key)
    print("Top-level keys:")
    for key in first_item.keys():
        value = first_item[key]
        shape = getattr(value, "shape", None)
        print(f"  {key:<30} type={type(value).__name__:<15} shape={shape}")

else:
    print("\n'E1_REGION_MAPS' not found.")


print("\n" + "=" * 100)
print("Inventory complete.")
print("No association or fusion rule has been applied.")
print("=" * 100)

STEP 13D.2a.1 — AVAILABLE C/D EVIDENCE INVENTORY

Relevant notebook objects:
  E1_REGION_CACHE_DIR                           type=PosixPath       shape=None
  E1_REGION_CACHE_INDEX_PATH                    type=PosixPath       shape=None
  E1_REGION_CACHE_ROOT                          type=PosixPath       shape=None
  E1_REGION_CONFIGS                             type=dict            shape=None
  E1_REGION_MAPS                                type=dict            shape=None
  E2_C_VARIANTS                                 type=list            shape=None
  E2_D_VARIANTS                                 type=list            shape=None
  E2_FREEZE_METADATA_PATH                       type=PosixPath       shape=None
  E2_GT_CLASSES                                 type=list            shape=None
  E2_POINT_TOLERANCE_FRAC                       type=float           shape=None
  E2_RESULTS_DIR                                type=PosixPath       shape=None
  FSC147_DEV_SAN_CACHE_DIR                 

In [61]:
# ============================================================
# Step 13D.2a.1 — Detailed C/D schema inventory
# ============================================================

print("=" * 100)
print("STEP 13D.2a.1 — DETAILED C/D SCHEMA INVENTORY")
print("=" * 100)


# ------------------------------------------------------------
# 1. Frozen Group-C handoff
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("GROUP-C HANDOFF")
print("-" * 100)

print("Shape:", group_c_handoff_df.shape)

print("\nColumns:")
for col in group_c_handoff_df.columns:
    print(f"  {col}")

print("\nCandidate IDs:")
print(sorted(group_c_handoff_df["candidate_id"].unique()))


# ------------------------------------------------------------
# 2. Frozen Group-D handoff
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("GROUP-D HANDOFF")
print("-" * 100)

print("Shape:", group_d_handoff_df.shape)

print("\nColumns:")
for col in group_d_handoff_df.columns:
    print(f"  {col}")

print("\nCandidate IDs:")
print(sorted(group_d_handoff_df["candidate_id"].unique()))


# ------------------------------------------------------------
# 3. Inspect one representative row from each D variant
#
# Only show fields that may describe:
#   - predicted points
#   - scores / confidence
#   - thresholds
#   - activation / peak configuration
# ------------------------------------------------------------

interesting_terms = (
    "point",
    "score",
    "conf",
    "activation",
    "peak",
    "threshold",
    "query",
)

interesting_d_cols = [
    col
    for col in group_d_handoff_df.columns
    if any(term in col.lower() for term in interesting_terms)
]

print("\nPotential D point/score/configuration fields:")
for col in interesting_d_cols:
    print(f"  {col}")


for candidate_id in sorted(
    group_d_handoff_df["candidate_id"].unique()
):

    row = (
        group_d_handoff_df[
            group_d_handoff_df["candidate_id"]
            == candidate_id
        ]
        .iloc[0]
    )

    print("\n" + "-" * 70)
    print(f"D variant: {candidate_id}")
    print("-" * 70)

    for col in interesting_d_cols:

        value = row[col]

        # Keep potentially large arrays/lists readable
        text = str(value)

        if len(text) > 300:
            text = text[:300] + " ..."

        print(f"{col}: {text}")


# ------------------------------------------------------------
# 4. Inspect reconstructed C-region structure
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("E1 REGION-MAP INTERNAL STRUCTURE")
print("=" * 100)

first_dataset_index = next(iter(E1_REGION_MAPS))
first_item = E1_REGION_MAPS[first_dataset_index]

print(
    f"\nExample dataset_index: {first_dataset_index}"
)

for c_variant in ["C1", "C2b", "C2c"]:

    region_obj = first_item[c_variant]

    print("\n" + "-" * 70)
    print(f"{c_variant}")
    print("-" * 70)

    print("Keys:")

    for key, value in region_obj.items():

        shape = getattr(value, "shape", None)

        print(
            f"  {key:<30} "
            f"type={type(value).__name__:<15} "
            f"shape={shape}"
        )


print("\n" + "=" * 100)
print("Detailed inventory complete.")
print("No GT information or fusion rule has been used.")
print("=" * 100)

STEP 13D.2a.1 — DETAILED C/D SCHEMA INVENTORY

----------------------------------------------------------------------------------------------------
GROUP-C HANDOFF
----------------------------------------------------------------------------------------------------
Shape: (300, 27)

Columns:
  candidate_id
  experiment_id
  candidate_family
  description
  dataset_index
  image_id
  category
  density_stratum
  width
  height
  a3c_percentile
  point_tolerance_frac
  radius_px
  gt_count
  pred_count
  signed_error
  abs_error
  squared_error
  pred_gt_ratio
  tp
  fp
  fn
  precision
  recall
  f1
  gt_points_xy
  pred_points_xy

Candidate IDs:
['C1', 'C2b', 'C2c']

----------------------------------------------------------------------------------------------------
GROUP-D HANDOFF
----------------------------------------------------------------------------------------------------
Shape: (500, 48)

Columns:
  candidate_id
  candidate_family
  experiment_id
  image_id
  dataset_index
  c

#### Step 13D.2a.2 — Construct and Validate C1+D1 Association Primitives

The association primitives are first implemented for the reference pair C1+D1 before being extended to all retained C×D combinations.

For each D1 point, this step records:

- whether it lies inside the C1 foreground;
- its containing C1 region, if any;
- D1 multiplicity within that containing region;
- distance to the containing-region centroid;
- nearest C1 region;
- distance to the nearest C1 region support;
- distance to the centroid of the nearest C1 region;
- image-normalised versions of the distance quantities.

A corresponding per-region table records C1 region geometry and D1 multiplicity.

The construction uses only frozen C1 region maps and frozen D1 predictions. No GT coordinates, GT counts, density strata, localisation matches, or evaluation-derived quantities are used.

The resulting structural counts are subsequently compared with the frozen E1 C1+D1 results as an implementation validation check only.

In [62]:
# ============================================================
# Step 13D.2a.2 — Construct and validate C1+D1
# association primitives
# ============================================================

import ast
import numpy as np
import pandas as pd

from scipy.ndimage import distance_transform_edt


# ------------------------------------------------------------
# Helper: robust point-array parsing
# ------------------------------------------------------------

def e3_parse_points_xy(value):
    """
    Convert stored point representation to float ndarray (N, 2).
    No GT-specific behaviour.
    """
    if value is None:
        return np.empty((0, 2), dtype=float)

    if isinstance(value, str):
        value = ast.literal_eval(value)

    arr = np.asarray(value, dtype=float)

    if arr.size == 0:
        return np.empty((0, 2), dtype=float)

    return arr.reshape(-1, 2)


# ------------------------------------------------------------
# Build lookup for frozen D1 rows
# ------------------------------------------------------------

d1_df = (
    group_d_handoff_df[
        group_d_handoff_df["candidate_id"] == "D1"
    ]
    .copy()
)

assert len(d1_df) == 100, (
    f"Expected 100 D1 rows, found {len(d1_df)}"
)

d1_lookup = {
    int(row["dataset_index"]): row
    for _, row in d1_df.iterrows()
}


# ------------------------------------------------------------
# Output records
# ------------------------------------------------------------

d_point_records = []
c_region_records = []


# ------------------------------------------------------------
# Process each frozen dev100 image
# ------------------------------------------------------------

for dataset_index, image_info in E1_REGION_MAPS.items():

    dataset_index = int(dataset_index)

    assert dataset_index in d1_lookup, (
        f"D1 row missing for dataset_index={dataset_index}"
    )

    d_row = d1_lookup[dataset_index]

    image_id = image_info["image_id"]
    width = int(image_info["width"])
    height = int(image_info["height"])

    short_side = min(width, height)
    image_area = width * height

    # --------------------------------------------------------
    # Frozen C1 representation
    # --------------------------------------------------------

    c_obj = image_info["C1"]

    labels = np.asarray(c_obj["labelled_regions"])
    c_points = np.asarray(c_obj["points"], dtype=float).reshape(-1, 2)
    region_areas = np.asarray(c_obj["region_areas"], dtype=float)

    n_regions = int(c_obj["num_regions"])

    assert labels.shape == (height, width)
    assert len(c_points) == n_regions
    assert len(region_areas) == n_regions

    # Region IDs are expected to be 1...n_regions.
    positive_labels = np.unique(labels)
    positive_labels = positive_labels[positive_labels > 0]

    assert np.array_equal(
        positive_labels,
        np.arange(1, n_regions + 1)
    ), (
        f"Unexpected C1 labels for dataset_index={dataset_index}"
    )

    # --------------------------------------------------------
    # Frozen D1 points
    # --------------------------------------------------------

    d_points = e3_parse_points_xy(d_row["pred_points_xy"])

    assert len(d_points) == int(d_row["pred_count"])

    # --------------------------------------------------------
    # Multiplicity: number of D points actually INSIDE
    # each C1 region.
    # --------------------------------------------------------

    d_inside_region_ids = []

    for x, y in d_points:

        # Frozen predictions are in image coordinates.
        # Use nearest pixel only for discrete region lookup.
        px = int(np.clip(np.rint(x), 0, width - 1))
        py = int(np.clip(np.rint(y), 0, height - 1))

        region_id = int(labels[py, px])

        d_inside_region_ids.append(region_id)

    d_inside_region_ids = np.asarray(
        d_inside_region_ids,
        dtype=int
    )

    multiplicity = np.bincount(
        d_inside_region_ids[d_inside_region_ids > 0],
        minlength=n_regions + 1
    )

    # --------------------------------------------------------
    # Distance-to-region infrastructure
    #
    # For each region, distance_transform_edt(~region_mask)
    # gives zero inside the region and Euclidean distance
    # outside it.
    #
    # With only 100 images and C1 validation this is kept
    # deliberately simple and explicit.
    # --------------------------------------------------------

    region_distance_maps = {}

    for region_id in range(1, n_regions + 1):

        region_mask = labels == region_id

        region_distance_maps[region_id] = (
            distance_transform_edt(~region_mask)
        )

    # --------------------------------------------------------
    # Per-C-region records
    # --------------------------------------------------------

    for region_id in range(1, n_regions + 1):

        centroid_x, centroid_y = c_points[region_id - 1]
        area_px = float(region_areas[region_id - 1])

        c_region_records.append({
            "dataset_index": dataset_index,
            "image_id": image_id,
            "category": d_row["category"],
            "c_variant": "C1",
            "d_variant": "D1",
            "region_id": region_id,
            "centroid_x": float(centroid_x),
            "centroid_y": float(centroid_y),
            "region_area_px": area_px,
            "region_area_frac": area_px / image_area,
            "n_d_inside": int(multiplicity[region_id]),
        })

    # --------------------------------------------------------
    # Per-D-point association records
    # --------------------------------------------------------

    for d_point_id, (x, y) in enumerate(d_points):

        px = int(np.clip(np.rint(x), 0, width - 1))
        py = int(np.clip(np.rint(y), 0, height - 1))

        containing_region_id = int(labels[py, px])

        inside = containing_region_id > 0

        # ----------------------------------------------------
        # Distance to containing-region centroid
        # ----------------------------------------------------

        if inside:

            containing_centroid = (
                c_points[containing_region_id - 1]
            )

            dist_containing_centroid_px = float(
                np.linalg.norm(
                    np.asarray([x, y]) -
                    containing_centroid
                )
            )

            n_d_in_containing_region = int(
                multiplicity[containing_region_id]
            )

        else:

            dist_containing_centroid_px = np.nan
            n_d_in_containing_region = 0

        # ----------------------------------------------------
        # Nearest C1 region by distance to REGION SUPPORT
        # ----------------------------------------------------

        if n_regions > 0:

            region_distances = np.asarray([
                region_distance_maps[rid][py, px]
                for rid in range(1, n_regions + 1)
            ], dtype=float)

            nearest_region_id = (
                int(np.argmin(region_distances)) + 1
            )

            dist_nearest_region_px = float(
                region_distances[nearest_region_id - 1]
            )

            nearest_centroid = (
                c_points[nearest_region_id - 1]
            )

            dist_nearest_centroid_px = float(
                np.linalg.norm(
                    np.asarray([x, y]) -
                    nearest_centroid
                )
            )

        else:

            nearest_region_id = np.nan
            dist_nearest_region_px = np.nan
            dist_nearest_centroid_px = np.nan

        d_point_records.append({
            "dataset_index": dataset_index,
            "image_id": image_id,
            "category": d_row["category"],
            "c_variant": "C1",
            "d_variant": "D1",
            "d_point_id": int(d_point_id),

            "d_x": float(x),
            "d_y": float(y),

            "inside_c_foreground": bool(inside),

            "containing_region_id": (
                containing_region_id
                if inside else np.nan
            ),

            "n_d_in_containing_region": (
                n_d_in_containing_region
            ),

            "distance_to_containing_centroid_px": (
                dist_containing_centroid_px
            ),

            "distance_to_containing_centroid_norm": (
                dist_containing_centroid_px / short_side
                if np.isfinite(dist_containing_centroid_px)
                else np.nan
            ),

            "nearest_region_id": nearest_region_id,

            "distance_to_nearest_region_px": (
                dist_nearest_region_px
            ),

            "distance_to_nearest_region_norm": (
                dist_nearest_region_px / short_side
                if np.isfinite(dist_nearest_region_px)
                else np.nan
            ),

            "distance_to_nearest_region_centroid_px": (
                dist_nearest_centroid_px
            ),

            "distance_to_nearest_region_centroid_norm": (
                dist_nearest_centroid_px / short_side
                if np.isfinite(dist_nearest_centroid_px)
                else np.nan
            ),
        })


# ------------------------------------------------------------
# Construct dataframes
# ------------------------------------------------------------

e3_c1_d1_region_df = pd.DataFrame(c_region_records)
e3_c1_d1_dpoint_df = pd.DataFrame(d_point_records)


# ------------------------------------------------------------
# Structural validation against frozen E1 findings
#
# IMPORTANT:
# These constants are validation targets only.
# They were NOT used to construct the associations.
# ------------------------------------------------------------

n_regions_total = len(e3_c1_d1_region_df)
n_d_total = len(e3_c1_d1_dpoint_df)

n_inside = int(
    e3_c1_d1_dpoint_df["inside_c_foreground"].sum()
)

n_outside = n_d_total - n_inside

n_zero = int(
    (e3_c1_d1_region_df["n_d_inside"] == 0).sum()
)

n_one = int(
    (e3_c1_d1_region_df["n_d_inside"] == 1).sum()
)

n_multi = int(
    (e3_c1_d1_region_df["n_d_inside"] >= 2).sum()
)


expected = {
    "C1 regions": 2554,
    "D1 points": 7652,
    "D1 inside C1": 4194,
    "D1 outside C1": 3458,
    "C1 zero-D regions": 399,
    "C1 one-D regions": 1528,
    "C1 multi-D regions": 627,
}

observed = {
    "C1 regions": n_regions_total,
    "D1 points": n_d_total,
    "D1 inside C1": n_inside,
    "D1 outside C1": n_outside,
    "C1 zero-D regions": n_zero,
    "C1 one-D regions": n_one,
    "C1 multi-D regions": n_multi,
}


print("=" * 100)
print("STEP 13D.2a.2 — C1+D1 ASSOCIATION VALIDATION")
print("=" * 100)

all_pass = True

for key in expected:

    ok = observed[key] == expected[key]
    all_pass &= ok

    print(
        f"{key:<25} "
        f"observed={observed[key]:>6}   "
        f"expected={expected[key]:>6}   "
        f"{'PASS' if ok else 'FAIL'}"
    )


# ------------------------------------------------------------
# Internal consistency checks for new distance primitives
# ------------------------------------------------------------

inside_df = e3_c1_d1_dpoint_df[
    e3_c1_d1_dpoint_df["inside_c_foreground"]
]

outside_df = e3_c1_d1_dpoint_df[
    ~e3_c1_d1_dpoint_df["inside_c_foreground"]
]

inside_region_distance_zero = np.allclose(
    inside_df["distance_to_nearest_region_px"],
    0.0
)

outside_region_distance_positive = (
    outside_df["distance_to_nearest_region_px"] > 0
).all()

inside_nearest_matches_containing = (
    inside_df["nearest_region_id"].astype(int).to_numpy()
    ==
    inside_df["containing_region_id"].astype(int).to_numpy()
).all()


print("\nDistance / association sanity checks:")

print(
    "  Inside D -> nearest-region distance = 0:",
    "PASS" if inside_region_distance_zero else "FAIL"
)

print(
    "  Outside D -> nearest-region distance > 0:",
    "PASS" if outside_region_distance_positive else "FAIL"
)

print(
    "  Inside D -> nearest region = containing region:",
    "PASS" if inside_nearest_matches_containing else "FAIL"
)


print("\nConstructed tables:")
print(
    "  e3_c1_d1_region_df:",
    e3_c1_d1_region_df.shape
)

print(
    "  e3_c1_d1_dpoint_df:",
    e3_c1_d1_dpoint_df.shape
)


assert all_pass, (
    "C1+D1 structural validation failed."
)

assert inside_region_distance_zero
assert outside_region_distance_positive
assert inside_nearest_matches_containing


print("\n" + "=" * 100)
print("Step 13D.2a.2 complete — C1+D1 association primitives validated.")
print("Construction is GT-free; E1 values were used only as post-construction validation targets.")
print("=" * 100)

STEP 13D.2a.2 — C1+D1 ASSOCIATION VALIDATION
C1 regions                observed=  2554   expected=  2554   PASS
D1 points                 observed=  7652   expected=  7652   PASS
D1 inside C1              observed=  4194   expected=  4194   PASS
D1 outside C1             observed=  3458   expected=  3458   PASS
C1 zero-D regions         observed=   399   expected=   399   PASS
C1 one-D regions          observed=  1528   expected=  1528   PASS
C1 multi-D regions        observed=   627   expected=   627   PASS

Distance / association sanity checks:
  Inside D -> nearest-region distance = 0: PASS
  Outside D -> nearest-region distance > 0: PASS
  Inside D -> nearest region = containing region: PASS

Constructed tables:
  e3_c1_d1_region_df: (2554, 11)
  e3_c1_d1_dpoint_df: (7652, 18)

Step 13D.2a.2 complete — C1+D1 association primitives validated.
Construction is GT-free; E1 values were used only as post-construction validation targets.


#### Step 13D.2a.3 — Generalise and Validate Association Primitives Across All C×D Pairs

The validated C1+D1 association construction is extended to all 15 retained C×D combinations:

\[
\{C1,C2b,C2c\}
\times
\{D1,D4,D5\text{-}768,D6B\text{-}T\text{-}native,D6B9\text{-}strict\}.
\]

The same GT-free association definitions are used for every representation pair.

For each D point, discrete C-region membership is determined from the frozen labelled region map. This is particularly important for the Taylor-refined D6 representations, whose coordinates may be subpixel-valued. The resulting all-pair structural summaries are therefore validated against the already-frozen E1 relationship analysis before the association infrastructure is accepted.

The validation checks membership and multiplicity quantities that overlap with E1. Newly introduced continuous distance primitives are checked independently for logical consistency.

No GT coordinates, GT counts, density strata, localisation matches, or evaluation-derived quantities are used to construct the associations.

In [63]:
# ============================================================
# Step 13D.2a.3 — Generalise and validate C–D association
# primitives across all 15 retained C×D pairs
# ============================================================

import numpy as np
import pandas as pd
from scipy.ndimage import distance_transform_edt


# ------------------------------------------------------------
# 1. Frozen representation inventory
# ------------------------------------------------------------

E3_C_VARIANTS = [
    "C1",
    "C2b",
    "C2c",
]

E3_D_VARIANTS = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]


# ------------------------------------------------------------
# 2. Build frozen D-row lookup
# ------------------------------------------------------------

e3_d_lookup = {}

for d_variant in E3_D_VARIANTS:

    subset = (
        group_d_handoff_df[
            group_d_handoff_df["candidate_id"] == d_variant
        ]
        .copy()
    )

    assert len(subset) == 100, (
        f"{d_variant}: expected 100 rows, found {len(subset)}"
    )

    e3_d_lookup[d_variant] = {
        int(row["dataset_index"]): row
        for _, row in subset.iterrows()
    }


# ------------------------------------------------------------
# 3. Construct GT-free association tables
# ------------------------------------------------------------

all_region_records = []
all_dpoint_records = []


for dataset_index, image_info in E1_REGION_MAPS.items():

    dataset_index = int(dataset_index)

    width = int(image_info["width"])
    height = int(image_info["height"])

    short_side = min(width, height)
    image_area = width * height

    # --------------------------------------------------------
    # Process each frozen C representation
    # --------------------------------------------------------

    for c_variant in E3_C_VARIANTS:

        c_obj = image_info[c_variant]

        labels = np.asarray(
            c_obj["labelled_regions"]
        )

        c_points = np.asarray(
            c_obj["points"],
            dtype=float,
        ).reshape(-1, 2)

        region_areas = np.asarray(
            c_obj["region_areas"],
            dtype=float,
        )

        n_regions = int(
            c_obj["num_regions"]
        )

        # Basic C-region integrity checks
        assert labels.shape == (height, width)

        assert len(c_points) == n_regions

        assert len(region_areas) == n_regions

        positive_labels = np.unique(labels)
        positive_labels = positive_labels[
            positive_labels > 0
        ]

        assert np.array_equal(
            positive_labels,
            np.arange(1, n_regions + 1),
        ), (
            f"Unexpected region labels for "
            f"dataset_index={dataset_index}, "
            f"C={c_variant}"
        )

        # ----------------------------------------------------
        # Region-distance maps depend only on C.
        #
        # distance_transform_edt(~region_mask):
        #   inside region  -> 0
        #   outside region -> Euclidean pixel distance
        # ----------------------------------------------------

        region_distance_maps = {}

        for region_id in range(1, n_regions + 1):

            region_mask = (
                labels == region_id
            )

            region_distance_maps[region_id] = (
                distance_transform_edt(
                    ~region_mask
                )
            )

        # ----------------------------------------------------
        # Pair this C representation with every frozen D
        # representation.
        # ----------------------------------------------------

        for d_variant in E3_D_VARIANTS:

            d_row = (
                e3_d_lookup[d_variant][dataset_index]
            )

            d_points = e3_parse_points_xy(
                d_row["pred_points_xy"]
            )

            assert len(d_points) == int(
                d_row["pred_count"]
            )

            # ------------------------------------------------
            # Discrete C-region membership
            #
            # D6 may contain Taylor-refined subpixel points.
            # We use nearest-pixel rounding here and validate
            # the resulting structure against frozen E1 below.
            # ------------------------------------------------

            containing_ids = []

            for x, y in d_points:

                px = int(
                    np.clip(
                        np.rint(x),
                        0,
                        width - 1,
                    )
                )

                py = int(
                    np.clip(
                        np.rint(y),
                        0,
                        height - 1,
                    )
                )

                containing_ids.append(
                    int(labels[py, px])
                )

            containing_ids = np.asarray(
                containing_ids,
                dtype=int,
            )

            # ------------------------------------------------
            # D multiplicity within actual containing regions
            # ------------------------------------------------

            multiplicity = np.bincount(
                containing_ids[
                    containing_ids > 0
                ],
                minlength=n_regions + 1,
            )

            # ------------------------------------------------
            # Per-C-region association records
            # ------------------------------------------------

            for region_id in range(
                1,
                n_regions + 1,
            ):

                centroid_x, centroid_y = (
                    c_points[region_id - 1]
                )

                area_px = float(
                    region_areas[region_id - 1]
                )

                all_region_records.append({
                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_info["image_id"],

                    "category":
                        d_row["category"],

                    "c_variant":
                        c_variant,

                    "d_variant":
                        d_variant,

                    "region_id":
                        region_id,

                    "centroid_x":
                        float(centroid_x),

                    "centroid_y":
                        float(centroid_y),

                    "region_area_px":
                        area_px,

                    "region_area_frac":
                        area_px / image_area,

                    "n_d_inside":
                        int(
                            multiplicity[
                                region_id
                            ]
                        ),
                })

            # ------------------------------------------------
            # Per-D-point association records
            # ------------------------------------------------

            for d_point_id, (x, y) in enumerate(
                d_points
            ):

                px = int(
                    np.clip(
                        np.rint(x),
                        0,
                        width - 1,
                    )
                )

                py = int(
                    np.clip(
                        np.rint(y),
                        0,
                        height - 1,
                    )
                )

                containing_region_id = int(
                    labels[py, px]
                )

                inside = (
                    containing_region_id > 0
                )

                # --------------------------------------------
                # Containing-region context
                # --------------------------------------------

                if inside:

                    containing_centroid = (
                        c_points[
                            containing_region_id - 1
                        ]
                    )

                    dist_containing_centroid_px = float(
                        np.linalg.norm(
                            np.asarray(
                                [x, y],
                                dtype=float,
                            )
                            -
                            containing_centroid
                        )
                    )

                    n_d_in_containing_region = int(
                        multiplicity[
                            containing_region_id
                        ]
                    )

                else:

                    dist_containing_centroid_px = (
                        np.nan
                    )

                    n_d_in_containing_region = 0

                # --------------------------------------------
                # Nearest-region context
                #
                # This does NOT imply membership.
                # --------------------------------------------

                if n_regions > 0:

                    region_distances = np.asarray(
                        [
                            region_distance_maps[
                                region_id
                            ][py, px]

                            for region_id in range(
                                1,
                                n_regions + 1,
                            )
                        ],
                        dtype=float,
                    )

                    nearest_region_id = (
                        int(
                            np.argmin(
                                region_distances
                            )
                        )
                        + 1
                    )

                    dist_nearest_region_px = float(
                        region_distances[
                            nearest_region_id - 1
                        ]
                    )

                    nearest_centroid = (
                        c_points[
                            nearest_region_id - 1
                        ]
                    )

                    dist_nearest_centroid_px = float(
                        np.linalg.norm(
                            np.asarray(
                                [x, y],
                                dtype=float,
                            )
                            -
                            nearest_centroid
                        )
                    )

                else:

                    nearest_region_id = np.nan
                    dist_nearest_region_px = np.nan
                    dist_nearest_centroid_px = np.nan

                # --------------------------------------------
                # Store D-point association record
                # --------------------------------------------

                all_dpoint_records.append({
                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_info["image_id"],

                    "category":
                        d_row["category"],

                    "c_variant":
                        c_variant,

                    "d_variant":
                        d_variant,

                    "d_point_id":
                        int(d_point_id),

                    "d_x":
                        float(x),

                    "d_y":
                        float(y),

                    "inside_c_foreground":
                        bool(inside),

                    "containing_region_id":
                        (
                            containing_region_id
                            if inside
                            else np.nan
                        ),

                    "n_d_in_containing_region":
                        n_d_in_containing_region,

                    "distance_to_containing_centroid_px":
                        dist_containing_centroid_px,

                    "distance_to_containing_centroid_norm":
                        (
                            dist_containing_centroid_px
                            / short_side

                            if np.isfinite(
                                dist_containing_centroid_px
                            )

                            else np.nan
                        ),

                    "nearest_region_id":
                        nearest_region_id,

                    "distance_to_nearest_region_px":
                        dist_nearest_region_px,

                    "distance_to_nearest_region_norm":
                        (
                            dist_nearest_region_px
                            / short_side

                            if np.isfinite(
                                dist_nearest_region_px
                            )

                            else np.nan
                        ),

                    "distance_to_nearest_region_centroid_px":
                        dist_nearest_centroid_px,

                    "distance_to_nearest_region_centroid_norm":
                        (
                            dist_nearest_centroid_px
                            / short_side

                            if np.isfinite(
                                dist_nearest_centroid_px
                            )

                            else np.nan
                        ),
                })


# ------------------------------------------------------------
# 4. Final GT-free association tables
# ------------------------------------------------------------

e3_all_pairs_region_df = pd.DataFrame(
    all_region_records
)

e3_all_pairs_dpoint_df = pd.DataFrame(
    all_dpoint_records
)


# ------------------------------------------------------------
# 5. Derive structural summary from new E3 infrastructure
# ------------------------------------------------------------

structural_records = []


for c_variant in E3_C_VARIANTS:

    for d_variant in E3_D_VARIANTS:

        r = e3_all_pairs_region_df[
            (
                e3_all_pairs_region_df[
                    "c_variant"
                ]
                == c_variant
            )
            &
            (
                e3_all_pairs_region_df[
                    "d_variant"
                ]
                == d_variant
            )
        ]

        d = e3_all_pairs_dpoint_df[
            (
                e3_all_pairs_dpoint_df[
                    "c_variant"
                ]
                == c_variant
            )
            &
            (
                e3_all_pairs_dpoint_df[
                    "d_variant"
                ]
                == d_variant
            )
        ]

        n_c_regions = len(r)
        n_d_points = len(d)

        n_d_inside = int(
            d[
                "inside_c_foreground"
            ].sum()
        )

        n_d_outside = (
            n_d_points - n_d_inside
        )

        n_zero = int(
            (
                r["n_d_inside"] == 0
            ).sum()
        )

        n_one = int(
            (
                r["n_d_inside"] == 1
            ).sum()
        )

        n_multi = int(
            (
                r["n_d_inside"] >= 2
            ).sum()
        )

        n_occupied = (
            n_one + n_multi
        )

        d_points_in_multi_regions = int(
            r.loc[
                r["n_d_inside"] >= 2,
                "n_d_inside",
            ].sum()
        )

        structural_records.append({
            "c_variant":
                c_variant,

            "d_variant":
                d_variant,

            "n_c_regions":
                n_c_regions,

            "n_d_points":
                n_d_points,

            "n_d_inside":
                n_d_inside,

            "n_d_outside":
                n_d_outside,

            "n_zero_regions":
                n_zero,

            "n_one_regions":
                n_one,

            "n_multi_regions":
                n_multi,

            "n_occupied_regions":
                n_occupied,

            "d_points_in_multi_regions":
                d_points_in_multi_regions,

            "d_inside_fraction":
                (
                    n_d_inside / n_d_points
                    if n_d_points > 0
                    else np.nan
                ),

            "d_outside_fraction":
                (
                    n_d_outside / n_d_points
                    if n_d_points > 0
                    else np.nan
                ),

            "c_zero_fraction":
                (
                    n_zero / n_c_regions
                    if n_c_regions > 0
                    else np.nan
                ),

            "c_one_fraction":
                (
                    n_one / n_c_regions
                    if n_c_regions > 0
                    else np.nan
                ),

            "c_multi_fraction":
                (
                    n_multi / n_c_regions
                    if n_c_regions > 0
                    else np.nan
                ),

            "c_occupied_fraction":
                (
                    n_occupied / n_c_regions
                    if n_c_regions > 0
                    else np.nan
                ),
        })


e3_all_pairs_structural_df = pd.DataFrame(
    structural_records
)


# ------------------------------------------------------------
# 6. Validate against ACTUAL frozen E1 schema
#
# E1 stores:
#   n_c_regions
#   n_d_points
#   d_inside_fraction
#   d_outside_fraction
#   c_zero_fraction
#   c_one_fraction
#   c_multi_fraction
#   c_occupied_fraction
#   n_multi_regions
#   d_points_in_multi_regions
#
# No guessed E1 column names are used here.
# ------------------------------------------------------------

E1_COUNT_FIELDS = [
    "n_c_regions",
    "n_d_points",
    "n_multi_regions",
    "d_points_in_multi_regions",
]

E1_FRACTION_FIELDS = [
    "d_inside_fraction",
    "d_outside_fraction",
    "c_zero_fraction",
    "c_one_fraction",
    "c_multi_fraction",
    "c_occupied_fraction",
]


required_e1_cols = (
    ["c_variant", "d_variant"]
    + E1_COUNT_FIELDS
    + E1_FRACTION_FIELDS
)

missing_e1_cols = [
    col
    for col in required_e1_cols
    if col not in e1_all_pairs_overall_df.columns
]

assert not missing_e1_cols, (
    "Missing expected frozen E1 columns: "
    + ", ".join(missing_e1_cols)
)


validation_records = []


for _, new_row in (
    e3_all_pairs_structural_df.iterrows()
):

    c_variant = new_row["c_variant"]
    d_variant = new_row["d_variant"]

    old_rows = e1_all_pairs_overall_df[
        (
            e1_all_pairs_overall_df[
                "c_variant"
            ]
            == c_variant
        )
        &
        (
            e1_all_pairs_overall_df[
                "d_variant"
            ]
            == d_variant
        )
    ]

    assert len(old_rows) == 1, (
        f"Expected exactly one frozen E1 row for "
        f"{c_variant}+{d_variant}; "
        f"found {len(old_rows)}."
    )

    old_row = old_rows.iloc[0]

    record = {
        "c_variant":
            c_variant,

        "d_variant":
            d_variant,
    }

    pair_pass = True

    # Exact count comparisons
    for field in E1_COUNT_FIELDS:

        new_value = int(
            new_row[field]
        )

        old_value = int(
            old_row[field]
        )

        passed = (
            new_value == old_value
        )

        record[
            f"{field}_pass"
        ] = passed

        pair_pass &= passed

    # Floating-point fraction comparisons
    for field in E1_FRACTION_FIELDS:

        new_value = float(
            new_row[field]
        )

        old_value = float(
            old_row[field]
        )

        passed = np.isclose(
            new_value,
            old_value,
            rtol=1e-10,
            atol=1e-12,
        )

        record[
            f"{field}_pass"
        ] = bool(passed)

        pair_pass &= bool(passed)

    record["pair_pass"] = (
        bool(pair_pass)
    )

    validation_records.append(
        record
    )


e3_all_pairs_validation_df = pd.DataFrame(
    validation_records
)


# ------------------------------------------------------------
# 7. Independent sanity checks for new distance primitives
# ------------------------------------------------------------

inside_df = e3_all_pairs_dpoint_df[
    e3_all_pairs_dpoint_df[
        "inside_c_foreground"
    ]
]

outside_df = e3_all_pairs_dpoint_df[
    ~e3_all_pairs_dpoint_df[
        "inside_c_foreground"
    ]
]


inside_region_distance_zero = np.allclose(
    inside_df[
        "distance_to_nearest_region_px"
    ],
    0.0,
)


outside_region_distance_positive = (
    outside_df[
        "distance_to_nearest_region_px"
    ]
    > 0
).all()


inside_nearest_matches_containing = (
    inside_df[
        "nearest_region_id"
    ]
    .astype(int)
    .to_numpy()
    ==
    inside_df[
        "containing_region_id"
    ]
    .astype(int)
    .to_numpy()
).all()


# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

print("=" * 110)

print(
    "STEP 13D.2a.3 — ALL-PAIR C–D "
    "ASSOCIATION VALIDATION"
)

print("=" * 110)


print("\nConstructed GT-free tables:")

print(
    "  e3_all_pairs_region_df:",
    e3_all_pairs_region_df.shape,
)

print(
    "  e3_all_pairs_dpoint_df:",
    e3_all_pairs_dpoint_df.shape,
)

print(
    "  e3_all_pairs_structural_df:",
    e3_all_pairs_structural_df.shape,
)


print("\nPer-pair frozen-E1 validation:")

print(
    e3_all_pairs_validation_df[
        [
            "c_variant",
            "d_variant",
            "pair_pass",
        ]
    ].to_string(index=False)
)


all_pairs_pass = bool(
    e3_all_pairs_validation_df[
        "pair_pass"
    ].all()
)


print(
    "\nDistance / association sanity checks:"
)

print(
    "  Inside D -> nearest-region distance = 0:",
    (
        "PASS"
        if inside_region_distance_zero
        else "FAIL"
    ),
)

print(
    "  Outside D -> nearest-region distance > 0:",
    (
        "PASS"
        if outside_region_distance_positive
        else "FAIL"
    ),
)

print(
    "  Inside D -> nearest region = containing region:",
    (
        "PASS"
        if inside_nearest_matches_containing
        else "FAIL"
    ),
)


print("\nD6 subpixel membership validation:")

d6_validation_df = (
    e3_all_pairs_validation_df[
        e3_all_pairs_validation_df[
            "d_variant"
        ].isin(
            [
                "D6B-T-native",
                "D6B9-strict",
            ]
        )
    ][
        [
            "c_variant",
            "d_variant",
            "pair_pass",
        ]
    ]
)

print(
    d6_validation_df.to_string(
        index=False
    )
)


print("\nOverall:")

print(
    "  All 15 pairs reproduce frozen E1 structure:",
    (
        "PASS"
        if all_pairs_pass
        else "FAIL"
    ),
)


# ------------------------------------------------------------
# 9. Freeze only if every validation passes
# ------------------------------------------------------------

assert all_pairs_pass, (
    "At least one C×D pair does not reproduce "
    "the frozen E1 structural analysis."
)

assert inside_region_distance_zero, (
    "Inside-D region-distance sanity check failed."
)

assert outside_region_distance_positive, (
    "Outside-D region-distance sanity check failed."
)

assert inside_nearest_matches_containing, (
    "Inside-D containing/nearest-region "
    "consistency check failed."
)


print("\n" + "=" * 110)

print(
    "Step 13D.2a.3 complete — all 15 C×D "
    "association structures validated."
)

print(
    "The D6 subpixel membership convention is "
    "consistent with the frozen E1 analysis."
)

print(
    "The GT-free association infrastructure is "
    "ready for subsequent E3 fusion experiments."
)

print("=" * 110)

STEP 13D.2a.3 — ALL-PAIR C–D ASSOCIATION VALIDATION

Constructed GT-free tables:
  e3_all_pairs_region_df: (63995, 11)
  e3_all_pairs_dpoint_df: (139305, 18)
  e3_all_pairs_structural_df: (15, 17)

Per-pair frozen-E1 validation:
c_variant    d_variant  pair_pass
       C1           D1       True
       C1           D4       True
       C1       D5-768       True
       C1 D6B-T-native       True
       C1  D6B9-strict       True
      C2b           D1       True
      C2b           D4       True
      C2b       D5-768       True
      C2b D6B-T-native       True
      C2b  D6B9-strict       True
      C2c           D1       True
      C2c           D4       True
      C2c       D5-768       True
      C2c D6B-T-native       True
      C2c  D6B9-strict       True

Distance / association sanity checks:
  Inside D -> nearest-region distance = 0: PASS
  Outside D -> nearest-region distance > 0: PASS
  Inside D -> nearest region = containing region: PASS

D6 subpixel membership validation:


#### Step 13D.2a.4 — Save and Freeze Validated C–D Association Artefacts

The validated GT-free C–D association infrastructure is saved before semantic-fusion experiments begin.

The frozen artefacts preserve:

- per-region C geometry and D-point multiplicity;
- per-D-point membership and nearest-region context;
- region- and centroid-distance primitives;
- all-pair structural summaries;
- validation results against the frozen E1 relationship analysis.

These artefacts contain no GT coordinates, GT counts, density strata, localisation matches, or other evaluation-derived information used as association inputs.

The saved association infrastructure is treated as fixed for the subsequent simple and structured semantic-fusion experiments. Any later extension, such as representation-specific activation confidence, will be introduced explicitly rather than silently modifying these frozen primitives.

In [64]:
# ============================================================
# Step 13D.2a.4 — Save and freeze validated GT-free
# C–D association artefacts
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_objects = [
    "e3_all_pairs_region_df",
    "e3_all_pairs_dpoint_df",
    "e3_all_pairs_structural_df",
    "e3_all_pairs_validation_df",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required E3 association objects: "
    + ", ".join(missing_objects)
)


# All 15 C×D combinations must have passed validation
assert len(e3_all_pairs_validation_df) == 15

assert bool(
    e3_all_pairs_validation_df[
        "pair_pass"
    ].all()
), (
    "Cannot freeze E3 association artefacts: "
    "not all 15 C×D pairs passed validation."
)


# ------------------------------------------------------------
# 2. Define output directory
#
# E2_RESULTS_DIR is already defined earlier as:
#   .../results/B1_experiments/Group_E/E2_GT_complementarity
#
# Therefore its parent is the existing Group_E directory.
# ------------------------------------------------------------

assert "E2_RESULTS_DIR" in globals(), (
    "E2_RESULTS_DIR is not defined. "
    "Run the earlier Group-E setup/E2 cells first."
)

GROUP_E_RESULTS_DIR = E2_RESULTS_DIR.parent

E3_RESULTS_DIR = (
    GROUP_E_RESULTS_DIR
    / "E3_semantic_integration"
)

E3_ASSOC_RESULTS_DIR = (
    E3_RESULTS_DIR
    / "association_primitives"
)

E3_ASSOC_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("GROUP_E_RESULTS_DIR :", GROUP_E_RESULTS_DIR)
print("E3_RESULTS_DIR      :", E3_RESULTS_DIR)
print("E3_ASSOC_RESULTS_DIR:", E3_ASSOC_RESULTS_DIR)



# ------------------------------------------------------------
# 3. Define output paths
# ------------------------------------------------------------

E3_ASSOC_REGION_PATH = (
    E3_ASSOC_RESULTS_DIR
    / "E3_association_per_region.csv"
)

E3_ASSOC_DPOINT_PATH = (
    E3_ASSOC_RESULTS_DIR
    / "E3_association_per_D_point.csv"
)

E3_ASSOC_STRUCTURAL_PATH = (
    E3_ASSOC_RESULTS_DIR
    / "E3_association_structural_summary.csv"
)

E3_ASSOC_VALIDATION_PATH = (
    E3_ASSOC_RESULTS_DIR
    / "E3_association_validation.csv"
)

E3_ASSOC_METADATA_PATH = (
    E3_ASSOC_RESULTS_DIR
    / "E3_association_metadata.json"
)


# ------------------------------------------------------------
# 4. Explicit GT-leakage check
#
# These columns must NOT appear in the frozen inference-side
# association tables.
# ------------------------------------------------------------

forbidden_columns = {
    "gt_count",
    "gt_points_xy",
    "density_stratum",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "radius_px",
}


for table_name, df in [
    (
        "per-region association table",
        e3_all_pairs_region_df,
    ),
    (
        "per-D-point association table",
        e3_all_pairs_dpoint_df,
    ),
]:

    leaked = sorted(
        forbidden_columns.intersection(
            df.columns
        )
    )

    assert not leaked, (
        f"{table_name} contains forbidden "
        f"evaluation-side columns: {leaked}"
    )


# ------------------------------------------------------------
# 5. Save validated association tables
# ------------------------------------------------------------

e3_all_pairs_region_df.to_csv(
    E3_ASSOC_REGION_PATH,
    index=False,
)

e3_all_pairs_dpoint_df.to_csv(
    E3_ASSOC_DPOINT_PATH,
    index=False,
)

e3_all_pairs_structural_df.to_csv(
    E3_ASSOC_STRUCTURAL_PATH,
    index=False,
)

e3_all_pairs_validation_df.to_csv(
    E3_ASSOC_VALIDATION_PATH,
    index=False,
)


# ------------------------------------------------------------
# 6. Freeze metadata
# ------------------------------------------------------------

e3_assoc_metadata = {

    "step":
        "13D.2a.4",

    "stage":
        "E3 semantic integration - association primitives",

    "dataset":
        "FSC147",

    "subset":
        "frozen validation dev100",

    "construction":
        "GT-free",

    "c_variants":
        E3_C_VARIANTS,

    "d_variants":
        E3_D_VARIANTS,

    "n_c_d_pairs":
        15,

    "n_images":
        100,

    "n_region_pair_rows":
        int(
            len(e3_all_pairs_region_df)
        ),

    "n_dpoint_pair_rows":
        int(
            len(e3_all_pairs_dpoint_df)
        ),

    "n_structural_summary_rows":
        int(
            len(e3_all_pairs_structural_df)
        ),

    "n_validation_rows":
        int(
            len(e3_all_pairs_validation_df)
        ),

    "all_pairs_validation_pass":
        bool(
            e3_all_pairs_validation_df[
                "pair_pass"
            ].all()
        ),

    "membership_rule":
        (
            "nearest-pixel lookup using "
            "np.rint(), clipped to image bounds"
        ),

    "membership_validation":
        (
            "all 15 CxD pairs reproduce frozen "
            "E1 structural relationship analysis"
        ),

    "d6_subpixel_validation":
        (
            "D6B-T-native and D6B9-strict "
            "membership results reproduce frozen E1"
        ),

    "core_association_primitives": [
        "C-region membership",
        "D multiplicity per C region",
        "distance to containing-region centroid",
        "nearest C-region identity",
        "distance to nearest C-region support",
        "distance to nearest C-region centroid",
        "image-normalised distance quantities",
        "C-region area",
        "image-normalised C-region area",
    ],

    "d_native_confidence":
        (
            "not included in common frozen association "
            "infrastructure; unavailable as a common "
            "per-point field in frozen D handoff"
        ),

    "gt_information_used_for_construction":
        False,

    "density_strata_used_for_construction":
        False,

    "fusion_rule_introduced":
        False,

    "notes": [
        (
            "Nearest-region context for an outside D point "
            "does not imply C-region membership."
        ),
        (
            "D multiplicity counts only D points actually "
            "inside each C region."
        ),
        (
            "Continuous distance quantities are preserved "
            "without applying fusion thresholds."
        ),
        (
            "Frozen E1 results are used only for "
            "post-construction implementation validation."
        ),
    ],
}


with open(
    E3_ASSOC_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        e3_assoc_metadata,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 7. Reload and verify exact saved artefacts
# ------------------------------------------------------------

check_region_df = pd.read_csv(
    E3_ASSOC_REGION_PATH
)

check_dpoint_df = pd.read_csv(
    E3_ASSOC_DPOINT_PATH
)

check_structural_df = pd.read_csv(
    E3_ASSOC_STRUCTURAL_PATH
)

check_validation_df = pd.read_csv(
    E3_ASSOC_VALIDATION_PATH
)

with open(
    E3_ASSOC_METADATA_PATH,
    "r",
    encoding="utf-8",
) as f:

    check_metadata = json.load(f)


assert check_region_df.shape == (
    e3_all_pairs_region_df.shape
)

assert check_dpoint_df.shape == (
    e3_all_pairs_dpoint_df.shape
)

assert check_structural_df.shape == (
    e3_all_pairs_structural_df.shape
)

assert check_validation_df.shape == (
    e3_all_pairs_validation_df.shape
)

assert bool(
    check_validation_df[
        "pair_pass"
    ].all()
)

assert (
    check_metadata[
        "all_pairs_validation_pass"
    ]
    is True
)

assert (
    check_metadata[
        "gt_information_used_for_construction"
    ]
    is False
)

assert (
    check_metadata[
        "fusion_rule_introduced"
    ]
    is False
)


# ------------------------------------------------------------
# 8. Final freeze report
# ------------------------------------------------------------

print("=" * 110)

print(
    "STEP 13D.2a.4 — E3 ASSOCIATION "
    "ARTEFACT FREEZE"
)

print("=" * 110)


print("\nFrozen association infrastructure:")

print(
    f"  C representations: "
    f"{E3_C_VARIANTS}"
)

print(
    f"  D representations: "
    f"{E3_D_VARIANTS}"
)

print(
    f"  C×D combinations: "
    f"{len(E3_C_VARIANTS) * len(E3_D_VARIANTS)}"
)

print(
    f"  Per-region rows: "
    f"{len(e3_all_pairs_region_df)}"
)

print(
    f"  Per-D-point rows: "
    f"{len(e3_all_pairs_dpoint_df)}"
)

print(
    f"  Structural-summary rows: "
    f"{len(e3_all_pairs_structural_df)}"
)

print(
    f"  Validation rows: "
    f"{len(e3_all_pairs_validation_df)}"
)


print("\nValidation:")

print(
    "  All 15 pairs reproduce frozen E1 structure:",
    "PASS"
)

print(
    "  GT/evaluation columns absent from "
    "association tables:",
    "PASS"
)

print(
    "  Saved-file reload validation:",
    "PASS"
)


print("\nSaved artefacts:")

for path in [
    E3_ASSOC_REGION_PATH,
    E3_ASSOC_DPOINT_PATH,
    E3_ASSOC_STRUCTURAL_PATH,
    E3_ASSOC_VALIDATION_PATH,
    E3_ASSOC_METADATA_PATH,
]:
    print(f"  {path}")


print("\n" + "=" * 110)

print(
    "Step 13D.2a.4 complete — validated GT-free "
    "C–D association infrastructure frozen."
)

print(
    "Step 13D.2a is complete."
)

print(
    "Next: Step 13D.2b — Simple Semantic-Fusion Baselines."
)

print("=" * 110)

GROUP_E_RESULTS_DIR : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_E
E3_RESULTS_DIR      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_E/E3_semantic_integration
E3_ASSOC_RESULTS_DIR: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_E/E3_semantic_integration/association_primitives
STEP 13D.2a.4 — E3 ASSOCIATION ARTEFACT FREEZE

Frozen association infrastructure:
  C representations: ['C1', 'C2b', 'C2c']
  D representations: ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']
  C×D combinations: 15
  Per-region rows: 63995
  Per-D-point rows: 139305
  Structural-summary rows: 15
  Validation rows: 15

Validation:
  All 15 pairs reproduce frozen E1 structure: PASS
  GT/evaluation columns absent from association tables: PASS
  Saved-file reload validation: PASS

Saved artefacts:
  /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_E/E3_semantic_integr

In [65]:
# ============================================================
# Step 13D.2b.1 — GT-Free C–D Point Association
# Part A: Confirm frozen 13D.2a association-table schemas
# ============================================================

print("=" * 110)
print("STEP 13D.2b.1 — FROZEN ASSOCIATION SCHEMA CHECK")
print("=" * 110)

tables = {
    "e3_all_pairs_region_df": e3_all_pairs_region_df,
    "e3_all_pairs_dpoint_df": e3_all_pairs_dpoint_df,
    "e3_all_pairs_structural_df": e3_all_pairs_structural_df,
}

for name, df in tables.items():
    print(f"\n{name}")
    print("-" * 110)
    print("Shape:", df.shape)
    print("Columns:")
    for col in df.columns:
        print(f"  {col}")

print("\n" + "=" * 110)
print("Schema check complete.")
print("No association threshold, fusion rule, or GT information has been applied.")
print("=" * 110)

STEP 13D.2b.1 — FROZEN ASSOCIATION SCHEMA CHECK

e3_all_pairs_region_df
--------------------------------------------------------------------------------------------------------------
Shape: (63995, 11)
Columns:
  dataset_index
  image_id
  category
  c_variant
  d_variant
  region_id
  centroid_x
  centroid_y
  region_area_px
  region_area_frac
  n_d_inside

e3_all_pairs_dpoint_df
--------------------------------------------------------------------------------------------------------------
Shape: (139305, 18)
Columns:
  dataset_index
  image_id
  category
  c_variant
  d_variant
  d_point_id
  d_x
  d_y
  inside_c_foreground
  containing_region_id
  n_d_in_containing_region
  distance_to_containing_centroid_px
  distance_to_containing_centroid_norm
  nearest_region_id
  distance_to_nearest_region_px
  distance_to_nearest_region_norm
  distance_to_nearest_region_centroid_px
  distance_to_nearest_region_centroid_norm

e3_all_pairs_structural_df
-------------------------------------------

In [66]:
# ============================================================
# Step 13D.2b.1 — Define and validate GT-free spatial
# consolidation rule
#
# Part B: Inspect outside-D-to-nearest-C-region distances
# before defining the bounded tau sweep.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 110)
print("STEP 13D.2b.1 — OUTSIDE-D / NEAREST-C DISTANCE DIAGNOSTIC")
print("=" * 110)


# ------------------------------------------------------------
# 1. Use frozen 13D.2a per-D-point association table
# ------------------------------------------------------------

assoc_df = e3_all_pairs_dpoint_df.copy()

required_cols = [
    "c_variant",
    "d_variant",
    "inside_c_foreground",
    "distance_to_nearest_region_px",
    "distance_to_nearest_region_norm",
]

missing = [
    col for col in required_cols
    if col not in assoc_df.columns
]

assert not missing, (
    "Missing required frozen association columns: "
    + ", ".join(missing)
)


# ------------------------------------------------------------
# 2. Separate inside and outside D points
#
# Inside points already have exact C-region membership.
# Only outside points require a distance-based association rule.
# ------------------------------------------------------------

inside_df = assoc_df[
    assoc_df["inside_c_foreground"].astype(bool)
].copy()

outside_df = assoc_df[
    ~assoc_df["inside_c_foreground"].astype(bool)
].copy()


print(f"\nAll D-point pair rows : {len(assoc_df):,}")
print(f"Inside C foreground   : {len(inside_df):,}")
print(f"Outside C foreground  : {len(outside_df):,}")

assert (
    len(inside_df) + len(outside_df)
    == len(assoc_df)
)


# ------------------------------------------------------------
# 3. Sanity checks inherited from 13D.2a
# ------------------------------------------------------------

assert np.allclose(
    inside_df["distance_to_nearest_region_px"],
    0.0,
), (
    "Some inside-C D points have non-zero "
    "nearest-region distance."
)

assert (
    outside_df["distance_to_nearest_region_px"] > 0
).all(), (
    "Some outside-C D points have non-positive "
    "nearest-region distance."
)

assert (
    outside_df["distance_to_nearest_region_norm"] > 0
).all()


# ------------------------------------------------------------
# 4. Overall outside-point distance distribution
# ------------------------------------------------------------

quantiles = [
    0.00,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1.00,
]

overall_quantiles = (
    outside_df[
        "distance_to_nearest_region_norm"
    ]
    .quantile(quantiles)
    .rename("distance_norm")
    .reset_index()
    .rename(columns={"index": "quantile"})
)

overall_quantiles["distance_percent_short_side"] = (
    100.0 * overall_quantiles["distance_norm"]
)


print("\n" + "-" * 110)
print("Outside-D distance to nearest C-region support")
print("Overall distribution")
print("-" * 110)

print(
    overall_quantiles.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 5. Distribution by C×D representation pair
# ------------------------------------------------------------

pair_summary = (
    outside_df
    .groupby(
        ["c_variant", "d_variant"],
        as_index=False,
    )
    .agg(
        n_outside=(
            "distance_to_nearest_region_norm",
            "size",
        ),
        mean_distance_norm=(
            "distance_to_nearest_region_norm",
            "mean",
        ),
        median_distance_norm=(
            "distance_to_nearest_region_norm",
            "median",
        ),
        p90_distance_norm=(
            "distance_to_nearest_region_norm",
            lambda x: x.quantile(0.90),
        ),
        p95_distance_norm=(
            "distance_to_nearest_region_norm",
            lambda x: x.quantile(0.95),
        ),
    )
)

for col in [
    "mean_distance_norm",
    "median_distance_norm",
    "p90_distance_norm",
    "p95_distance_norm",
]:
    pair_summary[
        col.replace("_norm", "_pct_short_side")
    ] = 100.0 * pair_summary[col]


print("\n" + "-" * 110)
print("Outside-D distance distribution by C×D pair")
print("-" * 110)

display_cols = [
    "c_variant",
    "d_variant",
    "n_outside",
    "median_distance_pct_short_side",
    "p90_distance_pct_short_side",
    "p95_distance_pct_short_side",
]

print(
    pair_summary[display_cols]
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Candidate bounded tau values
#
# These are NOT selected here.
# They are only reference thresholds whose coverage is
# measured GT-free.
# ------------------------------------------------------------

tau_candidates = [
    0.005,
    0.010,
    0.020,
    0.030,
    0.050,
]

coverage_records = []

for tau in tau_candidates:

    associated = (
        outside_df[
            "distance_to_nearest_region_norm"
        ] <= tau
    )

    coverage_records.append({
        "tau": tau,
        "tau_pct_short_side": 100.0 * tau,
        "n_outside_associated": int(
            associated.sum()
        ),
        "n_outside_total": int(
            len(outside_df)
        ),
        "outside_association_fraction": float(
            associated.mean()
        ),
    })


tau_coverage_df = pd.DataFrame(
    coverage_records
)


print("\n" + "-" * 110)
print("GT-free outside-point coverage at reference tau values")
print("-" * 110)

print(
    tau_coverage_df.to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print(
    "Diagnostic complete — no tau selected, "
    "no fusion performed, and no GT used."
)
print("=" * 110)

STEP 13D.2b.1 — OUTSIDE-D / NEAREST-C DISTANCE DIAGNOSTIC

All D-point pair rows : 139,305
Inside C foreground   : 50,535
Outside C foreground  : 88,770

--------------------------------------------------------------------------------------------------------------
Outside-D distance to nearest C-region support
Overall distribution
--------------------------------------------------------------------------------------------------------------
 quantile  distance_norm  distance_percent_short_side
     0.00       0.002604                     0.260417
     0.10       0.011646                     1.164619
     0.25       0.031250                     3.125000
     0.50       0.067758                     6.775839
     0.75       0.138119                    13.811907
     0.90       0.250122                    25.012204
     0.95       0.328166                    32.816633
     0.99       0.515316                    51.531583
     1.00       0.894993                    89.499264

---------------

In [67]:
# ============================================================
# Step 13D.2b.1 — Define and validate GT-free C–D association
#
# Part C: Bounded tau sweep
#
# Purpose:
#   Characterise how the frozen C–D association structure
#   changes as the outside-region distance tolerance tau varies.
#
# IMPORTANT:
#   - Inside-C D points are always associated with their
#     containing C region.
#   - Outside-C D points are associated with the nearest
#     C region only when:
#
#       distance_to_nearest_region_norm <= tau
#
#   - No GT is used.
#   - No F1/F2/F3/F4 fusion is performed.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 115)
print("STEP 13D.2b.1 PART C — BOUNDED GT-FREE C–D ASSOCIATION SWEEP")
print("=" * 115)


# ------------------------------------------------------------
# 1. Frozen tolerance sweep
# ------------------------------------------------------------

E3_ASSOC_TAU_VALUES = [
    0.005,
    0.010,
    0.020,
    0.030,
    0.050,
]

print("\nAssociation tolerances:")
for tau in E3_ASSOC_TAU_VALUES:
    print(
        f"  tau = {tau:.3f} "
        f"({100.0 * tau:.1f}% of image short side)"
    )


# ------------------------------------------------------------
# 2. Validate required frozen 13D.2a columns
# ------------------------------------------------------------

dpoint_df = e3_all_pairs_dpoint_df.copy()

required_cols = [
    "dataset_index",
    "image_id",
    "c_variant",
    "d_variant",
    "d_point_id",
    "inside_c_foreground",
    "containing_region_id",
    "nearest_region_id",
    "distance_to_nearest_region_norm",
]

missing = [
    col for col in required_cols
    if col not in dpoint_df.columns
]

assert not missing, (
    "Missing required columns: "
    + ", ".join(missing)
)


# ------------------------------------------------------------
# 3. Construct GT-free association assignments
#
# Association rule:
#
#   inside C:
#       associated with containing_region_id
#
#   outside C:
#       associated with nearest_region_id iff
#       nearest-region distance <= tau
#
# Otherwise:
#       unassociated
# ------------------------------------------------------------

assignment_frames = []
summary_records = []

for tau in E3_ASSOC_TAU_VALUES:

    work = dpoint_df.copy()

    inside = (
        work["inside_c_foreground"]
        .astype(bool)
    )

    outside_close = (
        (~inside)
        &
        (
            work["distance_to_nearest_region_norm"]
            <= tau
        )
    )

    associated = inside | outside_close


    # --------------------------------------------------------
    # Assigned C-region identity
    # --------------------------------------------------------

    work["tau"] = tau

    work["association_source"] = np.select(
        [
            inside,
            outside_close,
        ],
        [
            "inside_membership",
            "outside_nearest_region",
        ],
        default="unassociated",
    )

    work["associated_c_region_id"] = np.nan

    work.loc[
        inside,
        "associated_c_region_id"
    ] = work.loc[
        inside,
        "containing_region_id"
    ]

    work.loc[
        outside_close,
        "associated_c_region_id"
    ] = work.loc[
        outside_close,
        "nearest_region_id"
    ]

    work["is_associated"] = associated


    # Keep assignment table for later F1–F4 experiments
    assignment_frames.append(work)


    # --------------------------------------------------------
    # 4. Overall structural summary at this tau
    # --------------------------------------------------------

    n_total = len(work)

    n_inside = int(inside.sum())

    n_outside_close = int(
        outside_close.sum()
    )

    n_associated = int(
        associated.sum()
    )

    n_unassociated = (
        n_total - n_associated
    )

    summary_records.append({
        "tau": tau,
        "tau_pct_short_side": 100.0 * tau,

        "n_d_pair_rows": n_total,

        "n_inside_membership": n_inside,

        "n_outside_distance_associated":
            n_outside_close,

        "n_associated_total":
            n_associated,

        "n_unassociated":
            n_unassociated,

        "associated_fraction":
            n_associated / n_total,

        "unassociated_fraction":
            n_unassociated / n_total,
    })


# ------------------------------------------------------------
# 5. Combine assignments
# ------------------------------------------------------------

e3_tau_assignment_df = pd.concat(
    assignment_frames,
    ignore_index=True,
)

e3_tau_summary_df = pd.DataFrame(
    summary_records
)


# ------------------------------------------------------------
# 6. Calculate resulting C-region multiplicity
#
# For each image / C×D pair / tau / C region:
# how many D points are associated with that C region?
# ------------------------------------------------------------

associated_only = (
    e3_tau_assignment_df[
        e3_tau_assignment_df["is_associated"]
    ]
    .copy()
)

region_counts = (
    associated_only
    .groupby(
        [
            "tau",
            "dataset_index",
            "image_id",
            "c_variant",
            "d_variant",
            "associated_c_region_id",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_d_associated"
        }
    )
)


# ------------------------------------------------------------
# Need ALL C regions, including those with zero associated D.
# Cross the frozen region table with tau.
# ------------------------------------------------------------

tau_df = pd.DataFrame({
    "tau": E3_ASSOC_TAU_VALUES
})

all_regions = (
    e3_all_pairs_region_df.assign(
        _key=1
    )
    .merge(
        tau_df.assign(_key=1),
        on="_key",
    )
    .drop(columns="_key")
)

all_regions = all_regions.merge(
    region_counts,
    how="left",
    left_on=[
        "tau",
        "dataset_index",
        "image_id",
        "c_variant",
        "d_variant",
        "region_id",
    ],
    right_on=[
        "tau",
        "dataset_index",
        "image_id",
        "c_variant",
        "d_variant",
        "associated_c_region_id",
    ],
)

all_regions["n_d_associated"] = (
    all_regions["n_d_associated"]
    .fillna(0)
    .astype(int)
)

e3_tau_region_df = all_regions


# ------------------------------------------------------------
# 7. Region-level multiplicity summary by tau
# ------------------------------------------------------------

region_tau_summary = (
    e3_tau_region_df
    .groupby(
        "tau",
        as_index=False,
    )
    .agg(
        n_region_rows=(
            "region_id",
            "size",
        ),

        n_zero_regions=(
            "n_d_associated",
            lambda x: int((x == 0).sum()),
        ),

        n_one_regions=(
            "n_d_associated",
            lambda x: int((x == 1).sum()),
        ),

        n_multi_regions=(
            "n_d_associated",
            lambda x: int((x >= 2).sum()),
        ),

        mean_d_per_region=(
            "n_d_associated",
            "mean",
        ),

        median_d_per_region=(
            "n_d_associated",
            "median",
        ),

        max_d_per_region=(
            "n_d_associated",
            "max",
        ),
    )
)

region_tau_summary[
    "zero_region_fraction"
] = (
    region_tau_summary["n_zero_regions"]
    /
    region_tau_summary["n_region_rows"]
)

region_tau_summary[
    "one_region_fraction"
] = (
    region_tau_summary["n_one_regions"]
    /
    region_tau_summary["n_region_rows"]
)

region_tau_summary[
    "multi_region_fraction"
] = (
    region_tau_summary["n_multi_regions"]
    /
    region_tau_summary["n_region_rows"]
)


# ------------------------------------------------------------
# 8. Merge D-point and C-region summaries
# ------------------------------------------------------------

e3_tau_structural_summary_df = (
    e3_tau_summary_df
    .merge(
        region_tau_summary,
        on="tau",
        how="left",
    )
)


# ------------------------------------------------------------
# 9. Print compact result
# ------------------------------------------------------------

print("\n" + "-" * 115)
print("GT-free association structure across tau")
print("-" * 115)

display_cols = [
    "tau",
    "tau_pct_short_side",
    "associated_fraction",
    "n_outside_distance_associated",
    "zero_region_fraction",
    "one_region_fraction",
    "multi_region_fraction",
    "mean_d_per_region",
]

print(
    e3_tau_structural_summary_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 10. Sanity checks
# ------------------------------------------------------------

# Association must be monotonic as tau increases.
assert (
    e3_tau_structural_summary_df[
        "associated_fraction"
    ]
    .is_monotonic_increasing
)

# Number of zero-D C regions should not increase.
assert (
    e3_tau_structural_summary_df[
        "n_zero_regions"
    ]
    .is_monotonic_decreasing
)

# Number of assignment rows:
# original D-pair rows × number of tau values.
expected_assignment_rows = (
    len(e3_all_pairs_dpoint_df)
    *
    len(E3_ASSOC_TAU_VALUES)
)

assert (
    len(e3_tau_assignment_df)
    ==
    expected_assignment_rows
)

# Every associated point must have a C region.
assert (
    e3_tau_assignment_df.loc[
        e3_tau_assignment_df["is_associated"],
        "associated_c_region_id",
    ]
    .notna()
    .all()
)


print("\n" + "-" * 115)
print("Sanity checks")
print("-" * 115)

print("Association fraction monotonic with tau: PASS")
print("Zero-region count non-increasing with tau: PASS")
print("Assignment-row count: PASS")
print("All associated D points have a C-region identity: PASS")


print("\nConstructed objects:")
print(
    "  e3_tau_assignment_df           :",
    e3_tau_assignment_df.shape,
)
print(
    "  e3_tau_region_df               :",
    e3_tau_region_df.shape,
)
print(
    "  e3_tau_structural_summary_df   :",
    e3_tau_structural_summary_df.shape,
)


print("\n" + "=" * 115)
print(
    "Step 13D.2b.1 Part C complete — "
    "bounded GT-free association sweep constructed."
)
print(
    "No GT used. No F1/F2/F3/F4 fusion performed."
)
print("=" * 115)

STEP 13D.2b.1 PART C — BOUNDED GT-FREE C–D ASSOCIATION SWEEP

Association tolerances:
  tau = 0.005 (0.5% of image short side)
  tau = 0.010 (1.0% of image short side)
  tau = 0.020 (2.0% of image short side)
  tau = 0.030 (3.0% of image short side)
  tau = 0.050 (5.0% of image short side)

-------------------------------------------------------------------------------------------------------------------
GT-free association structure across tau
-------------------------------------------------------------------------------------------------------------------
   tau  tau_pct_short_side  associated_fraction  n_outside_distance_associated  zero_region_fraction  one_region_fraction  multi_region_fraction  mean_d_per_region
0.0050              0.5000               0.3831                           2832                0.4894               0.3802                 0.1304             0.8339
0.0100              1.0000               0.4151                           7287                0.4571       

In [68]:
# ============================================================
# Step 13D.2b.1 — Define and validate GT-free C–D association
#
# Part D: C×D pair-level tau characterisation
#
# Purpose:
#   Examine how the bounded association tolerance affects each
#   frozen C×D representation pair separately.
#
# Uses Part-C objects:
#   - E3_ASSOC_TAU_VALUES
#   - e3_tau_assignment_df
#   - e3_tau_region_df
#
# IMPORTANT:
#   - No GT is used.
#   - No F1/F2/F3/F4 fusion is performed.
#   - No tau is selected here.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.1 PART D — C×D PAIR-LEVEL TAU CHARACTERISATION")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required Part-C objects
# ------------------------------------------------------------

required_objects = [
    "E3_ASSOC_TAU_VALUES",
    "e3_tau_assignment_df",
    "e3_tau_region_df",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required Part-C objects: "
    + ", ".join(missing_objects)
)


# ------------------------------------------------------------
# 2. D-point association summary by C×D pair and tau
# ------------------------------------------------------------

dpoint_pair_summary = (
    e3_tau_assignment_df
    .groupby(
        [
            "c_variant",
            "d_variant",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        n_d_points=(
            "d_point_id",
            "size",
        ),

        n_inside_membership=(
            "association_source",
            lambda x: int(
                (x == "inside_membership").sum()
            ),
        ),

        n_outside_distance_associated=(
            "association_source",
            lambda x: int(
                (x == "outside_nearest_region").sum()
            ),
        ),

        n_unassociated=(
            "association_source",
            lambda x: int(
                (x == "unassociated").sum()
            ),
        ),

        n_associated_total=(
            "is_associated",
            lambda x: int(x.sum()),
        ),
    )
)


# Fractions relative to all D points in the pair
dpoint_pair_summary[
    "associated_fraction"
] = (
    dpoint_pair_summary["n_associated_total"]
    /
    dpoint_pair_summary["n_d_points"]
)

dpoint_pair_summary[
    "inside_fraction"
] = (
    dpoint_pair_summary["n_inside_membership"]
    /
    dpoint_pair_summary["n_d_points"]
)

dpoint_pair_summary[
    "outside_added_fraction_all_d"
] = (
    dpoint_pair_summary[
        "n_outside_distance_associated"
    ]
    /
    dpoint_pair_summary["n_d_points"]
)

dpoint_pair_summary[
    "unassociated_fraction"
] = (
    dpoint_pair_summary["n_unassociated"]
    /
    dpoint_pair_summary["n_d_points"]
)


# ------------------------------------------------------------
# 3. C-region multiplicity summary by C×D pair and tau
# ------------------------------------------------------------

region_pair_summary = (
    e3_tau_region_df
    .groupby(
        [
            "c_variant",
            "d_variant",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        n_c_regions=(
            "region_id",
            "size",
        ),

        n_zero_regions=(
            "n_d_associated",
            lambda x: int((x == 0).sum()),
        ),

        n_one_regions=(
            "n_d_associated",
            lambda x: int((x == 1).sum()),
        ),

        n_multi_regions=(
            "n_d_associated",
            lambda x: int((x >= 2).sum()),
        ),

        mean_d_per_region=(
            "n_d_associated",
            "mean",
        ),

        median_d_per_region=(
            "n_d_associated",
            "median",
        ),

        max_d_per_region=(
            "n_d_associated",
            "max",
        ),
    )
)


region_pair_summary[
    "zero_region_fraction"
] = (
    region_pair_summary["n_zero_regions"]
    /
    region_pair_summary["n_c_regions"]
)

region_pair_summary[
    "one_region_fraction"
] = (
    region_pair_summary["n_one_regions"]
    /
    region_pair_summary["n_c_regions"]
)

region_pair_summary[
    "multi_region_fraction"
] = (
    region_pair_summary["n_multi_regions"]
    /
    region_pair_summary["n_c_regions"]
)

region_pair_summary[
    "occupied_region_fraction"
] = (
    1.0
    -
    region_pair_summary["zero_region_fraction"]
)


# ------------------------------------------------------------
# 4. Merge D-point and C-region views
# ------------------------------------------------------------

e3_tau_pair_summary_df = (
    dpoint_pair_summary
    .merge(
        region_pair_summary,
        on=[
            "c_variant",
            "d_variant",
            "tau",
        ],
        how="inner",
        validate="one_to_one",
    )
)


e3_tau_pair_summary_df[
    "tau_pct_short_side"
] = (
    100.0
    *
    e3_tau_pair_summary_df["tau"]
)


# Put identifying columns first
front_cols = [
    "c_variant",
    "d_variant",
    "tau",
    "tau_pct_short_side",
]

other_cols = [
    col
    for col in e3_tau_pair_summary_df.columns
    if col not in front_cols
]

e3_tau_pair_summary_df = (
    e3_tau_pair_summary_df[
        front_cols + other_cols
    ]
    .sort_values(
        [
            "c_variant",
            "d_variant",
            "tau",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

expected_pairs = (
    len(E3_C_VARIANTS)
    *
    len(E3_D_VARIANTS)
)

expected_rows = (
    expected_pairs
    *
    len(E3_ASSOC_TAU_VALUES)
)

assert len(e3_tau_pair_summary_df) == expected_rows, (
    f"Expected {expected_rows} pair×tau rows, "
    f"found {len(e3_tau_pair_summary_df)}."
)


# Every C×D pair should have all five tau values
tau_counts = (
    e3_tau_pair_summary_df
    .groupby(
        ["c_variant", "d_variant"]
    )["tau"]
    .nunique()
)

assert (
    tau_counts
    ==
    len(E3_ASSOC_TAU_VALUES)
).all()


# Association fraction must be monotonic within every pair
association_monotonic = (
    e3_tau_pair_summary_df
    .sort_values("tau")
    .groupby(
        ["c_variant", "d_variant"]
    )["associated_fraction"]
    .apply(
        lambda x: x.is_monotonic_increasing
    )
)

assert association_monotonic.all()


# Zero-region count must be non-increasing within every pair
zero_monotonic = (
    e3_tau_pair_summary_df
    .sort_values("tau")
    .groupby(
        ["c_variant", "d_variant"]
    )["n_zero_regions"]
    .apply(
        lambda x: x.is_monotonic_decreasing
    )
)

assert zero_monotonic.all()


# Region partition identity:
# zero + one + multi = all C regions
assert (
    (
        e3_tau_pair_summary_df["n_zero_regions"]
        +
        e3_tau_pair_summary_df["n_one_regions"]
        +
        e3_tau_pair_summary_df["n_multi_regions"]
    )
    ==
    e3_tau_pair_summary_df["n_c_regions"]
).all()


# D-point partition identity:
# associated + unassociated = all D points
assert (
    (
        e3_tau_pair_summary_df["n_associated_total"]
        +
        e3_tau_pair_summary_df["n_unassociated"]
    )
    ==
    e3_tau_pair_summary_df["n_d_points"]
).all()


# ------------------------------------------------------------
# 6. Print compact 75-row comparison
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("Pair-level GT-free association structure")
print("-" * 120)

display_cols = [
    "c_variant",
    "d_variant",
    "tau_pct_short_side",
    "associated_fraction",
    "occupied_region_fraction",
    "zero_region_fraction",
    "one_region_fraction",
    "multi_region_fraction",
    "mean_d_per_region",
]

print(
    e3_tau_pair_summary_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 7. Compact delta view:
#    how much does 0.5% -> 5% change each C×D pair?
# ------------------------------------------------------------

tau_min = min(E3_ASSOC_TAU_VALUES)
tau_max = max(E3_ASSOC_TAU_VALUES)

low = (
    e3_tau_pair_summary_df[
        e3_tau_pair_summary_df["tau"] == tau_min
    ]
    .set_index(
        ["c_variant", "d_variant"]
    )
)

high = (
    e3_tau_pair_summary_df[
        e3_tau_pair_summary_df["tau"] == tau_max
    ]
    .set_index(
        ["c_variant", "d_variant"]
    )
)

delta_records = []

for pair in low.index:

    low_row = low.loc[pair]
    high_row = high.loc[pair]

    delta_records.append({
        "c_variant": pair[0],
        "d_variant": pair[1],

        "associated_fraction_at_0.5pct":
            low_row["associated_fraction"],

        "associated_fraction_at_5pct":
            high_row["associated_fraction"],

        "delta_associated_fraction":
            (
                high_row["associated_fraction"]
                -
                low_row["associated_fraction"]
            ),

        "multi_region_fraction_at_0.5pct":
            low_row["multi_region_fraction"],

        "multi_region_fraction_at_5pct":
            high_row["multi_region_fraction"],

        "delta_multi_region_fraction":
            (
                high_row["multi_region_fraction"]
                -
                low_row["multi_region_fraction"]
            ),
    })


e3_tau_pair_delta_df = pd.DataFrame(
    delta_records
)


print("\n" + "-" * 120)
print("Sensitivity from tau = 0.5% to tau = 5%")
print("-" * 120)

print(
    e3_tau_pair_delta_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 8. Final validation summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("Sanity checks")
print("-" * 120)

print(
    f"Expected C×D pairs: {expected_pairs} — PASS"
)
print(
    f"Expected pair×tau conditions: {expected_rows} — PASS"
)
print(
    "All pairs contain all tau values: PASS"
)
print(
    "Association fraction monotonic within every pair: PASS"
)
print(
    "Zero-region count non-increasing within every pair: PASS"
)
print(
    "C-region partition identities: PASS"
)
print(
    "D-point partition identities: PASS"
)


print("\nConstructed objects:")
print(
    "  e3_tau_pair_summary_df :",
    e3_tau_pair_summary_df.shape,
)
print(
    "  e3_tau_pair_delta_df   :",
    e3_tau_pair_delta_df.shape,
)


print("\n" + "=" * 120)
print(
    "Step 13D.2b.1 Part D complete — "
    "pair-level GT-free tau characterisation constructed."
)
print(
    "No GT used. No tau selected. "
    "No F1/F2/F3/F4 fusion performed."
)
print("=" * 120)

STEP 13D.2b.1 PART D — C×D PAIR-LEVEL TAU CHARACTERISATION

------------------------------------------------------------------------------------------------------------------------
Pair-level GT-free association structure
------------------------------------------------------------------------------------------------------------------------
c_variant    d_variant  tau_pct_short_side  associated_fraction  occupied_region_fraction  zero_region_fraction  one_region_fraction  multi_region_fraction  mean_d_per_region
       C1           D1              0.5000               0.5481                    0.8438                0.1562               0.5983                 0.2455             1.6421
       C1           D1              1.0000               0.5481                    0.8438                0.1562               0.5983                 0.2455             1.6421
       C1           D1              2.0000               0.5481                    0.8438                0.1562               0.5983

### Step 13D.2b.2 — Operational Definitions of Simple Semantic-Fusion Baselines

The frozen C–D association rule from Step 13D.2b.1 is used to define three simple semantic-fusion baselines.

For a C representation:

\[
P_C=\{c_1,\ldots,c_m\}
\]

where each \(c_i\) is the representative point of C region \(C_i\).

For a D representation:

\[
P_D=\{d_1,\ldots,d_n\}.
\]

A D point is associated with a C region,

\[
d_j\sim_{\tau}C_i,
\]

when it is either:

- inside \(C_i\); or
- outside the C foreground but within distance tolerance \(\tau\) of the nearest C-region support.

The bounded association tolerances remain:

\[
\tau\in\{0.5\%,1\%,2\%,3\%,5\%\}
\]

of the image short side.

No GT information is used to construct the fused reference points.

---

#### F1 — Union with association-aware consolidation

**Purpose:** preserve complementary evidence from both C and D while avoiding simple C–D duplication.

Rules:

- Retain all D points.
- If a C region contains/has an associated D point, its C representative point is not additionally retained.
- If a C region has no associated D point, retain its C representative point.
- Multiple D points associated with the same C region remain separate.

Therefore:

\[
P_{F1}
=
P_D
\cup
\{c_i:N_D(C_i)=0\}.
\]

F1 tests whether complementary C-only evidence can augment D without duplicating C hypotheses already supported by D.

---

#### F2 — C–D agreement

**Purpose:** test the effect of requiring support from both semantic representations.

Rules:

- Retain a D point only when it is associated with a C region.
- Discard unassociated D points.
- Discard C regions with no associated D evidence.
- Multiple associated D points remain separate.

Therefore:

\[
P_{F2}
=
\{d_j:d_j\sim_{\tau}C\}.
\]

F2 tests the precision–recall trade-off produced by cross-representation agreement.

---

#### F3 — C-anchored refinement

**Purpose:** test whether D can improve the localisation of C-derived hypotheses while preserving the C hypothesis count.

Rules:

- Each C region produces exactly one output point.
- If no D point is associated with a C region, retain the C representative point.
- If one or more D points are associated, use the associated D point nearest to the C representative point.
- Multiple D points do not create additional hypotheses.

Therefore:

\[
p_i^{F3}
=
\begin{cases}
c_i, & N_D(C_i)=0,\\
\arg\min_{d_j\sim_{\tau}C_i}\|d_j-c_i\|,
& N_D(C_i)\ge1.
\end{cases}
\]

Hence:

\[
|P_{F3}|=|P_C|.
\]

F3 can change localisation but cannot change the C-derived predicted count.

---

#### Deferred D-anchored structured integration

A simple D-anchored rule is not included as a separate F4 baseline.

- Keeping every D point unchanged reduces to the original D representation.
- Keeping only C-supported D points reduces to F2.
- Moving multiple D points to a C-region centroid can collapse distinct hypotheses.

A genuinely distinct D-anchored method therefore requires richer evidence such as C membership, C–D distance, multiplicity, region scale and/or D activation confidence.

Such methods are deferred to **Step 13D.3 — Structured Semantic Integration**.

---

#### Experimental controls

- C and D representations remain frozen.
- The C–D association mechanism remains frozen.
- The five bounded \(\tau\) values are evaluated without selecting one in advance.
- S1–S6 are evaluation strata only and are not used for fusion or routing.
- GT is used only after reference-point generation for performance evaluation.
- No Marigold/geometric prior is introduced.

In [70]:
# Step 13D.2b.3 — Schema check before fusion generation

print("e3_tau_assignment_df shape:")
print(e3_tau_assignment_df.shape)

print("\nColumns:")
for i, col in enumerate(e3_tau_assignment_df.columns):
    print(f"{i:2d}: {col}")

print("\nFirst row:")
display(e3_tau_assignment_df.head(1))

e3_tau_assignment_df shape:
(696525, 22)

Columns:
 0: dataset_index
 1: image_id
 2: category
 3: c_variant
 4: d_variant
 5: d_point_id
 6: d_x
 7: d_y
 8: inside_c_foreground
 9: containing_region_id
10: n_d_in_containing_region
11: distance_to_containing_centroid_px
12: distance_to_containing_centroid_norm
13: nearest_region_id
14: distance_to_nearest_region_px
15: distance_to_nearest_region_norm
16: distance_to_nearest_region_centroid_px
17: distance_to_nearest_region_centroid_norm
18: tau
19: association_source
20: associated_c_region_id
21: is_associated

First row:


,dataset_index,image_id,category,c_variant,d_variant,d_point_id,d_x,d_y,inside_c_foreground,containing_region_id,...,distance_to_containing_centroid_norm,nearest_region_id,distance_to_nearest_region_px,distance_to_nearest_region_norm,distance_to_nearest_region_centroid_px,distance_to_nearest_region_centroid_norm,tau,association_source,associated_c_region_id,is_associated
0,9,215.jpg,grapes,C1,D1,0,369.0,139.0,True,3.0,...,0.00829,3,0.0,0.0,3.183337,0.00829,0.005,inside_membership,3.0,True


In [73]:
# ============================================================
# Step 13D.2b.3 — Generate F1–F3 Semantic-Fusion Reference Points
#
# F1 — Union with association-aware consolidation
#      Keep all D points.
#      Add C representative only when that C region has no
#      associated D point.
#
# F2 — C–D agreement
#      Keep only D points associated with C.
#      Zero-point predictions are valid and retained.
#
# F3 — C-anchored refinement
#      Exactly one point per C region.
#      If associated D exists, use the associated D point
#      nearest to the C representative.
#      Otherwise retain the C representative.
#
# IMPORTANT:
#   - No GT is used.
#   - No performance evaluation is performed.
#   - No tau is selected.
#   - Empty F2 predictions are explicitly represented as:
#         pred_count = 0
#         pred_points_xy = []
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.3 — GENERATE F1–F3 SEMANTIC-FUSION REFERENCE POINTS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required objects and schema
# ------------------------------------------------------------

required_objects = [
    "E3_C_VARIANTS",
    "E3_D_VARIANTS",
    "E3_ASSOC_TAU_VALUES",
    "e3_tau_assignment_df",
    "e3_tau_region_df",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required objects: "
    + ", ".join(missing_objects)
)


required_assignment_cols = {
    "dataset_index",
    "image_id",
    "category",
    "c_variant",
    "d_variant",
    "tau",
    "d_point_id",
    "d_x",
    "d_y",
    "is_associated",
    "associated_c_region_id",
}

missing = (
    required_assignment_cols
    - set(e3_tau_assignment_df.columns)
)

assert not missing, (
    "Missing assignment columns: "
    + ", ".join(sorted(missing))
)


required_region_cols = {
    "dataset_index",
    "image_id",
    "category",
    "c_variant",
    "d_variant",
    "tau",
    "region_id",
    "centroid_x",
    "centroid_y",
    "n_d_associated",
}

missing = (
    required_region_cols
    - set(e3_tau_region_df.columns)
)

assert not missing, (
    "Missing region columns: "
    + ", ".join(sorted(missing))
)

print("Required schema validation: PASS")


# ------------------------------------------------------------
# 2. Define the complete image/configuration universe
#
# 100 images × 3 C × 5 D × 5 tau = 7,500 conditions.
#
# This table is essential because F2 may legitimately produce
# zero points for some conditions.
# ------------------------------------------------------------

condition_cols = [
    "dataset_index",
    "image_id",
    "category",
    "c_variant",
    "d_variant",
    "tau",
]

e3_all_conditions_df = (
    e3_tau_region_df[
        condition_cols
    ]
    .drop_duplicates()
    .sort_values(
        [
            "c_variant",
            "d_variant",
            "tau",
            "dataset_index",
        ]
    )
    .reset_index(drop=True)
)


n_images = (
    e3_all_conditions_df[
        ["dataset_index", "image_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

expected_base_conditions = (
    n_images
    * len(E3_C_VARIANTS)
    * len(E3_D_VARIANTS)
    * len(E3_ASSOC_TAU_VALUES)
)

assert len(e3_all_conditions_df) == expected_base_conditions, (
    f"Expected {expected_base_conditions} base conditions, "
    f"found {len(e3_all_conditions_df)}."
)

print(
    f"Complete base condition universe: "
    f"{len(e3_all_conditions_df)} rows — PASS"
)


# ------------------------------------------------------------
# 3. Generate F1
#
# F1 = all D points
#      + C representatives from regions with zero associated D
# ------------------------------------------------------------

f1_records = []


# 3a. Keep all D points
for row in e3_tau_assignment_df.itertuples(index=False):

    f1_records.append({
        "dataset_index": row.dataset_index,
        "image_id": row.image_id,
        "category": row.category,
        "c_variant": row.c_variant,
        "d_variant": row.d_variant,
        "tau": float(row.tau),

        "fusion_family": "F1",

        "point_x": float(row.d_x),
        "point_y": float(row.d_y),

        "point_source": "D",

        "source_d_point_id": row.d_point_id,

        "source_c_region_id": (
            row.associated_c_region_id
            if bool(row.is_associated)
            else np.nan
        ),
    })


# 3b. Add C representative only for C regions with no
#     associated D point
f1_empty_regions = (
    e3_tau_region_df[
        e3_tau_region_df["n_d_associated"] == 0
    ]
    .copy()
)


for row in f1_empty_regions.itertuples(index=False):

    f1_records.append({
        "dataset_index": row.dataset_index,
        "image_id": row.image_id,
        "category": row.category,
        "c_variant": row.c_variant,
        "d_variant": row.d_variant,
        "tau": float(row.tau),

        "fusion_family": "F1",

        "point_x": float(row.centroid_x),
        "point_y": float(row.centroid_y),

        "point_source": "C_unassociated_region",

        "source_d_point_id": np.nan,
        "source_c_region_id": row.region_id,
    })


f1_point_df = pd.DataFrame(f1_records)


# ------------------------------------------------------------
# 4. Generate F2
#
# F2 = associated D points only.
#
# IMPORTANT:
# Some image/configurations may produce zero F2 points.
# Those empty cases are added explicitly later.
# ------------------------------------------------------------

f2_source = (
    e3_tau_assignment_df[
        e3_tau_assignment_df["is_associated"]
    ]
    .copy()
)


f2_records = []


for row in f2_source.itertuples(index=False):

    f2_records.append({
        "dataset_index": row.dataset_index,
        "image_id": row.image_id,
        "category": row.category,
        "c_variant": row.c_variant,
        "d_variant": row.d_variant,
        "tau": float(row.tau),

        "fusion_family": "F2",

        "point_x": float(row.d_x),
        "point_y": float(row.d_y),

        "point_source": "D_associated",

        "source_d_point_id": row.d_point_id,
        "source_c_region_id": row.associated_c_region_id,
    })


f2_point_df = pd.DataFrame(f2_records)


# ------------------------------------------------------------
# 5. Generate F3
#
# F3 = exactly one output per C region.
#
# If associated D exists:
#   select associated D nearest to C representative.
#
# Otherwise:
#   retain C representative.
# ------------------------------------------------------------

f3_associated_d = (
    e3_tau_assignment_df[
        e3_tau_assignment_df["is_associated"]
    ]
    .copy()
)


# Join associated D points to their associated C regions
f3_associated = (
    f3_associated_d
    .merge(
        e3_tau_region_df[
            [
                "dataset_index",
                "image_id",
                "category",
                "c_variant",
                "d_variant",
                "tau",
                "region_id",
                "centroid_x",
                "centroid_y",
            ]
        ],
        left_on=[
            "dataset_index",
            "image_id",
            "category",
            "c_variant",
            "d_variant",
            "tau",
            "associated_c_region_id",
        ],
        right_on=[
            "dataset_index",
            "image_id",
            "category",
            "c_variant",
            "d_variant",
            "tau",
            "region_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert f3_associated["region_id"].notna().all(), (
    "Some associated D points could not be mapped "
    "to their associated C region."
)


# Distance from D point to C representative
f3_associated[
    "distance_to_c_representative_px"
] = np.sqrt(
    (
        f3_associated["d_x"]
        - f3_associated["centroid_x"]
    ) ** 2
    +
    (
        f3_associated["d_y"]
        - f3_associated["centroid_y"]
    ) ** 2
)


# Deterministic selection:
#   1. nearest associated D to C representative
#   2. smallest D point ID as tie-break
f3_associated = (
    f3_associated
    .sort_values(
        [
            "dataset_index",
            "image_id",
            "c_variant",
            "d_variant",
            "tau",
            "associated_c_region_id",
            "distance_to_c_representative_px",
            "d_point_id",
        ]
    )
)


f3_selected_d = (
    f3_associated
    .drop_duplicates(
        subset=[
            "dataset_index",
            "image_id",
            "c_variant",
            "d_variant",
            "tau",
            "associated_c_region_id",
        ],
        keep="first",
    )
)


f3_selected_lookup = (
    f3_selected_d[
        [
            "dataset_index",
            "image_id",
            "c_variant",
            "d_variant",
            "tau",
            "associated_c_region_id",
            "d_point_id",
            "d_x",
            "d_y",
            "distance_to_c_representative_px",
        ]
    ]
    .rename(
        columns={
            "associated_c_region_id": "region_id",
            "d_point_id": "selected_d_point_id",
            "d_x": "selected_d_x",
            "d_y": "selected_d_y",
        }
    )
)


# Attach selected D, where available, to every C region
f3_region_source = (
    e3_tau_region_df
    .merge(
        f3_selected_lookup,
        on=[
            "dataset_index",
            "image_id",
            "c_variant",
            "d_variant",
            "tau",
            "region_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


f3_records = []


for row in f3_region_source.itertuples(index=False):

    has_selected_d = pd.notna(
        row.selected_d_point_id
    )

    if has_selected_d:

        point_x = float(row.selected_d_x)
        point_y = float(row.selected_d_y)

        point_source = "D_nearest_to_C"
        source_d_point_id = row.selected_d_point_id

    else:

        point_x = float(row.centroid_x)
        point_y = float(row.centroid_y)

        point_source = "C_no_associated_D"
        source_d_point_id = np.nan


    f3_records.append({
        "dataset_index": row.dataset_index,
        "image_id": row.image_id,
        "category": row.category,
        "c_variant": row.c_variant,
        "d_variant": row.d_variant,
        "tau": float(row.tau),

        "fusion_family": "F3",

        "point_x": point_x,
        "point_y": point_y,

        "point_source": point_source,

        "source_d_point_id": source_d_point_id,
        "source_c_region_id": row.region_id,
    })


f3_point_df = pd.DataFrame(f3_records)


# ------------------------------------------------------------
# 6. Combine point-level F1–F3 outputs
#
# NOTE:
# e3_fusion_point_df intentionally contains only actual points.
# Therefore zero-point F2 predictions have no rows here.
# They are represented in the per-image table below.
# ------------------------------------------------------------

e3_fusion_point_df = (
    pd.concat(
        [
            f1_point_df,
            f2_point_df,
            f3_point_df,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
            "dataset_index",
            "point_y",
            "point_x",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Build observed point lists and counts
# ------------------------------------------------------------

group_cols = [
    "dataset_index",
    "image_id",
    "category",
    "c_variant",
    "d_variant",
    "tau",
    "fusion_family",
]


observed_counts_df = (
    e3_fusion_point_df
    .groupby(
        group_cols,
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "pred_count"
        }
    )
)


observed_points_df = (
    e3_fusion_point_df
    .groupby(
        group_cols,
        sort=False,
    )
    .apply(
        lambda g: [
            (float(x), float(y))
            for x, y in zip(
                g["point_x"],
                g["point_y"],
            )
        ],
        include_groups=False,
    )
    .rename("pred_points_xy")
    .reset_index()
)


observed_per_image_df = (
    observed_counts_df
    .merge(
        observed_points_df,
        on=group_cols,
        how="inner",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 8. Build COMPLETE F1/F2/F3 image/configuration universe
#
# This explicitly creates:
#
#   7,500 F1 rows
#   7,500 F2 rows
#   7,500 F3 rows
#
# = 22,500 rows total.
# ------------------------------------------------------------

fusion_families = [
    "F1",
    "F2",
    "F3",
]


complete_condition_frames = []


for fusion_family in fusion_families:

    temp = e3_all_conditions_df.copy()

    temp["fusion_family"] = fusion_family

    complete_condition_frames.append(temp)


complete_fusion_conditions_df = (
    pd.concat(
        complete_condition_frames,
        ignore_index=True,
    )
)


expected_fusion_rows = (
    expected_base_conditions
    * len(fusion_families)
)

assert len(complete_fusion_conditions_df) == expected_fusion_rows


# Left join actual predictions onto complete condition universe
e3_fusion_per_image_df = (
    complete_fusion_conditions_df
    .merge(
        observed_per_image_df,
        on=group_cols,
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 9. Explicitly represent zero-point predictions
# ------------------------------------------------------------

e3_fusion_per_image_df["pred_count"] = (
    e3_fusion_per_image_df["pred_count"]
    .fillna(0)
    .astype(int)
)


e3_fusion_per_image_df["pred_points_xy"] = (
    e3_fusion_per_image_df["pred_points_xy"]
    .apply(
        lambda x: (
            x
            if isinstance(x, list)
            else []
        )
    )
)


# Stable ordering
e3_fusion_per_image_df = (
    e3_fusion_per_image_df
    .sort_values(
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
            "dataset_index",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Structural validation — complete universe
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("STRUCTURAL VALIDATION")
print("-" * 120)


# Exactly 225 fusion configurations
expected_fusion_configs = (
    len(E3_C_VARIANTS)
    *
    len(E3_D_VARIANTS)
    *
    len(E3_ASSOC_TAU_VALUES)
    *
    len(fusion_families)
)


actual_fusion_configs = (
    e3_fusion_per_image_df[
        [
            "c_variant",
            "d_variant",
            "tau",
            "fusion_family",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


assert actual_fusion_configs == expected_fusion_configs, (
    f"Expected {expected_fusion_configs} fusion configurations, "
    f"found {actual_fusion_configs}."
)


# Exactly 22,500 image/configuration rows
assert len(e3_fusion_per_image_df) == expected_fusion_rows, (
    f"Expected {expected_fusion_rows} image/configuration rows, "
    f"found {len(e3_fusion_per_image_df)}."
)


# Exactly 7,500 rows per fusion family
rows_per_family = (
    e3_fusion_per_image_df
    .groupby("fusion_family")
    .size()
)


for fusion_family in fusion_families:

    assert (
        rows_per_family.loc[fusion_family]
        ==
        expected_base_conditions
    ), (
        f"{fusion_family} does not contain "
        f"{expected_base_conditions} rows."
    )


# Point-list length must always equal pred_count
point_count_consistency = (
    e3_fusion_per_image_df["pred_points_xy"]
    .apply(len)
    ==
    e3_fusion_per_image_df["pred_count"]
)

assert point_count_consistency.all(), (
    "Some pred_points_xy lengths do not match pred_count."
)


# ------------------------------------------------------------
# 11. F1 invariant
#
# N_F1 = N_D + number of C regions with zero associated D
# ------------------------------------------------------------

d_counts = (
    e3_tau_assignment_df
    .groupby(condition_cols, as_index=False)
    .size()
    .rename(
        columns={
            "size": "d_count"
        }
    )
)


zero_c_counts = (
    e3_tau_region_df[
        e3_tau_region_df["n_d_associated"] == 0
    ]
    .groupby(condition_cols, as_index=False)
    .size()
    .rename(
        columns={
            "size": "zero_c_regions"
        }
    )
)


f1_expected = (
    e3_all_conditions_df
    .merge(
        d_counts,
        on=condition_cols,
        how="left",
        validate="one_to_one",
    )
    .merge(
        zero_c_counts,
        on=condition_cols,
        how="left",
        validate="one_to_one",
    )
)


f1_expected["d_count"] = (
    f1_expected["d_count"]
    .fillna(0)
    .astype(int)
)

f1_expected["zero_c_regions"] = (
    f1_expected["zero_c_regions"]
    .fillna(0)
    .astype(int)
)

f1_expected["expected_f1_count"] = (
    f1_expected["d_count"]
    +
    f1_expected["zero_c_regions"]
)


f1_actual = (
    e3_fusion_per_image_df[
        e3_fusion_per_image_df["fusion_family"] == "F1"
    ][
        condition_cols + ["pred_count"]
    ]
    .rename(
        columns={
            "pred_count": "f1_count"
        }
    )
)


f1_check = (
    f1_expected
    .merge(
        f1_actual,
        on=condition_cols,
        how="inner",
        validate="one_to_one",
    )
)


assert (
    f1_check["f1_count"]
    ==
    f1_check["expected_f1_count"]
).all(), (
    "F1 count invariant failed."
)


# ------------------------------------------------------------
# 12. F2 invariant
#
# N_F2 = number of associated D points
#
# Zero associated D -> pred_count = 0 and points = []
# ------------------------------------------------------------

associated_d_counts = (
    e3_tau_assignment_df[
        e3_tau_assignment_df["is_associated"]
    ]
    .groupby(condition_cols, as_index=False)
    .size()
    .rename(
        columns={
            "size": "associated_d_count"
        }
    )
)


f2_expected = (
    e3_all_conditions_df
    .merge(
        associated_d_counts,
        on=condition_cols,
        how="left",
        validate="one_to_one",
    )
)


f2_expected["associated_d_count"] = (
    f2_expected["associated_d_count"]
    .fillna(0)
    .astype(int)
)


f2_actual = (
    e3_fusion_per_image_df[
        e3_fusion_per_image_df["fusion_family"] == "F2"
    ][
        condition_cols
        + [
            "pred_count",
            "pred_points_xy",
        ]
    ]
)


f2_check = (
    f2_expected
    .merge(
        f2_actual,
        on=condition_cols,
        how="inner",
        validate="one_to_one",
    )
)


assert (
    f2_check["pred_count"]
    ==
    f2_check["associated_d_count"]
).all(), (
    "F2 count invariant failed."
)


f2_zero_check = (
    f2_check[
        f2_check["associated_d_count"] == 0
    ]
)


assert (
    f2_zero_check["pred_count"] == 0
).all()

assert (
    f2_zero_check["pred_points_xy"]
    .apply(len)
    == 0
).all()


# ------------------------------------------------------------
# 13. F3 invariant
#
# N_F3 = number of C regions
# ------------------------------------------------------------

c_counts = (
    e3_tau_region_df
    .groupby(condition_cols, as_index=False)
    .size()
    .rename(
        columns={
            "size": "c_count"
        }
    )
)


f3_actual = (
    e3_fusion_per_image_df[
        e3_fusion_per_image_df["fusion_family"] == "F3"
    ][
        condition_cols + ["pred_count"]
    ]
    .rename(
        columns={
            "pred_count": "f3_count"
        }
    )
)


f3_check = (
    c_counts
    .merge(
        f3_actual,
        on=condition_cols,
        how="inner",
        validate="one_to_one",
    )
)


assert (
    f3_check["f3_count"]
    ==
    f3_check["c_count"]
).all(), (
    "F3 count invariant failed."
)


# ------------------------------------------------------------
# 14. Generation summary
# ------------------------------------------------------------

e3_fusion_generation_summary_df = (
    e3_fusion_per_image_df
    .groupby(
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        n_images=(
            "image_id",
            "nunique",
        ),

        total_points=(
            "pred_count",
            "sum",
        ),

        mean_points_per_image=(
            "pred_count",
            "mean",
        ),

        median_points_per_image=(
            "pred_count",
            "median",
        ),

        min_points_per_image=(
            "pred_count",
            "min",
        ),

        max_points_per_image=(
            "pred_count",
            "max",
        ),

        zero_prediction_images=(
            "pred_count",
            lambda x: int((x == 0).sum()),
        ),
    )
)


print("\n" + "-" * 120)
print("GENERATION SUMMARY")
print("-" * 120)

print(
    e3_fusion_generation_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 15. Point-source summary
# ------------------------------------------------------------

e3_fusion_source_summary_df = (
    e3_fusion_point_df
    .groupby(
        [
            "fusion_family",
            "point_source",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_points"
        }
    )
)


print("\n" + "-" * 120)
print("POINT-SOURCE SUMMARY")
print("-" * 120)

print(
    e3_fusion_source_summary_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 16. Zero-prediction summary
# ------------------------------------------------------------

e3_zero_prediction_summary_df = (
    e3_fusion_per_image_df[
        e3_fusion_per_image_df["pred_count"] == 0
    ]
    .groupby(
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_zero_prediction_images"
        }
    )
)


print("\n" + "-" * 120)
print("ZERO-PREDICTION SUMMARY")
print("-" * 120)

if len(e3_zero_prediction_summary_df) > 0:

    print(
        e3_zero_prediction_summary_df.to_string(
            index=False
        )
    )

else:

    print("No zero-point predictions.")


# ------------------------------------------------------------
# 17. Final validation summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print(
    f"Images: {n_images}"
)

print(
    f"Base C×D×tau conditions: "
    f"{expected_base_conditions} — PASS"
)

print(
    f"Fusion configurations: "
    f"{expected_fusion_configs} — PASS"
)

print(
    f"Complete image/configuration rows: "
    f"{expected_fusion_rows} — PASS"
)

print(
    f"Rows per family: "
    f"F1={rows_per_family['F1']}, "
    f"F2={rows_per_family['F2']}, "
    f"F3={rows_per_family['F3']} — PASS"
)

print(
    "Point-list length = pred_count for every row: PASS"
)

print(
    "F1 count = D count + zero-associated-D C regions: PASS"
)

print(
    "F2 count = associated D count: PASS"
)

print(
    f"F2 zero-point cases explicitly represented: "
    f"{len(f2_zero_check)} — PASS"
)

print(
    "F3 count = C region count: PASS"
)

print(
    "No GT used for fusion generation: PASS"
)


print("\nConstructed objects:")

print(
    "  e3_all_conditions_df             :",
    e3_all_conditions_df.shape,
)

print(
    "  f1_point_df                      :",
    f1_point_df.shape,
)

print(
    "  f2_point_df                      :",
    f2_point_df.shape,
)

print(
    "  f3_point_df                      :",
    f3_point_df.shape,
)

print(
    "  e3_fusion_point_df               :",
    e3_fusion_point_df.shape,
)

print(
    "  e3_fusion_per_image_df           :",
    e3_fusion_per_image_df.shape,
)

print(
    "  e3_fusion_generation_summary_df  :",
    e3_fusion_generation_summary_df.shape,
)

print(
    "  e3_fusion_source_summary_df      :",
    e3_fusion_source_summary_df.shape,
)

print(
    "  e3_zero_prediction_summary_df    :",
    e3_zero_prediction_summary_df.shape,
)


print("\n" + "=" * 120)

print(
    "Step 13D.2b.3 complete — "
    "F1–F3 semantic-fusion reference points generated."
)

print(
    "Legitimate zero-point predictions are retained explicitly."
)

print(
    "No GT evaluation performed. "
    "No tau selected."
)

print("=" * 120)

STEP 13D.2b.3 — GENERATE F1–F3 SEMANTIC-FUSION REFERENCE POINTS
Required schema validation: PASS
Complete base condition universe: 7500 rows — PASS

------------------------------------------------------------------------------------------------------------------------
STRUCTURAL VALIDATION
------------------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------------------------------------
GENERATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
fusion_family c_variant    d_variant    tau  n_images  total_points  mean_points_per_image  median_points_per_image  min_points_per_image  max_points_per_image  zero_prediction_images
           F1        C1           D1 0.0050       100          8051                80.5100                  74.0000                    16    

### 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines

Part A — Attach canonical GT and validate evaluation inputs

Part B — Evaluate all F1–F3 configurations

Part C — Add C-only and D-only reference baselines

Part D — Overall comparison

Part E — S1–S6 comparison

Part F — Fusion gains/losses relative to C and D

Part G — Findings and shortlist for structured integration

In [75]:
# ============================================================
# Step 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines
# Part A — Attach Canonical GT and Validate Evaluation Inputs
#
# Uses the EXISTING frozen B1 infrastructure:
#   fsc147_dev_100_df
#   fsc147_val_dataset
#   get_gt_points(...)
#
# IMPORTANT:
#   - GT is introduced ONLY for evaluation.
#   - Fusion predictions are not modified.
#   - No metric is computed yet.
#   - No tau is selected.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.4 PART A — ATTACH CANONICAL GT AND VALIDATE EVALUATION INPUTS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required existing notebook objects
# ------------------------------------------------------------

required_objects = [
    "fsc147_dev_100_df",
    "fsc147_val_dataset",
    "get_gt_points",
    "e3_fusion_per_image_df",
    "E3_C_VARIANTS",
    "E3_D_VARIANTS",
    "E3_ASSOC_TAU_VALUES",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required notebook objects: "
    + ", ".join(missing_objects)
)

print("Required notebook objects: PASS")


# ------------------------------------------------------------
# 2. Validate frozen dev100 dataframe
# ------------------------------------------------------------

required_dev_cols = {
    "dataset_index",
    "image_id",
    "category",
    "gt_count",
    "density_stratum",
}

missing_dev_cols = (
    required_dev_cols
    - set(fsc147_dev_100_df.columns)
)

assert not missing_dev_cols, (
    "fsc147_dev_100_df missing columns: "
    + ", ".join(sorted(missing_dev_cols))
)


e3_eval_dev100_df = (
    fsc147_dev_100_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "gt_count",
            "density_stratum",
        ]
    ]
    .copy()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


assert len(e3_eval_dev100_df) == 100

assert (
    e3_eval_dev100_df["dataset_index"].nunique()
    == 100
)

assert (
    e3_eval_dev100_df["image_id"].nunique()
    == 100
)


expected_strata = {
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
}

actual_strata = set(
    e3_eval_dev100_df[
        "density_stratum"
    ]
    .astype(str)
    .unique()
)

assert actual_strata == expected_strata, (
    f"Unexpected density strata: "
    f"{sorted(actual_strata)}"
)


expected_stratum_counts = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}

actual_stratum_counts = (
    e3_eval_dev100_df[
        "density_stratum"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
)

assert actual_stratum_counts == expected_stratum_counts, (
    "Frozen density-stratum allocation changed."
)

print("Frozen dev100 metadata: PASS")


# ------------------------------------------------------------
# 3. Validate fusion-table schema
# ------------------------------------------------------------

required_fusion_cols = {
    "dataset_index",
    "image_id",
    "category",
    "c_variant",
    "d_variant",
    "tau",
    "fusion_family",
    "pred_count",
    "pred_points_xy",
}

missing_fusion_cols = (
    required_fusion_cols
    - set(e3_fusion_per_image_df.columns)
)

assert not missing_fusion_cols, (
    "e3_fusion_per_image_df missing columns: "
    + ", ".join(sorted(missing_fusion_cols))
)

assert len(e3_fusion_per_image_df) == 22500

print("Fusion-table schema and size: PASS")


# ------------------------------------------------------------
# 4. Recover canonical GT points using EXISTING helper
# ------------------------------------------------------------

gt_records = []


for row in e3_eval_dev100_df.itertuples(index=False):

    dataset_index = int(row.dataset_index)

    sample = fsc147_val_dataset[
        dataset_index
    ]

    # Validate dataset identity
    assert sample["image_id"] == row.image_id, (
        f"Image mismatch at dataset_index={dataset_index}: "
        f"dev100={row.image_id}, "
        f"dataset={sample['image_id']}"
    )


    # FSC-147 categories is a dict keyed by category name
    assert row.category in sample["categories"], (
        f"Category '{row.category}' not found "
        f"for image {row.image_id}"
    )


    # Reuse the notebook's canonical GT helper
    gt_points_array = get_gt_points(
        sample,
        row.category,
    )


    assert (
        gt_points_array.ndim == 2
        and gt_points_array.shape[1] == 2
    ), (
        f"Unexpected GT point shape "
        f"for {row.image_id}: "
        f"{gt_points_array.shape}"
    )


    gt_points_xy = [
        (float(x), float(y))
        for x, y in gt_points_array
    ]


    gt_count_from_points = len(
        gt_points_xy
    )


    assert (
        gt_count_from_points
        == int(row.gt_count)
    ), (
        f"GT count mismatch for {row.image_id}: "
        f"metadata={int(row.gt_count)}, "
        f"points={gt_count_from_points}"
    )


    gt_records.append(
        {
            "dataset_index": dataset_index,
            "image_id": row.image_id,
            "category": row.category,
            "gt_count": int(row.gt_count),
            "density_stratum": str(
                row.density_stratum
            ),
            "gt_points_xy": gt_points_xy,
        }
    )


e3_eval_gt_df = pd.DataFrame(
    gt_records
)


# ------------------------------------------------------------
# 5. Validate GT table
# ------------------------------------------------------------

assert len(e3_eval_gt_df) == 100

assert (
    e3_eval_gt_df["dataset_index"].nunique()
    == 100
)

assert (
    e3_eval_gt_df["image_id"].nunique()
    == 100
)

assert (
    e3_eval_gt_df[
        "gt_points_xy"
    ].apply(len)
    ==
    e3_eval_gt_df["gt_count"]
).all()


total_gt_objects = int(
    e3_eval_gt_df[
        "gt_count"
    ].sum()
)

assert total_gt_objects == 10045, (
    f"Expected frozen dev100 GT total 10045, "
    f"found {total_gt_objects}."
)

print("Canonical GT extraction: PASS")


# ------------------------------------------------------------
# 6. Validate fusion image universe against GT universe
# ------------------------------------------------------------

fusion_images = (
    e3_fusion_per_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
        ]
    ]
    .drop_duplicates()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


gt_images = (
    e3_eval_gt_df[
        [
            "dataset_index",
            "image_id",
            "category",
        ]
    ]
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


assert len(fusion_images) == 100
assert len(gt_images) == 100


image_universe_check = (
    fusion_images
    .merge(
        gt_images,
        on=[
            "dataset_index",
            "image_id",
            "category",
        ],
        how="outer",
        indicator=True,
    )
)


assert (
    image_universe_check["_merge"]
    == "both"
).all(), (
    "Fusion and GT image universes differ."
)

print("Fusion ↔ GT image universe: PASS")


# ------------------------------------------------------------
# 7. Attach GT to all F1–F3 rows
# ------------------------------------------------------------

e3_fusion_eval_input_df = (
    e3_fusion_per_image_df
    .merge(
        e3_eval_gt_df,
        on=[
            "dataset_index",
            "image_id",
            "category",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert len(
    e3_fusion_eval_input_df
) == 22500


assert (
    e3_fusion_eval_input_df[
        "gt_count"
    ].notna().all()
)

assert (
    e3_fusion_eval_input_df[
        "density_stratum"
    ].notna().all()
)

assert (
    e3_fusion_eval_input_df[
        "gt_points_xy"
    ].notna().all()
)

print("GT attachment to F1–F3 table: PASS")


# ------------------------------------------------------------
# 8. Validate fusion-family completeness
# ------------------------------------------------------------

family_counts = (
    e3_fusion_eval_input_df
    .groupby(
        "fusion_family"
    )
    .size()
)


assert family_counts.to_dict() == {
    "F1": 7500,
    "F2": 7500,
    "F3": 7500,
}


# ------------------------------------------------------------
# 9. Ensure legitimate F2 zero predictions survive
# ------------------------------------------------------------

f2_zero_eval_df = (
    e3_fusion_eval_input_df[
        (
            e3_fusion_eval_input_df[
                "fusion_family"
            ] == "F2"
        )
        &
        (
            e3_fusion_eval_input_df[
                "pred_count"
            ] == 0
        )
    ]
    .copy()
)


assert len(f2_zero_eval_df) == 60, (
    f"Expected 60 F2 zero-point cases, "
    f"found {len(f2_zero_eval_df)}."
)


assert (
    f2_zero_eval_df[
        "pred_points_xy"
    ]
    .apply(len)
    == 0
).all()

print("F2 zero-point cases retained: 60 — PASS")


# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("GT EVALUATION INPUT SUMMARY")
print("-" * 120)

print(
    f"Images                  : "
    f"{len(e3_eval_gt_df)}"
)

print(
    f"Total GT objects        : "
    f"{total_gt_objects}"
)

print(
    f"Fusion evaluation rows  : "
    f"{len(e3_fusion_eval_input_df)}"
)

print(
    f"F1 rows                 : "
    f"{int(family_counts['F1'])}"
)

print(
    f"F2 rows                 : "
    f"{int(family_counts['F2'])}"
)

print(
    f"F3 rows                 : "
    f"{int(family_counts['F3'])}"
)

print(
    f"F2 zero-point rows      : "
    f"{len(f2_zero_eval_df)}"
)


print("\nDensity-stratum distribution:")

for stratum in [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]:

    print(
        f"  {stratum:<10}: "
        f"{actual_stratum_counts[stratum]}"
    )


print("\nConstructed objects:")

print(
    "  e3_eval_dev100_df       :",
    e3_eval_dev100_df.shape,
)

print(
    "  e3_eval_gt_df           :",
    e3_eval_gt_df.shape,
)

print(
    "  e3_fusion_eval_input_df :",
    e3_fusion_eval_input_df.shape,
)

print(
    "  f2_zero_eval_df         :",
    f2_zero_eval_df.shape,
)


print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print("Existing fsc147_dev_100_df reused: PASS")
print("Existing fsc147_val_dataset reused: PASS")
print("Existing get_gt_points() reused: PASS")
print("100-image GT universe validated: PASS")
print("Total GT objects = 10045: PASS")
print("GT point-list length = gt_count: PASS")
print("Fusion ↔ GT image universe identical: PASS")
print("Frozen S1–S6 allocation unchanged: PASS")
print("22,500 F1–F3 evaluation rows retained: PASS")
print("60 legitimate F2 zero-point cases retained: PASS")
print("No fusion prediction modified: PASS")
print("No performance metric computed: PASS")
print("No tau selected: PASS")


print("\n" + "=" * 120)

print(
    "Step 13D.2b.4 Part A complete — "
    "canonical GT attached and evaluation inputs validated."
)

print(
    "Ready for Part B — evaluate all F1–F3 "
    "configurations with the frozen canonical matcher."
)

print("=" * 120)

STEP 13D.2b.4 PART A — ATTACH CANONICAL GT AND VALIDATE EVALUATION INPUTS
Required notebook objects: PASS
Frozen dev100 metadata: PASS
Fusion-table schema and size: PASS
Canonical GT extraction: PASS
Fusion ↔ GT image universe: PASS
GT attachment to F1–F3 table: PASS
F2 zero-point cases retained: 60 — PASS

------------------------------------------------------------------------------------------------------------------------
GT EVALUATION INPUT SUMMARY
------------------------------------------------------------------------------------------------------------------------
Images                  : 100
Total GT objects        : 10045
Fusion evaluation rows  : 22500
F1 rows                 : 7500
F2 rows                 : 7500
F3 rows                 : 7500
F2 zero-point rows      : 60

Density-stratum distribution:
  S1_0_20   : 20
  S2_20_40  : 20
  S3_40_60  : 20
  S4_60_80  : 20
  S5_80_95  : 10
  S6_95_100 : 10

Constructed objects:
  e3_eval_dev100_df       : (100, 5)
  e3_eval_gt_

In [76]:
# ============================================================
# Step 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines
# Part B — Canonical Per-Image Evaluation of F1–F3
#
# Purpose:
#   Evaluate every F1/F2/F3 prediction generated in 13D.2b.3
#   against canonical FSC-147 GT using the SAME frozen matcher
#   and 3%-short-side tolerance used in corrected C/D/E2.
#
# Inputs:
#   e3_fusion_eval_input_df     (22,500 rows)
#   E1_REGION_MAPS
#   e2_match_radius_px(...)
#   e2_match_pred_to_gt(...)
#
# Outputs:
#   e3_fusion_eval_per_image_df (22,500 rows)
#
# IMPORTANT:
#   - Predictions are NOT modified.
#   - GT is used only for evaluation.
#   - No aggregation/ranking is performed here.
#   - No tau is selected here.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.4 PART B — CANONICAL PER-IMAGE EVALUATION OF F1–F3")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required existing objects
# ------------------------------------------------------------

required_objects = [
    "e3_fusion_eval_input_df",
    "E1_REGION_MAPS",
    "e2_match_radius_px",
    "e2_match_pred_to_gt",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required objects: "
    + ", ".join(missing_objects)
)


required_cols = {
    "dataset_index",
    "image_id",
    "category",
    "c_variant",
    "d_variant",
    "tau",
    "fusion_family",
    "pred_count",
    "pred_points_xy",
    "gt_count",
    "gt_points_xy",
    "density_stratum",
}

missing_cols = (
    required_cols
    - set(e3_fusion_eval_input_df.columns)
)

assert not missing_cols, (
    "e3_fusion_eval_input_df missing columns: "
    + ", ".join(sorted(missing_cols))
)


assert len(e3_fusion_eval_input_df) == 22500

print("Evaluation-input validation: PASS")


# ------------------------------------------------------------
# 2. Validate canonical matcher infrastructure
# ------------------------------------------------------------

assert callable(e2_match_radius_px)
assert callable(e2_match_pred_to_gt)

print("Frozen canonical matcher infrastructure: PASS")


# ------------------------------------------------------------
# 3. Precompute image metadata
#
# Avoid repeatedly recovering width/height for 22,500 rows.
# ------------------------------------------------------------

image_meta = {}


for dataset_index in sorted(
    e3_fusion_eval_input_df[
        "dataset_index"
    ].unique()
):

    dataset_index = int(dataset_index)

    assert dataset_index in E1_REGION_MAPS, (
        f"dataset_index={dataset_index} "
        f"missing from E1_REGION_MAPS."
    )

    region_item = E1_REGION_MAPS[
        dataset_index
    ]

    width = int(
        region_item["width"]
    )

    height = int(
        region_item["height"]
    )

    assert width > 0
    assert height > 0

    radius_px = int(
        e2_match_radius_px(
            image_width=width,
            image_height=height,
        )
    )

    image_meta[
        dataset_index
    ] = {
        "image_id":
            str(region_item["image_id"]),

        "width":
            width,

        "height":
            height,

        "short_side":
            min(width, height),

        "match_radius_px":
            radius_px,
    }


assert len(image_meta) == 100

print("100-image canonical size/radius metadata: PASS")


# ------------------------------------------------------------
# 4. Validate canonical radius
#
# Previous E2 work showed radius = 12 px for all dev100 images.
# We verify rather than assume.
# ------------------------------------------------------------

radius_values = sorted(
    {
        item["match_radius_px"]
        for item in image_meta.values()
    }
)


print(
    "Observed canonical match radii:",
    radius_values,
)


assert all(
    radius >= 1
    for radius in radius_values
)


# This is an expected consistency check from the frozen dev100.
assert radius_values == [12], (
    "Expected canonical dev100 match radius to be "
    f"12 px for all images, found {radius_values}."
)

print("Canonical radius = 12 px for all dev100 images: PASS")


# ------------------------------------------------------------
# 5. Evaluate all 22,500 F1–F3 image/configuration rows
# ------------------------------------------------------------

eval_records = []


for row in e3_fusion_eval_input_df.itertuples(
    index=False
):

    dataset_index = int(
        row.dataset_index
    )

    image_id = str(
        row.image_id
    )


    # --------------------------------------------------------
    # Image metadata / canonical tolerance
    # --------------------------------------------------------

    meta = image_meta[
        dataset_index
    ]


    assert meta["image_id"] == image_id, (
        f"Image mismatch at dataset_index="
        f"{dataset_index}: "
        f"{image_id} vs {meta['image_id']}"
    )


    width = int(
        meta["width"]
    )

    height = int(
        meta["height"]
    )

    radius_px = int(
        meta["match_radius_px"]
    )


    # --------------------------------------------------------
    # Prediction and GT points
    # --------------------------------------------------------

    pred_points = np.asarray(
        row.pred_points_xy,
        dtype=float,
    ).reshape(-1, 2)


    gt_points = np.asarray(
        row.gt_points_xy,
        dtype=float,
    ).reshape(-1, 2)


    pred_count = int(
        row.pred_count
    )

    gt_count = int(
        row.gt_count
    )


    assert len(pred_points) == pred_count, (
        f"Prediction count mismatch: "
        f"{image_id}, "
        f"{row.fusion_family}, "
        f"{row.c_variant}, "
        f"{row.d_variant}, "
        f"tau={row.tau}"
    )


    assert len(gt_points) == gt_count, (
        f"GT count mismatch for {image_id}"
    )


    # --------------------------------------------------------
    # Frozen canonical one-to-one matching
    # --------------------------------------------------------

    match = e2_match_pred_to_gt(
        pred_points_xy=pred_points,
        gt_points_xy=gt_points,
        radius_px=radius_px,
    )


    tp = int(
        match["tp"]
    )

    fp = int(
        match["fp"]
    )

    fn = int(
        match["fn"]
    )


    # --------------------------------------------------------
    # Structural matcher invariants
    # --------------------------------------------------------

    assert tp + fp == pred_count, (
        f"TP+FP != pred_count for {image_id}"
    )

    assert tp + fn == gt_count, (
        f"TP+FN != gt_count for {image_id}"
    )


    # --------------------------------------------------------
    # Per-image localisation metrics
    #
    # Precision:
    #   if no predictions -> 0
    #
    # Recall:
    #   FSC GT count is non-zero for this dev100, but retain
    #   safe handling anyway.
    #
    # F1:
    #   zero when P+R = 0.
    # --------------------------------------------------------

    precision = (
        tp / pred_count
        if pred_count > 0
        else 0.0
    )

    recall = (
        tp / gt_count
        if gt_count > 0
        else 0.0
    )

    f1 = (
        2.0
        * precision
        * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )


    # --------------------------------------------------------
    # Per-image count errors
    # --------------------------------------------------------

    signed_error = (
        pred_count
        - gt_count
    )

    abs_error = abs(
        signed_error
    )

    squared_error = (
        signed_error ** 2
    )


    # --------------------------------------------------------
    # Prediction / GT ratio
    # --------------------------------------------------------

    pred_gt_ratio = (
        pred_count / gt_count
        if gt_count > 0
        else np.nan
    )


    # --------------------------------------------------------
    # Store per-image evaluation record
    # --------------------------------------------------------

    eval_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "category":
                row.category,

            "density_stratum":
                str(row.density_stratum),

            "c_variant":
                row.c_variant,

            "d_variant":
                row.d_variant,

            "tau":
                float(row.tau),

            "fusion_family":
                row.fusion_family,

            "width":
                width,

            "height":
                height,

            "short_side":
                min(width, height),

            "match_radius_px":
                radius_px,

            "gt_count":
                gt_count,

            "pred_count":
                pred_count,

            "tp":
                tp,

            "fp":
                fp,

            "fn":
                fn,

            "precision":
                float(precision),

            "recall":
                float(recall),

            "f1":
                float(f1),

            "signed_error":
                int(signed_error),

            "abs_error":
                int(abs_error),

            "squared_error":
                int(squared_error),

            "pred_gt_ratio":
                float(pred_gt_ratio),
        }
    )


e3_fusion_eval_per_image_df = pd.DataFrame(
    eval_records
)


# ------------------------------------------------------------
# 6. Basic dataframe validation
# ------------------------------------------------------------

assert len(
    e3_fusion_eval_per_image_df
) == 22500


assert (
    e3_fusion_eval_per_image_df[
        "dataset_index"
    ].nunique()
    == 100
)


family_counts = (
    e3_fusion_eval_per_image_df
    .groupby(
        "fusion_family"
    )
    .size()
)


assert family_counts.to_dict() == {
    "F1": 7500,
    "F2": 7500,
    "F3": 7500,
}


print("22,500 per-image evaluations generated: PASS")


# ------------------------------------------------------------
# 7. Validate all 225 configurations
# ------------------------------------------------------------

configuration_cols = [
    "fusion_family",
    "c_variant",
    "d_variant",
    "tau",
]


n_configurations = (
    e3_fusion_eval_per_image_df[
        configuration_cols
    ]
    .drop_duplicates()
    .shape[0]
)


assert n_configurations == 225, (
    f"Expected 225 fusion configurations, "
    f"found {n_configurations}."
)


config_image_counts = (
    e3_fusion_eval_per_image_df
    .groupby(
        configuration_cols
    )[
        "dataset_index"
    ]
    .nunique()
)


assert (
    config_image_counts == 100
).all(), (
    "At least one configuration does not contain "
    "all 100 dev images."
)


print("225 configurations × 100 images: PASS")


# ------------------------------------------------------------
# 8. Validate matcher arithmetic globally
# ------------------------------------------------------------

assert (
    e3_fusion_eval_per_image_df[
        "tp"
    ]
    +
    e3_fusion_eval_per_image_df[
        "fp"
    ]
    ==
    e3_fusion_eval_per_image_df[
        "pred_count"
    ]
).all()


assert (
    e3_fusion_eval_per_image_df[
        "tp"
    ]
    +
    e3_fusion_eval_per_image_df[
        "fn"
    ]
    ==
    e3_fusion_eval_per_image_df[
        "gt_count"
    ]
).all()


assert (
    e3_fusion_eval_per_image_df[
        [
            "precision",
            "recall",
            "f1",
        ]
    ]
    .ge(0.0)
    .all()
    .all()
)


assert (
    e3_fusion_eval_per_image_df[
        [
            "precision",
            "recall",
            "f1",
        ]
    ]
    .le(1.0)
    .all()
    .all()
)


print("Matcher arithmetic and metric ranges: PASS")


# ------------------------------------------------------------
# 9. Explicitly validate F2 zero-point cases
# ------------------------------------------------------------

f2_zero_scored_df = (
    e3_fusion_eval_per_image_df[
        (
            e3_fusion_eval_per_image_df[
                "fusion_family"
            ]
            == "F2"
        )
        &
        (
            e3_fusion_eval_per_image_df[
                "pred_count"
            ]
            == 0
        )
    ]
    .copy()
)


assert len(
    f2_zero_scored_df
) == 60


assert (
    f2_zero_scored_df["tp"] == 0
).all()

assert (
    f2_zero_scored_df["fp"] == 0
).all()

assert (
    f2_zero_scored_df["fn"]
    ==
    f2_zero_scored_df["gt_count"]
).all()

assert (
    f2_zero_scored_df["precision"] == 0.0
).all()

assert (
    f2_zero_scored_df["recall"] == 0.0
).all()

assert (
    f2_zero_scored_df["f1"] == 0.0
).all()


print(
    "60 legitimate F2 zero-point cases "
    "scored correctly: PASS"
)


# ------------------------------------------------------------
# 10. F3 COUNT invariant
#
# F3 has exactly one output per C region.
# Therefore its count must be independent of:
#   - D variant
#   - tau
#
# within each image and C variant.
#
# We do NOT require localisation metrics to be invariant.
# ------------------------------------------------------------

f3_eval_df = (
    e3_fusion_eval_per_image_df[
        e3_fusion_eval_per_image_df[
            "fusion_family"
        ]
        == "F3"
    ]
)


f3_count_invariance = (
    f3_eval_df
    .groupby(
        [
            "dataset_index",
            "c_variant",
        ]
    )[
        "pred_count"
    ]
    .nunique()
)


assert (
    f3_count_invariance == 1
).all(), (
    "F3 count changed across D/tau for "
    "at least one image/C configuration."
)


print(
    "F3 count invariant across D variant and tau: PASS"
)


# ------------------------------------------------------------
# 11. GT consistency across all repeated configurations
# ------------------------------------------------------------

gt_consistency = (
    e3_fusion_eval_per_image_df
    .groupby(
        "dataset_index"
    )
    .agg(
        n_image_ids=(
            "image_id",
            "nunique",
        ),

        n_categories=(
            "category",
            "nunique",
        ),

        n_gt_counts=(
            "gt_count",
            "nunique",
        ),

        n_density_strata=(
            "density_stratum",
            "nunique",
        ),

        n_radii=(
            "match_radius_px",
            "nunique",
        ),
    )
)


assert (
    gt_consistency == 1
).all().all(), (
    "GT metadata changed across repeated "
    "fusion configurations."
)


print(
    "GT metadata consistent across all repeated configurations: PASS"
)


# ------------------------------------------------------------
# 12. Compact diagnostic summary
#
# NOT a ranking and NOT tau selection.
# This is only a structural evaluation summary.
# ------------------------------------------------------------

family_diagnostic_df = (
    e3_fusion_eval_per_image_df
    .groupby(
        "fusion_family",
        as_index=False,
    )
    .agg(
        n_rows=(
            "dataset_index",
            "size",
        ),

        total_tp=(
            "tp",
            "sum",
        ),

        total_fp=(
            "fp",
            "sum",
        ),

        total_fn=(
            "fn",
            "sum",
        ),

        mean_precision=(
            "precision",
            "mean",
        ),

        mean_recall=(
            "recall",
            "mean",
        ),

        mean_f1=(
            "f1",
            "mean",
        ),

        mean_abs_error=(
            "abs_error",
            "mean",
        ),
    )
)


print("\n" + "-" * 120)
print("FAMILY-LEVEL DIAGNOSTIC ONLY")
print("(averaged across all C/D/tau configurations — NOT for model selection)")
print("-" * 120)

print(
    family_diagnostic_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("PART B SUMMARY")
print("-" * 120)

print(
    f"Images                         : "
    f"{e3_fusion_eval_per_image_df['dataset_index'].nunique()}"
)

print(
    f"Fusion families                : "
    f"{e3_fusion_eval_per_image_df['fusion_family'].nunique()}"
)

print(
    f"Fusion configurations          : "
    f"{n_configurations}"
)

print(
    f"Per-image evaluation rows      : "
    f"{len(e3_fusion_eval_per_image_df)}"
)

print(
    f"Canonical match radius values  : "
    f"{radius_values}"
)

print(
    f"F2 zero-point rows scored      : "
    f"{len(f2_zero_scored_df)}"
)


print("\nConstructed objects:")

print(
    "  image_meta                       :",
    len(image_meta),
    "images",
)

print(
    "  e3_fusion_eval_per_image_df      :",
    e3_fusion_eval_per_image_df.shape,
)

print(
    "  f2_zero_scored_df                :",
    f2_zero_scored_df.shape,
)

print(
    "  family_diagnostic_df             :",
    family_diagnostic_df.shape,
)


print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print("Frozen canonical matcher reused: PASS")
print("Frozen 3%-short-side radius reused: PASS")
print("Canonical radius = 12 px on dev100: PASS")
print("225 configurations × 100 images: PASS")
print("TP + FP = prediction count: PASS")
print("TP + FN = GT count: PASS")
print("Precision/Recall/F1 within [0,1]: PASS")
print("60 F2 zero-point cases evaluated: PASS")
print("F3 count invariant across D/tau: PASS")
print("GT metadata consistent across configurations: PASS")
print("No prediction modified: PASS")
print("No configuration selected: PASS")
print("No tau selected: PASS")


print("\n" + "=" * 120)

print(
    "Step 13D.2b.4 Part B complete — "
    "all F1–F3 predictions evaluated with the "
    "frozen canonical matcher."
)

print(
    "Ready for Part C — attach/reconstruct the frozen "
    "C-only and D-only reference baselines."
)

print("=" * 120)

STEP 13D.2b.4 PART B — CANONICAL PER-IMAGE EVALUATION OF F1–F3
Evaluation-input validation: PASS
Frozen canonical matcher infrastructure: PASS
100-image canonical size/radius metadata: PASS
Observed canonical match radii: [12]
Canonical radius = 12 px for all dev100 images: PASS
22,500 per-image evaluations generated: PASS
225 configurations × 100 images: PASS
Matcher arithmetic and metric ranges: PASS
60 legitimate F2 zero-point cases scored correctly: PASS
F3 count invariant across D variant and tau: PASS
GT metadata consistent across all repeated configurations: PASS

------------------------------------------------------------------------------------------------------------------------
FAMILY-LEVEL DIAGNOSTIC ONLY
(averaged across all C/D/tau configurations — NOT for model selection)
------------------------------------------------------------------------------------------------------------------------
fusion_family  n_rows  total_tp  total_fp  total_fn  mean_precision  mean_recall

In [77]:
# ============================================================
# Step 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines
# Part C — Attach Frozen C-only and D-only Reference Baselines
#
# Purpose:
#   1. Reuse the canonical harmonised Group-E handoff created
#      in Step 13A.3.
#   2. Extract the frozen standalone C-only and D-only baselines.
#   3. Validate them against the same dev100 / GT / canonical
#      localisation protocol used in Parts A and B.
#   4. Create clean baseline tables for later controlled
#      fusion comparisons.
#
# IMPORTANT:
#   - C/D predictions are NOT recomputed.
#   - C/D localisation matches are NOT recomputed.
#   - Baselines do NOT depend on tau.
#   - No fusion configuration is selected here.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.4 PART C — ATTACH FROZEN C-ONLY AND D-ONLY REFERENCE BASELINES")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required existing notebook objects
# ------------------------------------------------------------

required_objects = [
    "group_e_handoff_df",
    "e3_eval_gt_df",
    "e3_fusion_eval_per_image_df",
    "E3_C_VARIANTS",
    "E3_D_VARIANTS",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required notebook objects: "
    + ", ".join(missing_objects)
)

print("Required notebook objects: PASS")


# ------------------------------------------------------------
# 2. Validate expected retained candidate identities
# ------------------------------------------------------------

expected_c_variants = [
    "C1",
    "C2b",
    "C2c",
]

expected_d_variants = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]


assert list(E3_C_VARIANTS) == expected_c_variants, (
    f"Unexpected E3_C_VARIANTS: {E3_C_VARIANTS}"
)

assert list(E3_D_VARIANTS) == expected_d_variants, (
    f"Unexpected E3_D_VARIANTS: {E3_D_VARIANTS}"
)

print("Retained C/D candidate identities: PASS")


# ------------------------------------------------------------
# 3. Validate canonical Group-E handoff schema
# ------------------------------------------------------------

required_baseline_cols = {
    "candidate_id",
    "source_group",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "width",
    "height",
    "tolerance_frac",
    "radius_px",
    "gt_count",
    "pred_count",
    "count_error",
    "abs_count_error",
    "squared_error",
    "pred_gt_ratio",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "gt_points_xy",
    "pred_points_xy",
}

missing_baseline_cols = (
    required_baseline_cols
    - set(group_e_handoff_df.columns)
)

assert not missing_baseline_cols, (
    "group_e_handoff_df missing required columns: "
    + ", ".join(sorted(missing_baseline_cols))
)


assert len(group_e_handoff_df) == 800, (
    f"Expected 800 canonical C/D rows, "
    f"found {len(group_e_handoff_df)}."
)

assert (
    group_e_handoff_df["candidate_id"].nunique()
    == 8
)

print("Canonical Group-E handoff schema: PASS")


# ------------------------------------------------------------
# 4. Extract only the eight retained standalone baselines
# ------------------------------------------------------------

baseline_candidate_order = (
    expected_c_variants
    + expected_d_variants
)


e3_baseline_reference_df = (
    group_e_handoff_df[
        group_e_handoff_df[
            "candidate_id"
        ].astype(str).isin(
            baseline_candidate_order
        )
    ]
    .copy()
)


assert len(e3_baseline_reference_df) == 800


candidate_counts = (
    e3_baseline_reference_df[
        "candidate_id"
    ]
    .astype(str)
    .value_counts()
)


for candidate_id in baseline_candidate_order:

    assert (
        int(candidate_counts[candidate_id])
        == 100
    ), (
        f"{candidate_id}: expected 100 rows, "
        f"found {candidate_counts[candidate_id]}."
    )


print("Eight frozen standalone baselines extracted: PASS")


# ------------------------------------------------------------
# 5. Create clean C-only and D-only tables
#
# We retain only fields required for evaluation/comparison.
# No tau is added because standalone baselines do not depend
# on the C–D association tolerance.
# ------------------------------------------------------------

baseline_core_cols = [
    "candidate_id",
    "source_group",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "width",
    "height",
    "radius_px",
    "gt_count",
    "pred_count",
    "count_error",
    "abs_count_error",
    "squared_error",
    "pred_gt_ratio",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
]


e3_c_baseline_df = (
    e3_baseline_reference_df[
        e3_baseline_reference_df[
            "candidate_id"
        ].astype(str).isin(
            expected_c_variants
        )
    ][baseline_core_cols]
    .copy()
    .reset_index(drop=True)
)


e3_d_baseline_df = (
    e3_baseline_reference_df[
        e3_baseline_reference_df[
            "candidate_id"
        ].astype(str).isin(
            expected_d_variants
        )
    ][baseline_core_cols]
    .copy()
    .reset_index(drop=True)
)


# Make candidate IDs ordinary strings in these new tables.
e3_c_baseline_df["candidate_id"] = (
    e3_c_baseline_df[
        "candidate_id"
    ].astype(str)
)

e3_d_baseline_df["candidate_id"] = (
    e3_d_baseline_df[
        "candidate_id"
    ].astype(str)
)


assert len(e3_c_baseline_df) == 300
assert len(e3_d_baseline_df) == 500

print("C-only table: 300 rows — PASS")
print("D-only table: 500 rows — PASS")


# ------------------------------------------------------------
# 6. Validate exactly one row per candidate × image
# ------------------------------------------------------------

assert not (
    e3_c_baseline_df[
        [
            "candidate_id",
            "dataset_index",
        ]
    ]
    .duplicated()
    .any()
)


assert not (
    e3_d_baseline_df[
        [
            "candidate_id",
            "dataset_index",
        ]
    ]
    .duplicated()
    .any()
)


assert (
    e3_c_baseline_df
    .groupby(
        "candidate_id"
    )[
        "dataset_index"
    ]
    .nunique()
    .eq(100)
    .all()
)


assert (
    e3_d_baseline_df
    .groupby(
        "candidate_id"
    )[
        "dataset_index"
    ]
    .nunique()
    .eq(100)
    .all()
)


print("One baseline row per candidate × image: PASS")


# ------------------------------------------------------------
# 7. Validate baseline image universe against Part-A GT
# ------------------------------------------------------------

baseline_image_keys = (
    e3_baseline_reference_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
        ]
    ]
    .drop_duplicates()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


current_gt_keys = (
    e3_eval_gt_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
        ]
    ]
    .drop_duplicates()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


assert len(baseline_image_keys) == 100
assert len(current_gt_keys) == 100


baseline_gt_universe_check = (
    baseline_image_keys
    .merge(
        current_gt_keys,
        on=[
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
        ],
        how="outer",
        indicator=True,
    )
)


assert (
    baseline_gt_universe_check[
        "_merge"
    ] == "both"
).all(), (
    "Frozen C/D baselines and current Part-A GT "
    "do not use the same dev100 image universe."
)


print("Baseline ↔ current GT image universe: PASS")


# ------------------------------------------------------------
# 8. Validate GT count consistency
# ------------------------------------------------------------

current_gt_count_lookup = (
    e3_eval_gt_df[
        [
            "dataset_index",
            "gt_count",
        ]
    ]
    .drop_duplicates()
)


baseline_gt_check = (
    e3_baseline_reference_df[
        [
            "candidate_id",
            "dataset_index",
            "gt_count",
        ]
    ]
    .copy()
)


baseline_gt_check["candidate_id"] = (
    baseline_gt_check[
        "candidate_id"
    ].astype(str)
)


baseline_gt_check = (
    baseline_gt_check
    .merge(
        current_gt_count_lookup,
        on="dataset_index",
        how="left",
        suffixes=(
            "_baseline",
            "_current",
        ),
        validate="many_to_one",
    )
)


assert (
    baseline_gt_check[
        "gt_count_baseline"
    ].astype(int)
    ==
    baseline_gt_check[
        "gt_count_current"
    ].astype(int)
).all(), (
    "GT count mismatch between frozen baselines "
    "and current Part-A GT."
)


print("Baseline ↔ current GT counts: PASS")


# ------------------------------------------------------------
# 9. Validate image dimensions and canonical radius against Part B
# ------------------------------------------------------------

part_b_image_meta_df = (
    e3_fusion_eval_per_image_df[
        [
            "dataset_index",
            "image_id",
            "width",
            "height",
            "match_radius_px",
        ]
    ]
    .drop_duplicates()
    .sort_values("dataset_index")
    .reset_index(drop=True)
)


assert len(part_b_image_meta_df) == 100


baseline_meta_check = (
    e3_baseline_reference_df[
        [
            "candidate_id",
            "dataset_index",
            "image_id",
            "width",
            "height",
            "radius_px",
        ]
    ]
    .copy()
)


baseline_meta_check["candidate_id"] = (
    baseline_meta_check[
        "candidate_id"
    ].astype(str)
)


baseline_meta_check = (
    baseline_meta_check
    .merge(
        part_b_image_meta_df,
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        suffixes=(
            "_baseline",
            "_partB",
        ),
        validate="many_to_one",
    )
)


assert (
    baseline_meta_check[
        "width_baseline"
    ].astype(int)
    ==
    baseline_meta_check[
        "width_partB"
    ].astype(int)
).all()


assert (
    baseline_meta_check[
        "height_baseline"
    ].astype(int)
    ==
    baseline_meta_check[
        "height_partB"
    ].astype(int)
).all()


assert (
    baseline_meta_check[
        "radius_px"
    ].astype(int)
    ==
    baseline_meta_check[
        "match_radius_px"
    ].astype(int)
).all()


assert (
    baseline_meta_check[
        "radius_px"
    ].astype(int)
    == 12
).all()


print("Image dimensions and canonical radius match Part B: PASS")


# ------------------------------------------------------------
# 10. Validate baseline metric arithmetic
# ------------------------------------------------------------

for baseline_df, baseline_name in [
    (
        e3_c_baseline_df,
        "C",
    ),
    (
        e3_d_baseline_df,
        "D",
    ),
]:

    # Count error
    assert np.allclose(
        baseline_df[
            "count_error"
        ].astype(float),
        (
            baseline_df[
                "pred_count"
            ].astype(float)
            -
            baseline_df[
                "gt_count"
            ].astype(float)
        ),
    ), (
        f"{baseline_name}: count_error mismatch."
    )


    # Absolute count error
    assert np.allclose(
        baseline_df[
            "abs_count_error"
        ].astype(float),
        np.abs(
            baseline_df[
                "count_error"
            ].astype(float)
        ),
    ), (
        f"{baseline_name}: abs_count_error mismatch."
    )


    # Squared count error
    assert np.allclose(
        baseline_df[
            "squared_error"
        ].astype(float),
        (
            baseline_df[
                "count_error"
            ].astype(float)
            ** 2
        ),
    ), (
        f"{baseline_name}: squared_error mismatch."
    )


    # Matcher arithmetic
    assert (
        baseline_df[
            "tp"
        ].astype(int)
        +
        baseline_df[
            "fp"
        ].astype(int)
        ==
        baseline_df[
            "pred_count"
        ].astype(int)
    ).all(), (
        f"{baseline_name}: TP+FP != pred_count."
    )


    assert (
        baseline_df[
            "tp"
        ].astype(int)
        +
        baseline_df[
            "fn"
        ].astype(int)
        ==
        baseline_df[
            "gt_count"
        ].astype(int)
    ).all(), (
        f"{baseline_name}: TP+FN != gt_count."
    )


    # Metric ranges
    assert (
        baseline_df[
            [
                "precision",
                "recall",
                "f1",
            ]
        ]
        .astype(float)
        .ge(0.0)
        .all()
        .all()
    )


    assert (
        baseline_df[
            [
                "precision",
                "recall",
                "f1",
            ]
        ]
        .astype(float)
        .le(1.0)
        .all()
        .all()
    )


print("Frozen C/D metric arithmetic: PASS")


# ------------------------------------------------------------
# 11. Build compact overall baseline summary
#
# Same macro localisation convention used previously:
# mean per-image Precision / Recall / F1.
#
# This is a reference summary only.
# No ranking or selection is performed.
# ------------------------------------------------------------

def summarise_baseline_candidate(candidate_df):

    gt = (
        candidate_df[
            "gt_count"
        ].astype(float)
    )

    pred = (
        candidate_df[
            "pred_count"
        ].astype(float)
    )

    count_error = (
        candidate_df[
            "count_error"
        ].astype(float)
    )

    return pd.Series(
        {
            "n_images":
                len(candidate_df),

            "sum_gt":
                int(gt.sum()),

            "sum_pred":
                int(pred.sum()),

            "mae":
                float(
                    candidate_df[
                        "abs_count_error"
                    ]
                    .astype(float)
                    .mean()
                ),

            "rmse":
                float(
                    np.sqrt(
                        candidate_df[
                            "squared_error"
                        ]
                        .astype(float)
                        .mean()
                    )
                ),

            "mean_signed_error":
                float(
                    count_error.mean()
                ),

            "mean_pred_gt_ratio":
                float(
                    candidate_df[
                        "pred_gt_ratio"
                    ]
                    .astype(float)
                    .mean()
                ),

            "aggregate_pred_gt_ratio":
                float(
                    pred.sum()
                    /
                    gt.sum()
                ),

            "macro_precision":
                float(
                    candidate_df[
                        "precision"
                    ]
                    .astype(float)
                    .mean()
                ),

            "macro_recall":
                float(
                    candidate_df[
                        "recall"
                    ]
                    .astype(float)
                    .mean()
                ),

            "macro_f1":
                float(
                    candidate_df[
                        "f1"
                    ]
                    .astype(float)
                    .mean()
                ),
        }
    )


e3_baseline_overall_summary_df = (
    pd.concat(
        [
            e3_c_baseline_df,
            e3_d_baseline_df,
        ],
        ignore_index=True,
    )
    .groupby(
        [
            "source_group",
            "candidate_id",
        ],
        sort=False,
    )
    .apply(
        summarise_baseline_candidate,
        include_groups=False,
    )
    .reset_index()
)


# Preserve intended candidate order
candidate_order_map = {
    candidate_id: i
    for i, candidate_id
    in enumerate(
        baseline_candidate_order
    )
}


e3_baseline_overall_summary_df[
    "_order"
] = (
    e3_baseline_overall_summary_df[
        "candidate_id"
    ]
    .map(candidate_order_map)
)


e3_baseline_overall_summary_df = (
    e3_baseline_overall_summary_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. Validate against frozen E0 overall values
#
# These values were already established in Step 13A/E0.
# This is a regression check, not a new result.
# ------------------------------------------------------------

expected_frozen_overall = {
    "C1": {
        "mae": 79.67,
        "rmse": 263.5414,
        "macro_f1": 0.2687,
    },

    "C2b": {
        "mae": 88.79,
        "rmse": 257.5129,
        "macro_f1": 0.2861,
    },

    "C2c": {
        "mae": 80.08,
        "rmse": 260.1300,
        "macro_f1": 0.3178,
    },

    "D1": {
        "mae": 94.25,
        "rmse": 254.9215,
        "macro_f1": 0.3452,
    },

    "D4": {
        "mae": 80.77,
        "rmse": 260.6916,
        "macro_f1": 0.3247,
    },

    "D5-768": {
        "mae": 138.20,
        "rmse": 249.4203,
        "macro_f1": 0.2852,
    },

    "D6B-T-native": {
        "mae": 166.72,
        "rmse": 288.7728,
        "macro_f1": 0.3136,
    },

    "D6B9-strict": {
        "mae": 90.02,
        "rmse": 265.9052,
        "macro_f1": 0.3277,
    },
}


for row in (
    e3_baseline_overall_summary_df
    .itertuples(index=False)
):

    expected = (
        expected_frozen_overall[
            row.candidate_id
        ]
    )

    assert np.isclose(
        row.mae,
        expected["mae"],
        atol=1e-10,
    ), (
        f"{row.candidate_id}: MAE changed."
    )

    assert np.isclose(
        row.rmse,
        expected["rmse"],
        atol=5e-5,
    ), (
        f"{row.candidate_id}: RMSE changed."
    )

    assert np.isclose(
        row.macro_f1,
        expected["macro_f1"],
        atol=5e-5,
    ), (
        f"{row.candidate_id}: macro F1 changed."
    )


print("Frozen E0 overall baseline regression check: PASS")


# ------------------------------------------------------------
# 13. Display baseline summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("FROZEN C-ONLY / D-ONLY REFERENCE SUMMARY")
print("-" * 120)

print(
    e3_baseline_overall_summary_df[
        [
            "source_group",
            "candidate_id",
            "n_images",
            "sum_gt",
            "sum_pred",
            "mae",
            "rmse",
            "mean_signed_error",
            "mean_pred_gt_ratio",
            "aggregate_pred_gt_ratio",
            "macro_precision",
            "macro_recall",
            "macro_f1",
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("PART C SUMMARY")
print("-" * 120)

print(
    f"C-only candidates           : "
    f"{len(expected_c_variants)}"
)

print(
    f"D-only candidates           : "
    f"{len(expected_d_variants)}"
)

print(
    f"C-only per-image rows       : "
    f"{len(e3_c_baseline_df)}"
)

print(
    f"D-only per-image rows       : "
    f"{len(e3_d_baseline_df)}"
)

print(
    f"Total baseline rows         : "
    f"{len(e3_c_baseline_df) + len(e3_d_baseline_df)}"
)

print(
    f"Baseline summary rows       : "
    f"{len(e3_baseline_overall_summary_df)}"
)


print("\nConstructed objects:")

print(
    "  e3_baseline_reference_df       :",
    e3_baseline_reference_df.shape,
)

print(
    "  e3_c_baseline_df               :",
    e3_c_baseline_df.shape,
)

print(
    "  e3_d_baseline_df               :",
    e3_d_baseline_df.shape,
)

print(
    "  e3_baseline_overall_summary_df :",
    e3_baseline_overall_summary_df.shape,
)


print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print("Canonical Step-13A Group-E handoff reused: PASS")
print("3 frozen C candidates × 100 images: PASS")
print("5 frozen D candidates × 100 images: PASS")
print("Same dev100 image universe as Part A: PASS")
print("GT counts identical to Part A: PASS")
print("Image dimensions identical to Part B: PASS")
print("Canonical radius = 12 px: PASS")
print("TP + FP = prediction count: PASS")
print("TP + FN = GT count: PASS")
print("Frozen E0 overall metrics reproduced: PASS")
print("No C/D prediction recomputed: PASS")
print("No C/D localisation match recomputed: PASS")
print("No tau attached to standalone baselines: PASS")
print("No configuration selected: PASS")


print("\n" + "=" * 120)

print(
    "Step 13D.2b.4 Part C complete — "
    "frozen C-only and D-only reference baselines attached."
)

print(
    "Ready for Part D — overall controlled comparison of "
    "C-only, D-only, F1, F2 and F3."
)

print("=" * 120)

STEP 13D.2b.4 PART C — ATTACH FROZEN C-ONLY AND D-ONLY REFERENCE BASELINES
Required notebook objects: PASS
Retained C/D candidate identities: PASS
Canonical Group-E handoff schema: PASS
Eight frozen standalone baselines extracted: PASS
C-only table: 300 rows — PASS
D-only table: 500 rows — PASS
One baseline row per candidate × image: PASS
Baseline ↔ current GT image universe: PASS
Baseline ↔ current GT counts: PASS
Image dimensions and canonical radius match Part B: PASS
Frozen C/D metric arithmetic: PASS
Frozen E0 overall baseline regression check: PASS

------------------------------------------------------------------------------------------------------------------------
FROZEN C-ONLY / D-ONLY REFERENCE SUMMARY
------------------------------------------------------------------------------------------------------------------------
source_group candidate_id  n_images     sum_gt   sum_pred      mae     rmse  mean_signed_error  mean_pred_gt_ratio  aggregate_pred_gt_ratio  macro_precisio

In [78]:
# ============================================================
# Step 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines
# Part D — Overall Controlled Comparison
#
# Purpose:
#   1. Aggregate the 22,500 per-image fusion evaluations into
#      225 overall fusion configurations:
#
#         3 C × 5 D × 5 tau × 3 fusion families
#
#   2. Compute the same overall metrics used for the frozen
#      C-only / D-only baselines.
#
#   3. Attach the corresponding standalone C_i and D_j
#      reference metrics to every fusion configuration.
#
#   4. Validate important structural invariants, especially:
#         - 100 images per fusion configuration
#         - GT total = 10,045 per configuration
#         - F3 count metrics = corresponding C-only metrics
#
# IMPORTANT:
#   - No configuration is selected.
#   - No tau is selected.
#   - No density-stratum analysis is performed here.
#   - No predictions or matching results are recomputed.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.4 PART D — OVERALL CONTROLLED COMPARISON")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required objects from Parts B and C
# ------------------------------------------------------------

required_objects = [
    "e3_fusion_eval_per_image_df",
    "e3_c_baseline_df",
    "e3_d_baseline_df",
    "e3_baseline_overall_summary_df",
    "E3_C_VARIANTS",
    "E3_D_VARIANTS",
    "E3_ASSOC_TAU_VALUES",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required objects: "
    + ", ".join(missing_objects)
)


required_fusion_cols = {
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "c_variant",
    "d_variant",
    "tau",
    "fusion_family",
    "gt_count",
    "pred_count",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "signed_error",
    "abs_error",
    "squared_error",
    "pred_gt_ratio",
}

missing_cols = (
    required_fusion_cols
    - set(e3_fusion_eval_per_image_df.columns)
)

assert not missing_cols, (
    "e3_fusion_eval_per_image_df missing columns: "
    + ", ".join(sorted(missing_cols))
)


assert len(e3_fusion_eval_per_image_df) == 22500

print("Required Part-B / Part-C objects: PASS")


# ------------------------------------------------------------
# 2. Define canonical configuration columns
# ------------------------------------------------------------

fusion_config_cols = [
    "fusion_family",
    "c_variant",
    "d_variant",
    "tau",
]


expected_n_configs = (
    len(E3_C_VARIANTS)
    *
    len(E3_D_VARIANTS)
    *
    len(E3_ASSOC_TAU_VALUES)
    *
    3
)

assert expected_n_configs == 225


# ------------------------------------------------------------
# 3. Aggregate one fusion configuration
#
# Macro localisation convention:
#   mean per-image precision / recall / F1
#
# This matches the frozen C/D summaries.
# ------------------------------------------------------------

def summarise_fusion_configuration(config_df):

    gt = (
        config_df[
            "gt_count"
        ].astype(float)
    )

    pred = (
        config_df[
            "pred_count"
        ].astype(float)
    )

    signed_error = (
        config_df[
            "signed_error"
        ].astype(float)
    )


    return pd.Series(
        {
            "n_images":
                int(len(config_df)),

            "sum_gt":
                int(gt.sum()),

            "sum_pred":
                int(pred.sum()),

            "mae":
                float(
                    config_df[
                        "abs_error"
                    ]
                    .astype(float)
                    .mean()
                ),

            "rmse":
                float(
                    np.sqrt(
                        config_df[
                            "squared_error"
                        ]
                        .astype(float)
                        .mean()
                    )
                ),

            "mean_signed_error":
                float(
                    signed_error.mean()
                ),

            "mean_pred_gt_ratio":
                float(
                    config_df[
                        "pred_gt_ratio"
                    ]
                    .astype(float)
                    .mean()
                ),

            "aggregate_pred_gt_ratio":
                float(
                    pred.sum()
                    /
                    gt.sum()
                ),

            "macro_precision":
                float(
                    config_df[
                        "precision"
                    ]
                    .astype(float)
                    .mean()
                ),

            "macro_recall":
                float(
                    config_df[
                        "recall"
                    ]
                    .astype(float)
                    .mean()
                ),

            "macro_f1":
                float(
                    config_df[
                        "f1"
                    ]
                    .astype(float)
                    .mean()
                ),

            # Diagnostic totals only.
            # These are useful for consistency checks but
            # macro P/R/F1 remain the primary localisation
            # metrics used in the experiment.
            "total_tp":
                int(
                    config_df[
                        "tp"
                    ].sum()
                ),

            "total_fp":
                int(
                    config_df[
                        "fp"
                    ].sum()
                ),

            "total_fn":
                int(
                    config_df[
                        "fn"
                    ].sum()
                ),
        }
    )


# ------------------------------------------------------------
# 4. Aggregate all 225 fusion configurations
# ------------------------------------------------------------

e3_fusion_overall_summary_df = (
    e3_fusion_eval_per_image_df
    .groupby(
        fusion_config_cols,
        sort=False,
        observed=True,
    )
    .apply(
        summarise_fusion_configuration,
        include_groups=False,
    )
    .reset_index()
)


assert len(
    e3_fusion_overall_summary_df
) == expected_n_configs, (
    f"Expected {expected_n_configs} fusion configurations, "
    f"found {len(e3_fusion_overall_summary_df)}."
)


print("225 overall fusion configurations generated: PASS")


# ------------------------------------------------------------
# 5. Validate each configuration contains the same frozen
#    100-image / 10,045-object evaluation universe
# ------------------------------------------------------------

assert (
    e3_fusion_overall_summary_df[
        "n_images"
    ].astype(int)
    == 100
).all()


assert (
    e3_fusion_overall_summary_df[
        "sum_gt"
    ].astype(int)
    == 10045
).all()


assert (
    e3_fusion_overall_summary_df[
        "total_tp"
    ].astype(int)
    +
    e3_fusion_overall_summary_df[
        "total_fn"
    ].astype(int)
    ==
    10045
).all()


print("100 images / 10,045 GT objects per configuration: PASS")


# ------------------------------------------------------------
# 6. Validate configuration coverage
#
# Every fusion family must contain:
#   3 C × 5 D × 5 tau = 75 configurations.
# ------------------------------------------------------------

family_config_counts = (
    e3_fusion_overall_summary_df
    .groupby(
        "fusion_family"
    )
    .size()
)


assert family_config_counts.to_dict() == {
    "F1": 75,
    "F2": 75,
    "F3": 75,
}


for fusion_family in [
    "F1",
    "F2",
    "F3",
]:

    family_df = (
        e3_fusion_overall_summary_df[
            e3_fusion_overall_summary_df[
                "fusion_family"
            ]
            == fusion_family
        ]
    )

    assert (
        family_df[
            "c_variant"
        ].nunique()
        == len(E3_C_VARIANTS)
    )

    assert (
        family_df[
            "d_variant"
        ].nunique()
        == len(E3_D_VARIANTS)
    )

    assert (
        family_df[
            "tau"
        ].nunique()
        == len(E3_ASSOC_TAU_VALUES)
    )


print("75 configurations per fusion family: PASS")


# ------------------------------------------------------------
# 7. Prepare C-only overall reference metrics for joining
# ------------------------------------------------------------

baseline_metric_cols = [
    "n_images",
    "sum_gt",
    "sum_pred",
    "mae",
    "rmse",
    "mean_signed_error",
    "mean_pred_gt_ratio",
    "aggregate_pred_gt_ratio",
    "macro_precision",
    "macro_recall",
    "macro_f1",
]


c_reference_df = (
    e3_baseline_overall_summary_df[
        e3_baseline_overall_summary_df[
            "source_group"
        ]
        == "C"
    ][
        [
            "candidate_id",
        ]
        + baseline_metric_cols
    ]
    .copy()
)


c_reference_df["candidate_id"] = (
    c_reference_df[
        "candidate_id"
    ].astype(str)
)


c_reference_df = (
    c_reference_df
    .rename(
        columns={
            "candidate_id":
                "c_variant",

            **{
                col:
                    f"c_{col}"
                for col
                in baseline_metric_cols
            },
        }
    )
)


assert len(c_reference_df) == 3


# ------------------------------------------------------------
# 8. Prepare D-only overall reference metrics for joining
# ------------------------------------------------------------

d_reference_df = (
    e3_baseline_overall_summary_df[
        e3_baseline_overall_summary_df[
            "source_group"
        ]
        == "D"
    ][
        [
            "candidate_id",
        ]
        + baseline_metric_cols
    ]
    .copy()
)


d_reference_df["candidate_id"] = (
    d_reference_df[
        "candidate_id"
    ].astype(str)
)


d_reference_df = (
    d_reference_df
    .rename(
        columns={
            "candidate_id":
                "d_variant",

            **{
                col:
                    f"d_{col}"
                for col
                in baseline_metric_cols
            },
        }
    )
)


assert len(d_reference_df) == 5


# ------------------------------------------------------------
# 9. Attach corresponding C_i and D_j references
#
# Example:
#
#   F2(C2c, D1, tau=.02)
#
# receives:
#   C reference = C2c
#   D reference = D1
#
# Tau affects only fusion; the standalone references remain
# unchanged.
# ------------------------------------------------------------

e3_overall_controlled_comparison_df = (
    e3_fusion_overall_summary_df
    .merge(
        c_reference_df,
        on="c_variant",
        how="left",
        validate="many_to_one",
    )
    .merge(
        d_reference_df,
        on="d_variant",
        how="left",
        validate="many_to_one",
    )
)


assert len(
    e3_overall_controlled_comparison_df
) == 225


assert not (
    e3_overall_controlled_comparison_df[
        [
            "c_mae",
            "c_macro_f1",
            "d_mae",
            "d_macro_f1",
        ]
    ]
    .isna()
    .any()
    .any()
)


print("Corresponding C-only and D-only references attached: PASS")


# ------------------------------------------------------------
# 10. Validate reference invariance across tau/fusion family
#
# C-only metrics must depend only on c_variant.
# D-only metrics must depend only on d_variant.
# ------------------------------------------------------------

c_reference_check = (
    e3_overall_controlled_comparison_df
    .groupby(
        "c_variant"
    )[
        [
            "c_mae",
            "c_rmse",
            "c_macro_precision",
            "c_macro_recall",
            "c_macro_f1",
        ]
    ]
    .nunique()
)


assert (
    c_reference_check == 1
).all().all()


d_reference_check = (
    e3_overall_controlled_comparison_df
    .groupby(
        "d_variant"
    )[
        [
            "d_mae",
            "d_rmse",
            "d_macro_precision",
            "d_macro_recall",
            "d_macro_f1",
        ]
    ]
    .nunique()
)


assert (
    d_reference_check == 1
).all().all()


print("Standalone baseline references invariant across fusion/tau: PASS")


# ------------------------------------------------------------
# 11. F3 count-metric regression check
#
# F3 outputs exactly one point per C region.
# Therefore for every (C,D,tau):
#
#   F3 sum_pred           == C sum_pred
#   F3 MAE                == C MAE
#   F3 RMSE               == C RMSE
#   F3 signed error       == C signed error
#   F3 count ratios       == C count ratios
#
# Localisation metrics are NOT expected to be equal.
# ------------------------------------------------------------

f3_overall_check_df = (
    e3_overall_controlled_comparison_df[
        e3_overall_controlled_comparison_df[
            "fusion_family"
        ]
        == "F3"
    ]
)


assert len(f3_overall_check_df) == 75


assert np.allclose(
    f3_overall_check_df[
        "sum_pred"
    ].astype(float),
    f3_overall_check_df[
        "c_sum_pred"
    ].astype(float),
)


assert np.allclose(
    f3_overall_check_df[
        "mae"
    ].astype(float),
    f3_overall_check_df[
        "c_mae"
    ].astype(float),
)


assert np.allclose(
    f3_overall_check_df[
        "rmse"
    ].astype(float),
    f3_overall_check_df[
        "c_rmse"
    ].astype(float),
)


assert np.allclose(
    f3_overall_check_df[
        "mean_signed_error"
    ].astype(float),
    f3_overall_check_df[
        "c_mean_signed_error"
    ].astype(float),
)


assert np.allclose(
    f3_overall_check_df[
        "mean_pred_gt_ratio"
    ].astype(float),
    f3_overall_check_df[
        "c_mean_pred_gt_ratio"
    ].astype(float),
)


assert np.allclose(
    f3_overall_check_df[
        "aggregate_pred_gt_ratio"
    ].astype(float),
    f3_overall_check_df[
        "c_aggregate_pred_gt_ratio"
    ].astype(float),
)


print("F3 count metrics exactly reproduce corresponding C-only baseline: PASS")


# ------------------------------------------------------------
# 12. Add descriptive paired-reference columns
#
# These are NOT selection scores.
# They simply make later controlled comparisons easier to read.
# ------------------------------------------------------------

e3_overall_controlled_comparison_df[
    "paired_c_reference"
] = (
    e3_overall_controlled_comparison_df[
        "c_variant"
    ]
)


e3_overall_controlled_comparison_df[
    "paired_d_reference"
] = (
    e3_overall_controlled_comparison_df[
        "d_variant"
    ]
)


# ------------------------------------------------------------
# 13. Stable ordering
# ------------------------------------------------------------

fusion_family_order = {
    "F1": 0,
    "F2": 1,
    "F3": 2,
}

c_order = {
    candidate: i
    for i, candidate
    in enumerate(E3_C_VARIANTS)
}

d_order = {
    candidate: i
    for i, candidate
    in enumerate(E3_D_VARIANTS)
}

tau_order = {
    float(tau): i
    for i, tau
    in enumerate(E3_ASSOC_TAU_VALUES)
}


e3_overall_controlled_comparison_df[
    "_family_order"
] = (
    e3_overall_controlled_comparison_df[
        "fusion_family"
    ]
    .map(fusion_family_order)
)


e3_overall_controlled_comparison_df[
    "_c_order"
] = (
    e3_overall_controlled_comparison_df[
        "c_variant"
    ]
    .map(c_order)
)


e3_overall_controlled_comparison_df[
    "_d_order"
] = (
    e3_overall_controlled_comparison_df[
        "d_variant"
    ]
    .map(d_order)
)


e3_overall_controlled_comparison_df[
    "_tau_order"
] = (
    e3_overall_controlled_comparison_df[
        "tau"
    ]
    .astype(float)
    .map(tau_order)
)


e3_overall_controlled_comparison_df = (
    e3_overall_controlled_comparison_df
    .sort_values(
        [
            "_family_order",
            "_c_order",
            "_d_order",
            "_tau_order",
        ]
    )
    .drop(
        columns=[
            "_family_order",
            "_c_order",
            "_d_order",
            "_tau_order",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. Create compact display table
#
# Keep the main fusion metrics together with the most useful
# paired C/D references.
#
# This is NOT ranked.
# ------------------------------------------------------------

e3_overall_display_df = (
    e3_overall_controlled_comparison_df[
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",

            "sum_pred",

            "mae",
            "rmse",
            "mean_signed_error",

            "mean_pred_gt_ratio",
            "aggregate_pred_gt_ratio",

            "macro_precision",
            "macro_recall",
            "macro_f1",

            "c_mae",
            "c_macro_precision",
            "c_macro_recall",
            "c_macro_f1",

            "d_mae",
            "d_macro_precision",
            "d_macro_recall",
            "d_macro_f1",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 15. Display all 225 rows in family blocks
#
# This preserves the complete controlled results without
# ranking or selecting configurations.
# ------------------------------------------------------------

for fusion_family in [
    "F1",
    "F2",
    "F3",
]:

    print("\n" + "-" * 120)

    print(
        f"{fusion_family} — OVERALL CONTROLLED RESULTS "
        f"(75 configurations)"
    )

    print("-" * 120)

    family_display = (
        e3_overall_display_df[
            e3_overall_display_df[
                "fusion_family"
            ]
            == fusion_family
        ]
    )

    print(
        family_display.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )


# ------------------------------------------------------------
# 16. Compact family-range diagnostic
#
# This reports the observed range across the 75 configurations
# within each family. It is descriptive only and does not
# identify a winner.
# ------------------------------------------------------------

e3_family_metric_range_df = (
    e3_fusion_overall_summary_df
    .groupby(
        "fusion_family",
        as_index=False,
    )
    .agg(
        mae_min=(
            "mae",
            "min",
        ),

        mae_max=(
            "mae",
            "max",
        ),

        precision_min=(
            "macro_precision",
            "min",
        ),

        precision_max=(
            "macro_precision",
            "max",
        ),

        recall_min=(
            "macro_recall",
            "min",
        ),

        recall_max=(
            "macro_recall",
            "max",
        ),

        f1_min=(
            "macro_f1",
            "min",
        ),

        f1_max=(
            "macro_f1",
            "max",
        ),
    )
)


print("\n" + "-" * 120)
print("FUSION-FAMILY METRIC RANGES — DESCRIPTIVE ONLY")
print("-" * 120)

print(
    e3_family_metric_range_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 17. Final structural summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("PART D SUMMARY")
print("-" * 120)

print(
    f"Fusion configurations            : "
    f"{len(e3_fusion_overall_summary_df)}"
)

print(
    f"Controlled comparison rows       : "
    f"{len(e3_overall_controlled_comparison_df)}"
)

print(
    f"F1 configurations                : "
    f"{int(family_config_counts['F1'])}"
)

print(
    f"F2 configurations                : "
    f"{int(family_config_counts['F2'])}"
)

print(
    f"F3 configurations                : "
    f"{int(family_config_counts['F3'])}"
)

print(
    "Images per configuration         : 100"
)

print(
    "GT objects per configuration     : 10045"
)


print("\nConstructed objects:")

print(
    "  e3_fusion_overall_summary_df          :",
    e3_fusion_overall_summary_df.shape,
)

print(
    "  c_reference_df                        :",
    c_reference_df.shape,
)

print(
    "  d_reference_df                        :",
    d_reference_df.shape,
)

print(
    "  e3_overall_controlled_comparison_df   :",
    e3_overall_controlled_comparison_df.shape,
)

print(
    "  e3_overall_display_df                 :",
    e3_overall_display_df.shape,
)

print(
    "  e3_family_metric_range_df             :",
    e3_family_metric_range_df.shape,
)


print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print("22,500 Part-B evaluations reused: PASS")
print("225 overall fusion configurations: PASS")
print("75 configurations per F1/F2/F3 family: PASS")
print("100 images per fusion configuration: PASS")
print("10,045 GT objects per fusion configuration: PASS")
print("Corresponding C-only references attached: PASS")
print("Corresponding D-only references attached: PASS")
print("Standalone references independent of tau: PASS")
print("F3 count metrics = corresponding C-only metrics: PASS")
print("No prediction recomputed: PASS")
print("No localisation matching recomputed: PASS")
print("No density routing used: PASS")
print("No configuration selected: PASS")
print("No tau selected: PASS")


print("\n" + "=" * 120)

print(
    "Step 13D.2b.4 Part D complete — "
    "overall controlled comparison table established."
)

print(
    "Ready for Part E — S1–S6 density-stratified "
    "controlled comparison."
)

print("=" * 120)

STEP 13D.2b.4 PART D — OVERALL CONTROLLED COMPARISON
Required Part-B / Part-C objects: PASS
225 overall fusion configurations generated: PASS
100 images / 10,045 GT objects per configuration: PASS
75 configurations per fusion family: PASS
Corresponding C-only and D-only references attached: PASS
Standalone baseline references invariant across fusion/tau: PASS
F3 count metrics exactly reproduce corresponding C-only baseline: PASS

------------------------------------------------------------------------------------------------------------------------
F1 — OVERALL CONTROLLED RESULTS (75 configurations)
------------------------------------------------------------------------------------------------------------------------
fusion_family c_variant    d_variant    tau   sum_pred      mae     rmse  mean_signed_error  mean_pred_gt_ratio  aggregate_pred_gt_ratio  macro_precision  macro_recall  macro_f1   c_mae  c_macro_precision  c_macro_recall  c_macro_f1    d_mae  d_macro_precision  d_macro_re

In [79]:
# ============================================================
# Step 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines
# Part E — S1–S6 Density-Stratified Controlled Comparison
#
# Purpose:
#   1. Aggregate all 225 fusion configurations separately
#      within frozen density strata S1–S6.
#   2. Build matching C-only and D-only density references.
#   3. Attach paired C/D references to every fusion row.
#   4. Preserve all configurations — no selection yet.
#
# Output:
#   225 configs × 6 strata = 1,350 fusion-stratum rows
#
# IMPORTANT:
#   - Existing per-image matching is reused.
#   - No predictions are recomputed.
#   - No tau/configuration is selected.
#   - Density strata are evaluation groups ONLY.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.4 PART E — S1–S6 DENSITY-STRATIFIED CONTROLLED COMPARISON")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required objects
# ------------------------------------------------------------

required_objects = [
    "e3_fusion_eval_per_image_df",
    "e3_c_baseline_df",
    "e3_d_baseline_df",
    "DENSITY_ORDER",
    "E3_C_VARIANTS",
    "E3_D_VARIANTS",
    "E3_ASSOC_TAU_VALUES",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required objects: "
    + ", ".join(missing_objects)
)


expected_density_order = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]

assert list(DENSITY_ORDER) == expected_density_order

print("Required objects and frozen density order: PASS")


# ------------------------------------------------------------
# 2. Expected number of images per stratum
# ------------------------------------------------------------

EXPECTED_STRATUM_N = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}


observed_density_counts = (
    e3_fusion_eval_per_image_df[
        ["dataset_index", "density_stratum"]
    ]
    .drop_duplicates()
    ["density_stratum"]
    .astype(str)
    .value_counts()
    .to_dict()
)


assert observed_density_counts == EXPECTED_STRATUM_N

print("Frozen 20/20/20/20/10/10 density allocation: PASS")


# ------------------------------------------------------------
# 3. Shared summariser
#
# Same metric convention as Parts C/D:
#   - MAE / RMSE
#   - mean signed error
#   - mean image-level Pred/GT ratio
#   - aggregate Pred/GT ratio
#   - macro per-image Precision / Recall / F1
# ------------------------------------------------------------

def summarise_density_group(df):

    gt = df["gt_count"].astype(float)
    pred = df["pred_count"].astype(float)

    # Part-B fusion table uses signed_error / abs_error.
    # Part-C baseline tables use count_error / abs_count_error.
    if "signed_error" in df.columns:
        signed_error = df["signed_error"].astype(float)
    else:
        signed_error = df["count_error"].astype(float)

    if "abs_error" in df.columns:
        abs_error = df["abs_error"].astype(float)
    else:
        abs_error = df["abs_count_error"].astype(float)

    squared_error = df["squared_error"].astype(float)

    return pd.Series(
        {
            "n_images":
                int(len(df)),

            "sum_gt":
                int(gt.sum()),

            "sum_pred":
                int(pred.sum()),

            "mae":
                float(abs_error.mean()),

            "rmse":
                float(np.sqrt(squared_error.mean())),

            "mean_signed_error":
                float(signed_error.mean()),

            "mean_pred_gt_ratio":
                float(
                    df["pred_gt_ratio"]
                    .astype(float)
                    .mean()
                ),

            "aggregate_pred_gt_ratio":
                float(pred.sum() / gt.sum()),

            "macro_precision":
                float(
                    df["precision"]
                    .astype(float)
                    .mean()
                ),

            "macro_recall":
                float(
                    df["recall"]
                    .astype(float)
                    .mean()
                ),

            "macro_f1":
                float(
                    df["f1"]
                    .astype(float)
                    .mean()
                ),

            "total_tp":
                int(df["tp"].sum()),

            "total_fp":
                int(df["fp"].sum()),

            "total_fn":
                int(df["fn"].sum()),
        }
    )


# ------------------------------------------------------------
# 4. Fusion density-stratified summary
#
# 225 configurations × 6 strata = 1,350 rows
# ------------------------------------------------------------

fusion_density_group_cols = [
    "fusion_family",
    "c_variant",
    "d_variant",
    "tau",
    "density_stratum",
]


e3_fusion_density_summary_df = (
    e3_fusion_eval_per_image_df
    .groupby(
        fusion_density_group_cols,
        sort=False,
        observed=True,
    )
    .apply(
        summarise_density_group,
        include_groups=False,
    )
    .reset_index()
)


assert len(e3_fusion_density_summary_df) == 1350, (
    f"Expected 1,350 fusion-stratum rows, "
    f"found {len(e3_fusion_density_summary_df)}."
)

print("1,350 fusion configuration × density rows generated: PASS")


# ------------------------------------------------------------
# 5. Validate image count per fusion × stratum
# ------------------------------------------------------------

for stratum, expected_n in EXPECTED_STRATUM_N.items():

    sdf = (
        e3_fusion_density_summary_df[
            e3_fusion_density_summary_df[
                "density_stratum"
            ].astype(str)
            == stratum
        ]
    )

    assert len(sdf) == 225

    assert (
        sdf["n_images"].astype(int)
        == expected_n
    ).all()


print("Correct image count per fusion configuration × stratum: PASS")


# ------------------------------------------------------------
# 6. Build C-only density-stratified references
#
# 3 C × 6 strata = 18 rows
# ------------------------------------------------------------

e3_c_density_reference_df = (
    e3_c_baseline_df
    .groupby(
        [
            "candidate_id",
            "density_stratum",
        ],
        sort=False,
        observed=True,
    )
    .apply(
        summarise_density_group,
        include_groups=False,
    )
    .reset_index()
    .rename(
        columns={
            "candidate_id":
                "c_variant"
        }
    )
)


assert len(e3_c_density_reference_df) == 18


# ------------------------------------------------------------
# 7. Build D-only density-stratified references
#
# 5 D × 6 strata = 30 rows
# ------------------------------------------------------------

e3_d_density_reference_df = (
    e3_d_baseline_df
    .groupby(
        [
            "candidate_id",
            "density_stratum",
        ],
        sort=False,
        observed=True,
    )
    .apply(
        summarise_density_group,
        include_groups=False,
    )
    .reset_index()
    .rename(
        columns={
            "candidate_id":
                "d_variant"
        }
    )
)


assert len(e3_d_density_reference_df) == 30

print("C-only and D-only density references generated: PASS")


# ------------------------------------------------------------
# 8. Prefix baseline metric columns before joining
# ------------------------------------------------------------

density_metric_cols = [
    "n_images",
    "sum_gt",
    "sum_pred",
    "mae",
    "rmse",
    "mean_signed_error",
    "mean_pred_gt_ratio",
    "aggregate_pred_gt_ratio",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "total_tp",
    "total_fp",
    "total_fn",
]


c_density_join_df = (
    e3_c_density_reference_df[
        [
            "c_variant",
            "density_stratum",
        ]
        + density_metric_cols
    ]
    .copy()
    .rename(
        columns={
            col: f"c_{col}"
            for col in density_metric_cols
        }
    )
)


d_density_join_df = (
    e3_d_density_reference_df[
        [
            "d_variant",
            "density_stratum",
        ]
        + density_metric_cols
    ]
    .copy()
    .rename(
        columns={
            col: f"d_{col}"
            for col in density_metric_cols
        }
    )
)


# ------------------------------------------------------------
# 9. Attach paired C/D density references
# ------------------------------------------------------------

e3_density_controlled_comparison_df = (
    e3_fusion_density_summary_df
    .merge(
        c_density_join_df,
        on=[
            "c_variant",
            "density_stratum",
        ],
        how="left",
        validate="many_to_one",
    )
    .merge(
        d_density_join_df,
        on=[
            "d_variant",
            "density_stratum",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert len(e3_density_controlled_comparison_df) == 1350


assert not (
    e3_density_controlled_comparison_df[
        [
            "c_mae",
            "c_macro_f1",
            "d_mae",
            "d_macro_f1",
        ]
    ]
    .isna()
    .any()
    .any()
)


print("Paired C/D density references attached: PASS")


# ------------------------------------------------------------
# 10. Validate GT totals within each stratum
#
# All 225 configurations must see identical GT totals
# within a given stratum.
# ------------------------------------------------------------

stratum_gt_check = (
    e3_density_controlled_comparison_df
    .groupby(
        "density_stratum",
        observed=True,
    )[
        "sum_gt"
    ]
    .nunique()
)


assert (
    stratum_gt_check == 1
).all()


stratum_gt_totals_df = (
    e3_density_controlled_comparison_df[
        [
            "density_stratum",
            "sum_gt",
            "n_images",
        ]
    ]
    .drop_duplicates()
    .copy()
)


print("GT totals invariant across configurations within each stratum: PASS")


# ------------------------------------------------------------
# 11. F3 density-level count invariant
#
# Within every stratum:
#   F3 count metrics must equal the paired C-only baseline.
# ------------------------------------------------------------

f3_density_check_df = (
    e3_density_controlled_comparison_df[
        e3_density_controlled_comparison_df[
            "fusion_family"
        ]
        == "F3"
    ]
)


assert len(f3_density_check_df) == 450


for fusion_col, c_col in [
    ("sum_pred", "c_sum_pred"),
    ("mae", "c_mae"),
    ("rmse", "c_rmse"),
    (
        "mean_signed_error",
        "c_mean_signed_error",
    ),
    (
        "mean_pred_gt_ratio",
        "c_mean_pred_gt_ratio",
    ),
    (
        "aggregate_pred_gt_ratio",
        "c_aggregate_pred_gt_ratio",
    ),
]:

    assert np.allclose(
        f3_density_check_df[
            fusion_col
        ].astype(float),
        f3_density_check_df[
            c_col
        ].astype(float),
    ), (
        f"F3 density invariant failed for {fusion_col}"
    )


print("F3 count metrics = paired C baseline within every stratum: PASS")


# ------------------------------------------------------------
# 12. Stable categorical ordering
# ------------------------------------------------------------

e3_density_controlled_comparison_df[
    "density_stratum"
] = pd.Categorical(
    e3_density_controlled_comparison_df[
        "density_stratum"
    ].astype(str),
    categories=DENSITY_ORDER,
    ordered=True,
)


e3_c_density_reference_df[
    "density_stratum"
] = pd.Categorical(
    e3_c_density_reference_df[
        "density_stratum"
    ].astype(str),
    categories=DENSITY_ORDER,
    ordered=True,
)


e3_d_density_reference_df[
    "density_stratum"
] = pd.Categorical(
    e3_d_density_reference_df[
        "density_stratum"
    ].astype(str),
    categories=DENSITY_ORDER,
    ordered=True,
)


# ------------------------------------------------------------
# 13. Descriptive metric ranges by fusion family × density
#
# IMPORTANT:
#   This shows the range across the 75 configurations in each
#   family. It does NOT select a configuration.
# ------------------------------------------------------------

e3_density_family_range_df = (
    e3_density_controlled_comparison_df
    .groupby(
        [
            "fusion_family",
            "density_stratum",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        mae_min=("mae", "min"),
        mae_max=("mae", "max"),

        precision_min=(
            "macro_precision",
            "min",
        ),
        precision_max=(
            "macro_precision",
            "max",
        ),

        recall_min=(
            "macro_recall",
            "min",
        ),
        recall_max=(
            "macro_recall",
            "max",
        ),

        f1_min=(
            "macro_f1",
            "min",
        ),
        f1_max=(
            "macro_f1",
            "max",
        ),
    )
)


# ------------------------------------------------------------
# 14. Display family ranges
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("FUSION-FAMILY METRIC RANGES BY DENSITY — DESCRIPTIVE ONLY")
print("-" * 120)

print(
    e3_density_family_range_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 15. Display frozen C-only / D-only F1 by density
#
# This provides useful context for interpreting the fusion
# ranges without selecting any fusion configuration.
# ------------------------------------------------------------

baseline_density_f1_df = (
    pd.concat(
        [
            e3_c_density_reference_df[
                [
                    "c_variant",
                    "density_stratum",
                    "macro_f1",
                ]
            ]
            .rename(
                columns={
                    "c_variant": "candidate_id"
                }
            )
            .assign(source_group="C"),

            e3_d_density_reference_df[
                [
                    "d_variant",
                    "density_stratum",
                    "macro_f1",
                ]
            ]
            .rename(
                columns={
                    "d_variant": "candidate_id"
                }
            )
            .assign(source_group="D"),
        ],
        ignore_index=True,
    )
)


baseline_density_f1_pivot_df = (
    baseline_density_f1_df
    .pivot(
        index=[
            "source_group",
            "candidate_id",
        ],
        columns="density_stratum",
        values="macro_f1",
    )
    .reindex(
        columns=DENSITY_ORDER
    )
)


print("\n" + "-" * 120)
print("FROZEN C/D MACRO-F1 BY DENSITY")
print("-" * 120)

print(
    baseline_density_f1_pivot_df.to_string(
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 16. Compact structural summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("PART E SUMMARY")
print("-" * 120)

print(
    "Fusion configurations             : 225"
)

print(
    "Density strata                    : 6"
)

print(
    f"Fusion × density rows             : "
    f"{len(e3_fusion_density_summary_df)}"
)

print(
    f"C-only density reference rows     : "
    f"{len(e3_c_density_reference_df)}"
)

print(
    f"D-only density reference rows     : "
    f"{len(e3_d_density_reference_df)}"
)

print(
    f"Controlled comparison rows        : "
    f"{len(e3_density_controlled_comparison_df)}"
)


print("\nGT evaluation universe by density:")

print(
    stratum_gt_totals_df
    .sort_values("density_stratum")
    .to_string(index=False)
)


print("\nConstructed objects:")

print(
    "  e3_fusion_density_summary_df        :",
    e3_fusion_density_summary_df.shape,
)

print(
    "  e3_c_density_reference_df           :",
    e3_c_density_reference_df.shape,
)

print(
    "  e3_d_density_reference_df           :",
    e3_d_density_reference_df.shape,
)

print(
    "  e3_density_controlled_comparison_df :",
    e3_density_controlled_comparison_df.shape,
)

print(
    "  e3_density_family_range_df          :",
    e3_density_family_range_df.shape,
)

print(
    "  baseline_density_f1_pivot_df        :",
    baseline_density_f1_pivot_df.shape,
)


print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print("Existing Part-B matching reused: PASS")
print("Frozen S1–S6 definitions reused: PASS")
print("20/20/20/20/10/10 allocation preserved: PASS")
print("225 configurations × 6 strata = 1,350 rows: PASS")
print("Paired C-only density references attached: PASS")
print("Paired D-only density references attached: PASS")
print("GT totals invariant within each stratum: PASS")
print("F3 count metrics = paired C within each stratum: PASS")
print("No prediction recomputed: PASS")
print("No localisation matching recomputed: PASS")
print("Density used for evaluation only: PASS")
print("No density routing used: PASS")
print("No configuration selected: PASS")
print("No tau selected: PASS")


print("\n" + "=" * 120)

print(
    "Step 13D.2b.4 Part E complete — "
    "S1–S6 controlled comparison established."
)

print(
    "Ready for Part F — quantify fusion gains/losses "
    "relative to paired C and D baselines."
)

print("=" * 120)

STEP 13D.2b.4 PART E — S1–S6 DENSITY-STRATIFIED CONTROLLED COMPARISON
Required objects and frozen density order: PASS
Frozen 20/20/20/20/10/10 density allocation: PASS
1,350 fusion configuration × density rows generated: PASS
Correct image count per fusion configuration × stratum: PASS
C-only and D-only density references generated: PASS
Paired C/D density references attached: PASS
GT totals invariant across configurations within each stratum: PASS
F3 count metrics = paired C baseline within every stratum: PASS

------------------------------------------------------------------------------------------------------------------------
FUSION-FAMILY METRIC RANGES BY DENSITY — DESCRIPTIVE ONLY
------------------------------------------------------------------------------------------------------------------------
fusion_family density_stratum  mae_min  mae_max  precision_min  precision_max  recall_min  recall_max  f1_min  f1_max
           F1         S1_0_20  21.7000 114.7000         0.0633  

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [81]:
# ============================================================
# Step 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines
# Part F — Fusion Gains/Losses Relative to Paired C and D
#
# Purpose:
#   1. Quantify fusion changes relative to its exact paired
#      C-only and D-only parents.
#   2. Do this for:
#        A. Overall dev100 performance
#        B. S1–S6 density-stratified performance
#   3. Measure:
#        - MAE change / MAE reduction
#        - Precision change
#        - Recall change
#        - F1 change
#   4. Identify whether fusion improves:
#        - C parent
#        - D parent
#        - BOTH parents
#
# IMPORTANT:
#   - Negative delta_MAE is an improvement.
#   - Positive MAE reduction is an improvement.
#   - Positive delta Precision/Recall/F1 is an improvement.
#   - No configuration is selected here.
#   - Density remains evaluation-only.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.4 PART F — FUSION GAINS/LOSSES RELATIVE TO PAIRED C AND D")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required Part-D / Part-E objects
# ------------------------------------------------------------

required_objects = [
    "e3_overall_controlled_comparison_df",
    "e3_density_controlled_comparison_df",
    "DENSITY_ORDER",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required objects: "
    + ", ".join(missing_objects)
)


assert len(e3_overall_controlled_comparison_df) == 225
assert len(e3_density_controlled_comparison_df) == 1350

print("Required Part-D / Part-E objects: PASS")


# ------------------------------------------------------------
# 2. Helper — add paired C/D gains and losses
#
# Conventions:
#
#   MAE:
#       delta_mae_vs_X = fusion MAE - X MAE
#       negative = better
#
#       mae_reduction_vs_X = X MAE - fusion MAE
#       positive = better
#
#   P/R/F1:
#       delta_metric_vs_X = fusion metric - X metric
#       positive = better
# ------------------------------------------------------------

def add_paired_gain_columns(df):

    out = df.copy()

    # -----------------------------
    # MAE relative to C
    # -----------------------------

    out["delta_mae_vs_c"] = (
        out["mae"]
        - out["c_mae"]
    )

    out["mae_reduction_vs_c"] = (
        out["c_mae"]
        - out["mae"]
    )

    out["mae_reduction_pct_vs_c"] = (
        100.0
        * out["mae_reduction_vs_c"]
        / out["c_mae"]
    )


    # -----------------------------
    # MAE relative to D
    # -----------------------------

    out["delta_mae_vs_d"] = (
        out["mae"]
        - out["d_mae"]
    )

    out["mae_reduction_vs_d"] = (
        out["d_mae"]
        - out["mae"]
    )

    out["mae_reduction_pct_vs_d"] = (
        100.0
        * out["mae_reduction_vs_d"]
        / out["d_mae"]
    )


    # -----------------------------
    # Precision deltas
    # -----------------------------

    out["delta_precision_vs_c"] = (
        out["macro_precision"]
        - out["c_macro_precision"]
    )

    out["delta_precision_vs_d"] = (
        out["macro_precision"]
        - out["d_macro_precision"]
    )


    # -----------------------------
    # Recall deltas
    # -----------------------------

    out["delta_recall_vs_c"] = (
        out["macro_recall"]
        - out["c_macro_recall"]
    )

    out["delta_recall_vs_d"] = (
        out["macro_recall"]
        - out["d_macro_recall"]
    )


    # -----------------------------
    # F1 deltas
    # -----------------------------

    out["delta_f1_vs_c"] = (
        out["macro_f1"]
        - out["c_macro_f1"]
    )

    out["delta_f1_vs_d"] = (
        out["macro_f1"]
        - out["d_macro_f1"]
    )


    # -----------------------------
    # Relative F1 changes (%)
    # -----------------------------

    out["f1_change_pct_vs_c"] = (
        100.0
        * out["delta_f1_vs_c"]
        / out["c_macro_f1"]
    )

    out["f1_change_pct_vs_d"] = (
        100.0
        * out["delta_f1_vs_d"]
        / out["d_macro_f1"]
    )


    # -----------------------------
    # Improvement flags
    # -----------------------------

    out["mae_better_than_c"] = (
        out["mae"]
        < out["c_mae"]
    )

    out["mae_better_than_d"] = (
        out["mae"]
        < out["d_mae"]
    )

    out["mae_better_than_both"] = (
        out["mae_better_than_c"]
        &
        out["mae_better_than_d"]
    )


    out["f1_better_than_c"] = (
        out["macro_f1"]
        > out["c_macro_f1"]
    )

    out["f1_better_than_d"] = (
        out["macro_f1"]
        > out["d_macro_f1"]
    )

    out["f1_better_than_both"] = (
        out["f1_better_than_c"]
        &
        out["f1_better_than_d"]
    )


    # Strict joint improvement:
    # lower MAE AND higher F1 than both parents.
    out["mae_and_f1_better_than_both"] = (
        out["mae_better_than_both"]
        &
        out["f1_better_than_both"]
    )

    return out


# ------------------------------------------------------------
# 3. Overall paired gains/losses
# ------------------------------------------------------------

e3_overall_gain_loss_df = (
    add_paired_gain_columns(
        e3_overall_controlled_comparison_df
    )
)


assert len(e3_overall_gain_loss_df) == 225

print("Overall paired gain/loss table: PASS")


# ------------------------------------------------------------
# 4. Density-stratified paired gains/losses
# ------------------------------------------------------------

e3_density_gain_loss_df = (
    add_paired_gain_columns(
        e3_density_controlled_comparison_df
    )
)


assert len(e3_density_gain_loss_df) == 1350

print("S1–S6 paired gain/loss table: PASS")


# ------------------------------------------------------------
# 5. Sanity check:
#    F3 count MAE must never improve over C because
#    F3 has exactly the same count as C.
# ------------------------------------------------------------

f3_overall = (
    e3_overall_gain_loss_df[
        e3_overall_gain_loss_df[
            "fusion_family"
        ]
        == "F3"
    ]
)


assert np.allclose(
    f3_overall["delta_mae_vs_c"],
    0.0,
)


f3_density = (
    e3_density_gain_loss_df[
        e3_density_gain_loss_df[
            "fusion_family"
        ]
        == "F3"
    ]
)


assert np.allclose(
    f3_density["delta_mae_vs_c"],
    0.0,
)


print("F3 MAE delta vs C = 0 overall and within every stratum: PASS")


# ------------------------------------------------------------
# 6. Overall improvement-frequency summary
#
# Count how often each fusion family improves its paired
# parents across the 75 configurations.
# ------------------------------------------------------------

def improvement_frequency_summary(df):

    rows = []

    for family, g in df.groupby(
        "fusion_family",
        sort=False,
    ):

        n = len(g)

        rows.append(
            {
                "fusion_family": family,
                "n_configs": n,

                "mae_better_C_n":
                    int(g["mae_better_than_c"].sum()),

                "mae_better_D_n":
                    int(g["mae_better_than_d"].sum()),

                "mae_better_both_n":
                    int(g["mae_better_than_both"].sum()),

                "f1_better_C_n":
                    int(g["f1_better_than_c"].sum()),

                "f1_better_D_n":
                    int(g["f1_better_than_d"].sum()),

                "f1_better_both_n":
                    int(g["f1_better_than_both"].sum()),

                "joint_better_both_n":
                    int(
                        g[
                            "mae_and_f1_better_than_both"
                        ].sum()
                    ),
            }
        )

    result = pd.DataFrame(rows)

    # Percentages
    for col in [
        "mae_better_C_n",
        "mae_better_D_n",
        "mae_better_both_n",
        "f1_better_C_n",
        "f1_better_D_n",
        "f1_better_both_n",
        "joint_better_both_n",
    ]:

        result[
            col.replace("_n", "_pct")
        ] = (
            100.0
            * result[col]
            / result["n_configs"]
        )

    return result


e3_overall_improvement_frequency_df = (
    improvement_frequency_summary(
        e3_overall_gain_loss_df
    )
)


print("\n" + "-" * 120)
print("OVERALL IMPROVEMENT FREQUENCY — RELATIVE TO PAIRED PARENTS")
print("-" * 120)

print(
    e3_overall_improvement_frequency_df
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}",
    )
)


# ------------------------------------------------------------
# 7. Overall gain/loss ranges
#
# Shows how much each family can help/hurt relative to its
# paired C and D parents.
# ------------------------------------------------------------

e3_overall_gain_range_df = (
    e3_overall_gain_loss_df
    .groupby(
        "fusion_family",
        as_index=False,
    )
    .agg(
        mae_reduction_vs_c_min=(
            "mae_reduction_vs_c",
            "min",
        ),
        mae_reduction_vs_c_max=(
            "mae_reduction_vs_c",
            "max",
        ),

        mae_reduction_vs_d_min=(
            "mae_reduction_vs_d",
            "min",
        ),
        mae_reduction_vs_d_max=(
            "mae_reduction_vs_d",
            "max",
        ),

        delta_f1_vs_c_min=(
            "delta_f1_vs_c",
            "min",
        ),
        delta_f1_vs_c_max=(
            "delta_f1_vs_c",
            "max",
        ),

        delta_f1_vs_d_min=(
            "delta_f1_vs_d",
            "min",
        ),
        delta_f1_vs_d_max=(
            "delta_f1_vs_d",
            "max",
        ),
    )
)


print("\n" + "-" * 120)
print("OVERALL GAIN/LOSS RANGES")
print("Positive MAE reduction = better; positive ΔF1 = better")
print("-" * 120)

print(
    e3_overall_gain_range_df
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 8. Configurations that improve F1 over BOTH paired parents
#
# This is descriptive evidence, not final selection.
# ------------------------------------------------------------

e3_f1_better_both_df = (
    e3_overall_gain_loss_df[
        e3_overall_gain_loss_df[
            "f1_better_than_both"
        ]
    ][
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",

            "mae",
            "macro_precision",
            "macro_recall",
            "macro_f1",

            "c_macro_f1",
            "d_macro_f1",

            "delta_f1_vs_c",
            "delta_f1_vs_d",

            "f1_change_pct_vs_c",
            "f1_change_pct_vs_d",

            "mae_reduction_vs_c",
            "mae_reduction_vs_d",
        ]
    ]
    .sort_values(
        [
            "macro_f1",
            "delta_f1_vs_d",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


print("\n" + "-" * 120)
print("CONFIGURATIONS WITH F1 HIGHER THAN BOTH PAIRED C AND D")
print("(descriptive only — not yet a shortlist)")
print("-" * 120)

if len(e3_f1_better_both_df) == 0:

    print("None.")

else:

    print(
        e3_f1_better_both_df
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )


# ------------------------------------------------------------
# 9. Strict joint improvement:
#    lower MAE AND higher F1 than BOTH paired parents
# ------------------------------------------------------------

e3_joint_better_both_df = (
    e3_overall_gain_loss_df[
        e3_overall_gain_loss_df[
            "mae_and_f1_better_than_both"
        ]
    ][
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",

            "mae",
            "macro_f1",

            "c_mae",
            "d_mae",

            "c_macro_f1",
            "d_macro_f1",

            "mae_reduction_vs_c",
            "mae_reduction_vs_d",

            "delta_f1_vs_c",
            "delta_f1_vs_d",
        ]
    ]
    .sort_values(
        [
            "macro_f1",
            "mae",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print("\n" + "-" * 120)
print("STRICT JOINT IMPROVEMENT")
print("Lower MAE AND higher F1 than BOTH paired parents")
print("-" * 120)

if len(e3_joint_better_both_df) == 0:

    print("None.")

else:

    print(
        e3_joint_better_both_df
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )


# ------------------------------------------------------------
# 10. Density-level improvement frequencies
#
# For each family × stratum:
# how often does fusion improve over BOTH parents?
# ------------------------------------------------------------

density_frequency_rows = []

for (
    family,
    stratum
), g in (
    e3_density_gain_loss_df
    .groupby(
        [
            "fusion_family",
            "density_stratum",
        ],
        sort=False,
        observed=True,
    )
):

    n = len(g)

    density_frequency_rows.append(
        {
            "fusion_family": family,
            "density_stratum": stratum,
            "n_configs": n,

            "mae_better_both_n":
                int(
                    g[
                        "mae_better_than_both"
                    ].sum()
                ),

            "f1_better_both_n":
                int(
                    g[
                        "f1_better_than_both"
                    ].sum()
                ),

            "joint_better_both_n":
                int(
                    g[
                        "mae_and_f1_better_than_both"
                    ].sum()
                ),

            "mae_better_both_pct":
                100.0
                * g[
                    "mae_better_than_both"
                ].mean(),

            "f1_better_both_pct":
                100.0
                * g[
                    "f1_better_than_both"
                ].mean(),

            "joint_better_both_pct":
                100.0
                * g[
                    "mae_and_f1_better_than_both"
                ].mean(),
        }
    )


e3_density_improvement_frequency_df = (
    pd.DataFrame(
        density_frequency_rows
    )
)


e3_density_improvement_frequency_df[
    "density_stratum"
] = pd.Categorical(
    e3_density_improvement_frequency_df[
        "density_stratum"
    ].astype(str),
    categories=DENSITY_ORDER,
    ordered=True,
)


e3_density_improvement_frequency_df = (
    e3_density_improvement_frequency_df
    .sort_values(
        [
            "fusion_family",
            "density_stratum",
        ]
    )
    .reset_index(drop=True)
)


print("\n" + "-" * 120)
print("DENSITY-LEVEL IMPROVEMENT FREQUENCY")
print("Percentage of configurations improving over BOTH paired parents")
print("-" * 120)

print(
    e3_density_improvement_frequency_df
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}",
    )
)


# ------------------------------------------------------------
# 11. Density-level maximum paired gains
#
# Unlike Part E's raw family maxima, these are actual
# paired-parent deltas.
# ------------------------------------------------------------

e3_density_gain_range_df = (
    e3_density_gain_loss_df
    .groupby(
        [
            "fusion_family",
            "density_stratum",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        best_mae_reduction_vs_c=(
            "mae_reduction_vs_c",
            "max",
        ),

        best_mae_reduction_vs_d=(
            "mae_reduction_vs_d",
            "max",
        ),

        best_delta_f1_vs_c=(
            "delta_f1_vs_c",
            "max",
        ),

        best_delta_f1_vs_d=(
            "delta_f1_vs_d",
            "max",
        ),
    )
)


print("\n" + "-" * 120)
print("DENSITY-LEVEL MAXIMUM PAIRED GAINS")
print("These are paired-parent gains, not raw family maxima.")
print("-" * 120)

print(
    e3_density_gain_range_df
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 12. Configuration-level density consistency
#
# Count, for each exact overall configuration, in how many
# of the six strata it improves F1 over BOTH paired parents,
# and in how many it jointly improves MAE + F1 over both.
#
# This will be useful for Part G.
# ------------------------------------------------------------

e3_density_consistency_df = (
    e3_density_gain_loss_df
    .groupby(
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        n_strata_f1_better_both=(
            "f1_better_than_both",
            "sum",
        ),

        n_strata_mae_better_both=(
            "mae_better_than_both",
            "sum",
        ),

        n_strata_joint_better_both=(
            "mae_and_f1_better_than_both",
            "sum",
        ),

        mean_delta_f1_vs_c=(
            "delta_f1_vs_c",
            "mean",
        ),

        mean_delta_f1_vs_d=(
            "delta_f1_vs_d",
            "mean",
        ),

        mean_mae_reduction_vs_c=(
            "mae_reduction_vs_c",
            "mean",
        ),

        mean_mae_reduction_vs_d=(
            "mae_reduction_vs_d",
            "mean",
        ),
    )
)


for col in [
    "n_strata_f1_better_both",
    "n_strata_mae_better_both",
    "n_strata_joint_better_both",
]:

    e3_density_consistency_df[col] = (
        e3_density_consistency_df[col]
        .astype(int)
    )


assert len(e3_density_consistency_df) == 225


# ------------------------------------------------------------
# 13. Attach overall metrics to density consistency table
# ------------------------------------------------------------

overall_for_consistency = (
    e3_overall_gain_loss_df[
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",

            "mae",
            "macro_f1",

            "delta_f1_vs_c",
            "delta_f1_vs_d",

            "mae_reduction_vs_c",
            "mae_reduction_vs_d",

            "f1_better_than_both",
            "mae_better_than_both",
            "mae_and_f1_better_than_both",
        ]
    ]
    .copy()
)


e3_part_f_configuration_summary_df = (
    overall_for_consistency
    .merge(
        e3_density_consistency_df,
        on=[
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "_overall",
            "_density_mean",
        ),
    )
)


assert len(e3_part_f_configuration_summary_df) == 225


# ------------------------------------------------------------
# 14. Show configurations with strongest cross-density
#     F1 consistency.
#
# IMPORTANT:
#   This is descriptive sorting, not final selection.
# ------------------------------------------------------------

e3_cross_density_f1_view_df = (
    e3_part_f_configuration_summary_df
    .sort_values(
        [
            "n_strata_f1_better_both",
            "macro_f1",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(25)
)


print("\n" + "-" * 120)
print("TOP CROSS-DENSITY F1-CONSISTENCY VIEW")
print("Sorted descriptively; NOT a Part-G shortlist yet.")
print("-" * 120)

print(
    e3_cross_density_f1_view_df[
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",

            "mae",
            "macro_f1",

            "n_strata_f1_better_both",
            "n_strata_mae_better_both",
            "n_strata_joint_better_both",

            # Correct column names:
            "delta_f1_vs_c",
            "delta_f1_vs_d",

            "mae_reduction_vs_c",
            "mae_reduction_vs_d",
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 15. Final summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("PART F SUMMARY")
print("-" * 120)

print(
    f"Overall fusion configurations       : "
    f"{len(e3_overall_gain_loss_df)}"
)

print(
    f"Density-stratified rows             : "
    f"{len(e3_density_gain_loss_df)}"
)

print(
    f"Overall F1 > both parents           : "
    f"{int(e3_overall_gain_loss_df['f1_better_than_both'].sum())}"
)

print(
    f"Overall MAE < both parents          : "
    f"{int(e3_overall_gain_loss_df['mae_better_than_both'].sum())}"
)

print(
    f"Overall MAE+F1 better than both     : "
    f"{int(e3_overall_gain_loss_df['mae_and_f1_better_than_both'].sum())}"
)


print("\nConstructed objects:")

print(
    "  e3_overall_gain_loss_df               :",
    e3_overall_gain_loss_df.shape,
)

print(
    "  e3_density_gain_loss_df               :",
    e3_density_gain_loss_df.shape,
)

print(
    "  e3_overall_improvement_frequency_df   :",
    e3_overall_improvement_frequency_df.shape,
)

print(
    "  e3_overall_gain_range_df              :",
    e3_overall_gain_range_df.shape,
)

print(
    "  e3_f1_better_both_df                  :",
    e3_f1_better_both_df.shape,
)

print(
    "  e3_joint_better_both_df               :",
    e3_joint_better_both_df.shape,
)

print(
    "  e3_density_improvement_frequency_df   :",
    e3_density_improvement_frequency_df.shape,
)

print(
    "  e3_density_gain_range_df              :",
    e3_density_gain_range_df.shape,
)

print(
    "  e3_density_consistency_df             :",
    e3_density_consistency_df.shape,
)

print(
    "  e3_part_f_configuration_summary_df    :",
    e3_part_f_configuration_summary_df.shape,
)


print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print("Part-D overall comparisons reused: PASS")
print("Part-E density comparisons reused: PASS")
print("Paired C/D parents preserved: PASS")
print("Positive MAE reduction means improvement: PASS")
print("Positive ΔF1 means improvement: PASS")
print("F3 MAE delta vs C = 0: PASS")
print("Density remains evaluation-only: PASS")
print("No prediction recomputed: PASS")
print("No matching recomputed: PASS")
print("No configuration selected: PASS")
print("No tau selected: PASS")


print("\n" + "=" * 120)

print(
    "Step 13D.2b.4 Part F complete — "
    "fusion gains/losses quantified relative to paired parents."
)

print(
    "Ready for Part G — findings and shortlist "
    "for structured integration / end-to-end checkpoint."
)

print("=" * 120)



STEP 13D.2b.4 PART F — FUSION GAINS/LOSSES RELATIVE TO PAIRED C AND D
Required Part-D / Part-E objects: PASS
Overall paired gain/loss table: PASS
S1–S6 paired gain/loss table: PASS
F3 MAE delta vs C = 0 overall and within every stratum: PASS

------------------------------------------------------------------------------------------------------------------------
OVERALL IMPROVEMENT FREQUENCY — RELATIVE TO PAIRED PARENTS
------------------------------------------------------------------------------------------------------------------------
fusion_family  n_configs  mae_better_C_n  mae_better_D_n  mae_better_both_n  f1_better_C_n  f1_better_D_n  f1_better_both_n  joint_better_both_n  mae_better_C_pct  mae_better_D_pct  mae_better_both_pct  f1_better_C_pct  f1_better_D_pct  f1_better_both_pct  joint_better_both_pct
           F1         75               0              18                  0             40             10                10                    0              0.00             24

In [82]:
# ============================================================
# Diagnostic before Part G:
# Which exact configurations work best in S1-S3?
# Existing Part-F results only — no new experiment.
# ============================================================

sparse_strata = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
]

sparse_df = (
    e3_density_gain_loss_df[
        e3_density_gain_loss_df["density_stratum"]
        .astype(str)
        .isin(sparse_strata)
    ]
    .copy()
)

# ------------------------------------------------------------
# A. Exact rows jointly better than BOTH parents
# ------------------------------------------------------------

sparse_joint_df = (
    sparse_df[
        sparse_df["mae_and_f1_better_than_both"]
    ]
    .sort_values(
        ["density_stratum", "macro_f1"],
        ascending=[True, False],
    )
)

print("=" * 110)
print("S1-S3 — CONFIGURATIONS WITH LOWER MAE AND HIGHER F1 THAN BOTH PARENTS")
print("=" * 110)

print(
    sparse_joint_df[
        [
            "density_stratum",
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
            "mae",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "c_mae",
            "d_mae",
            "c_macro_f1",
            "d_macro_f1",
            "mae_reduction_vs_c",
            "mae_reduction_vs_d",
            "delta_f1_vs_c",
            "delta_f1_vs_d",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# B. How consistently does each exact configuration work
#    across S1, S2, S3?
# ------------------------------------------------------------

sparse_consistency_df = (
    sparse_df
    .groupby(
        [
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        n_sparse_f1_better_both=(
            "f1_better_than_both",
            "sum",
        ),
        n_sparse_mae_better_both=(
            "mae_better_than_both",
            "sum",
        ),
        n_sparse_joint_better_both=(
            "mae_and_f1_better_than_both",
            "sum",
        ),
        mean_sparse_f1=("macro_f1", "mean"),
        mean_sparse_mae=("mae", "mean"),
    )
)

for col in [
    "n_sparse_f1_better_both",
    "n_sparse_mae_better_both",
    "n_sparse_joint_better_both",
]:
    sparse_consistency_df[col] = (
        sparse_consistency_df[col].astype(int)
    )

sparse_consistency_df = (
    sparse_consistency_df
    .sort_values(
        [
            "n_sparse_joint_better_both",
            "n_sparse_f1_better_both",
            "mean_sparse_f1",
        ],
        ascending=[False, False, False],
    )
)

print("\n" + "=" * 110)
print("S1-S3 — CROSS-SPARSE CONSISTENCY")
print("=" * 110)

print(
    sparse_consistency_df.head(20).to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# C. Specifically inspect our overall F2 representative
# ------------------------------------------------------------

f2_rep_sparse_df = (
    sparse_df[
        (sparse_df["fusion_family"] == "F2")
        & (sparse_df["c_variant"] == "C2b")
        & (sparse_df["d_variant"] == "D6B-T-native")
        & np.isclose(sparse_df["tau"], 0.01)
    ]
)

print("\n" + "=" * 110)
print("F2 REPRESENTATIVE — C2b + D6B-T-native + tau=.01")
print("=" * 110)

print(
    f2_rep_sparse_df[
        [
            "density_stratum",
            "mae",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "c_mae",
            "d_mae",
            "c_macro_f1",
            "d_macro_f1",
            "mae_reduction_vs_c",
            "mae_reduction_vs_d",
            "delta_f1_vs_c",
            "delta_f1_vs_d",
            "mae_and_f1_better_than_both",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

S1-S3 — CONFIGURATIONS WITH LOWER MAE AND HIGHER F1 THAN BOTH PARENTS
density_stratum fusion_family c_variant    d_variant    tau     mae  macro_precision  macro_recall  macro_f1   c_mae    d_mae  c_macro_f1  d_macro_f1  mae_reduction_vs_c  mae_reduction_vs_d  delta_f1_vs_c  delta_f1_vs_d
        S1_0_20            F2       C2b  D6B9-strict 0.0500 10.1000           0.3330        0.4256    0.3353 28.3000  18.3000      0.2208      0.3242             18.2000              8.2000         0.1145         0.0111
        S1_0_20            F2       C2c  D6B9-strict 0.0500 10.1000           0.3330        0.4256    0.3353 15.6500  18.3000      0.2546      0.3242              5.5500              8.2000         0.0807         0.0111
        S1_0_20            F2       C2b  D6B9-strict 0.0300  9.1500           0.3325        0.4061    0.3290 28.3000  18.3000      0.2208      0.3242             19.1500              9.1500         0.1082         0.0048
        S1_0_20            F2       C2c  D6B9-stri

In [83]:
# ============================================================
# Step 13D.2b.4 — GT Evaluation of Simple Semantic-Fusion Baselines
# Part G — Findings and Frozen Diagnostic Shortlist
#
# Purpose:
#   1. Close the simple semantic-fusion experiment.
#   2. Record the main findings from Parts D–F.
#   3. Freeze a small diagnostic shortlist for the later
#      end-to-end SAM2 checkpoint.
#
# IMPORTANT:
#   - No new predictions are generated.
#   - No new matching is performed.
#   - No new parameter search is performed.
#   - S1–S6 are evaluation strata ONLY.
#   - The shortlist is NOT a density-routing policy.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 13D.2b.4 PART G — FINDINGS AND FROZEN DIAGNOSTIC SHORTLIST")
print("=" * 120)


# ------------------------------------------------------------
# 1. Validate required Part-F objects
# ------------------------------------------------------------

required_objects = [
    "e3_overall_gain_loss_df",
    "e3_density_gain_loss_df",
    "e3_part_f_configuration_summary_df",
    "DENSITY_ORDER",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Missing required objects: "
    + ", ".join(missing_objects)
)

assert len(e3_overall_gain_loss_df) == 225
assert len(e3_density_gain_loss_df) == 1350

print("Required Part-F objects: PASS")


# ------------------------------------------------------------
# 2. Helper — retrieve one exact overall configuration
# ------------------------------------------------------------

def get_overall_config(family, c_variant, d_variant, tau):

    mask = (
        (e3_overall_gain_loss_df["fusion_family"] == family)
        & (e3_overall_gain_loss_df["c_variant"] == c_variant)
        & (e3_overall_gain_loss_df["d_variant"] == d_variant)
        & np.isclose(
            e3_overall_gain_loss_df["tau"].astype(float),
            float(tau),
        )
    )

    out = e3_overall_gain_loss_df.loc[mask].copy()

    assert len(out) == 1, (
        f"Expected exactly one overall row for "
        f"{family}/{c_variant}/{d_variant}/tau={tau}; "
        f"found {len(out)}."
    )

    return out.iloc[0]


# ------------------------------------------------------------
# 3. Helper — retrieve density rows for one exact config
# ------------------------------------------------------------

def get_density_config(family, c_variant, d_variant, tau):

    mask = (
        (e3_density_gain_loss_df["fusion_family"] == family)
        & (e3_density_gain_loss_df["c_variant"] == c_variant)
        & (e3_density_gain_loss_df["d_variant"] == d_variant)
        & np.isclose(
            e3_density_gain_loss_df["tau"].astype(float),
            float(tau),
        )
    )

    out = e3_density_gain_loss_df.loc[mask].copy()

    assert len(out) == 6, (
        f"Expected six density rows for "
        f"{family}/{c_variant}/{d_variant}/tau={tau}; "
        f"found {len(out)}."
    )

    return out


# ------------------------------------------------------------
# 4. Freeze the three diagnostic representatives
#
# These have DIFFERENT experimental roles.
#
# A. Overall joint-improvement representative
# B. Sparse-regime diagnostic representative
# C. Dense-regime diagnostic representative
#
# They are NOT an inference-time routing policy.
# ------------------------------------------------------------

E3_DIAGNOSTIC_SHORTLIST = [
    {
        "checkpoint_id": "overall_joint",
        "fusion_family": "F2",
        "c_variant": "C2b",
        "d_variant": "D6B-T-native",
        "tau": 0.010,
        "selection_basis":
            "Only simple-fusion configuration with lower overall "
            "MAE and higher overall macro-F1 than both paired parents.",
    },
    {
        "checkpoint_id": "sparse_joint",
        "fusion_family": "F2",
        "c_variant": "C2b",
        "d_variant": "D6B-T-native",
        "tau": 0.005,
        "selection_basis":
            "Only observed configuration jointly improving MAE and "
            "macro-F1 over both paired parents in all S1-S3 strata.",
    },
    {
        "checkpoint_id": "dense_recovery",
        "fusion_family": "F1",
        "c_variant": "C1",
        "d_variant": "D6B9-strict",
        "tau": 0.050,
        "selection_basis":
            "Recovery-oriented representative with strongest overall "
            "simple-fusion macro-F1 and joint improvement over both "
            "paired parents in three density strata.",
    },
]


# ------------------------------------------------------------
# 5. Build frozen overall shortlist table
# ------------------------------------------------------------

shortlist_rows = []

for cfg in E3_DIAGNOSTIC_SHORTLIST:

    r = get_overall_config(
        cfg["fusion_family"],
        cfg["c_variant"],
        cfg["d_variant"],
        cfg["tau"],
    )

    shortlist_rows.append(
        {
            "checkpoint_id":
                cfg["checkpoint_id"],

            "fusion_family":
                cfg["fusion_family"],

            "c_variant":
                cfg["c_variant"],

            "d_variant":
                cfg["d_variant"],

            "tau":
                cfg["tau"],

            "selection_basis":
                cfg["selection_basis"],

            "mae":
                float(r["mae"]),

            "rmse":
                float(r["rmse"]),

            "macro_precision":
                float(r["macro_precision"]),

            "macro_recall":
                float(r["macro_recall"]),

            "macro_f1":
                float(r["macro_f1"]),

            "c_mae":
                float(r["c_mae"]),

            "d_mae":
                float(r["d_mae"]),

            "c_macro_f1":
                float(r["c_macro_f1"]),

            "d_macro_f1":
                float(r["d_macro_f1"]),

            "mae_reduction_vs_c":
                float(r["mae_reduction_vs_c"]),

            "mae_reduction_vs_d":
                float(r["mae_reduction_vs_d"]),

            "delta_f1_vs_c":
                float(r["delta_f1_vs_c"]),

            "delta_f1_vs_d":
                float(r["delta_f1_vs_d"]),

            "overall_mae_better_both":
                bool(r["mae_better_than_both"]),

            "overall_f1_better_both":
                bool(r["f1_better_than_both"]),

            "overall_joint_better_both":
                bool(r["mae_and_f1_better_than_both"]),
        }
    )


e3_diagnostic_shortlist_df = pd.DataFrame(shortlist_rows)

assert len(e3_diagnostic_shortlist_df) == 3

print("Three diagnostic representatives frozen: PASS")


# ------------------------------------------------------------
# 6. Build density profile for the three representatives
# ------------------------------------------------------------

density_rows = []

for cfg in E3_DIAGNOSTIC_SHORTLIST:

    ddf = get_density_config(
        cfg["fusion_family"],
        cfg["c_variant"],
        cfg["d_variant"],
        cfg["tau"],
    )

    for _, r in ddf.iterrows():

        density_rows.append(
            {
                "checkpoint_id":
                    cfg["checkpoint_id"],

                "fusion_family":
                    cfg["fusion_family"],

                "c_variant":
                    cfg["c_variant"],

                "d_variant":
                    cfg["d_variant"],

                "tau":
                    cfg["tau"],

                "density_stratum":
                    str(r["density_stratum"]),

                "mae":
                    float(r["mae"]),

                "macro_precision":
                    float(r["macro_precision"]),

                "macro_recall":
                    float(r["macro_recall"]),

                "macro_f1":
                    float(r["macro_f1"]),

                "c_mae":
                    float(r["c_mae"]),

                "d_mae":
                    float(r["d_mae"]),

                "c_macro_f1":
                    float(r["c_macro_f1"]),

                "d_macro_f1":
                    float(r["d_macro_f1"]),

                "mae_reduction_vs_c":
                    float(r["mae_reduction_vs_c"]),

                "mae_reduction_vs_d":
                    float(r["mae_reduction_vs_d"]),

                "delta_f1_vs_c":
                    float(r["delta_f1_vs_c"]),

                "delta_f1_vs_d":
                    float(r["delta_f1_vs_d"]),

                "mae_better_both":
                    bool(r["mae_better_than_both"]),

                "f1_better_both":
                    bool(r["f1_better_than_both"]),

                "joint_better_both":
                    bool(
                        r["mae_and_f1_better_than_both"]
                    ),
            }
        )


e3_diagnostic_shortlist_density_df = pd.DataFrame(
    density_rows
)

assert len(e3_diagnostic_shortlist_density_df) == 18

print("Three-representative S1-S6 profiles constructed: PASS")


# ------------------------------------------------------------
# 7. Validate the evidence used to define the roles
# ------------------------------------------------------------

overall_row = e3_diagnostic_shortlist_df[
    e3_diagnostic_shortlist_df["checkpoint_id"]
    == "overall_joint"
].iloc[0]

assert bool(
    overall_row["overall_joint_better_both"]
), (
    "Overall representative no longer satisfies "
    "strict joint improvement."
)


sparse_profile = (
    e3_diagnostic_shortlist_density_df[
        (
            e3_diagnostic_shortlist_density_df[
                "checkpoint_id"
            ]
            == "sparse_joint"
        )
        &
        (
            e3_diagnostic_shortlist_density_df[
                "density_stratum"
            ].isin(
                [
                    "S1_0_20",
                    "S2_20_40",
                    "S3_40_60",
                ]
            )
        )
    ]
)

assert len(sparse_profile) == 3

assert sparse_profile[
    "joint_better_both"
].all(), (
    "Sparse representative does not jointly improve "
    "over both parents in all S1-S3 strata."
)


dense_profile = (
    e3_diagnostic_shortlist_density_df[
        (
            e3_diagnostic_shortlist_density_df[
                "checkpoint_id"
            ]
            == "dense_recovery"
        )
        &
        (
            e3_diagnostic_shortlist_density_df[
                "density_stratum"
            ].isin(
                [
                    "S4_60_80",
                    "S5_80_95",
                    "S6_95_100",
                ]
            )
        )
    ]
)

assert len(dense_profile) == 3

assert dense_profile[
    "joint_better_both"
].all(), (
    "Dense representative does not jointly improve "
    "over both parents in all S4-S6 strata."
)


print("Overall representative evidence: PASS")
print("Sparse S1-S3 representative evidence: PASS")
print("Dense S4-S6 representative evidence: PASS")


# ------------------------------------------------------------
# 8. Confirm shortlist is diagnostic, not density routing
# ------------------------------------------------------------

assert {
    cfg["checkpoint_id"]
    for cfg in E3_DIAGNOSTIC_SHORTLIST
} == {
    "overall_joint",
    "sparse_joint",
    "dense_recovery",
}

print("Shortlist defined as diagnostic variants, not routing rules: PASS")


# ------------------------------------------------------------
# 9. Display frozen shortlist
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("FROZEN DIAGNOSTIC SHORTLIST — OVERALL DEV100")
print("-" * 120)

display_cols = [
    "checkpoint_id",
    "fusion_family",
    "c_variant",
    "d_variant",
    "tau",
    "mae",
    "rmse",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "c_mae",
    "d_mae",
    "c_macro_f1",
    "d_macro_f1",
    "mae_reduction_vs_c",
    "mae_reduction_vs_d",
    "delta_f1_vs_c",
    "delta_f1_vs_d",
    "overall_joint_better_both",
]

print(
    e3_diagnostic_shortlist_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 10. Display density profiles
# ------------------------------------------------------------

density_display_df = (
    e3_diagnostic_shortlist_density_df[
        [
            "checkpoint_id",
            "density_stratum",
            "mae",
            "macro_f1",
            "c_mae",
            "d_mae",
            "c_macro_f1",
            "d_macro_f1",
            "mae_reduction_vs_c",
            "mae_reduction_vs_d",
            "delta_f1_vs_c",
            "delta_f1_vs_d",
            "joint_better_both",
        ]
    ]
    .copy()
)


density_display_df["density_stratum"] = pd.Categorical(
    density_display_df["density_stratum"],
    categories=DENSITY_ORDER,
    ordered=True,
)


density_display_df = (
    density_display_df
    .sort_values(
        [
            "checkpoint_id",
            "density_stratum",
        ]
    )
)


print("\n" + "-" * 120)
print("FROZEN SHORTLIST — S1-S6 PROFILES")
print("-" * 120)

print(
    density_display_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ------------------------------------------------------------
# 11. Record the empirical conclusions of 13D.2b.4
# ------------------------------------------------------------

E3_SIMPLE_FUSION_FINDINGS = {
    "overall":
        "Simple semantic fusion produced only limited overall "
        "joint improvement in count MAE and localisation F1.",

    "f1":
        "F1 acts primarily as a recovery/complementarity strategy. "
        "Its benefit increases in higher-density strata, but its "
        "overall count MAE does not beat the strongest C-only count "
        "baseline.",

    "f2":
        "F2 acts primarily as an agreement/filtering strategy. "
        "It can improve sparse-to-moderate scenes by suppressing "
        "unsupported D hypotheses, but hard agreement becomes "
        "increasingly restrictive in dense scenes.",

    "f3":
        "F3 can refine localisation but cannot recover object "
        "multiplicity because its predicted count is constrained "
        "to the number of C regions.",

    "complementarity":
        "C and D contain complementary semantic evidence, but "
        "simple point-list fusion does not yet convert that "
        "complementarity into a large overall counting gain.",

    "next_step":
        "Use the frozen representatives as diagnostic end-to-end "
        "variants before committing to more complex structured "
        "semantic integration.",
}


print("\n" + "-" * 120)
print("EMPIRICAL FINDINGS — SIMPLE SEMANTIC FUSION")
print("-" * 120)

for key, value in E3_SIMPLE_FUSION_FINDINGS.items():
    print(f"{key:>16}: {value}")


# ------------------------------------------------------------
# 12. Explicitly record what is NOT concluded
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("BOUNDARIES OF THE CONCLUSION")
print("-" * 120)

print(
    "1. S1-S6 are evaluation strata derived from GT counts; "
    "they are NOT available for inference-time routing."
)

print(
    "2. The three representatives are diagnostic variants; "
    "they do NOT define a final adaptive pipeline."
)

print(
    "3. The current results evaluate semantic reference-point "
    "proxies, not the final SAM2 instance-counting pipeline."
)

print(
    "4. No Marigold/geometric prior has been used in this "
    "simple-fusion evaluation."
)

print(
    "5. No claim is made that the current semantic fusion "
    "outperforms B0 end-to-end."
)


# ------------------------------------------------------------
# 13. Final Part-G summary
# ------------------------------------------------------------

print("\n" + "-" * 120)
print("PART G SUMMARY")
print("-" * 120)

print("Simple-fusion families evaluated : F1, F2, F3")
print("Frozen diagnostic variants       : 3")

print(
    "  Overall : F2 / C2b / D6B-T-native / tau=.010"
)

print(
    "  Sparse  : F2 / C2b / D6B-T-native / tau=.005"
)

print(
    "  Dense   : F1 / C1  / D6B9-strict   / tau=.050"
)


print("\nConstructed objects:")

print(
    "  E3_DIAGNOSTIC_SHORTLIST              :",
    len(E3_DIAGNOSTIC_SHORTLIST),
)

print(
    "  e3_diagnostic_shortlist_df           :",
    e3_diagnostic_shortlist_df.shape,
)

print(
    "  e3_diagnostic_shortlist_density_df   :",
    e3_diagnostic_shortlist_density_df.shape,
)

print(
    "  E3_SIMPLE_FUSION_FINDINGS            :",
    len(E3_SIMPLE_FUSION_FINDINGS),
)


print("\n" + "-" * 120)
print("SANITY CHECKS")
print("-" * 120)

print("Existing Part-F results reused: PASS")
print("No predictions recomputed: PASS")
print("No localisation matching recomputed: PASS")
print("No new parameter sweep performed: PASS")
print("Overall representative criterion validated: PASS")
print("Sparse S1-S3 criterion validated: PASS")
print("Dense S4-S6 criterion validated: PASS")
print("Density used for evaluation only: PASS")
print("No inference-time density routing defined: PASS")


print("\n" + "=" * 120)

print(
    "Step 13D.2b.4 Part G complete — "
    "simple semantic-fusion evaluation closed."
)

print(
    "Three diagnostic representatives frozen for "
    "subsequent end-to-end checkpoint design."
)

print("=" * 120)

STEP 13D.2b.4 PART G — FINDINGS AND FROZEN DIAGNOSTIC SHORTLIST
Required Part-F objects: PASS
Three diagnostic representatives frozen: PASS
Three-representative S1-S6 profiles constructed: PASS
Overall representative evidence: PASS
Sparse S1-S3 representative evidence: PASS
Dense S4-S6 representative evidence: PASS
Shortlist defined as diagnostic variants, not routing rules: PASS

------------------------------------------------------------------------------------------------------------------------
FROZEN DIAGNOSTIC SHORTLIST — OVERALL DEV100
------------------------------------------------------------------------------------------------------------------------
 checkpoint_id fusion_family c_variant    d_variant    tau     mae     rmse  macro_precision  macro_recall  macro_f1   c_mae    d_mae  c_macro_f1  d_macro_f1  mae_reduction_vs_c  mae_reduction_vs_d  delta_f1_vs_c  delta_f1_vs_d  overall_joint_better_both
 overall_joint            F2       C2b D6B-T-native 0.0100 88.1700 262.730

In [86]:
# ============================================================
# Step 13D.2b — FINAL DRIVE-BACKED E3 ARTEFACT FREEZE
#
# Purpose:
#   Persist all important E3 semantic-integration artefacts
#   to the REAL Google Drive project directory.
#
# No predictions, matching, or experiments are recomputed.
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# 1. Define the ONLY valid E3 results directory
# ------------------------------------------------------------

assert "PROJECT_ROOT" in globals()
assert "B1_RESULTS_ROOT" in globals()

E3_RESULTS_DIR = (
    B1_RESULTS_ROOT
    / "Group_E"
    / "E3_semantic_integration"
)

E3_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Critical protection against accidentally writing to
# /content or /content/sam2.
assert str(E3_RESULTS_DIR).startswith(
    str(PROJECT_ROOT)
)

assert "/content/drive/MyDrive/" in str(
    E3_RESULTS_DIR
)

print("PROJECT_ROOT  :", PROJECT_ROOT)
print("E3_RESULTS_DIR:", E3_RESULTS_DIR)


# ------------------------------------------------------------
# 2. Organise E3 results into reproducible subdirectories
# ------------------------------------------------------------

ASSOCIATION_DIR = (
    E3_RESULTS_DIR
    / "association_primitives"
)

FUSION_GEN_DIR = (
    E3_RESULTS_DIR
    / "fusion_generation"
)

GT_EVAL_DIR = (
    E3_RESULTS_DIR
    / "gt_evaluation"
)

FINAL_DIR = (
    E3_RESULTS_DIR
    / "final"
)

for d in [
    ASSOCIATION_DIR,
    FUSION_GEN_DIR,
    GT_EVAL_DIR,
    FINAL_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 3. Tables to save
#
# These are the important E3 experimental artefacts currently
# present in the notebook.
# ------------------------------------------------------------

tables_to_save = {

    # ========================================================
    # 13D.2b.1 — spatial association / tau sweep
    # ========================================================

    FUSION_GEN_DIR
    / "E3_tau_assignment.csv":
        e3_tau_assignment_df,

    FUSION_GEN_DIR
    / "E3_tau_region.csv":
        e3_tau_region_df,

    FUSION_GEN_DIR
    / "E3_tau_structural_summary.csv":
        e3_tau_structural_summary_df,

    FUSION_GEN_DIR
    / "E3_tau_pair_summary.csv":
        e3_tau_pair_summary_df,

    FUSION_GEN_DIR
    / "E3_tau_pair_delta.csv":
        e3_tau_pair_delta_df,


    # ========================================================
    # 13D.2b.3 — F1/F2/F3 generation
    # ========================================================

    FUSION_GEN_DIR
    / "E3_fusion_per_image.csv":
        e3_fusion_per_image_df,

    FUSION_GEN_DIR
    / "E3_fusion_generation_summary.csv":
        e3_fusion_generation_summary_df,

    FUSION_GEN_DIR
    / "E3_fusion_source_summary.csv":
        e3_fusion_source_summary_df,

    FUSION_GEN_DIR
    / "E3_zero_prediction_summary.csv":
        e3_zero_prediction_summary_df,


    # ========================================================
    # 13D.2b.4 Part A/B — GT evaluation
    # ========================================================

    GT_EVAL_DIR
    / "E3_eval_dev100.csv":
        e3_eval_dev100_df,

    GT_EVAL_DIR
    / "E3_fusion_eval_per_image.csv":
        e3_fusion_eval_per_image_df,


    # ========================================================
    # Part C — reference baselines
    # ========================================================

    GT_EVAL_DIR
    / "E3_baseline_reference.csv":
        e3_baseline_reference_df,

    GT_EVAL_DIR
    / "E3_baseline_overall_summary.csv":
        e3_baseline_overall_summary_df,


    # ========================================================
    # Part D — overall controlled comparison
    # ========================================================

    GT_EVAL_DIR
    / "E3_fusion_overall_summary.csv":
        e3_fusion_overall_summary_df,

    GT_EVAL_DIR
    / "E3_overall_controlled_comparison.csv":
        e3_overall_controlled_comparison_df,

    GT_EVAL_DIR
    / "E3_family_metric_range.csv":
        e3_family_metric_range_df,


    # ========================================================
    # Part E — S1-S6 controlled comparison
    # ========================================================

    GT_EVAL_DIR
    / "E3_fusion_density_summary.csv":
        e3_fusion_density_summary_df,

    GT_EVAL_DIR
    / "E3_density_controlled_comparison.csv":
        e3_density_controlled_comparison_df,

    GT_EVAL_DIR
    / "E3_density_family_range.csv":
        e3_density_family_range_df,


    # ========================================================
    # Part F — paired gain/loss analysis
    # ========================================================

    GT_EVAL_DIR
    / "E3_overall_gain_loss.csv":
        e3_overall_gain_loss_df,

    GT_EVAL_DIR
    / "E3_density_gain_loss.csv":
        e3_density_gain_loss_df,

    GT_EVAL_DIR
    / "E3_overall_improvement_frequency.csv":
        e3_overall_improvement_frequency_df,

    GT_EVAL_DIR
    / "E3_overall_gain_range.csv":
        e3_overall_gain_range_df,

    GT_EVAL_DIR
    / "E3_density_improvement_frequency.csv":
        e3_density_improvement_frequency_df,

    GT_EVAL_DIR
    / "E3_density_gain_range.csv":
        e3_density_gain_range_df,

    GT_EVAL_DIR
    / "E3_density_consistency.csv":
        e3_density_consistency_df,

    GT_EVAL_DIR
    / "E3_part_f_configuration_summary.csv":
        e3_part_f_configuration_summary_df,


    # ========================================================
    # Part G — frozen diagnostic handoff
    # ========================================================

    FINAL_DIR
    / "E3_diagnostic_shortlist.csv":
        e3_diagnostic_shortlist_df,

    FINAL_DIR
    / "E3_diagnostic_shortlist_density.csv":
        e3_diagnostic_shortlist_density_df,
}


# ------------------------------------------------------------
# 4. Validate all required objects before writing anything
# ------------------------------------------------------------

print("\nTables scheduled for Drive freeze:")

for path, df in tables_to_save.items():

    assert isinstance(
        df,
        pd.DataFrame,
    ), (
        f"Expected DataFrame for {path.name}"
    )

    print(
        f"  {path.name:<45} "
        f"{df.shape}"
    )


# ------------------------------------------------------------
# 5. Save all tables
# ------------------------------------------------------------

print("\nSaving to Google Drive...")

for path, df in tables_to_save.items():

    df.to_csv(
        path,
        index=False,
    )

    assert path.exists()

    print(
        f"  SAVED: {path}"
    )


# ------------------------------------------------------------
# 6. Save final experiment manifest
# ------------------------------------------------------------

manifest = {

    "experiment":
        "Group E — E3 Semantic Integration",

    "final_step":
        "13D.2b.4 Part G",

    "status":
        "complete_frozen",

    "project_root":
        str(PROJECT_ROOT),

    "results_directory":
        str(E3_RESULTS_DIR),

    "evaluation_dataset":
        "FSC147 frozen validation dev100",

    "semantic_fusion_families":
        ["F1", "F2", "F3"],

    "association_tau_values":
        [
            float(x)
            for x in E3_ASSOC_TAU_VALUES
        ],

    "diagnostic_shortlist": [
        {
            "checkpoint_id":
                cfg["checkpoint_id"],

            "fusion_family":
                cfg["fusion_family"],

            "c_variant":
                cfg["c_variant"],

            "d_variant":
                cfg["d_variant"],

            "tau":
                float(cfg["tau"]),

            "selection_basis":
                cfg["selection_basis"],
        }
        for cfg in E3_DIAGNOSTIC_SHORTLIST
    ],

    "findings":
        E3_SIMPLE_FUSION_FINDINGS,

    "constraints": [
        (
            "S1-S6 are GT-derived evaluation "
            "strata only."
        ),
        (
            "No inference-time density routing "
            "is defined."
        ),
        (
            "Diagnostic shortlist is not a final "
            "adaptive pipeline."
        ),
        (
            "Current evaluation concerns semantic "
            "reference-point proxies before SAM2."
        ),
        (
            "No Marigold/geometric prior is used "
            "in this experiment."
        ),
        (
            "No claim is made that semantic fusion "
            "outperforms B0 end-to-end."
        ),
    ],

    "saved_tables": [
        str(path.relative_to(E3_RESULTS_DIR))
        for path in tables_to_save
    ],
}


MANIFEST_PATH = (
    FINAL_DIR
    / "E3_simple_fusion_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )

assert MANIFEST_PATH.exists()


# ------------------------------------------------------------
# 7. Round-trip validation
#
# Verify every saved CSV is readable and preserves shape.
# ------------------------------------------------------------

print("\nRound-trip validation:")

validation_rows = []

for path, original_df in tables_to_save.items():

    loaded_df = pd.read_csv(
        path
    )

    shape_match = (
        loaded_df.shape
        ==
        original_df.shape
    )

    assert shape_match, (
        f"Shape mismatch after reload: {path}"
    )

    validation_rows.append(
        {
            "file":
                str(
                    path.relative_to(
                        E3_RESULTS_DIR
                    )
                ),

            "rows":
                int(
                    original_df.shape[0]
                ),

            "columns":
                int(
                    original_df.shape[1]
                ),

            "reload_shape_match":
                bool(
                    shape_match
                ),
        }
    )

    print(
        f"  PASS: {path.name:<45} "
        f"{loaded_df.shape}"
    )


e3_drive_freeze_index_df = pd.DataFrame(
    validation_rows
)


FREEZE_INDEX_PATH = (
    FINAL_DIR
    / "E3_drive_freeze_index.csv"
)

e3_drive_freeze_index_df.to_csv(
    FREEZE_INDEX_PATH,
    index=False,
)

assert FREEZE_INDEX_PATH.exists()


# ------------------------------------------------------------
# 8. Verify existing association-primitives freeze
# ------------------------------------------------------------

expected_association_files = [
    ASSOCIATION_DIR
    / "E3_association_per_region.csv",

    ASSOCIATION_DIR
    / "E3_association_per_D_point.csv",

    ASSOCIATION_DIR
    / "E3_association_structural_summary.csv",

    ASSOCIATION_DIR
    / "E3_association_validation.csv",

    ASSOCIATION_DIR
    / "E3_association_metadata.json",
]


print("\nExisting association-primitives artefacts:")

for path in expected_association_files:

    assert path.exists(), (
        f"Missing existing association artefact: "
        f"{path}"
    )

    print(
        f"  PASS: {path}"
    )


# ------------------------------------------------------------
# 9. Final absolute-path verification
# ------------------------------------------------------------

all_final_paths = (
    list(
        tables_to_save.keys()
    )
    + expected_association_files
    + [
        MANIFEST_PATH,
        FREEZE_INDEX_PATH,
    ]
)

assert all(
    "/content/drive/MyDrive/"
    in str(path)
    for path in all_final_paths
)

assert all(
    path.exists()
    for path in all_final_paths
)


print("\n" + "=" * 110)
print("E3 GOOGLE-DRIVE ARTEFACT FREEZE COMPLETE")
print("=" * 110)

print(
    "Drive directory:",
    E3_RESULTS_DIR,
)

print(
    "New/re-saved result tables:",
    len(tables_to_save),
)

print(
    "Existing association artefacts verified:",
    len(expected_association_files),
)

print(
    "Manifest:",
    MANIFEST_PATH,
)

print(
    "Freeze index:",
    FREEZE_INDEX_PATH,
)

print(
    "\nAll checked paths are under "
    "/content/drive/MyDrive/: PASS"
)

print(
    "All saved CSVs reload with expected shape: PASS"
)

print(
    "13D.2b.4 remains COMPLETE AND FROZEN."
)

print("=" * 110)

PROJECT_ROOT  : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
E3_RESULTS_DIR: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_E/E3_semantic_integration

Tables scheduled for Drive freeze:
  E3_tau_assignment.csv                         (696525, 22)
  E3_tau_region.csv                             (319975, 14)
  E3_tau_structural_summary.csv                 (5, 19)
  E3_tau_pair_summary.csv                       (75, 24)
  E3_tau_pair_delta.csv                         (15, 8)
  E3_fusion_per_image.csv                       (22500, 9)
  E3_fusion_generation_summary.csv              (225, 11)
  E3_fusion_source_summary.csv                  (5, 3)
  E3_zero_prediction_summary.csv                (30, 5)
  E3_eval_dev100.csv                            (100, 5)
  E3_fusion_eval_per_image.csv                  (22500, 24)
  E3_baseline_reference.csv                     (800, 51)
  E3_baseline_overall_summary.csv               (8, 13)
  E3_fusion_overall_s